# Performance of Linear Regression Model in Predicting Fuel Consumption for the Government Fleets
**Guimaras State University — College of Science and Technology**

This notebook reproduces the model-development workflow of the study (Chapter III) in Google Colab:

1. Merge the local trip records (Monthly Report of Official Travel) with the vehicle records by plate number,
   and add the supplementary dataset with the same variables
2. Data cleaning: drop missing values and zero distance; IQR outliers of fuel per 100 km within each engine-power group
3. Encoding (fuel type: 1 = Gasoline, 0 = Diesel) and the interaction term Distance × Engine Power
4. Multicollinearity check with the Variance Inflation Factor (VIF above 5 = high collinearity), with distance and
   engine power mean-centered before the interaction term is formed
5. Train-test split (80/20) and Linear Regression training:
   **ŷ = β0 + β1(Distance) + β2(Engine Power) + β3(Fuel Type) + β4(Distance × Engine Power)**
6. Evaluation with R², MAE and RMSE, and 5-fold cross-validation
7. Residual plot, residual histogram and Durbin-Watson test on the local trips arranged by date of travel
8. Saving the trained model with Joblib for the Flask web application

The code is the same preprocessing and diagnostics code used by the web application
(`model/preprocess.py`, `model/diagnostics.py`, `model/train_model.py`).

In [ ]:
# Colab already includes pandas, numpy, scikit-learn, matplotlib and seaborn.
!pip -q install statsmodels joblib

In [ ]:
import json, os, warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
sns.set_theme(style="whitegrid")
RANDOM_STATE, CV_FOLDS = 42, 5

## 1. Data collection — the datasets (built into this notebook)
This version already contains the data exported from the web application (Model Performance → Training Details):
**trip_records.csv** (Monthly Report of Official Travel), **vehicle_records.csv** (plate number, model, fuel type,
engine power) and **supplementary_data.csv** (6,000 real-world gasoline trips from Fan & Song, 2022, Mendeley Data,
https://doi.org/10.17632/jzjxcdn6k4.2, and 1,260 real-world diesel vans from the European Environment Agency,
2024, https://doi.org/10.2909/13e6fad9-149c-4b86-9d83-cdf3569ccfee; both CC BY 4.0).
The next cell writes the three files, so **Runtime → Run all** trains the model with no file upload.

In [ ]:
# Data exported from the web application. For newer records, download the three CSV files again from
# Model Performance and use fuel_consumption_linear_regression.ipynb (the upload version).
TRIPS_PATH, VEHICLES_PATH, SUPPLEMENTARY_PATH = "trip_records.csv", "vehicle_records.csv", "supplementary_data.csv"
TRIPS_CSV = "trip_date,plate_number,distance_km,actual_fuel_consumption\n2026-01-05,SFA 4714,192.0,19.2\n2026-01-06,SFA 4714,128.0,12.8\n2026-01-07,SFA 4714,192.0,19.2\n2026-01-08,SFA 4714,208.0,20.8\n2026-01-12,SFA 4714,64.0,6.4\n2026-01-13,SFA 4714,100.0,10.0\n2026-01-15,SFA 4714,182.0,18.2\n2026-01-16,SFA 4714,93.0,9.3\n2026-01-17,SFA 4714,64.0,6.4\n2026-01-19,SFA 4714,89.0,8.9\n2026-01-20,SFA 4714,262.0,26.2\n2026-01-21,SFA 4714,332.0,33.2\n2026-01-22,SFA 4714,184.0,18.4\n2026-01-23,SFA 4714,278.0,27.8\n2026-01-28,SFA 4714,196.0,19.6\n2026-01-29,SFA 4714,67.0,6.7\n2026-01-30,SFA 4714,154.0,15.4\n2026-02-03,SFA 4714,32.0,3.2\n2026-02-10,SFA 4714,229.0,22.9\n2026-02-11,SFA 4714,208.0,20.8\n2026-02-13,SFA 4714,64.0,6.4\n2026-02-14,SFA 4714,168.0,16.8\n2026-02-16,SFA 4714,283.0,28.3\n2026-02-18,SFA 4714,202.0,20.2\n2026-02-19,SFA 4714,269.0,26.9\n2026-02-20,SFA 4714,188.0,18.8\n2026-02-21,SFA 4714,262.0,26.2\n2026-02-22,SFA 4714,208.0,20.8\n2026-02-23,SFA 4714,215.0,21.5\n2026-02-24,SFA 4714,233.0,23.3\n2026-02-25,SFA 4714,220.0,22.0\n2026-02-27,SFA 4714,132.0,13.2\n2026-03-02,SFA 4714,256.0,25.6\n2026-03-03,SFA 4714,128.0,12.8\n2026-03-04,SFA 4714,248.0,24.8\n2026-03-05,SFA 4714,270.0,27.0\n2026-03-06,SFA 4714,131.0,13.1\n2026-03-10,SFA 4714,112.0,11.2\n2026-03-11,SFA 4714,151.0,15.1\n2026-03-12,SFA 4714,106.0,10.6\n2026-03-19,SFA 4714,169.0,16.9\n2026-03-20,SFA 4714,362.0,36.2\n2026-03-23,SFA 4714,128.0,12.8\n2026-03-25,SFA 4714,209.0,20.9\n2026-03-30,SFA 4714,134.0,13.4\n2026-04-06,SFA 4714,222.0,22.2\n2026-04-07,SFA 4714,156.0,15.6\n2026-04-14,SFA 4714,67.0,6.7\n2026-04-15,SFA 4714,131.0,13.1\n2026-04-17,SFA 4714,64.0,6.4\n2026-04-20,SFA 4714,136.0,13.6\n2026-04-21,SFA 4714,200.0,20.0\n2026-04-22,SFA 4714,364.0,36.4\n2026-04-27,SFA 4714,134.0,13.4\n2026-04-28,SFA 4714,188.0,18.8\n2026-04-29,SFA 4714,138.0,13.8\n2026-05-04,SFA 4714,210.0,21.0\n2026-05-05,SFA 4714,257.0,25.7\n2026-05-06,SFA 4714,154.0,15.4\n2026-05-11,SFA 4714,146.0,14.6\n2026-05-12,SFA 4714,92.0,9.2\n2026-05-13,SFA 4714,184.0,18.4\n2026-05-14,SFA 4714,196.0,19.6\n2026-05-18,SFA 4714,122.0,12.2\n2026-05-20,SFA 4714,232.0,23.2\n2026-05-21,SFA 4714,126.0,12.6\n2026-05-22,SFA 4714,90.0,9.0\n2026-05-23,SFA 4714,64.0,6.4\n2026-05-24,SFA 4714,76.0,7.6\n2026-05-25,SFA 4714,147.0,14.7\n2026-05-26,SFA 4714,84.0,8.4\n2026-05-30,SFA 4714,148.0,14.8\n2026-06-01,SFA 4714,158.0,15.8\n2026-06-02,SFA 4714,372.0,37.2\n2026-06-04,SFA 4714,202.0,20.2\n2026-06-05,SFA 4714,244.0,24.4\n2026-06-08,SFA 4714,104.0,10.4\n2026-06-09,SFA 4714,163.0,16.3\n2026-06-10,SFA 4714,176.0,17.6\n2026-06-11,SFA 4714,257.0,25.7\n2026-06-13,SFA 4714,173.0,17.3\n2026-06-15,SFA 4714,204.0,20.4\n2026-06-16,SFA 4714,338.0,33.8\n2026-06-18,SFA 4714,315.0,31.5\n2026-06-20,SFA 4714,116.0,11.6\n2026-06-22,SFA 4714,218.0,21.8\n2026-06-24,SFA 4714,158.0,15.8\n2026-06-25,SFA 4714,240.0,24.0\n2026-06-26,SFA 4714,67.0,6.7\n2026-06-27,SFA 4714,131.0,13.1\n2026-06-28,SFA 4714,64.0,6.4\n2026-06-29,SFA 4714,190.0,19.0\n2026-06-30,SFA 4714,248.0,24.8\n"
VEHICLES_CSV = "plate_number,brand_model,fuel_type,engine_power_kw\nSEF 535,L300,Diesel,51.0\nSFA 1172,Toyota - Grandia,Diesel,130.0\nSFA 1173,Toyota Innova,Diesel,128.0\nSFA 1741,Hino Bus,Diesel,177.0\nSFA 4712,Nissan Van,Diesel,95.0\nSFA 4713,Nissan Van,Diesel,95.0\nSFA 4714,Nissan Van,Diesel,95.0\nSGF 810,Revo,Diesel,63.0\nSJC 955,Crosswind,Diesel,62.0\n"
SUPPLEMENTARY_CSV = "brand_model,engine_code,distance_km,engine_power_kw,fuel_type,actual_fuel_consumption,source\nAudi A6L,0230320,215.48,132,Gasoline,16.0883,Fan & Song 2022\nAudi A6L,0230320,40.48,132,Gasoline,3.5847,Fan & Song 2022\nAudi A6L,0230320,26.48,132,Gasoline,2.2697,Fan & Song 2022\nAudi A6L,0230320,1.76,132,Gasoline,0.2609,Fan & Song 2022\nAudi A6L,0230320,11.51,132,Gasoline,1.5043,Fan & Song 2022\nAudi A6L,0230320,365.59,132,Gasoline,27.7548,Fan & Song 2022\nAudi A6L,0230320,247.57,132,Gasoline,20.2132,Fan & Song 2022\nAudi A6L,0230320,190.45,132,Gasoline,15.3557,Fan & Song 2022\nAudi A6L,0230320,107.03,132,Gasoline,8.8785,Fan & Song 2022\nAudi A6L,0230320,3.22,132,Gasoline,0.3864,Fan & Song 2022\nAudi A6L,0230320,32.44,132,Gasoline,2.7163,Fan & Song 2022\nAudi A6L,0230320,9.51,132,Gasoline,0.9584,Fan & Song 2022\nAudi A6L,0230320,16.78,132,Gasoline,1.847,Fan & Song 2022\nAudi A6L,0230320,44.35,132,Gasoline,3.7785,Fan & Song 2022\nAudi A6L,0230320,21.86,132,Gasoline,2.1835,Fan & Song 2022\nAudi A6L,0230320,3.9,132,Gasoline,0.457,Fan & Song 2022\nAudi A6L,0230320,38.12,132,Gasoline,2.4397,Fan & Song 2022\nAudi A6L,0230320,2.02,132,Gasoline,0.1775,Fan & Song 2022\nAudi A6L,0230320,4.05,132,Gasoline,0.5339,Fan & Song 2022\nAudi A6L,0230320,39.66,132,Gasoline,3.3925,Fan & Song 2022\nAudi A6L,0230320,54.18,132,Gasoline,4.2552,Fan & Song 2022\nAudi A6L,0230320,30.56,132,Gasoline,2.9804,Fan & Song 2022\nAudi A6L,0230320,1.79,132,Gasoline,0.2229,Fan & Song 2022\nAudi A6L,0230320,1.87,132,Gasoline,0.3021,Fan & Song 2022\nAudi A6L,0230320,22.19,132,Gasoline,2.5329,Fan & Song 2022\nAudi A6L,0230320,19.32,132,Gasoline,1.7322,Fan & Song 2022\nAudi A6L,0230320,64.15,132,Gasoline,5.2771,Fan & Song 2022\nAudi A6L,0230320,1.66,132,Gasoline,0.2613,Fan & Song 2022\nAudi A6L,0230320,29.89,132,Gasoline,2.4037,Fan & Song 2022\nAudi A6L,0230320,48.64,132,Gasoline,3.84,Fan & Song 2022\nAudi A6L,0230320,111.94,132,Gasoline,7.7716,Fan & Song 2022\nAudi A6L,0230320,25.11,132,Gasoline,3.2213,Fan & Song 2022\nAudi A6L,0230320,16.53,132,Gasoline,1.5772,Fan & Song 2022\nAudi A6L,0230320,109.67,132,Gasoline,8.069,Fan & Song 2022\nAudi A6L,0230320,14.01,132,Gasoline,1.3735,Fan & Song 2022\nAudi A6L,0230320,11.39,132,Gasoline,0.7677,Fan & Song 2022\nAudi A6L,0230320,241.96,132,Gasoline,15.6353,Fan & Song 2022\nAudi A6L,0230320,2.84,132,Gasoline,0.316,Fan & Song 2022\nAudi A6L,0230320,145.41,132,Gasoline,10.6009,Fan & Song 2022\nAudi A6L,0230320,29.21,132,Gasoline,2.687,Fan & Song 2022\nAudi A6L,0230320,78.39,132,Gasoline,5.6223,Fan & Song 2022\nAudi A6L,0230320,129.26,132,Gasoline,12.1256,Fan & Song 2022\nAudi A6L,0230320,12.82,132,Gasoline,1.117,Fan & Song 2022\nAudi A6L,0230320,74.63,132,Gasoline,5.068,Fan & Song 2022\nAudi A6L,0230320,17.39,132,Gasoline,1.5978,Fan & Song 2022\nAudi A6L,0230320,15.74,132,Gasoline,1.2103,Fan & Song 2022\nAudi A6L,0230320,3.1,132,Gasoline,0.4657,Fan & Song 2022\nAudi A6L,0230320,13.62,132,Gasoline,1.5662,Fan & Song 2022\nAudi A6L,0230320,53.76,132,Gasoline,3.5291,Fan & Song 2022\nAudi A6L,0230320,11.37,132,Gasoline,0.8677,Fan & Song 2022\nAudi A6L,0230320,224.27,132,Gasoline,18.6642,Fan & Song 2022\nAudi A6L,0230320,120.97,132,Gasoline,7.7497,Fan & Song 2022\nAudi A6L,0230320,33.77,132,Gasoline,2.6521,Fan & Song 2022\nAudi A6L,0230320,123.77,132,Gasoline,9.1391,Fan & Song 2022\nAudi A6L,0230320,176.85,132,Gasoline,13.8691,Fan & Song 2022\nAudi A6L,0230320,108.06,132,Gasoline,8.7847,Fan & Song 2022\nAudi A6L,0230320,178.4,132,Gasoline,9.7168,Fan & Song 2022\nAudi A6L,0230320,18.36,132,Gasoline,1.5851,Fan & Song 2022\nAudi A6L,0230320,101.58,132,Gasoline,8.0194,Fan & Song 2022\nAudi A6L,0230320,11.33,132,Gasoline,1.1549,Fan & Song 2022\nAudi A6L,0230320,130.41,132,Gasoline,11.9789,Fan & Song 2022\nAudi A6L,0230320,187.03,132,Gasoline,14.2775,Fan & Song 2022\nAudi A6L,0230320,14.61,132,Gasoline,1.2908,Fan & Song 2022\nAudi A6L,0230320,1.56,132,Gasoline,0.2491,Fan & Song 2022\nAudi A6L,0230320,15.03,132,Gasoline,1.3071,Fan & Song 2022\nAudi A6L,0230320,224.48,132,Gasoline,16.6309,Fan & Song 2022\nAudi A6L,0230320,36.76,132,Gasoline,2.2316,Fan & Song 2022\nAudi A6L,0230320,2.72,132,Gasoline,0.5366,Fan & Song 2022\nAudi A6L,0230320,34.69,132,Gasoline,2.7991,Fan & Song 2022\nAudi A6L,0230320,3.53,132,Gasoline,0.427,Fan & Song 2022\nAudi A6L,0230320,1.57,132,Gasoline,0.3124,Fan & Song 2022\nAudi A6L,0230320,262.92,132,Gasoline,20.433,Fan & Song 2022\nAudi A6L,0230320,2.35,132,Gasoline,0.2913,Fan & Song 2022\nAudi A6L,0230320,60.81,132,Gasoline,4.9964,Fan & Song 2022\nAudi A6L,0230320,10.46,132,Gasoline,1.107,Fan & Song 2022\nAudi A6L,0230320,65.41,132,Gasoline,5.2389,Fan & Song 2022\nAudi A6L,0230320,180.78,132,Gasoline,15.8678,Fan & Song 2022\nAudi A6L,0230320,40.6,132,Gasoline,2.7386,Fan & Song 2022\nAudi A6L,0230320,14.1,132,Gasoline,1.2683,Fan & Song 2022\nAudi A6L,0230320,4.16,132,Gasoline,0.4995,Fan & Song 2022\nAudi A6L,0230320,1.66,132,Gasoline,0.2257,Fan & Song 2022\nAudi A6L,0230320,10.17,132,Gasoline,0.8963,Fan & Song 2022\nAudi A6L,0230320,113.05,132,Gasoline,7.9507,Fan & Song 2022\nAudi A6L,0230320,4.83,132,Gasoline,0.5886,Fan & Song 2022\nAudi A6L,0230320,31.27,132,Gasoline,2.2226,Fan & Song 2022\nAudi A6L,0230320,18.58,132,Gasoline,2.1159,Fan & Song 2022\nAudi A6L,0230320,16.47,132,Gasoline,1.4051,Fan & Song 2022\nAudi A6L,0230320,2.16,132,Gasoline,0.2541,Fan & Song 2022\nAudi A6L,0230320,48.47,132,Gasoline,3.7628,Fan & Song 2022\nAudi A6L,0230320,41.74,132,Gasoline,2.8347,Fan & Song 2022\nAudi A6L,0230320,12.12,132,Gasoline,1.0585,Fan & Song 2022\nAudi A6L,0230320,7.03,132,Gasoline,0.7141,Fan & Song 2022\nAudi A6L,0230320,107.53,132,Gasoline,7.1811,Fan & Song 2022\nAudi A6L,0230320,33.65,132,Gasoline,2.624,Fan & Song 2022\nAudi A6L,0230320,38.47,132,Gasoline,3.3452,Fan & Song 2022\nAudi A6L,0230320,24.47,132,Gasoline,2.6662,Fan & Song 2022\nAudi A6L,0230320,8.34,132,Gasoline,0.9545,Fan & Song 2022\nAudi A6L,0230320,3.97,132,Gasoline,0.4315,Fan & Song 2022\nAudi A6L,0230320,139.08,132,Gasoline,10.1337,Fan & Song 2022\nAudi A6L,0230320,30.4,132,Gasoline,2.6953,Fan & Song 2022\nAudi A6L,0230320,27.75,132,Gasoline,2.0094,Fan & Song 2022\nAudi A6L,0230320,5.79,132,Gasoline,0.5075,Fan & Song 2022\nAudi A6L,0230320,213.5,132,Gasoline,16.5538,Fan & Song 2022\nAudi A6L,0230320,9.48,132,Gasoline,1.5891,Fan & Song 2022\nAudi A6L,0230320,110.2,132,Gasoline,8.6012,Fan & Song 2022\nAudi A6L,0230320,35.92,132,Gasoline,2.6773,Fan & Song 2022\nAudi A6L,0230320,130.01,132,Gasoline,7.9117,Fan & Song 2022\nAudi A6L,0230320,37.28,132,Gasoline,3.9644,Fan & Song 2022\nAudi A6L,0230320,4.84,132,Gasoline,0.5299,Fan & Song 2022\nAudi A6L,0230320,3.24,132,Gasoline,0.4708,Fan & Song 2022\nAudi A6L,0230320,148.48,132,Gasoline,11.0496,Fan & Song 2022\nAudi A6L,0230320,18.03,132,Gasoline,2.0502,Fan & Song 2022\nAudi A6L,0230320,6.93,132,Gasoline,0.9642,Fan & Song 2022\nAudi A6L,0230320,310.66,132,Gasoline,27.0892,Fan & Song 2022\nAudi A6L,0230320,116.53,132,Gasoline,7.1921,Fan & Song 2022\nAudi A6L,0230320,76.04,132,Gasoline,5.6172,Fan & Song 2022\nAudi A6L,0230320,5.68,132,Gasoline,0.9818,Fan & Song 2022\nAudi A6L,0230320,50.8,132,Gasoline,4.4675,Fan & Song 2022\nAudi A6L,0230320,291.67,132,Gasoline,22.1184,Fan & Song 2022\nAudi A6L,0230320,8.19,132,Gasoline,0.6438,Fan & Song 2022\nAudi A6L,0230320,111.91,132,Gasoline,8.4068,Fan & Song 2022\nAudi A6L,0230320,236.5,132,Gasoline,19.94,Fan & Song 2022\nAudi A6L,0230320,34.9,132,Gasoline,3.7289,Fan & Song 2022\nAudi A6L,0230320,125.24,132,Gasoline,10.4103,Fan & Song 2022\nAudi A6L,0230320,12.42,132,Gasoline,1.2801,Fan & Song 2022\nAudi A6L,0230320,12.58,132,Gasoline,1.6393,Fan & Song 2022\nAudi A6L,0230320,9.83,132,Gasoline,1.3175,Fan & Song 2022\nAudi A6L,0230320,3.7,132,Gasoline,0.5073,Fan & Song 2022\nAudi A6L,0230320,56.74,132,Gasoline,4.4455,Fan & Song 2022\nAudi A6L,0230320,35.53,132,Gasoline,3.1047,Fan & Song 2022\nAudi A6L,0230320,9.74,132,Gasoline,1.0661,Fan & Song 2022\nAudi A6L,0230320,11.2,132,Gasoline,1.3557,Fan & Song 2022\nAudi A6L,0230320,104.2,132,Gasoline,8.624,Fan & Song 2022\nAudi A6L,0230320,38.57,132,Gasoline,2.7532,Fan & Song 2022\nAudi A6L,0230320,45.82,132,Gasoline,3.0239,Fan & Song 2022\nAudi A6L,0230320,111.33,132,Gasoline,7.0721,Fan & Song 2022\nAudi A6L,0230320,123.28,132,Gasoline,8.2513,Fan & Song 2022\nAudi A6L,0230320,35.77,132,Gasoline,2.475,Fan & Song 2022\nAudi A6L,0230320,10.18,132,Gasoline,1.0,Fan & Song 2022\nAudi A6L,0230320,268.99,132,Gasoline,21.1727,Fan & Song 2022\nAudi A6L,0230320,34.53,132,Gasoline,2.7078,Fan & Song 2022\nAudi A6L,0230320,7.34,132,Gasoline,0.7113,Fan & Song 2022\nAudi A6L,0230320,245.18,132,Gasoline,18.378,Fan & Song 2022\nAudi A6L,0230320,4.55,132,Gasoline,0.453,Fan & Song 2022\nAudi A6L,0230320,45.79,132,Gasoline,4.3665,Fan & Song 2022\nAudi A6L,0230320,13.94,132,Gasoline,1.0691,Fan & Song 2022\nAudi A6L,0230320,7.77,132,Gasoline,0.73,Fan & Song 2022\nAudi A6L,0230320,13.84,132,Gasoline,1.1089,Fan & Song 2022\nAudi A6L,0230320,5.5,132,Gasoline,0.7176,Fan & Song 2022\nAudi A6L,0230320,26.49,132,Gasoline,2.0863,Fan & Song 2022\nAudi A6L,0230320,18.57,132,Gasoline,1.5704,Fan & Song 2022\nAudi A6L,0230320,118.9,132,Gasoline,8.5196,Fan & Song 2022\nAudi A6L,0230320,173.3,132,Gasoline,11.6554,Fan & Song 2022\nAudi A6L,0230320,171.79,132,Gasoline,12.2762,Fan & Song 2022\nAudi A6L,0230320,14.77,132,Gasoline,1.3773,Fan & Song 2022\nAudi A6L,0230320,14.01,132,Gasoline,1.5948,Fan & Song 2022\nAudi A6L,0230320,64.55,132,Gasoline,4.593,Fan & Song 2022\nAudi A6L,0230320,40.7,132,Gasoline,3.4677,Fan & Song 2022\nAudi A6L,0230320,14.09,132,Gasoline,1.2103,Fan & Song 2022\nAudi A6L,0230320,8.7,132,Gasoline,0.7054,Fan & Song 2022\nAudi A6L,0230320,117.03,132,Gasoline,8.6118,Fan & Song 2022\nAudi A6L,0230320,107.42,132,Gasoline,10.8329,Fan & Song 2022\nAudi A6L,0230320,228.12,132,Gasoline,20.1162,Fan & Song 2022\nAudi A6L,0230320,16.0,132,Gasoline,1.229,Fan & Song 2022\nAudi A6L,0230320,67.76,132,Gasoline,7.3445,Fan & Song 2022\nAudi A6L,0230320,131.31,132,Gasoline,10.5113,Fan & Song 2022\nAudi A6L,0230320,72.88,132,Gasoline,5.719,Fan & Song 2022\nAudi A6L,0230320,14.38,132,Gasoline,1.0647,Fan & Song 2022\nAudi A6L,0230320,3.66,132,Gasoline,0.5916,Fan & Song 2022\nAudi A6L,0230320,9.71,132,Gasoline,1.4472,Fan & Song 2022\nAudi A6L,0230320,7.37,132,Gasoline,0.7618,Fan & Song 2022\nAudi A6L,0230320,20.2,132,Gasoline,3.6218,Fan & Song 2022\nAudi A6L,0230320,192.22,132,Gasoline,14.1293,Fan & Song 2022\nAudi A6L,0230320,25.48,132,Gasoline,1.9923,Fan & Song 2022\nAudi A6L,0230320,21.23,132,Gasoline,1.8974,Fan & Song 2022\nAudi A6L,0230320,101.08,132,Gasoline,6.375,Fan & Song 2022\nAudi A6L,0230320,1.75,132,Gasoline,0.1914,Fan & Song 2022\nAudi A6L,0230320,114.61,132,Gasoline,9.8444,Fan & Song 2022\nAudi A6L,0230320,1.63,132,Gasoline,0.1714,Fan & Song 2022\nAudi A6L,0230320,18.87,132,Gasoline,1.7714,Fan & Song 2022\nAudi A6L,0230320,53.17,132,Gasoline,4.262,Fan & Song 2022\nAudi A6L,0230320,139.45,132,Gasoline,10.3853,Fan & Song 2022\nAudi A6L,0230320,33.82,132,Gasoline,2.3666,Fan & Song 2022\nAudi A6L,0230320,132.62,132,Gasoline,9.0778,Fan & Song 2022\nAudi A6L,0230320,3.11,132,Gasoline,0.3203,Fan & Song 2022\nAudi A6L,0230320,126.88,132,Gasoline,9.609,Fan & Song 2022\nAudi A6L,0230320,206.2,132,Gasoline,14.0531,Fan & Song 2022\nAudi A6L,0230320,236.58,132,Gasoline,20.3705,Fan & Song 2022\nAudi A6L,0230320,26.11,132,Gasoline,2.0573,Fan & Song 2022\nAudi A6L,0230320,107.83,132,Gasoline,10.2177,Fan & Song 2022\nAudi A6L,0230320,6.7,132,Gasoline,0.8193,Fan & Song 2022\nAudi A6L,0230320,116.81,132,Gasoline,8.8965,Fan & Song 2022\nAudi A6L,0230320,40.26,132,Gasoline,4.5143,Fan & Song 2022\nAudi A6L,0230320,108.29,132,Gasoline,8.0304,Fan & Song 2022\nAudi A6L,0230320,63.18,132,Gasoline,4.7048,Fan & Song 2022\nAudi A6L,0230320,39.28,132,Gasoline,2.5892,Fan & Song 2022\nAudi A6L,0230320,1.95,132,Gasoline,0.3132,Fan & Song 2022\nAudi A6L,0230320,176.2,132,Gasoline,10.7678,Fan & Song 2022\nAudi A6L,0230320,125.8,132,Gasoline,9.2207,Fan & Song 2022\nAudi A6L,0230320,135.11,132,Gasoline,10.2032,Fan & Song 2022\nAudi A6L,0230320,180.44,132,Gasoline,12.908,Fan & Song 2022\nAudi A6L,0230320,40.26,132,Gasoline,3.0448,Fan & Song 2022\nAudi A6L,0230320,51.02,132,Gasoline,2.9633,Fan & Song 2022\nAudi A6L,0230320,71.71,132,Gasoline,4.4702,Fan & Song 2022\nAudi A6L,0230320,192.97,132,Gasoline,12.4272,Fan & Song 2022\nAudi A6L,0230320,133.74,132,Gasoline,10.1712,Fan & Song 2022\nAudi A6L,0230320,60.5,132,Gasoline,3.977,Fan & Song 2022\nAudi A6L,0230320,179.52,132,Gasoline,14.3018,Fan & Song 2022\nAudi A6L,0230320,3.72,132,Gasoline,0.4887,Fan & Song 2022\nAudi A6L,0230320,14.54,132,Gasoline,1.3609,Fan & Song 2022\nAudi A6L,0230320,56.37,132,Gasoline,4.1154,Fan & Song 2022\nAudi A6L,0230320,28.52,132,Gasoline,1.869,Fan & Song 2022\nAudi A6L,0230320,2.62,132,Gasoline,0.2709,Fan & Song 2022\nAudi A6L,0230320,109.78,132,Gasoline,8.1106,Fan & Song 2022\nAudi A6L,0230320,38.67,132,Gasoline,2.9346,Fan & Song 2022\nAudi A6L,0230320,21.68,132,Gasoline,2.3623,Fan & Song 2022\nAudi A6L,0230320,8.27,132,Gasoline,0.6579,Fan & Song 2022\nAudi A6L,0230320,20.26,132,Gasoline,1.981,Fan & Song 2022\nAudi A6L,0230320,33.33,132,Gasoline,3.2525,Fan & Song 2022\nAudi A6L,0230320,44.0,132,Gasoline,3.5871,Fan & Song 2022\nAudi A6L,0230320,115.74,132,Gasoline,8.4691,Fan & Song 2022\nAudi A6L,0230320,72.48,132,Gasoline,6.2348,Fan & Song 2022\nAudi A6L,0230320,7.1,132,Gasoline,0.716,Fan & Song 2022\nAudi A6L,0230320,27.08,132,Gasoline,2.3751,Fan & Song 2022\nAudi A6L,0230320,4.16,132,Gasoline,0.3305,Fan & Song 2022\nAudi A6L,0230320,9.35,132,Gasoline,1.1529,Fan & Song 2022\nAudi A6L,0230320,173.95,132,Gasoline,13.9229,Fan & Song 2022\nAudi A6L,0230320,25.57,132,Gasoline,2.323,Fan & Song 2022\nAudi A6L,0230320,132.12,132,Gasoline,10.0345,Fan & Song 2022\nAudi A6L,0230320,46.83,132,Gasoline,4.4998,Fan & Song 2022\nAudi A6L,0230320,11.55,132,Gasoline,1.3823,Fan & Song 2022\nAudi A6L,0230320,108.55,132,Gasoline,7.8673,Fan & Song 2022\nAudi A6L,0230320,102.66,132,Gasoline,7.3959,Fan & Song 2022\nAudi A6L,0230320,11.4,132,Gasoline,1.3623,Fan & Song 2022\nAudi A6L,0230320,10.03,132,Gasoline,1.8521,Fan & Song 2022\nAudi A6L,0230320,100.7,132,Gasoline,6.1142,Fan & Song 2022\nAudi A6L,0230320,38.01,132,Gasoline,3.017,Fan & Song 2022\nAudi A6L,0230320,5.66,132,Gasoline,0.6358,Fan & Song 2022\nAudi A6L,0230320,13.6,132,Gasoline,1.4658,Fan & Song 2022\nAudi A6L,0230320,53.74,132,Gasoline,3.6898,Fan & Song 2022\nAudi A6L,0230320,63.77,132,Gasoline,3.9333,Fan & Song 2022\nAudi A6L,0230320,26.73,132,Gasoline,2.3538,Fan & Song 2022\nAudi A6L,0230320,3.39,132,Gasoline,0.4882,Fan & Song 2022\nAudi A6L,0230320,14.71,132,Gasoline,1.9567,Fan & Song 2022\nAudi A6L,0230320,115.54,132,Gasoline,8.5386,Fan & Song 2022\nAudi A6L,0230320,120.9,132,Gasoline,9.6646,Fan & Song 2022\nAudi A6L,0230320,11.26,132,Gasoline,1.1441,Fan & Song 2022\nAudi A6L,0230320,2.88,132,Gasoline,0.3909,Fan & Song 2022\nAudi A6L,0230320,13.9,132,Gasoline,1.3107,Fan & Song 2022\nAudi A6L,0230320,2.09,132,Gasoline,0.4946,Fan & Song 2022\nAudi A6L,0230320,22.68,132,Gasoline,1.281,Fan & Song 2022\nAudi A6L,0230320,15.76,132,Gasoline,1.5477,Fan & Song 2022\nAudi A6L,0230320,14.64,132,Gasoline,1.4685,Fan & Song 2022\nAudi A6L,0230320,247.58,132,Gasoline,16.2676,Fan & Song 2022\nAudi A6L,0230320,4.85,132,Gasoline,0.4833,Fan & Song 2022\nAudi A6L,0230320,51.2,132,Gasoline,3.9707,Fan & Song 2022\nAudi A6L,0230320,11.78,132,Gasoline,1.2744,Fan & Song 2022\nAudi A6L,0230320,15.08,132,Gasoline,2.1501,Fan & Song 2022\nAudi A6L,0230320,13.81,132,Gasoline,1.4074,Fan & Song 2022\nAudi A6L,0230320,7.8,132,Gasoline,0.9612,Fan & Song 2022\nAudi A6L,0230320,3.72,132,Gasoline,0.4413,Fan & Song 2022\nAudi A6L,0230320,4.72,132,Gasoline,0.8199,Fan & Song 2022\nAudi A6L,0230320,1.53,132,Gasoline,0.1923,Fan & Song 2022\nAudi A6L,0230320,32.29,132,Gasoline,2.1023,Fan & Song 2022\nAudi A6L,0230320,12.8,132,Gasoline,2.1154,Fan & Song 2022\nAudi A6L,0230320,19.4,132,Gasoline,2.2957,Fan & Song 2022\nAudi A6L,0230320,3.31,132,Gasoline,0.3848,Fan & Song 2022\nAudi A6L,0230320,11.64,132,Gasoline,1.1026,Fan & Song 2022\nAudi A6L,0230320,5.7,132,Gasoline,0.553,Fan & Song 2022\nAudi A6L,0230320,52.54,132,Gasoline,3.445,Fan & Song 2022\nAudi A6L,0230320,57.61,132,Gasoline,3.9151,Fan & Song 2022\nAudi A6L,0230320,21.84,132,Gasoline,1.5382,Fan & Song 2022\nAudi A6L,0230320,3.1,132,Gasoline,0.3203,Fan & Song 2022\nAudi A6L,0230320,2.9,132,Gasoline,0.4789,Fan & Song 2022\nAudi A6L,0230320,220.8,132,Gasoline,20.9861,Fan & Song 2022\nAudi A6L,0230320,29.55,132,Gasoline,3.0092,Fan & Song 2022\nAudi A6L,0230320,102.32,132,Gasoline,8.5006,Fan & Song 2022\nAudi A6L,0230320,2.29,132,Gasoline,0.3074,Fan & Song 2022\nAudi A6L,0230320,5.3,132,Gasoline,0.9445,Fan & Song 2022\nAudi A6L,0230320,54.18,132,Gasoline,4.6178,Fan & Song 2022\nAudi A6L,0230320,18.8,132,Gasoline,1.8567,Fan & Song 2022\nAudi A6L,0230320,316.3,132,Gasoline,23.9566,Fan & Song 2022\nAudi A6L,0230320,29.17,132,Gasoline,2.2288,Fan & Song 2022\nAudi A6L,0230320,107.65,132,Gasoline,8.6522,Fan & Song 2022\nAudi A6L,0230320,109.14,132,Gasoline,6.3804,Fan & Song 2022\nAudi A6L,0230320,2.76,132,Gasoline,0.4158,Fan & Song 2022\nAudi A6L,0230320,3.69,132,Gasoline,0.4057,Fan & Song 2022\nAudi A6L,0230320,1.9,132,Gasoline,0.2822,Fan & Song 2022\nAudi A6L,0230320,45.68,132,Gasoline,2.9552,Fan & Song 2022\nAudi A6L,0230320,51.28,132,Gasoline,4.3882,Fan & Song 2022\nAudi A6L,0230320,15.3,132,Gasoline,1.4257,Fan & Song 2022\nAudi A6L,0230320,36.28,132,Gasoline,2.9148,Fan & Song 2022\nAudi A6L,0230320,22.47,132,Gasoline,2.23,Fan & Song 2022\nAudi A6L,0230320,143.85,132,Gasoline,11.0219,Fan & Song 2022\nAudi A6L,0230320,3.11,132,Gasoline,0.2956,Fan & Song 2022\nAudi A6L,0230320,3.24,132,Gasoline,0.2725,Fan & Song 2022\nAudi A6L,0230320,46.28,132,Gasoline,4.0697,Fan & Song 2022\nAudi A6L,0230320,12.17,132,Gasoline,1.1873,Fan & Song 2022\nAudi A6L,0230320,300.7,132,Gasoline,19.3971,Fan & Song 2022\nAudi A6L,0230320,27.12,132,Gasoline,3.1641,Fan & Song 2022\nAudi A6L,0230320,25.44,132,Gasoline,2.7205,Fan & Song 2022\nAudi A6L,0230320,109.37,132,Gasoline,8.6245,Fan & Song 2022\nAudi A6L,0230320,134.23,132,Gasoline,9.4628,Fan & Song 2022\nAudi A6L,0230320,3.22,132,Gasoline,0.3299,Fan & Song 2022\nAudi A6L,0230320,14.49,132,Gasoline,2.2192,Fan & Song 2022\nAudi A6L,0230320,272.18,132,Gasoline,19.0231,Fan & Song 2022\nAudi A6L,0230320,189.74,132,Gasoline,16.8495,Fan & Song 2022\nAudi A6L,0230320,6.41,132,Gasoline,0.6172,Fan & Song 2022\nAudi A6L,0230320,30.5,132,Gasoline,2.892,Fan & Song 2022\nAudi A6L,0230320,348.32,132,Gasoline,28.2983,Fan & Song 2022\nAudi A6L,0230320,106.84,132,Gasoline,8.8672,Fan & Song 2022\nAudi A6L,0230320,2.88,132,Gasoline,0.5467,Fan & Song 2022\nAudi A6L,0230320,3.15,132,Gasoline,0.4052,Fan & Song 2022\nAudi A6L,0230320,7.97,132,Gasoline,0.8404,Fan & Song 2022\nAudi A6L,0230320,248.7,132,Gasoline,17.1284,Fan & Song 2022\nAudi A6L,0230320,132.51,132,Gasoline,7.8378,Fan & Song 2022\nAudi A6L,0230320,131.55,132,Gasoline,8.6656,Fan & Song 2022\nAudi A6L,0230320,39.92,132,Gasoline,3.4628,Fan & Song 2022\nAudi A6L,0230320,217.16,132,Gasoline,13.9794,Fan & Song 2022\nAudi A6L,0230320,436.12,132,Gasoline,39.223,Fan & Song 2022\nAudi A6L,0230320,2.72,132,Gasoline,0.2077,Fan & Song 2022\nAudi A6L,0230320,1.76,132,Gasoline,0.288,Fan & Song 2022\nAudi A6L,0230320,115.93,132,Gasoline,8.3081,Fan & Song 2022\nAudi A6L,0230320,43.87,132,Gasoline,3.146,Fan & Song 2022\nAudi A6L,0230320,88.47,132,Gasoline,7.6298,Fan & Song 2022\nAudi A6L,0230320,368.06,132,Gasoline,29.4592,Fan & Song 2022\nAudi A6L,0230320,12.76,132,Gasoline,1.3307,Fan & Song 2022\nAudi A6L,0230320,1.85,132,Gasoline,0.3069,Fan & Song 2022\nAudi A6L,0230320,3.46,132,Gasoline,0.5199,Fan & Song 2022\nAudi A6L,0230320,100.44,132,Gasoline,7.392,Fan & Song 2022\nAudi A6L,0230320,2.56,132,Gasoline,0.343,Fan & Song 2022\nAudi A6L,0230320,35.04,132,Gasoline,2.906,Fan & Song 2022\nAudi A6L,0230320,37.92,132,Gasoline,3.2537,Fan & Song 2022\nAudi A6L,0230320,6.74,132,Gasoline,0.9846,Fan & Song 2022\nAudi A6L,0230320,11.15,132,Gasoline,1.1161,Fan & Song 2022\nAudi A6L,0230320,1.94,132,Gasoline,0.3104,Fan & Song 2022\nAudi A6L,0230320,37.69,132,Gasoline,3.9224,Fan & Song 2022\nAudi A6L,0230320,5.16,132,Gasoline,0.5998,Fan & Song 2022\nAudi A6L,0230320,3.06,132,Gasoline,0.4577,Fan & Song 2022\nAudi A6L,0230320,5.28,132,Gasoline,0.9769,Fan & Song 2022\nAudi A6L,0230320,27.99,132,Gasoline,2.6935,Fan & Song 2022\nAudi A6L,0230320,101.84,132,Gasoline,8.0598,Fan & Song 2022\nAudi A6L,0230320,82.18,132,Gasoline,6.6167,Fan & Song 2022\nAudi A6L,0230320,115.06,132,Gasoline,6.8374,Fan & Song 2022\nAudi A6L,0230320,16.34,132,Gasoline,1.2201,Fan & Song 2022\nAudi A6L,0230320,15.85,132,Gasoline,1.742,Fan & Song 2022\nAudi A6L,0230320,77.45,132,Gasoline,8.3384,Fan & Song 2022\nAudi A6L,0230320,3.67,132,Gasoline,0.3783,Fan & Song 2022\nAudi A6L,0230320,34.14,132,Gasoline,2.6273,Fan & Song 2022\nAudi A6L,0230320,51.56,132,Gasoline,4.5136,Fan & Song 2022\nAudi A6L,0230320,11.6,132,Gasoline,0.8774,Fan & Song 2022\nAudi A6L,0230320,63.21,132,Gasoline,6.2665,Fan & Song 2022\nAudi A6L,0230320,8.54,132,Gasoline,0.8703,Fan & Song 2022\nAudi A6L,0230320,7.63,132,Gasoline,0.966,Fan & Song 2022\nAudi A6L,0230320,3.49,132,Gasoline,0.3962,Fan & Song 2022\nAudi A6L,0230320,11.02,132,Gasoline,1.1251,Fan & Song 2022\nAudi A6L,0230320,12.26,132,Gasoline,1.3877,Fan & Song 2022\nAudi A6L,0230320,38.31,132,Gasoline,2.4162,Fan & Song 2022\nAudi A6L,0230320,35.38,132,Gasoline,3.3516,Fan & Song 2022\nAudi A6L,0230320,57.89,132,Gasoline,6.6367,Fan & Song 2022\nAudi A6L,0230320,104.22,132,Gasoline,8.9912,Fan & Song 2022\nAudi A6L,0230320,41.43,132,Gasoline,3.2512,Fan & Song 2022\nAudi A6L,0230320,185.41,132,Gasoline,11.8851,Fan & Song 2022\nAudi A6L,0230320,10.09,132,Gasoline,1.3963,Fan & Song 2022\nAudi A6L,0230320,49.95,132,Gasoline,3.9009,Fan & Song 2022\nAudi A6L,0230320,44.35,132,Gasoline,3.6481,Fan & Song 2022\nAudi A6L,0230320,181.44,132,Gasoline,12.8012,Fan & Song 2022\nAudi A6L,0230320,112.46,132,Gasoline,7.7936,Fan & Song 2022\nAudi A6L,0230320,5.31,132,Gasoline,0.7344,Fan & Song 2022\nAudi A6L,0230320,112.18,132,Gasoline,8.2846,Fan & Song 2022\nAudi A6L,0230320,21.81,132,Gasoline,2.9564,Fan & Song 2022\nAudi A6L,0230320,117.77,132,Gasoline,9.382,Fan & Song 2022\nAudi A6L,0230320,31.8,132,Gasoline,2.565,Fan & Song 2022\nAudi A6L,0230320,39.43,132,Gasoline,3.2216,Fan & Song 2022\nAudi A6L,0230320,114.46,132,Gasoline,10.078,Fan & Song 2022\nAudi A6L,0230320,83.46,132,Gasoline,6.5011,Fan & Song 2022\nAudi A6L,0230320,38.29,132,Gasoline,2.5789,Fan & Song 2022\nAudi A6L,0230320,218.28,132,Gasoline,19.4136,Fan & Song 2022\nAudi A6L,0230320,19.7,132,Gasoline,1.881,Fan & Song 2022\nAudi A6L,0230320,214.51,132,Gasoline,19.1199,Fan & Song 2022\nAudi A6L,0230320,8.2,132,Gasoline,0.766,Fan & Song 2022\nAudi A6L,0230320,106.68,132,Gasoline,7.3155,Fan & Song 2022\nAudi A6L,0230320,78.35,132,Gasoline,5.0278,Fan & Song 2022\nAudi A6L,0230320,46.86,132,Gasoline,3.179,Fan & Song 2022\nAudi A6L,0230320,192.72,132,Gasoline,11.8529,Fan & Song 2022\nAudi A6L,0230320,111.66,132,Gasoline,9.0867,Fan & Song 2022\nAudi A6L,0230320,36.77,132,Gasoline,3.7966,Fan & Song 2022\nAudi A6L,0230320,73.67,132,Gasoline,7.7337,Fan & Song 2022\nAudi A6L,0230320,27.66,132,Gasoline,2.6146,Fan & Song 2022\nAudi A6L,0230320,36.4,132,Gasoline,2.4063,Fan & Song 2022\nAudi A6L,0230320,38.69,132,Gasoline,3.2153,Fan & Song 2022\nAudi A6L,0230320,9.01,132,Gasoline,0.9538,Fan & Song 2022\nAudi A6L,0230320,38.19,132,Gasoline,2.4118,Fan & Song 2022\nAudi A6L,0230320,48.55,132,Gasoline,3.3733,Fan & Song 2022\nAudi A6L,0230320,120.81,132,Gasoline,10.7917,Fan & Song 2022\nAudi A6L,0230320,38.47,132,Gasoline,3.255,Fan & Song 2022\nAudi A6L,0230320,9.14,132,Gasoline,0.756,Fan & Song 2022\nAudi A6L,0230320,31.69,132,Gasoline,3.0481,Fan & Song 2022\nAudi A6L,0230320,9.22,132,Gasoline,0.82,Fan & Song 2022\nAudi A6L,0230320,1.62,132,Gasoline,0.2187,Fan & Song 2022\nAudi A6L,0230320,51.59,132,Gasoline,4.4359,Fan & Song 2022\nAudi A6L,0230320,10.69,132,Gasoline,0.811,Fan & Song 2022\nAudi A6L,0230320,172.1,132,Gasoline,12.9876,Fan & Song 2022\nAudi A6L,0230320,53.93,132,Gasoline,4.0613,Fan & Song 2022\nAudi A6L,0230320,8.48,132,Gasoline,1.1132,Fan & Song 2022\nAudi A6L,0230320,106.13,132,Gasoline,6.6247,Fan & Song 2022\nAudi A6L,0230320,65.89,132,Gasoline,6.0436,Fan & Song 2022\nAudi A6L,0230320,203.51,132,Gasoline,17.7436,Fan & Song 2022\nAudi A6L,0230320,12.2,132,Gasoline,1.0568,Fan & Song 2022\nAudi A6L,0230320,5.93,132,Gasoline,0.8834,Fan & Song 2022\nAudi A6L,0230320,57.33,132,Gasoline,3.5716,Fan & Song 2022\nAudi A6L,0230320,2.64,132,Gasoline,0.3694,Fan & Song 2022\nAudi A6L,0230320,24.3,132,Gasoline,2.2863,Fan & Song 2022\nAudi A6L,0230320,21.75,132,Gasoline,2.0833,Fan & Song 2022\nAudi A6L,0230320,16.52,132,Gasoline,1.6085,Fan & Song 2022\nAudi A6L,0230320,4.61,132,Gasoline,0.43,Fan & Song 2022\nAudi A6L,0230320,39.19,132,Gasoline,2.5581,Fan & Song 2022\nAudi A6L,0230320,37.15,132,Gasoline,2.5563,Fan & Song 2022\nAudi A6L,0230320,1.8,132,Gasoline,0.3066,Fan & Song 2022\nAudi A6L,0230320,14.62,132,Gasoline,1.5482,Fan & Song 2022\nAudi A6L,0230320,11.28,132,Gasoline,1.4191,Fan & Song 2022\nAudi A6L,0230320,30.05,132,Gasoline,1.7631,Fan & Song 2022\nAudi A6L,0230320,10.83,132,Gasoline,1.2294,Fan & Song 2022\nAudi A6L,0230320,70.92,132,Gasoline,5.2745,Fan & Song 2022\nAudi A6L,0230320,12.2,132,Gasoline,2.31,Fan & Song 2022\nAudi A6L,0230320,1.95,132,Gasoline,0.3195,Fan & Song 2022\nAudi A6L,0230320,24.07,132,Gasoline,1.9433,Fan & Song 2022\nAudi A6L,0230320,3.96,132,Gasoline,0.4283,Fan & Song 2022\nAudi A6L,0230320,16.09,132,Gasoline,2.1265,Fan & Song 2022\nAudi A6L,0230320,15.57,132,Gasoline,1.4013,Fan & Song 2022\nAudi A6L,0230320,18.78,132,Gasoline,1.7511,Fan & Song 2022\nAudi A6L,0230320,30.02,132,Gasoline,1.9299,Fan & Song 2022\nAudi A6L,0230320,2.9,132,Gasoline,0.3366,Fan & Song 2022\nAudi A6L,0230320,105.08,132,Gasoline,6.6875,Fan & Song 2022\nAudi A6L,0230320,14.93,132,Gasoline,1.1678,Fan & Song 2022\nAudi A6L,0230320,12.8,132,Gasoline,1.4845,Fan & Song 2022\nAudi A6L,0230320,5.67,132,Gasoline,0.8137,Fan & Song 2022\nAudi A6L,0230320,34.27,132,Gasoline,3.1533,Fan & Song 2022\nAudi A6L,0230320,59.15,132,Gasoline,6.1616,Fan & Song 2022\nAudi A6L,0230320,80.69,132,Gasoline,5.9152,Fan & Song 2022\nAudi A6L,0230320,2.51,132,Gasoline,0.3597,Fan & Song 2022\nAudi A6L,0230320,41.84,132,Gasoline,3.8418,Fan & Song 2022\nAudi A6L,0230320,137.45,132,Gasoline,9.7069,Fan & Song 2022\nAudi A6L,0230320,34.65,132,Gasoline,2.7641,Fan & Song 2022\nAudi A6L,0230320,1.85,132,Gasoline,0.2509,Fan & Song 2022\nAudi A6L,0230320,50.08,132,Gasoline,3.4452,Fan & Song 2022\nAudi A6L,0230320,21.1,132,Gasoline,2.1327,Fan & Song 2022\nAudi A6L,0230320,2.05,132,Gasoline,0.4034,Fan & Song 2022\nAudi A6L,0230320,43.76,132,Gasoline,3.3217,Fan & Song 2022\nAudi A6L,0230320,3.02,132,Gasoline,0.4575,Fan & Song 2022\nAudi A6L,0230320,30.04,132,Gasoline,1.7355,Fan & Song 2022\nAudi A6L,0230320,12.95,132,Gasoline,1.1632,Fan & Song 2022\nAudi A6L,0230320,4.45,132,Gasoline,0.9413,Fan & Song 2022\nAudi A6L,0230320,19.14,132,Gasoline,1.6608,Fan & Song 2022\nAudi A6L,0230320,109.49,132,Gasoline,7.7395,Fan & Song 2022\nAudi A6L,0230320,70.58,132,Gasoline,5.2323,Fan & Song 2022\nAudi A6L,0230320,12.12,132,Gasoline,1.3823,Fan & Song 2022\nAudi A6L,0230320,209.8,132,Gasoline,15.5746,Fan & Song 2022\nAudi A6L,0230320,105.56,132,Gasoline,6.5057,Fan & Song 2022\nAudi A6L,0230320,157.44,132,Gasoline,11.2472,Fan & Song 2022\nAudi A6L,0230320,688.5,132,Gasoline,52.5943,Fan & Song 2022\nAudi A6L,0230320,116.39,132,Gasoline,8.919,Fan & Song 2022\nAudi A6L,0230320,227.34,132,Gasoline,16.555,Fan & Song 2022\nAudi A6L,0230320,5.33,132,Gasoline,0.5805,Fan & Song 2022\nAudi A6L,0230320,34.65,132,Gasoline,2.8072,Fan & Song 2022\nAudi A6L,0230320,58.93,132,Gasoline,4.5212,Fan & Song 2022\nAudi A6L,0230320,32.7,132,Gasoline,2.6604,Fan & Song 2022\nAudi A6L,0230320,11.89,132,Gasoline,1.1423,Fan & Song 2022\nAudi A6L,0230320,117.84,132,Gasoline,8.8975,Fan & Song 2022\nAudi A6L,0230320,36.48,132,Gasoline,2.4482,Fan & Song 2022\nAudi A6L,0230320,15.78,132,Gasoline,1.3893,Fan & Song 2022\nAudi A6L,0230320,8.02,132,Gasoline,0.6241,Fan & Song 2022\nAudi A6L,0230320,134.9,132,Gasoline,11.7876,Fan & Song 2022\nAudi A6L,0230320,150.63,132,Gasoline,11.56,Fan & Song 2022\nAudi A6L,0230320,28.99,132,Gasoline,2.8687,Fan & Song 2022\nAudi A6L,0230320,8.72,132,Gasoline,0.9234,Fan & Song 2022\nAudi A6L,0230320,38.27,132,Gasoline,2.9671,Fan & Song 2022\nAudi A6L,0230320,6.3,132,Gasoline,0.9715,Fan & Song 2022\nAudi A6L,0230320,19.84,132,Gasoline,2.1133,Fan & Song 2022\nAudi A6L,0230320,249.56,132,Gasoline,19.1281,Fan & Song 2022\nAudi A6L,0230320,24.34,132,Gasoline,2.6748,Fan & Song 2022\nAudi A6L,0230320,6.28,132,Gasoline,0.8705,Fan & Song 2022\nAudi A6L,0230320,111.52,132,Gasoline,7.7422,Fan & Song 2022\nAudi A6L,0230320,240.03,132,Gasoline,17.7071,Fan & Song 2022\nAudi A6L,0230320,43.29,132,Gasoline,4.4971,Fan & Song 2022\nAudi A6L,0230320,7.48,132,Gasoline,0.6048,Fan & Song 2022\nAudi A6L,0230320,31.75,132,Gasoline,2.7867,Fan & Song 2022\nAudi A6L,0230320,11.84,132,Gasoline,1.2596,Fan & Song 2022\nAudi A6L,0230320,16.96,132,Gasoline,2.4749,Fan & Song 2022\nAudi A6L,0230320,32.19,132,Gasoline,2.8324,Fan & Song 2022\nAudi A6L,0230320,3.12,132,Gasoline,0.4498,Fan & Song 2022\nAudi A6L,0230320,282.26,132,Gasoline,24.5944,Fan & Song 2022\nAudi A6L,0230320,179.88,132,Gasoline,15.9647,Fan & Song 2022\nAudi A6L,0230320,2.39,132,Gasoline,0.4255,Fan & Song 2022\nAudi A6L,0230320,3.52,132,Gasoline,0.3081,Fan & Song 2022\nAudi A6L,0230320,4.88,132,Gasoline,0.3973,Fan & Song 2022\nAudi A6L,0230320,21.87,132,Gasoline,2.547,Fan & Song 2022\nAudi A6L,0230320,16.26,132,Gasoline,1.6089,Fan & Song 2022\nAudi A6L,0230320,48.49,132,Gasoline,3.3045,Fan & Song 2022\nAudi A6L,0230320,406.93,132,Gasoline,34.3304,Fan & Song 2022\nAudi A6L,0230320,36.99,132,Gasoline,2.4066,Fan & Song 2022\nAudi A6L,0230320,9.95,132,Gasoline,1.0429,Fan & Song 2022\nAudi A6L,0230320,1.97,132,Gasoline,0.2563,Fan & Song 2022\nAudi A6L,0230320,108.29,132,Gasoline,8.989,Fan & Song 2022\nAudi A6L,0230320,82.22,132,Gasoline,4.7207,Fan & Song 2022\nAudi A6L,0230320,38.19,132,Gasoline,3.9237,Fan & Song 2022\nAudi A6L,0230320,11.01,132,Gasoline,0.9516,Fan & Song 2022\nAudi A6L,0230320,14.14,132,Gasoline,1.1029,Fan & Song 2022\nAudi A6L,0230320,1.65,132,Gasoline,0.2146,Fan & Song 2022\nAudi A6L,0230320,66.02,132,Gasoline,6.8009,Fan & Song 2022\nAudi A6L,0230320,3.8,132,Gasoline,0.3915,Fan & Song 2022\nAudi A6L,0230320,5.52,132,Gasoline,0.8386,Fan & Song 2022\nAudi A6L,0230320,14.37,132,Gasoline,1.3496,Fan & Song 2022\nAudi A6L,0230320,137.63,132,Gasoline,12.7663,Fan & Song 2022\nAudi A6L,0230320,39.37,132,Gasoline,4.9225,Fan & Song 2022\nAudi A6L,0230320,106.98,132,Gasoline,7.4942,Fan & Song 2022\nAudi A6L,0230320,14.84,132,Gasoline,1.2853,Fan & Song 2022\nAudi A6L,0230320,124.73,132,Gasoline,9.8281,Fan & Song 2022\nAudi A6L,0230320,1.96,132,Gasoline,0.2903,Fan & Song 2022\nAudi A6L,0230320,12.67,132,Gasoline,1.2545,Fan & Song 2022\nAudi A6L,0230320,70.28,132,Gasoline,5.4759,Fan & Song 2022\nAudi A6L,0230320,34.39,132,Gasoline,3.2519,Fan & Song 2022\nAudi A6L,0230320,16.53,132,Gasoline,2.1946,Fan & Song 2022\nAudi A6L,0230320,110.83,132,Gasoline,7.5094,Fan & Song 2022\nAudi A6L,0230320,1.65,132,Gasoline,0.2311,Fan & Song 2022\nAudi A6L,0230320,6.44,132,Gasoline,0.8208,Fan & Song 2022\nAudi A6L,0230320,74.79,132,Gasoline,4.5513,Fan & Song 2022\nAudi A6L,0230320,35.83,132,Gasoline,3.7367,Fan & Song 2022\nAudi A6L,0230320,61.81,132,Gasoline,4.7685,Fan & Song 2022\nAudi A6L,0230320,3.25,132,Gasoline,0.2993,Fan & Song 2022\nAudi A6L,0230320,167.41,132,Gasoline,11.9213,Fan & Song 2022\nAudi A6L,0230320,8.24,132,Gasoline,0.8941,Fan & Song 2022\nAudi A6L,0230320,24.7,132,Gasoline,2.4554,Fan & Song 2022\nAudi A6L,0230320,39.97,132,Gasoline,2.5589,Fan & Song 2022\nAudi A6L,0230320,8.68,132,Gasoline,1.1932,Fan & Song 2022\nAudi A6L,0230320,2.29,132,Gasoline,0.2824,Fan & Song 2022\nAudi A6L,0230320,39.61,132,Gasoline,3.4225,Fan & Song 2022\nAudi A6L,0230320,23.42,132,Gasoline,2.1838,Fan & Song 2022\nAudi A6L,0230320,35.58,132,Gasoline,2.6459,Fan & Song 2022\nAudi A6L,0230320,51.83,132,Gasoline,3.7581,Fan & Song 2022\nAudi A6L,0230320,167.97,132,Gasoline,11.3649,Fan & Song 2022\nAudi A6L,0230320,17.1,132,Gasoline,1.176,Fan & Song 2022\nAudi A6L,0230320,19.26,132,Gasoline,1.2893,Fan & Song 2022\nAudi A6L,0230320,15.25,132,Gasoline,1.5534,Fan & Song 2022\nAudi A6L,0230320,2.22,132,Gasoline,0.2398,Fan & Song 2022\nAudi A6L,0230320,5.67,132,Gasoline,0.6421,Fan & Song 2022\nAudi A6L,0230320,57.45,132,Gasoline,3.347,Fan & Song 2022\nAudi A6L,0230320,17.39,132,Gasoline,1.7296,Fan & Song 2022\nAudi A6L,0230320,13.49,132,Gasoline,0.8977,Fan & Song 2022\nAudi A6L,0230320,2.82,132,Gasoline,0.3617,Fan & Song 2022\nAudi A6L,0230320,2.54,132,Gasoline,0.3318,Fan & Song 2022\nAudi A6L,0230320,65.58,132,Gasoline,5.6098,Fan & Song 2022\nAudi A6L,0230320,6.93,132,Gasoline,0.5998,Fan & Song 2022\nAudi A6L,0230320,19.66,132,Gasoline,1.7047,Fan & Song 2022\nAudi A6L,0230320,5.72,132,Gasoline,0.5852,Fan & Song 2022\nAudi A6L,0230320,37.09,132,Gasoline,2.9986,Fan & Song 2022\nAudi A6L,0230320,2.85,132,Gasoline,0.3664,Fan & Song 2022\nAudi A6L,0230320,3.87,132,Gasoline,0.4543,Fan & Song 2022\nAudi A6L,0230320,87.4,132,Gasoline,7.547,Fan & Song 2022\nAudi A6L,0230320,17.91,132,Gasoline,1.7502,Fan & Song 2022\nAudi A6L,0230320,71.12,132,Gasoline,5.8845,Fan & Song 2022\nAudi A6L,0230320,120.22,132,Gasoline,9.3538,Fan & Song 2022\nAudi A6L,0230320,10.85,132,Gasoline,1.0188,Fan & Song 2022\nAudi A6L,0230320,19.36,132,Gasoline,2.0177,Fan & Song 2022\nAudi A6L,0230320,20.27,132,Gasoline,1.8864,Fan & Song 2022\nAudi A6L,0230320,13.7,132,Gasoline,1.186,Fan & Song 2022\nAudi A6L,0230320,29.86,132,Gasoline,2.339,Fan & Song 2022\nAudi A6L,0230320,22.72,132,Gasoline,2.0908,Fan & Song 2022\nAudi A6L,0230320,3.87,132,Gasoline,0.4315,Fan & Song 2022\nAudi A6L,0230320,6.6,132,Gasoline,0.938,Fan & Song 2022\nAudi A6L,0230320,21.86,132,Gasoline,2.2223,Fan & Song 2022\nAudi A6L,0230320,101.97,132,Gasoline,7.8444,Fan & Song 2022\nAudi A6L,0230320,6.28,132,Gasoline,0.7994,Fan & Song 2022\nAudi A6L,0230320,29.96,132,Gasoline,2.2743,Fan & Song 2022\nAudi A6L,0230320,18.19,132,Gasoline,1.5144,Fan & Song 2022\nAudi A6L,0230320,207.73,132,Gasoline,14.3691,Fan & Song 2022\nAudi A6L,0230320,5.61,132,Gasoline,0.8353,Fan & Song 2022\nAudi A6L,0230320,41.16,132,Gasoline,4.1724,Fan & Song 2022\nAudi A6L,0230320,21.34,132,Gasoline,2.2066,Fan & Song 2022\nAudi A6L,0230320,113.83,132,Gasoline,7.7057,Fan & Song 2022\nAudi A6L,0230320,30.84,132,Gasoline,2.124,Fan & Song 2022\nAudi A6L,0230320,67.39,132,Gasoline,6.1728,Fan & Song 2022\nAudi A6L,0230320,105.37,132,Gasoline,6.9805,Fan & Song 2022\nAudi A6L,0230320,64.93,132,Gasoline,4.2989,Fan & Song 2022\nAudi A6L,0230320,5.81,132,Gasoline,0.5844,Fan & Song 2022\nAudi A6L,0230320,8.46,132,Gasoline,0.9015,Fan & Song 2022\nAudi A6L,0230320,112.28,132,Gasoline,8.5624,Fan & Song 2022\nAudi A6L,0230320,101.98,132,Gasoline,7.7148,Fan & Song 2022\nAudi A6L,0230320,35.49,132,Gasoline,3.298,Fan & Song 2022\nAudi A6L,0230320,13.59,132,Gasoline,1.3572,Fan & Song 2022\nAudi A6L,0230320,7.05,132,Gasoline,0.9102,Fan & Song 2022\nAudi A6L,0230320,10.05,132,Gasoline,1.4385,Fan & Song 2022\nAudi A6L,0230320,112.05,132,Gasoline,8.5982,Fan & Song 2022\nAudi A6L,0230320,17.87,132,Gasoline,1.8764,Fan & Song 2022\nAudi A6L,0230320,168.95,132,Gasoline,13.9068,Fan & Song 2022\nAudi A6L,0230320,134.26,132,Gasoline,11.216,Fan & Song 2022\nAudi A6L,0230320,2.84,132,Gasoline,0.3876,Fan & Song 2022\nAudi A6L,0230320,116.82,132,Gasoline,8.4575,Fan & Song 2022\nAudi A6L,0230320,2.34,132,Gasoline,0.3958,Fan & Song 2022\nAudi A6L,0230320,17.33,132,Gasoline,1.6484,Fan & Song 2022\nBuick GL8,0B2020W,32.03,123,Gasoline,2.8621,Fan & Song 2022\nBuick GL8,0B2020W,3.28,123,Gasoline,0.6342,Fan & Song 2022\nBuick GL8,0B2020W,61.66,123,Gasoline,5.5259,Fan & Song 2022\nBuick GL8,0B2020W,59.82,123,Gasoline,5.6814,Fan & Song 2022\nBuick GL8,0B2021Y,8.1,147,Gasoline,1.1003,Fan & Song 2022\nBuick GL8,0B2020W,3.48,123,Gasoline,0.5445,Fan & Song 2022\nBuick GL8,0B2021Y,103.66,147,Gasoline,11.5496,Fan & Song 2022\nBuick GL8,0B2021Y,4.92,147,Gasoline,0.5473,Fan & Song 2022\nBuick GL8,0B2020W,1.54,123,Gasoline,0.1856,Fan & Song 2022\nBuick GL8,0B2020W,1.73,123,Gasoline,0.3877,Fan & Song 2022\nBuick GL8,0B2020W,65.58,123,Gasoline,6.8058,Fan & Song 2022\nBuick GL8,0B2020W,42.64,123,Gasoline,5.908,Fan & Song 2022\nBuick GL8,0B2020W,12.59,123,Gasoline,1.5907,Fan & Song 2022\nBuick GL8,0B2020W,32.52,123,Gasoline,2.85,Fan & Song 2022\nBuick GL8,0B2020W,78.22,123,Gasoline,8.1535,Fan & Song 2022\nBuick GL8,0B2020W,5.66,123,Gasoline,0.5749,Fan & Song 2022\nBuick GL8,0B2021Y,189.04,147,Gasoline,15.2896,Fan & Song 2022\nBuick GL8,0B2020W,170.14,123,Gasoline,17.2718,Fan & Song 2022\nBuick GL8,0B2020W,19.91,123,Gasoline,2.5226,Fan & Song 2022\nBuick GL8,0B2020W,100.44,123,Gasoline,13.6065,Fan & Song 2022\nBuick GL8,0B2020W,15.82,123,Gasoline,1.8115,Fan & Song 2022\nBuick GL8,0B2020W,6.8,123,Gasoline,0.9099,Fan & Song 2022\nBuick GL8,0B2020W,16.43,123,Gasoline,2.3731,Fan & Song 2022\nBuick GL8,0B2021Y,2.8,147,Gasoline,0.4813,Fan & Song 2022\nBuick GL8,0B2021Y,4.43,147,Gasoline,0.5208,Fan & Song 2022\nBuick GL8,0B2020W,3.68,123,Gasoline,0.6225,Fan & Song 2022\nBuick GL8,0B2020W,7.68,123,Gasoline,1.0144,Fan & Song 2022\nBuick GL8,0B2020W,32.5,123,Gasoline,3.984,Fan & Song 2022\nBuick GL8,0B2020W,9.1,123,Gasoline,1.4152,Fan & Song 2022\nBuick GL8,0B2020W,11.91,123,Gasoline,1.2387,Fan & Song 2022\nBuick GL8,0B2021Y,16.38,147,Gasoline,1.5768,Fan & Song 2022\nBuick GL8,0B2020W,13.09,123,Gasoline,1.7538,Fan & Song 2022\nBuick GL8,0B2020W,1.82,123,Gasoline,0.4336,Fan & Song 2022\nBuick GL8,0B2020W,17.56,123,Gasoline,3.2227,Fan & Song 2022\nBuick GL8,0B2020W,101.74,123,Gasoline,9.685,Fan & Song 2022\nBuick GL8,0B2021Y,48.78,147,Gasoline,5.0196,Fan & Song 2022\nBuick GL8,0B2021Y,34.4,147,Gasoline,3.4112,Fan & Song 2022\nBuick GL8,0B2020W,51.12,123,Gasoline,6.3248,Fan & Song 2022\nBuick GL8,0B2021Y,11.16,147,Gasoline,1.5322,Fan & Song 2022\nBuick GL8,0B2021Y,41.3,147,Gasoline,4.1334,Fan & Song 2022\nBuick GL8,0B2020W,51.21,123,Gasoline,5.9031,Fan & Song 2022\nBuick GL8,0B2020W,22.28,123,Gasoline,2.646,Fan & Song 2022\nBuick GL8,0B2021Y,133.96,147,Gasoline,12.4539,Fan & Song 2022\nBuick GL8,0B2020W,21.47,123,Gasoline,1.9311,Fan & Song 2022\nBuick GL8,0B2020W,20.75,123,Gasoline,2.7851,Fan & Song 2022\nBuick GL8,0B2021Y,52.92,147,Gasoline,4.5203,Fan & Song 2022\nBuick GL8,0B2020W,41.77,123,Gasoline,4.8844,Fan & Song 2022\nBuick GL8,0B2021Y,225.29,147,Gasoline,23.6756,Fan & Song 2022\nBuick GL8,0B2020W,5.52,123,Gasoline,0.7838,Fan & Song 2022\nBuick GL8,0B2021Y,1.56,147,Gasoline,0.2683,Fan & Song 2022\nBuick GL8,0B2021Y,131.84,147,Gasoline,9.8366,Fan & Song 2022\nBuick GL8,0B2020W,5.02,123,Gasoline,0.8153,Fan & Song 2022\nBuick GL8,0B2021Y,188.44,147,Gasoline,16.6265,Fan & Song 2022\nBuick GL8,0B2020W,70.46,123,Gasoline,6.9574,Fan & Song 2022\nBuick GL8,0B2021Y,50.6,147,Gasoline,5.6772,Fan & Song 2022\nBuick GL8,0B2021Y,111.86,147,Gasoline,10.0301,Fan & Song 2022\nBuick GL8,0B2021Y,100.1,147,Gasoline,9.475,Fan & Song 2022\nBuick GL8,0B2020W,106.82,123,Gasoline,10.5875,Fan & Song 2022\nBuick GL8,0B2020W,199.26,123,Gasoline,17.2246,Fan & Song 2022\nBuick GL8,0B2020W,30.78,123,Gasoline,3.771,Fan & Song 2022\nBuick GL8,0B2021Y,83.34,147,Gasoline,6.8582,Fan & Song 2022\nBuick GL8,0B2020W,195.18,123,Gasoline,17.8397,Fan & Song 2022\nBuick GL8,0B2021Y,26.68,147,Gasoline,3.3397,Fan & Song 2022\nBuick GL8,0B2021Y,64.84,147,Gasoline,5.414,Fan & Song 2022\nBuick GL8,0B2021Y,155.34,147,Gasoline,20.6463,Fan & Song 2022\nBuick GL8,0B2021Y,166.35,147,Gasoline,22.847,Fan & Song 2022\nBuick GL8,0B2021Y,3.55,147,Gasoline,0.5103,Fan & Song 2022\nBuick GL8,0B2020W,100.44,123,Gasoline,11.9268,Fan & Song 2022\nBuick GL8,0B2020W,75.48,123,Gasoline,8.3765,Fan & Song 2022\nBuick GL8,0B2020W,29.85,123,Gasoline,3.3705,Fan & Song 2022\nBuick GL8,0B2020W,24.57,123,Gasoline,2.3666,Fan & Song 2022\nBuick GL8,0B2020W,8.53,123,Gasoline,0.9812,Fan & Song 2022\nBuick GL8,0B2020W,107.63,123,Gasoline,12.05,Fan & Song 2022\nBuick GL8,0B2020W,43.49,123,Gasoline,4.2879,Fan & Song 2022\nBuick GL8,0B2020W,106.1,123,Gasoline,9.765,Fan & Song 2022\nBuick GL8,0B2020W,3.91,123,Gasoline,0.6697,Fan & Song 2022\nBuick GL8,0B2020W,46.36,123,Gasoline,4.3505,Fan & Song 2022\nBuick GL8,0B2021Y,11.84,147,Gasoline,1.1332,Fan & Song 2022\nBuick GL8,0B2021Y,70.19,147,Gasoline,7.0728,Fan & Song 2022\nBuick GL8,0B2021Y,6.74,147,Gasoline,1.0529,Fan & Song 2022\nBuick GL8,0B2020W,2.54,123,Gasoline,0.3825,Fan & Song 2022\nBuick GL8,0B2021Y,322.79,147,Gasoline,37.4627,Fan & Song 2022\nBuick GL8,0B2021Y,21.5,147,Gasoline,1.8315,Fan & Song 2022\nBuick GL8,0B2021Y,189.3,147,Gasoline,19.8654,Fan & Song 2022\nBuick GL8,0B2021Y,125.26,147,Gasoline,10.2955,Fan & Song 2022\nBuick GL8,0B2021Y,5.32,147,Gasoline,0.6958,Fan & Song 2022\nBuick GL8,0B2021Y,99.69,147,Gasoline,8.2512,Fan & Song 2022\nBuick GL8,0B2020W,30.98,123,Gasoline,2.5041,Fan & Song 2022\nBuick GL8,0B2020W,104.07,123,Gasoline,10.1994,Fan & Song 2022\nBuick GL8,0B2020W,141.23,123,Gasoline,13.3416,Fan & Song 2022\nBuick GL8,0B2020W,22.26,123,Gasoline,2.8598,Fan & Song 2022\nBuick GL8,0B2020W,67.82,123,Gasoline,6.4243,Fan & Song 2022\nBuick GL8,0B2021Y,18.02,147,Gasoline,2.5237,Fan & Song 2022\nBuick GL8,0B2020W,10.37,123,Gasoline,1.328,Fan & Song 2022\nBuick GL8,0B2021Y,37.44,147,Gasoline,2.9517,Fan & Song 2022\nBuick GL8,0B2020W,115.44,123,Gasoline,11.8472,Fan & Song 2022\nBuick GL8,0B2021Y,44.79,147,Gasoline,3.8577,Fan & Song 2022\nBuick GL8,0B2021Y,46.42,147,Gasoline,3.7332,Fan & Song 2022\nBuick GL8,0B2021Y,24.63,147,Gasoline,2.7582,Fan & Song 2022\nBuick GL8,0B2021Y,50.0,147,Gasoline,3.9213,Fan & Song 2022\nBuick GL8,0B2020W,13.69,123,Gasoline,1.9997,Fan & Song 2022\nBuick GL8,0B2021Y,4.55,147,Gasoline,0.6037,Fan & Song 2022\nBuick GL8,0B2021Y,3.14,147,Gasoline,0.3778,Fan & Song 2022\nBuick GL8,0B2020W,17.93,123,Gasoline,1.8871,Fan & Song 2022\nBuick GL8,0B2020W,5.7,123,Gasoline,0.791,Fan & Song 2022\nBuick GL8,0B2020W,13.06,123,Gasoline,1.3647,Fan & Song 2022\nBuick GL8,0B2021Y,60.62,147,Gasoline,5.4672,Fan & Song 2022\nBuick GL8,0B2020W,38.45,123,Gasoline,4.0734,Fan & Song 2022\nBuick GL8,0B2020W,14.72,123,Gasoline,1.6998,Fan & Song 2022\nBuick GL8,0B2020W,11.6,123,Gasoline,1.436,Fan & Song 2022\nBuick GL8,0B2021Y,1.91,147,Gasoline,0.3143,Fan & Song 2022\nBuick GL8,0B2020W,10.03,123,Gasoline,1.2039,Fan & Song 2022\nBuick GL8,0B2021Y,21.03,147,Gasoline,4.495,Fan & Song 2022\nBuick GL8,0B2020W,138.42,123,Gasoline,10.4512,Fan & Song 2022\nBuick GL8,0B2020W,116.7,123,Gasoline,9.9751,Fan & Song 2022\nBuick GL8,0B2021Y,33.56,147,Gasoline,4.2311,Fan & Song 2022\nBuick GL8,0B2020W,21.92,123,Gasoline,2.412,Fan & Song 2022\nBuick GL8,0B2020W,127.95,123,Gasoline,13.1827,Fan & Song 2022\nBuick GL8,0B2021Y,3.08,147,Gasoline,0.4646,Fan & Song 2022\nBuick GL8,0B2020W,25.45,123,Gasoline,2.574,Fan & Song 2022\nBuick GL8,0B2020W,12.85,123,Gasoline,1.4102,Fan & Song 2022\nBuick GL8,0B2021Y,148.17,147,Gasoline,16.7204,Fan & Song 2022\nBuick GL8,0B2020W,91.05,123,Gasoline,13.0069,Fan & Song 2022\nBuick GL8,0B2021Y,9.42,147,Gasoline,1.0923,Fan & Song 2022\nBuick GL8,0B2020W,113.82,123,Gasoline,9.3879,Fan & Song 2022\nBuick GL8,0B2021Y,10.84,147,Gasoline,1.2548,Fan & Song 2022\nBuick GL8,0B2021Y,5.93,147,Gasoline,1.0018,Fan & Song 2022\nBuick GL8,0B2020W,4.56,123,Gasoline,0.8029,Fan & Song 2022\nBuick GL8,0B2021Y,10.09,147,Gasoline,1.2454,Fan & Song 2022\nBuick GL8,0B2021Y,37.37,147,Gasoline,3.0294,Fan & Song 2022\nBuick GL8,0B2020W,1.86,123,Gasoline,0.2388,Fan & Song 2022\nBuick GL8,0B2021Y,49.77,147,Gasoline,4.1163,Fan & Song 2022\nBuick GL8,0B2021Y,175.31,147,Gasoline,14.6262,Fan & Song 2022\nBuick GL8,0B2021Y,3.44,147,Gasoline,0.5498,Fan & Song 2022\nBuick GL8,0B2021Y,204.79,147,Gasoline,15.6764,Fan & Song 2022\nBuick GL8,0B2021Y,41.46,147,Gasoline,3.4331,Fan & Song 2022\nBuick GL8,0B2021Y,2.31,147,Gasoline,0.298,Fan & Song 2022\nBuick GL8,0B2020W,35.13,123,Gasoline,3.6602,Fan & Song 2022\nBuick GL8,0B2021Y,1.77,147,Gasoline,0.2676,Fan & Song 2022\nBuick GL8,0B2020W,20.52,123,Gasoline,2.0562,Fan & Song 2022\nBuick GL8,0B2020W,5.26,123,Gasoline,0.7233,Fan & Song 2022\nBuick GL8,0B2020W,31.22,123,Gasoline,3.0199,Fan & Song 2022\nBuick GL8,0B2020W,380.85,123,Gasoline,33.3197,Fan & Song 2022\nBuick GL8,0B2020W,119.74,123,Gasoline,9.3943,Fan & Song 2022\nBuick GL8,0B2021Y,3.86,147,Gasoline,0.6665,Fan & Song 2022\nBuick GL8,0B2020W,14.59,123,Gasoline,2.9305,Fan & Song 2022\nBuick GL8,0B2020W,5.46,123,Gasoline,0.5489,Fan & Song 2022\nBuick GL8,0B2021Y,1.68,147,Gasoline,0.2421,Fan & Song 2022\nBuick GL8,0B2021Y,4.39,147,Gasoline,0.7118,Fan & Song 2022\nBuick GL8,0B2021Y,8.82,147,Gasoline,0.842,Fan & Song 2022\nBuick GL8,0B2020W,260.58,123,Gasoline,21.0177,Fan & Song 2022\nBuick GL8,0B2020W,2.44,123,Gasoline,0.3929,Fan & Song 2022\nBuick GL8,0B2020W,107.65,123,Gasoline,9.2457,Fan & Song 2022\nBuick GL8,0B2021Y,17.08,147,Gasoline,1.9332,Fan & Song 2022\nBuick GL8,0B2021Y,33.24,147,Gasoline,3.4099,Fan & Song 2022\nBuick GL8,0B2020W,72.82,123,Gasoline,6.6083,Fan & Song 2022\nBuick GL8,0B2020W,24.28,123,Gasoline,2.5928,Fan & Song 2022\nBuick GL8,0B2021Y,16.96,147,Gasoline,1.7697,Fan & Song 2022\nBuick GL8,0B2021Y,4.23,147,Gasoline,0.5264,Fan & Song 2022\nBuick GL8,0B2020W,12.24,123,Gasoline,1.6262,Fan & Song 2022\nBuick GL8,0B2021Y,57.14,147,Gasoline,5.5523,Fan & Song 2022\nBuick GL8,0B2020W,132.11,123,Gasoline,14.4132,Fan & Song 2022\nBuick GL8,0B2021Y,41.31,147,Gasoline,4.2506,Fan & Song 2022\nBuick GL8,0B2021Y,37.86,147,Gasoline,4.8409,Fan & Song 2022\nBuick GL8,0B2020W,98.82,123,Gasoline,10.5994,Fan & Song 2022\nBuick GL8,0B2021Y,3.74,147,Gasoline,0.5993,Fan & Song 2022\nBuick GL8,0B2020W,107.75,123,Gasoline,11.2488,Fan & Song 2022\nBuick GL8,0B2020W,217.61,123,Gasoline,18.9063,Fan & Song 2022\nBuick GL8,0B2020W,132.68,123,Gasoline,11.576,Fan & Song 2022\nBuick GL8,0B2021Y,35.84,147,Gasoline,3.9753,Fan & Song 2022\nBuick GL8,0B2021Y,6.17,147,Gasoline,0.8579,Fan & Song 2022\nBuick GL8,0B2020W,2.2,123,Gasoline,0.3811,Fan & Song 2022\nBuick GL8,0B2020W,20.13,123,Gasoline,2.1981,Fan & Song 2022\nBuick GL8,0B2020W,195.77,123,Gasoline,18.1264,Fan & Song 2022\nBuick GL8,0B2020W,30.38,123,Gasoline,2.6451,Fan & Song 2022\nBuick GL8,0B2021Y,39.73,147,Gasoline,3.8986,Fan & Song 2022\nBuick GL8,0B2020W,111.54,123,Gasoline,9.9106,Fan & Song 2022\nBuick GL8,0B2020W,14.03,123,Gasoline,1.5726,Fan & Song 2022\nBuick GL8,0B2020W,7.15,123,Gasoline,0.9078,Fan & Song 2022\nBuick GL8,0B2021Y,15.7,147,Gasoline,1.8747,Fan & Song 2022\nBuick GL8,0B2021Y,38.28,147,Gasoline,3.2118,Fan & Song 2022\nBuick GL8,0B2021Y,2.18,147,Gasoline,0.2401,Fan & Song 2022\nBuick GL8,0B2021Y,1.86,147,Gasoline,0.324,Fan & Song 2022\nBuick GL8,0B2020W,116.73,123,Gasoline,15.4418,Fan & Song 2022\nBuick GL8,0B2021Y,40.58,147,Gasoline,3.1775,Fan & Song 2022\nBuick GL8,0B2021Y,36.16,147,Gasoline,3.1633,Fan & Song 2022\nBuick GL8,0B2021Y,3.6,147,Gasoline,0.5282,Fan & Song 2022\nBuick GL8,0B2020W,22.57,123,Gasoline,3.1621,Fan & Song 2022\nBuick GL8,0B2020W,22.27,123,Gasoline,2.7286,Fan & Song 2022\nBuick GL8,0B2020W,8.96,123,Gasoline,1.2552,Fan & Song 2022\nBuick GL8,0B2021Y,21.14,147,Gasoline,1.9834,Fan & Song 2022\nBuick GL8,0B2020W,106.22,123,Gasoline,10.0855,Fan & Song 2022\nBuick GL8,0B2021Y,133.13,147,Gasoline,12.2205,Fan & Song 2022\nBuick GL8,0B2021Y,64.13,147,Gasoline,4.706,Fan & Song 2022\nBuick GL8,0B2020W,10.44,123,Gasoline,1.3621,Fan & Song 2022\nBuick GL8,0B2021Y,23.82,147,Gasoline,1.8361,Fan & Song 2022\nBuick GL8,0B2020W,21.9,123,Gasoline,2.6751,Fan & Song 2022\nBuick GL8,0B2021Y,214.12,147,Gasoline,15.2199,Fan & Song 2022\nBuick GL8,0B2020W,155.08,123,Gasoline,15.2779,Fan & Song 2022\nBuick GL8,0B2020W,1.57,123,Gasoline,0.2655,Fan & Song 2022\nBuick GL8,0B2021Y,5.15,147,Gasoline,0.563,Fan & Song 2022\nBuick GL8,0B2020W,18.52,123,Gasoline,2.2625,Fan & Song 2022\nBuick GL8,0B2020W,7.3,123,Gasoline,1.1806,Fan & Song 2022\nBuick GL8,0B2020W,22.04,123,Gasoline,2.9569,Fan & Song 2022\nBuick GL8,0B2021Y,58.96,147,Gasoline,6.0183,Fan & Song 2022\nBuick GL8,0B2021Y,210.74,147,Gasoline,25.1073,Fan & Song 2022\nBuick GL8,0B2021Y,266.2,147,Gasoline,32.3257,Fan & Song 2022\nBuick GL8,0B2020W,16.99,123,Gasoline,1.9442,Fan & Song 2022\nBuick GL8,0B2020W,12.6,123,Gasoline,1.3305,Fan & Song 2022\nBuick GL8,0B2021Y,1.97,147,Gasoline,0.2714,Fan & Song 2022\nBuick GL8,0B2020W,153.21,123,Gasoline,15.9668,Fan & Song 2022\nBuick GL8,0B2020W,21.66,123,Gasoline,2.5744,Fan & Song 2022\nBuick GL8,0B2021Y,12.39,147,Gasoline,1.5171,Fan & Song 2022\nBuick GL8,0B2021Y,26.92,147,Gasoline,1.6857,Fan & Song 2022\nBuick GL8,0B2020W,13.53,123,Gasoline,2.1327,Fan & Song 2022\nBuick GL8,0B2020W,33.5,123,Gasoline,3.62,Fan & Song 2022\nBuick GL8,0B2020W,31.51,123,Gasoline,4.5181,Fan & Song 2022\nBuick GL8,0B2020W,119.45,123,Gasoline,10.5076,Fan & Song 2022\nBuick GL8,0B2021Y,29.7,147,Gasoline,3.1852,Fan & Song 2022\nBuick GL8,0B2021Y,30.47,147,Gasoline,2.8734,Fan & Song 2022\nBuick GL8,0B2020W,26.78,123,Gasoline,2.5301,Fan & Song 2022\nBuick GL8,0B2020W,90.7,123,Gasoline,10.3015,Fan & Song 2022\nBuick GL8,0B2021Y,118.38,147,Gasoline,9.5183,Fan & Song 2022\nBuick GL8,0B2020W,13.94,123,Gasoline,1.4837,Fan & Song 2022\nBuick GL8,0B2020W,157.39,123,Gasoline,14.7349,Fan & Song 2022\nBuick GL8,0B2021Y,3.96,147,Gasoline,0.6995,Fan & Song 2022\nBuick GL8,0B2021Y,166.18,147,Gasoline,12.9602,Fan & Song 2022\nBuick GL8,0B2020W,71.7,123,Gasoline,7.5882,Fan & Song 2022\nBuick GL8,0B2021Y,20.81,147,Gasoline,2.0132,Fan & Song 2022\nBuick GL8,0B2021Y,23.24,147,Gasoline,2.8962,Fan & Song 2022\nBuick GL8,0B2020W,2.98,123,Gasoline,0.5036,Fan & Song 2022\nBuick GL8,0B2020W,111.68,123,Gasoline,10.6012,Fan & Song 2022\nBuick GL8,0B2020W,8.9,123,Gasoline,1.5288,Fan & Song 2022\nBuick GL8,0B2020W,3.21,123,Gasoline,0.6666,Fan & Song 2022\nBuick GL8,0B2020W,27.56,123,Gasoline,3.3941,Fan & Song 2022\nBuick GL8,0B2020W,24.86,123,Gasoline,3.7289,Fan & Song 2022\nBuick GL8,0B2020W,14.16,123,Gasoline,1.4735,Fan & Song 2022\nBuick GL8,0B2021Y,30.23,147,Gasoline,4.3987,Fan & Song 2022\nBuick GL8,0B2021Y,3.14,147,Gasoline,0.3605,Fan & Song 2022\nBuick GL8,0B2020W,11.55,123,Gasoline,1.8158,Fan & Song 2022\nBuick GL8,0B2020W,125.12,123,Gasoline,16.7153,Fan & Song 2022\nBuick GL8,0B2020W,157.66,123,Gasoline,12.1326,Fan & Song 2022\nBuick GL8,0B2020W,31.01,123,Gasoline,4.6104,Fan & Song 2022\nBuick GL8,0B2020W,10.31,123,Gasoline,1.4972,Fan & Song 2022\nBuick GL8,0B2020W,107.89,123,Gasoline,10.5108,Fan & Song 2022\nBuick GL8,0B2020W,74.09,123,Gasoline,8.1476,Fan & Song 2022\nBuick GL8,0B2021Y,1.71,147,Gasoline,0.2863,Fan & Song 2022\nBuick GL8,0B2021Y,13.7,147,Gasoline,2.0995,Fan & Song 2022\nBuick GL8,0B2020W,47.28,123,Gasoline,6.3834,Fan & Song 2022\nBuick GL8,0B2021Y,32.75,147,Gasoline,6.4456,Fan & Song 2022\nBuick GL8,0B2020W,45.25,123,Gasoline,4.3415,Fan & Song 2022\nBuick GL8,0B2020W,39.94,123,Gasoline,4.5683,Fan & Song 2022\nBuick GL8,0B2020W,115.73,123,Gasoline,16.0197,Fan & Song 2022\nBuick GL8,0B2021Y,7.09,147,Gasoline,0.8169,Fan & Song 2022\nBuick GL8,0B2021Y,39.12,147,Gasoline,3.3118,Fan & Song 2022\nBuick GL8,0B2020W,5.34,123,Gasoline,1.1647,Fan & Song 2022\nBuick GL8,0B2021Y,137.02,147,Gasoline,14.9525,Fan & Song 2022\nBuick GL8,0B2021Y,109.06,147,Gasoline,10.363,Fan & Song 2022\nBuick GL8,0B2021Y,148.15,147,Gasoline,14.4418,Fan & Song 2022\nBuick GL8,0B2021Y,46.06,147,Gasoline,4.6817,Fan & Song 2022\nBuick GL8,0B2020W,13.94,123,Gasoline,2.0033,Fan & Song 2022\nBuick GL8,0B2021Y,127.36,147,Gasoline,10.9606,Fan & Song 2022\nBuick GL8,0B2021Y,77.01,147,Gasoline,6.8807,Fan & Song 2022\nBuick GL8,0B2021Y,24.8,147,Gasoline,1.6845,Fan & Song 2022\nBuick GL8,0B2020W,10.5,123,Gasoline,1.2459,Fan & Song 2022\nBuick GL8,0B2020W,3.68,123,Gasoline,0.726,Fan & Song 2022\nBuick GL8,0B2020W,1.96,123,Gasoline,0.3517,Fan & Song 2022\nBuick GL8,0B2021Y,45.22,147,Gasoline,5.5701,Fan & Song 2022\nBuick GL8,0B2020W,47.88,123,Gasoline,4.6983,Fan & Song 2022\nBuick GL8,0B2020W,133.6,123,Gasoline,17.276,Fan & Song 2022\nBuick GL8,0B2021Y,7.52,147,Gasoline,1.6862,Fan & Song 2022\nBuick GL8,0B2021Y,159.4,147,Gasoline,14.9768,Fan & Song 2022\nBuick GL8,0B2021Y,28.42,147,Gasoline,3.1575,Fan & Song 2022\nBuick GL8,0B2021Y,27.16,147,Gasoline,2.8582,Fan & Song 2022\nBuick GL8,0B2021Y,14.01,147,Gasoline,1.6411,Fan & Song 2022\nBuick GL8,0B2021Y,73.96,147,Gasoline,7.6635,Fan & Song 2022\nBuick GL8,0B2021Y,8.92,147,Gasoline,0.9259,Fan & Song 2022\nBuick GL8,0B2020W,30.13,123,Gasoline,3.2396,Fan & Song 2022\nBuick GL8,0B2021Y,12.83,147,Gasoline,0.9235,Fan & Song 2022\nBuick GL8,0B2021Y,16.98,147,Gasoline,2.5899,Fan & Song 2022\nBuick GL8,0B2021Y,50.78,147,Gasoline,4.3304,Fan & Song 2022\nBuick GL8,0B2021Y,192.36,147,Gasoline,15.3455,Fan & Song 2022\nBuick GL8,0B2020W,110.09,123,Gasoline,14.5,Fan & Song 2022\nBuick GL8,0B2020W,329.13,123,Gasoline,34.8571,Fan & Song 2022\nBuick GL8,0B2021Y,52.66,147,Gasoline,7.1166,Fan & Song 2022\nBuick GL8,0B2020W,23.91,123,Gasoline,2.5773,Fan & Song 2022\nBuick GL8,0B2021Y,16.28,147,Gasoline,3.0673,Fan & Song 2022\nBuick GL8,0B2020W,11.27,123,Gasoline,1.82,Fan & Song 2022\nBuick GL8,0B2021Y,128.82,147,Gasoline,9.6531,Fan & Song 2022\nBuick GL8,0B2021Y,59.34,147,Gasoline,6.2547,Fan & Song 2022\nBuick GL8,0B2020W,119.62,123,Gasoline,11.4877,Fan & Song 2022\nBuick GL8,0B2021Y,52.06,147,Gasoline,4.9537,Fan & Song 2022\nBuick GL8,0B2020W,6.72,123,Gasoline,0.7651,Fan & Song 2022\nBuick GL8,0B2021Y,110.1,147,Gasoline,12.8429,Fan & Song 2022\nBuick GL8,0B2020W,185.49,123,Gasoline,16.4408,Fan & Song 2022\nBuick GL8,0B2020W,35.29,123,Gasoline,3.3783,Fan & Song 2022\nBuick GL8,0B2020W,32.45,123,Gasoline,4.6203,Fan & Song 2022\nBuick GL8,0B2020W,11.3,123,Gasoline,1.821,Fan & Song 2022\nBuick GL8,0B2021Y,147.62,147,Gasoline,9.5274,Fan & Song 2022\nBuick GL8,0B2021Y,146.93,147,Gasoline,11.7595,Fan & Song 2022\nBuick GL8,0B2021Y,39.38,147,Gasoline,3.2912,Fan & Song 2022\nBuick GL8,0B2020W,24.72,123,Gasoline,4.1131,Fan & Song 2022\nBuick GL8,0B2021Y,23.79,147,Gasoline,2.6323,Fan & Song 2022\nBuick GL8,0B2020W,6.72,123,Gasoline,1.1282,Fan & Song 2022\nBuick GL8,0B2021Y,2.12,147,Gasoline,0.3924,Fan & Song 2022\nBuick GL8,0B2021Y,13.85,147,Gasoline,1.5039,Fan & Song 2022\nBuick GL8,0B2020W,9.44,123,Gasoline,1.2187,Fan & Song 2022\nBuick GL8,0B2021Y,6.66,147,Gasoline,1.2597,Fan & Song 2022\nBuick GL8,0B2020W,4.6,123,Gasoline,1.1556,Fan & Song 2022\nBuick GL8,0B2021Y,1.62,147,Gasoline,0.2183,Fan & Song 2022\nBuick GL8,0B2020W,4.83,123,Gasoline,0.8243,Fan & Song 2022\nBuick GL8,0B2020W,161.14,123,Gasoline,16.0872,Fan & Song 2022\nBuick GL8,0B2021Y,5.53,147,Gasoline,0.6767,Fan & Song 2022\nBuick GL8,0B2020W,33.24,123,Gasoline,4.1619,Fan & Song 2022\nBuick GL8,0B2021Y,3.34,147,Gasoline,0.4545,Fan & Song 2022\nBuick GL8,0B2020W,21.26,123,Gasoline,3.967,Fan & Song 2022\nBuick GL8,0B2021Y,3.16,147,Gasoline,0.6266,Fan & Song 2022\nBuick GL8,0B2021Y,61.97,147,Gasoline,6.6172,Fan & Song 2022\nBuick GL8,0B2021Y,89.5,147,Gasoline,6.8834,Fan & Song 2022\nBuick GL8,0B2021Y,9.84,147,Gasoline,1.0789,Fan & Song 2022\nBuick GL8,0B2021Y,25.55,147,Gasoline,3.3027,Fan & Song 2022\nBuick GL8,0B2021Y,110.33,147,Gasoline,8.1589,Fan & Song 2022\nBuick GL8,0B2021Y,2.56,147,Gasoline,0.3728,Fan & Song 2022\nBuick GL8,0B2021Y,5.72,147,Gasoline,1.012,Fan & Song 2022\nBuick GL8,0B2021Y,351.53,147,Gasoline,30.6194,Fan & Song 2022\nBuick GL8,0B2020W,83.43,123,Gasoline,9.6112,Fan & Song 2022\nBuick GL8,0B2020W,120.62,123,Gasoline,14.5073,Fan & Song 2022\nBuick GL8,0B2021Y,13.9,147,Gasoline,1.5524,Fan & Song 2022\nBuick GL8,0B2021Y,38.14,147,Gasoline,3.9761,Fan & Song 2022\nBuick GL8,0B2020W,221.72,123,Gasoline,20.8831,Fan & Song 2022\nBuick GL8,0B2021Y,149.62,147,Gasoline,12.6716,Fan & Song 2022\nBuick GL8,0B2020W,10.14,123,Gasoline,2.3169,Fan & Song 2022\nBuick GL8,0B2020W,52.16,123,Gasoline,5.636,Fan & Song 2022\nBuick GL8,0B2021Y,84.82,147,Gasoline,14.7463,Fan & Song 2022\nBuick GL8,0B2021Y,24.62,147,Gasoline,5.5604,Fan & Song 2022\nBuick GL8,0B2021Y,12.5,147,Gasoline,1.2935,Fan & Song 2022\nBuick GL8,0B2021Y,40.31,147,Gasoline,3.8087,Fan & Song 2022\nBuick GL8,0B2021Y,177.9,147,Gasoline,16.0466,Fan & Song 2022\nBuick GL8,0B2021Y,121.38,147,Gasoline,13.6399,Fan & Song 2022\nBuick GL8,0B2020W,7.51,123,Gasoline,1.8528,Fan & Song 2022\nBuick GL8,0B2021Y,155.74,147,Gasoline,14.1131,Fan & Song 2022\nBuick GL8,0B2020W,3.52,123,Gasoline,0.6679,Fan & Song 2022\nBuick GL8,0B2021Y,125.33,147,Gasoline,9.8632,Fan & Song 2022\nBuick GL8,0B2020W,17.85,123,Gasoline,1.798,Fan & Song 2022\nBuick GL8,0B2020W,182.45,123,Gasoline,16.1925,Fan & Song 2022\nBuick GL8,0B2020W,3.44,123,Gasoline,0.4378,Fan & Song 2022\nBuick GL8,0B2020W,37.66,123,Gasoline,9.4218,Fan & Song 2022\nBuick GL8,0B2021Y,156.38,147,Gasoline,16.5919,Fan & Song 2022\nBuick GL8,0B2021Y,150.97,147,Gasoline,12.6195,Fan & Song 2022\nBuick GL8,0B2020W,42.03,123,Gasoline,4.3637,Fan & Song 2022\nBuick GL8,0B2021Y,49.96,147,Gasoline,6.7482,Fan & Song 2022\nBuick GL8,0B2021Y,3.83,147,Gasoline,0.7422,Fan & Song 2022\nBuick GL8,0B2020W,32.15,123,Gasoline,5.9276,Fan & Song 2022\nBuick GL8,0B2021Y,95.08,147,Gasoline,9.0969,Fan & Song 2022\nBuick GL8,0B2020W,60.26,123,Gasoline,6.7604,Fan & Song 2022\nBuick GL8,0B2021Y,215.29,147,Gasoline,18.589,Fan & Song 2022\nBuick GL8,0B2021Y,12.25,147,Gasoline,1.5288,Fan & Song 2022\nBuick GL8,0B2021Y,1.61,147,Gasoline,0.1701,Fan & Song 2022\nBuick GL8,0B2021Y,4.16,147,Gasoline,0.6129,Fan & Song 2022\nBuick GL8,0B2020W,23.7,123,Gasoline,2.3839,Fan & Song 2022\nBuick GL8,0B2021Y,6.03,147,Gasoline,0.7405,Fan & Song 2022\nBuick GL8,0B2021Y,83.03,147,Gasoline,6.3043,Fan & Song 2022\nBuick GL8,0B2021Y,11.06,147,Gasoline,1.1921,Fan & Song 2022\nBuick GL8,0B2020W,59.85,123,Gasoline,6.1602,Fan & Song 2022\nBuick GL8,0B2021Y,28.1,147,Gasoline,2.7185,Fan & Song 2022\nBuick GL8,0B2021Y,117.15,147,Gasoline,11.3563,Fan & Song 2022\nBuick GL8,0B2020W,58.59,123,Gasoline,7.8417,Fan & Song 2022\nBuick GL8,0B2020W,31.06,123,Gasoline,3.5473,Fan & Song 2022\nBuick GL8,0B2021Y,47.93,147,Gasoline,5.8616,Fan & Song 2022\nBuick GL8,0B2021Y,106.55,147,Gasoline,11.6342,Fan & Song 2022\nBuick GL8,0B2020W,1.74,123,Gasoline,0.2617,Fan & Song 2022\nBuick GL8,0B2021Y,6.78,147,Gasoline,1.4877,Fan & Song 2022\nBuick GL8,0B2021Y,3.28,147,Gasoline,0.3878,Fan & Song 2022\nBuick GL8,0B2021Y,25.99,147,Gasoline,3.002,Fan & Song 2022\nBuick GL8,0B2021Y,5.81,147,Gasoline,0.6315,Fan & Song 2022\nBuick GL8,0B2021Y,18.38,147,Gasoline,2.5482,Fan & Song 2022\nBuick GL8,0B2020W,3.82,123,Gasoline,0.5366,Fan & Song 2022\nBuick GL8,0B2020W,230.85,123,Gasoline,23.0162,Fan & Song 2022\nBuick GL8,0B2020W,392.32,123,Gasoline,39.9268,Fan & Song 2022\nBuick GL8,0B2020W,201.2,123,Gasoline,18.0992,Fan & Song 2022\nBuick GL8,0B2021Y,131.42,147,Gasoline,12.0202,Fan & Song 2022\nBuick GL8,0B2020W,7.93,123,Gasoline,0.9802,Fan & Song 2022\nBuick GL8,0B2021Y,87.96,147,Gasoline,9.6522,Fan & Song 2022\nBuick GL8,0B2021Y,108.21,147,Gasoline,9.1663,Fan & Song 2022\nBuick GL8,0B2020W,14.06,123,Gasoline,1.5941,Fan & Song 2022\nBuick GL8,0B2021Y,191.14,147,Gasoline,14.7748,Fan & Song 2022\nBuick GL8,0B2020W,3.58,123,Gasoline,0.4511,Fan & Song 2022\nBuick GL8,0B2020W,241.92,123,Gasoline,23.7302,Fan & Song 2022\nBuick GL8,0B2021Y,119.21,147,Gasoline,8.9343,Fan & Song 2022\nBuick GL8,0B2020W,1.63,123,Gasoline,0.3324,Fan & Song 2022\nBuick GL8,0B2021Y,167.77,147,Gasoline,15.9666,Fan & Song 2022\nBuick GL8,0B2021Y,109.34,147,Gasoline,10.5588,Fan & Song 2022\nBuick GL8,0B2021Y,54.49,147,Gasoline,5.3582,Fan & Song 2022\nBuick GL8,0B2020W,38.75,123,Gasoline,6.8513,Fan & Song 2022\nBuick GL8,0B2020W,6.44,123,Gasoline,0.7995,Fan & Song 2022\nBuick GL8,0B2021Y,2.94,147,Gasoline,0.5044,Fan & Song 2022\nBuick GL8,0B2021Y,4.34,147,Gasoline,0.9267,Fan & Song 2022\nBuick GL8,0B2020W,116.42,123,Gasoline,10.0411,Fan & Song 2022\nBuick GL8,0B2021Y,160.36,147,Gasoline,11.8301,Fan & Song 2022\nBuick GL8,0B2021Y,112.12,147,Gasoline,12.1699,Fan & Song 2022\nBuick GL8,0B2021Y,21.32,147,Gasoline,3.1504,Fan & Song 2022\nBuick GL8,0B2020W,4.94,123,Gasoline,1.0498,Fan & Song 2022\nBuick GL8,0B2021Y,200.85,147,Gasoline,18.6536,Fan & Song 2022\nBuick GL8,0B2021Y,185.21,147,Gasoline,15.2448,Fan & Song 2022\nBuick GL8,0B2020W,6.18,123,Gasoline,0.9804,Fan & Song 2022\nBuick GL8,0B2020W,1.94,123,Gasoline,0.3036,Fan & Song 2022\nBuick GL8,0B2020W,44.03,123,Gasoline,4.5824,Fan & Song 2022\nBuick GL8,0B2021Y,4.04,147,Gasoline,0.6041,Fan & Song 2022\nBuick GL8,0B2021Y,11.36,147,Gasoline,1.0955,Fan & Song 2022\nBuick GL8,0B2020W,119.07,123,Gasoline,10.9639,Fan & Song 2022\nBuick GL8,0B2021Y,29.22,147,Gasoline,2.3487,Fan & Song 2022\nBuick GL8,0B2020W,190.92,123,Gasoline,17.8352,Fan & Song 2022\nBuick GL8,0B2021Y,40.84,147,Gasoline,3.6595,Fan & Song 2022\nBuick GL8,0B2021Y,6.09,147,Gasoline,1.4307,Fan & Song 2022\nBuick GL8,0B2020W,22.27,123,Gasoline,3.2374,Fan & Song 2022\nBuick GL8,0B2021Y,26.73,147,Gasoline,3.1714,Fan & Song 2022\nBuick GL8,0B2021Y,4.17,147,Gasoline,0.4953,Fan & Song 2022\nBuick GL8,0B2021Y,16.79,147,Gasoline,1.3011,Fan & Song 2022\nBuick GL8,0B2020W,39.38,123,Gasoline,3.6149,Fan & Song 2022\nBuick GL8,0B2021Y,7.3,147,Gasoline,0.8539,Fan & Song 2022\nBuick GL8,0B2020W,11.71,123,Gasoline,1.775,Fan & Song 2022\nBuick GL8,0B2020W,69.86,123,Gasoline,6.11,Fan & Song 2022\nBuick GL8,0B2020W,16.66,123,Gasoline,1.9932,Fan & Song 2022\nBuick GL8,0B2021Y,132.66,147,Gasoline,13.8929,Fan & Song 2022\nBuick GL8,0B2021Y,2.7,147,Gasoline,0.4147,Fan & Song 2022\nBuick GL8,0B2021Y,40.4,147,Gasoline,3.084,Fan & Song 2022\nBuick GL8,0B2020W,18.33,123,Gasoline,1.6217,Fan & Song 2022\nBuick GL8,0B2020W,12.46,123,Gasoline,2.0218,Fan & Song 2022\nBuick GL8,0B2021Y,10.67,147,Gasoline,1.3992,Fan & Song 2022\nBuick GL8,0B2020W,101.39,123,Gasoline,10.0487,Fan & Song 2022\nBuick GL8,0B2020W,17.42,123,Gasoline,2.8031,Fan & Song 2022\nBuick GL8,0B2021Y,10.74,147,Gasoline,1.0783,Fan & Song 2022\nBuick GL8,0B2021Y,10.94,147,Gasoline,1.083,Fan & Song 2022\nBuick GL8,0B2021Y,16.02,147,Gasoline,2.6894,Fan & Song 2022\nBuick GL8,0B2020W,132.03,123,Gasoline,12.1678,Fan & Song 2022\nBuick GL8,0B2020W,14.62,123,Gasoline,1.3401,Fan & Song 2022\nBuick GL8,0B2020W,112.48,123,Gasoline,10.2247,Fan & Song 2022\nBuick GL8,0B2020W,3.5,123,Gasoline,0.5362,Fan & Song 2022\nBuick GL8,0B2021Y,3.99,147,Gasoline,0.5382,Fan & Song 2022\nBuick GL8,0B2021Y,3.13,147,Gasoline,0.8665,Fan & Song 2022\nBuick GL8,0B2021Y,18.99,147,Gasoline,3.5379,Fan & Song 2022\nBuick GL8,0B2020W,7.28,123,Gasoline,0.937,Fan & Song 2022\nBuick GL8,0B2020W,9.63,123,Gasoline,1.285,Fan & Song 2022\nBuick GL8,0B2020W,47.56,123,Gasoline,6.6905,Fan & Song 2022\nBuick GL8,0B2020W,3.63,123,Gasoline,0.4891,Fan & Song 2022\nBuick GL8,0B2020W,127.18,123,Gasoline,11.8517,Fan & Song 2022\nBuick GL8,0B2020W,12.1,123,Gasoline,1.3438,Fan & Song 2022\nBuick GL8,0B2021Y,17.03,147,Gasoline,1.6988,Fan & Song 2022\nBuick GL8,0B2021Y,32.52,147,Gasoline,3.5723,Fan & Song 2022\nBuick GL8,0B2021Y,217.82,147,Gasoline,17.8105,Fan & Song 2022\nBuick GL8,0B2020W,296.6,123,Gasoline,28.5127,Fan & Song 2022\nBuick GL8,0B2021Y,39.52,147,Gasoline,3.9455,Fan & Song 2022\nBuick GL8,0B2021Y,11.36,147,Gasoline,1.6081,Fan & Song 2022\nBuick GL8,0B2021Y,3.63,147,Gasoline,0.495,Fan & Song 2022\nBuick GL8,0B2021Y,124.11,147,Gasoline,12.6536,Fan & Song 2022\nBuick GL8,0B2021Y,169.04,147,Gasoline,15.1286,Fan & Song 2022\nBuick GL8,0B2021Y,183.86,147,Gasoline,13.8595,Fan & Song 2022\nBuick GL8,0B2021Y,16.48,147,Gasoline,1.43,Fan & Song 2022\nBuick GL8,0B2020W,13.4,123,Gasoline,1.5562,Fan & Song 2022\nBuick GL8,0B2020W,176.94,123,Gasoline,16.3844,Fan & Song 2022\nBuick GL8,0B2020W,61.08,123,Gasoline,5.6601,Fan & Song 2022\nBuick GL8,0B2021Y,175.38,147,Gasoline,14.4917,Fan & Song 2022\nBuick GL8,0B2020W,52.91,123,Gasoline,4.5585,Fan & Song 2022\nBuick GL8,0B2020W,49.99,123,Gasoline,5.0662,Fan & Song 2022\nBuick GL8,0B2021Y,10.99,147,Gasoline,1.4374,Fan & Song 2022\nBuick GL8,0B2020W,193.72,123,Gasoline,19.5438,Fan & Song 2022\nBuick GL8,0B2021Y,114.43,147,Gasoline,8.9926,Fan & Song 2022\nBuick GL8,0B2021Y,7.69,147,Gasoline,1.074,Fan & Song 2022\nBuick GL8,0B2021Y,190.09,147,Gasoline,16.202,Fan & Song 2022\nBuick GL8,0B2021Y,17.54,147,Gasoline,2.0964,Fan & Song 2022\nBuick GL8,0B2021Y,80.96,147,Gasoline,6.2055,Fan & Song 2022\nBuick GL8,0B2021Y,115.0,147,Gasoline,9.8514,Fan & Song 2022\nBuick GL8,0B2020W,6.86,123,Gasoline,0.9103,Fan & Song 2022\nBuick GL8,0B2021Y,6.55,147,Gasoline,0.7824,Fan & Song 2022\nBuick GL8,0B2020W,166.41,123,Gasoline,13.6403,Fan & Song 2022\nBuick GL8,0B2020W,78.81,123,Gasoline,6.963,Fan & Song 2022\nBuick GL8,0B2020W,2.66,123,Gasoline,0.3904,Fan & Song 2022\nBuick GL8,0B2021Y,43.01,147,Gasoline,3.8007,Fan & Song 2022\nBuick GL8,0B2020W,103.92,123,Gasoline,10.628,Fan & Song 2022\nBuick GL8,0B2020W,46.74,123,Gasoline,5.12,Fan & Song 2022\nBuick GL8,0B2021Y,302.69,147,Gasoline,28.9497,Fan & Song 2022\nBuick GL8,0B2021Y,28.99,147,Gasoline,3.3332,Fan & Song 2022\nBuick GL8,0B2020W,35.3,123,Gasoline,3.3858,Fan & Song 2022\nBuick GL8,0B2021Y,15.46,147,Gasoline,1.6237,Fan & Song 2022\nBuick GL8,0B2020W,30.88,123,Gasoline,3.9864,Fan & Song 2022\nBuick GL8,0B2020W,8.15,123,Gasoline,0.8995,Fan & Song 2022\nBuick GL8,0B2021Y,103.44,147,Gasoline,8.5873,Fan & Song 2022\nBuick GL8,0B2020W,7.05,123,Gasoline,0.8648,Fan & Song 2022\nBuick GL8,0B2020W,41.68,123,Gasoline,4.2582,Fan & Song 2022\nBuick GL8,0B2021Y,164.13,147,Gasoline,15.7428,Fan & Song 2022\nBuick GL8,0B2021Y,58.11,147,Gasoline,5.4889,Fan & Song 2022\nBuick GL8,0B2020W,2.88,123,Gasoline,0.5226,Fan & Song 2022\nBuick GL8,0B2020W,27.78,123,Gasoline,3.1536,Fan & Song 2022\nBuick GL8,0B2021Y,12.83,147,Gasoline,1.4847,Fan & Song 2022\nBuick GL8,0B2020W,64.44,123,Gasoline,9.2853,Fan & Song 2022\nBuick GL8,0B2020W,17.09,123,Gasoline,1.9701,Fan & Song 2022\nBuick GL8,0B2020W,105.16,123,Gasoline,9.7343,Fan & Song 2022\nBuick GL8,0B2020W,49.3,123,Gasoline,4.492,Fan & Song 2022\nBuick GL8,0B2020W,48.05,123,Gasoline,5.2815,Fan & Song 2022\nBuick GL8,0B2020W,199.73,123,Gasoline,21.2806,Fan & Song 2022\nBuick GL8,0B2020W,148.92,123,Gasoline,14.6767,Fan & Song 2022\nBuick GL8,0B2020W,33.2,123,Gasoline,3.1247,Fan & Song 2022\nBuick GL8,0B2021Y,51.42,147,Gasoline,4.2178,Fan & Song 2022\nBuick GL8,0B2021Y,33.2,147,Gasoline,3.707,Fan & Song 2022\nBuick GL8,0B2020W,10.6,123,Gasoline,1.2829,Fan & Song 2022\nBuick GL8,0B2021Y,5.25,147,Gasoline,0.5256,Fan & Song 2022\nBuick GL8,0B2020W,2.06,123,Gasoline,0.4307,Fan & Song 2022\nBuick GL8,0B2021Y,119.58,147,Gasoline,8.9499,Fan & Song 2022\nBuick GL8,0B2021Y,4.74,147,Gasoline,0.5758,Fan & Song 2022\nBuick GL8,0B2021Y,101.51,147,Gasoline,8.1147,Fan & Song 2022\nBuick GL8,0B2020W,161.06,123,Gasoline,15.1646,Fan & Song 2022\nBuick GL8,0B2021Y,49.84,147,Gasoline,4.7007,Fan & Song 2022\nBuick GL8,0B2020W,1.78,123,Gasoline,0.4352,Fan & Song 2022\nBuick GL8,0B2020W,26.65,123,Gasoline,2.5729,Fan & Song 2022\nBuick GL8,0B2020W,110.72,123,Gasoline,8.0267,Fan & Song 2022\nBuick GL8,0B2020W,66.31,123,Gasoline,9.9328,Fan & Song 2022\nBuick GL8,0B2021Y,15.46,147,Gasoline,1.5196,Fan & Song 2022\nBuick GL8,0B2020W,104.96,123,Gasoline,10.514,Fan & Song 2022\nBuick GL8,0B2020W,119.14,123,Gasoline,14.7159,Fan & Song 2022\nBuick GL8,0B2020W,4.04,123,Gasoline,0.5584,Fan & Song 2022\nBuick GL8,0B2020W,143.42,123,Gasoline,13.9952,Fan & Song 2022\nBuick GL8,0B2021Y,3.58,147,Gasoline,0.3864,Fan & Song 2022\nBuick GL8,0B2020W,72.66,123,Gasoline,9.9622,Fan & Song 2022\nBuick GL8,0B2021Y,33.68,147,Gasoline,2.8669,Fan & Song 2022\nBuick GL8,0B2020W,35.67,123,Gasoline,4.2838,Fan & Song 2022\nBuick GL8,0B2021Y,22.0,147,Gasoline,1.9362,Fan & Song 2022\nBuick GL8,0B2020W,42.52,123,Gasoline,4.0376,Fan & Song 2022\nBuick GL8,0B2021Y,42.8,147,Gasoline,4.1358,Fan & Song 2022\nBuick GL8,0B2021Y,178.97,147,Gasoline,13.0918,Fan & Song 2022\nBuick GL8,0B2021Y,53.42,147,Gasoline,4.7179,Fan & Song 2022\nBuick GL8,0B2020W,2.45,123,Gasoline,0.3641,Fan & Song 2022\nBuick GL8,0B2021Y,4.25,147,Gasoline,0.792,Fan & Song 2022\nBuick GL8,0B2021Y,11.02,147,Gasoline,1.1354,Fan & Song 2022\nBuick GL8,0B2020W,37.48,123,Gasoline,4.9226,Fan & Song 2022\nBuick GL8,0B2020W,31.05,123,Gasoline,4.7206,Fan & Song 2022\nBuick GL8,0B2021Y,22.21,147,Gasoline,3.5017,Fan & Song 2022\nBuick GL8,0B2020W,6.18,123,Gasoline,0.7945,Fan & Song 2022\nBuick GL8,0B2020W,8.7,123,Gasoline,1.1524,Fan & Song 2022\nBuick GL8,0B2020W,114.01,123,Gasoline,13.4385,Fan & Song 2022\nBuick GL8,0B2021Y,23.68,147,Gasoline,2.5109,Fan & Song 2022\nBuick GL8,0B2021Y,24.58,147,Gasoline,2.6137,Fan & Song 2022\nBuick GL8,0B2020W,3.58,123,Gasoline,0.5282,Fan & Song 2022\nBuick GL8,0B2021Y,323.52,147,Gasoline,21.7584,Fan & Song 2022\nBuick GL8,0B2020W,6.13,123,Gasoline,0.8245,Fan & Song 2022\nBuick GL8,0B2021Y,13.13,147,Gasoline,1.3532,Fan & Song 2022\nBuick GL8,0B2020W,28.01,123,Gasoline,2.912,Fan & Song 2022\nBuick GL8,0B2021Y,14.8,147,Gasoline,1.7433,Fan & Song 2022\nBuick GL8,0B2021Y,90.35,147,Gasoline,6.4998,Fan & Song 2022\nBuick GL8,0B2021Y,30.17,147,Gasoline,3.7095,Fan & Song 2022\nBuick GL8,0B2021Y,110.51,147,Gasoline,7.6787,Fan & Song 2022\nBuick GL8,0B2021Y,32.15,147,Gasoline,2.8579,Fan & Song 2022\nBuick GL8,0B2020W,37.99,123,Gasoline,3.2903,Fan & Song 2022\nBuick GL8,0B2021Y,20.97,147,Gasoline,1.7356,Fan & Song 2022\nBuick GL8,0B2021Y,4.78,147,Gasoline,0.7355,Fan & Song 2022\nBuick GL8,0B2020W,18.52,123,Gasoline,3.0752,Fan & Song 2022\nBuick GL8,0B2020W,10.96,123,Gasoline,1.398,Fan & Song 2022\nBuick GL8,0B2021Y,17.89,147,Gasoline,1.5795,Fan & Song 2022\nBuick GL8,0B2020W,115.08,123,Gasoline,10.1271,Fan & Song 2022\nBuick GL8,0B2020W,51.95,123,Gasoline,6.5228,Fan & Song 2022\nBuick GL8,0B2020W,154.15,123,Gasoline,16.3858,Fan & Song 2022\nBuick GL8,0B2021Y,141.95,147,Gasoline,14.2803,Fan & Song 2022\nBuick GL8,0B2020W,16.26,123,Gasoline,2.5954,Fan & Song 2022\nBuick GL8,0B2020W,10.49,123,Gasoline,1.2489,Fan & Song 2022\nBuick GL8,0B2020W,91.87,123,Gasoline,8.7891,Fan & Song 2022\nBuick GL8,0B2020W,9.43,123,Gasoline,1.2579,Fan & Song 2022\nBuick GL8,0B2020W,2.43,123,Gasoline,0.4827,Fan & Song 2022\nBuick GL8,0B2021Y,27.38,147,Gasoline,3.5591,Fan & Song 2022\nBuick GL8,0B2021Y,30.44,147,Gasoline,3.1892,Fan & Song 2022\nBuick GL8,0B2020W,10.32,123,Gasoline,1.8039,Fan & Song 2022\nBuick GL8,0B2021Y,2.57,147,Gasoline,0.4197,Fan & Song 2022\nBuick GL8,0B2020W,210.7,123,Gasoline,20.8631,Fan & Song 2022\nBuick GL8,0B2020W,3.06,123,Gasoline,0.3484,Fan & Song 2022\nBuick GL8,0B2020W,9.89,123,Gasoline,1.0887,Fan & Song 2022\nBuick GL8,0B2020W,110.78,123,Gasoline,11.7315,Fan & Song 2022\nBuick GL8,0B2020W,38.99,123,Gasoline,4.8086,Fan & Song 2022\nBuick GL8,0B2020W,9.27,123,Gasoline,1.7611,Fan & Song 2022\nBuick GL8,0B2020W,6.72,123,Gasoline,1.0156,Fan & Song 2022\nBuick GL8,0B2021Y,4.3,147,Gasoline,1.0155,Fan & Song 2022\nBuick GL8,0B2021Y,18.81,147,Gasoline,2.0244,Fan & Song 2022\nBuick GL8,0B2021Y,2.72,147,Gasoline,0.3975,Fan & Song 2022\nBuick GL8,0B2021Y,263.7,147,Gasoline,15.4713,Fan & Song 2022\nBuick GL8,0B2021Y,108.75,147,Gasoline,8.5298,Fan & Song 2022\nBuick GL8,0B2020W,6.08,123,Gasoline,0.8301,Fan & Song 2022\nBuick GL8,0B2020W,5.84,123,Gasoline,0.8204,Fan & Song 2022\nBuick GL8,0B2020W,8.03,123,Gasoline,0.7566,Fan & Song 2022\nBuick GL8,0B2021Y,2.44,147,Gasoline,0.4633,Fan & Song 2022\nBuick GL8,0B2020W,12.23,123,Gasoline,1.4924,Fan & Song 2022\nBuick GL8,0B2020W,22.15,123,Gasoline,1.9688,Fan & Song 2022\nBuick GL8,0B2020W,60.28,123,Gasoline,6.7709,Fan & Song 2022\nBuick GL8,0B2021Y,32.83,147,Gasoline,2.8632,Fan & Song 2022\nBuick GL8,0B2021Y,3.76,147,Gasoline,0.6115,Fan & Song 2022\nBuick GL8,0B2021Y,108.12,147,Gasoline,9.4507,Fan & Song 2022\nBuick GL8,0B2020W,38.84,123,Gasoline,4.7121,Fan & Song 2022\nBuick GL8,0B2020W,9.49,123,Gasoline,1.821,Fan & Song 2022\nBuick GL8,0B2020W,21.51,123,Gasoline,2.0882,Fan & Song 2022\nBuick GL8,0B2020W,46.92,123,Gasoline,4.6808,Fan & Song 2022\nBuick GL8,0B2020W,16.37,123,Gasoline,1.5653,Fan & Song 2022\nBuick GL8,0B2021Y,28.11,147,Gasoline,2.9827,Fan & Song 2022\nBuick GL8,0B2020W,65.8,123,Gasoline,6.7434,Fan & Song 2022\nBuick GL8,0B2020W,3.42,123,Gasoline,0.5902,Fan & Song 2022\nChevrolet Captiva,3K10409,16.7,123,Gasoline,1.7574,Fan & Song 2022\nChevrolet Captiva,3K10409,87.17,123,Gasoline,8.1683,Fan & Song 2022\nChevrolet Captiva,3K10409,3.25,123,Gasoline,0.8101,Fan & Song 2022\nChevrolet Captiva,3K10409,4.39,123,Gasoline,0.7346,Fan & Song 2022\nChevrolet Captiva,3K10409,13.74,123,Gasoline,1.8568,Fan & Song 2022\nChevrolet Captiva,3K10409,3.98,123,Gasoline,0.7316,Fan & Song 2022\nChevrolet Captiva,3K10409,2.72,123,Gasoline,0.4248,Fan & Song 2022\nChevrolet Captiva,3K10409,197.69,123,Gasoline,21.7249,Fan & Song 2022\nChevrolet Captiva,3K10409,15.76,123,Gasoline,1.7545,Fan & Song 2022\nChevrolet Captiva,3K10409,184.42,123,Gasoline,20.1791,Fan & Song 2022\nChevrolet Captiva,3K10409,5.9,123,Gasoline,0.9728,Fan & Song 2022\nChevrolet Captiva,3K10409,2.82,123,Gasoline,0.3906,Fan & Song 2022\nChevrolet Captiva,3K10409,13.86,123,Gasoline,1.3675,Fan & Song 2022\nChevrolet Captiva,3K10409,4.98,123,Gasoline,0.9078,Fan & Song 2022\nChevrolet Captiva,3K10409,16.71,123,Gasoline,1.9354,Fan & Song 2022\nChevrolet Captiva,3K10409,17.59,123,Gasoline,1.8372,Fan & Song 2022\nChevrolet Captiva,3K10409,6.86,123,Gasoline,1.2393,Fan & Song 2022\nChevrolet Captiva,3K10409,5.65,123,Gasoline,0.8124,Fan & Song 2022\nChevrolet Captiva,3K10409,14.21,123,Gasoline,2.4304,Fan & Song 2022\nChevrolet Captiva,3K10409,11.17,123,Gasoline,1.4088,Fan & Song 2022\nChevrolet Captiva,3K10409,36.5,123,Gasoline,4.6156,Fan & Song 2022\nChevrolet Captiva,3K10409,20.04,123,Gasoline,2.856,Fan & Song 2022\nChevrolet Captiva,3K10409,57.18,123,Gasoline,6.4709,Fan & Song 2022\nChevrolet Captiva,3K10409,4.08,123,Gasoline,0.7332,Fan & Song 2022\nChevrolet Captiva,3K10409,53.11,123,Gasoline,5.9097,Fan & Song 2022\nChevrolet Captiva,3K10409,4.52,123,Gasoline,0.568,Fan & Song 2022\nChevrolet Captiva,3K10409,32.7,123,Gasoline,3.1705,Fan & Song 2022\nChevrolet Captiva,3K10409,45.22,123,Gasoline,4.0776,Fan & Song 2022\nChevrolet Captiva,3K10409,10.0,123,Gasoline,1.304,Fan & Song 2022\nChevrolet Captiva,3K10409,12.32,123,Gasoline,1.7522,Fan & Song 2022\nChevrolet Captiva,3K10409,17.2,123,Gasoline,1.9833,Fan & Song 2022\nChevrolet Captiva,3K10409,5.53,123,Gasoline,1.0252,Fan & Song 2022\nChevrolet Captiva,3K10409,2.7,123,Gasoline,0.5631,Fan & Song 2022\nChevrolet Captiva,3K10409,101.59,123,Gasoline,9.5314,Fan & Song 2022\nChevrolet Captiva,3K10409,54.57,123,Gasoline,5.7989,Fan & Song 2022\nChevrolet Captiva,3K10409,102.36,123,Gasoline,9.1788,Fan & Song 2022\nChevrolet Captiva,3K10409,83.21,123,Gasoline,8.8694,Fan & Song 2022\nChevrolet Captiva,3K10409,4.58,123,Gasoline,0.7396,Fan & Song 2022\nChevrolet Captiva,3K10409,6.16,123,Gasoline,0.8319,Fan & Song 2022\nChevrolet Captiva,3K10409,7.83,123,Gasoline,1.0066,Fan & Song 2022\nChevrolet Captiva,3K10409,21.66,123,Gasoline,3.2525,Fan & Song 2022\nChevrolet Captiva,3K10409,56.77,123,Gasoline,6.0744,Fan & Song 2022\nChevrolet Captiva,3K10409,8.36,123,Gasoline,1.2329,Fan & Song 2022\nChevrolet Captiva,3K10409,13.97,123,Gasoline,1.8611,Fan & Song 2022\nChevrolet Captiva,3K10409,51.41,123,Gasoline,6.0557,Fan & Song 2022\nChevrolet Captiva,3K10409,94.26,123,Gasoline,9.4303,Fan & Song 2022\nChevrolet Captiva,3K10409,17.45,123,Gasoline,2.0028,Fan & Song 2022\nChevrolet Captiva,3K10409,5.28,123,Gasoline,0.9424,Fan & Song 2022\nChevrolet Captiva,3K10409,60.92,123,Gasoline,5.5932,Fan & Song 2022\nChevrolet Captiva,3K10409,2.53,123,Gasoline,0.4831,Fan & Song 2022\nChevrolet Captiva,3K10409,27.22,123,Gasoline,2.9508,Fan & Song 2022\nChevrolet Captiva,3K10409,503.81,123,Gasoline,46.6664,Fan & Song 2022\nChevrolet Captiva,3K10409,9.53,123,Gasoline,1.2544,Fan & Song 2022\nChevrolet Captiva,3K10409,112.94,123,Gasoline,10.4417,Fan & Song 2022\nChevrolet Captiva,3K10409,32.91,123,Gasoline,3.2557,Fan & Song 2022\nChevrolet Captiva,3K10409,30.9,123,Gasoline,3.3893,Fan & Song 2022\nChevrolet Captiva,3K10409,67.28,123,Gasoline,9.283,Fan & Song 2022\nChevrolet Captiva,3K10409,36.47,123,Gasoline,4.3558,Fan & Song 2022\nChevrolet Captiva,3K10409,79.59,123,Gasoline,9.2591,Fan & Song 2022\nChevrolet Captiva,3K10409,100.81,123,Gasoline,12.6724,Fan & Song 2022\nChevrolet Captiva,3K10409,16.84,123,Gasoline,1.8468,Fan & Song 2022\nChevrolet Captiva,3K10409,50.48,123,Gasoline,5.6997,Fan & Song 2022\nChevrolet Captiva,3K10409,183.0,123,Gasoline,16.399,Fan & Song 2022\nChevrolet Captiva,3K10409,6.39,123,Gasoline,1.0857,Fan & Song 2022\nChevrolet Captiva,3K10409,223.66,123,Gasoline,23.804,Fan & Song 2022\nChevrolet Captiva,3K10409,2.95,123,Gasoline,0.51,Fan & Song 2022\nChevrolet Captiva,3K10409,5.67,123,Gasoline,0.8731,Fan & Song 2022\nChevrolet Captiva,3K10409,253.24,123,Gasoline,26.9195,Fan & Song 2022\nChevrolet Captiva,3K10409,235.1,123,Gasoline,23.9095,Fan & Song 2022\nChevrolet Captiva,3K10409,173.32,123,Gasoline,16.8545,Fan & Song 2022\nChevrolet Captiva,3K10409,3.94,123,Gasoline,0.7852,Fan & Song 2022\nChevrolet Captiva,3K10409,192.01,123,Gasoline,19.926,Fan & Song 2022\nChevrolet Captiva,3K10409,12.86,123,Gasoline,1.4154,Fan & Song 2022\nChevrolet Captiva,3K10409,4.12,123,Gasoline,0.6014,Fan & Song 2022\nChevrolet Captiva,3K10409,10.93,123,Gasoline,1.5797,Fan & Song 2022\nChevrolet Captiva,3K10409,74.87,123,Gasoline,7.0307,Fan & Song 2022\nChevrolet Captiva,3K10409,21.45,123,Gasoline,3.9895,Fan & Song 2022\nChevrolet Captiva,3K10409,37.34,123,Gasoline,4.2527,Fan & Song 2022\nChevrolet Captiva,3K10409,217.57,123,Gasoline,20.7313,Fan & Song 2022\nChevrolet Captiva,3K10409,2.55,123,Gasoline,0.5574,Fan & Song 2022\nChevrolet Captiva,3K10409,148.73,123,Gasoline,15.6379,Fan & Song 2022\nChevrolet Captiva,3K10409,114.4,123,Gasoline,11.7508,Fan & Song 2022\nChevrolet Captiva,3K10409,348.8,123,Gasoline,31.8371,Fan & Song 2022\nChevrolet Captiva,3K10409,116.52,123,Gasoline,10.7386,Fan & Song 2022\nChevrolet Captiva,3K10409,6.93,123,Gasoline,1.0228,Fan & Song 2022\nChevrolet Captiva,3K10409,113.75,123,Gasoline,10.6058,Fan & Song 2022\nChevrolet Captiva,3K10409,484.54,123,Gasoline,45.5469,Fan & Song 2022\nChevrolet Captiva,3K10409,8.22,123,Gasoline,1.1798,Fan & Song 2022\nChevrolet Captiva,3K10409,367.0,123,Gasoline,37.7565,Fan & Song 2022\nChevrolet Captiva,3K10409,1.88,123,Gasoline,0.3607,Fan & Song 2022\nChevrolet Captiva,3K10409,32.84,123,Gasoline,3.6277,Fan & Song 2022\nChevrolet Captiva,3K10409,182.99,123,Gasoline,16.8455,Fan & Song 2022\nChevrolet Captiva,3K10409,18.8,123,Gasoline,2.1345,Fan & Song 2022\nChevrolet Captiva,3K10409,5.9,123,Gasoline,0.8642,Fan & Song 2022\nChevrolet Captiva,3K10409,31.78,123,Gasoline,3.383,Fan & Song 2022\nChevrolet Captiva,3K10409,32.45,123,Gasoline,3.3408,Fan & Song 2022\nChevrolet Captiva,3K10409,1.87,123,Gasoline,0.4751,Fan & Song 2022\nChevrolet Captiva,3K10409,41.49,123,Gasoline,4.2372,Fan & Song 2022\nChevrolet Captiva,3K10409,54.32,123,Gasoline,5.7568,Fan & Song 2022\nChevrolet Captiva,3K10409,5.07,123,Gasoline,0.5204,Fan & Song 2022\nChevrolet Captiva,3K10409,2.96,123,Gasoline,0.4804,Fan & Song 2022\nChevrolet Captiva,3K10409,53.78,123,Gasoline,4.805,Fan & Song 2022\nChevrolet Captiva,3K10409,14.43,123,Gasoline,1.5915,Fan & Song 2022\nChevrolet Captiva,3K10409,4.3,123,Gasoline,0.5568,Fan & Song 2022\nChevrolet Captiva,3K10409,3.61,123,Gasoline,0.5583,Fan & Song 2022\nChevrolet Captiva,3K10409,147.75,123,Gasoline,13.1742,Fan & Song 2022\nChevrolet Captiva,3K10409,14.25,123,Gasoline,1.68,Fan & Song 2022\nChevrolet Captiva,3K10409,3.1,123,Gasoline,0.6256,Fan & Song 2022\nChevrolet Captiva,3K10409,12.93,123,Gasoline,1.5491,Fan & Song 2022\nChevrolet Captiva,3K10409,12.54,123,Gasoline,1.4296,Fan & Song 2022\nChevrolet Captiva,3K10409,80.96,123,Gasoline,9.6864,Fan & Song 2022\nChevrolet Captiva,3K10409,6.8,123,Gasoline,0.946,Fan & Song 2022\nChevrolet Captiva,3K10409,103.1,123,Gasoline,9.1019,Fan & Song 2022\nChevrolet Captiva,3K10409,4.49,123,Gasoline,0.4916,Fan & Song 2022\nChevrolet Captiva,3K10409,17.54,123,Gasoline,1.845,Fan & Song 2022\nChevrolet Captiva,3K10409,16.42,123,Gasoline,1.5438,Fan & Song 2022\nChevrolet Captiva,3K10409,19.29,123,Gasoline,2.0456,Fan & Song 2022\nChevrolet Captiva,3K10409,60.35,123,Gasoline,5.3868,Fan & Song 2022\nChevrolet Captiva,3K10409,44.61,123,Gasoline,4.3604,Fan & Song 2022\nChevrolet Captiva,3K10409,6.98,123,Gasoline,0.9606,Fan & Song 2022\nChevrolet Captiva,3K10409,140.39,123,Gasoline,14.0952,Fan & Song 2022\nChevrolet Captiva,3K10409,82.47,123,Gasoline,7.6256,Fan & Song 2022\nChevrolet Captiva,3K10409,103.28,123,Gasoline,10.4551,Fan & Song 2022\nChevrolet Captiva,3K10409,109.43,123,Gasoline,10.5348,Fan & Song 2022\nChevrolet Captiva,3K10409,34.76,123,Gasoline,3.274,Fan & Song 2022\nChevrolet Captiva,3K10409,7.28,123,Gasoline,1.1792,Fan & Song 2022\nChevrolet Captiva,3K10409,34.78,123,Gasoline,2.8257,Fan & Song 2022\nChevrolet Captiva,3K10409,49.63,123,Gasoline,4.3269,Fan & Song 2022\nChevrolet Captiva,3K10409,14.34,123,Gasoline,1.697,Fan & Song 2022\nChevrolet Captiva,3K10409,45.82,123,Gasoline,4.8421,Fan & Song 2022\nChevrolet Captiva,3K10409,5.33,123,Gasoline,0.8994,Fan & Song 2022\nChevrolet Captiva,3K10409,50.07,123,Gasoline,4.6914,Fan & Song 2022\nChevrolet Captiva,3K10409,216.2,123,Gasoline,16.982,Fan & Song 2022\nChevrolet Captiva,3K10409,5.19,123,Gasoline,0.7867,Fan & Song 2022\nChevrolet Captiva,3K10409,124.26,123,Gasoline,11.2669,Fan & Song 2022\nChevrolet Captiva,3K10409,63.64,123,Gasoline,6.5264,Fan & Song 2022\nChevrolet Captiva,3K10409,3.33,123,Gasoline,0.6193,Fan & Song 2022\nChevrolet Captiva,3K10409,172.42,123,Gasoline,14.7318,Fan & Song 2022\nChevrolet Captiva,3K10409,25.97,123,Gasoline,2.8369,Fan & Song 2022\nChevrolet Captiva,3K10409,50.52,123,Gasoline,4.7339,Fan & Song 2022\nChevrolet Captiva,3K10409,43.88,123,Gasoline,4.7201,Fan & Song 2022\nChevrolet Captiva,3K10409,3.78,123,Gasoline,0.4064,Fan & Song 2022\nChevrolet Captiva,3K10409,17.64,123,Gasoline,2.2943,Fan & Song 2022\nChevrolet Captiva,3K10409,123.84,123,Gasoline,11.3632,Fan & Song 2022\nChevrolet Captiva,3K10409,2.72,123,Gasoline,0.6069,Fan & Song 2022\nChevrolet Captiva,3K10409,10.65,123,Gasoline,1.2526,Fan & Song 2022\nChevrolet Captiva,3K10409,1.63,123,Gasoline,0.3112,Fan & Song 2022\nChevrolet Captiva,3K10409,13.07,123,Gasoline,1.9215,Fan & Song 2022\nChevrolet Captiva,3K10409,117.09,123,Gasoline,12.1576,Fan & Song 2022\nChevrolet Captiva,3K10409,78.29,123,Gasoline,8.2745,Fan & Song 2022\nChevrolet Captiva,3K10409,9.66,123,Gasoline,1.4837,Fan & Song 2022\nChevrolet Captiva,3K10409,11.47,123,Gasoline,1.9157,Fan & Song 2022\nChevrolet Captiva,3K10409,125.24,123,Gasoline,11.1949,Fan & Song 2022\nChevrolet Captiva,3K10409,20.93,123,Gasoline,2.423,Fan & Song 2022\nChevrolet Captiva,3K10409,11.8,123,Gasoline,1.3549,Fan & Song 2022\nChevrolet Captiva,3K10409,10.48,123,Gasoline,1.242,Fan & Song 2022\nChevrolet Captiva,3K10409,108.86,123,Gasoline,8.0699,Fan & Song 2022\nChevrolet Captiva,3K10409,13.83,123,Gasoline,1.7436,Fan & Song 2022\nChevrolet Captiva,3K10409,51.61,123,Gasoline,4.6723,Fan & Song 2022\nChevrolet Captiva,3K10409,14.26,123,Gasoline,2.0875,Fan & Song 2022\nChevrolet Captiva,3K10409,27.1,123,Gasoline,3.0071,Fan & Song 2022\nChevrolet Captiva,3K10409,14.71,123,Gasoline,1.8922,Fan & Song 2022\nChevrolet Captiva,3K10409,45.37,123,Gasoline,4.1051,Fan & Song 2022\nChevrolet Captiva,3K10409,18.0,123,Gasoline,2.1809,Fan & Song 2022\nChevrolet Captiva,3K10409,8.54,123,Gasoline,1.1466,Fan & Song 2022\nChevrolet Captiva,3K10409,61.66,123,Gasoline,5.8796,Fan & Song 2022\nChevrolet Captiva,3K10409,132.38,123,Gasoline,12.8864,Fan & Song 2022\nChevrolet Captiva,3K10409,18.03,123,Gasoline,1.8646,Fan & Song 2022\nChevrolet Captiva,3K10409,33.11,123,Gasoline,3.0852,Fan & Song 2022\nChevrolet Captiva,3K10409,26.05,123,Gasoline,3.3349,Fan & Song 2022\nChevrolet Captiva,3K10409,4.16,123,Gasoline,0.7954,Fan & Song 2022\nChevrolet Captiva,3K10409,105.3,123,Gasoline,9.0941,Fan & Song 2022\nChevrolet Captiva,3K10409,96.86,123,Gasoline,9.6541,Fan & Song 2022\nChevrolet Captiva,3K10409,1.58,123,Gasoline,0.3489,Fan & Song 2022\nChevrolet Captiva,3K10409,1.76,123,Gasoline,0.3905,Fan & Song 2022\nChevrolet Captiva,3K10409,48.65,123,Gasoline,4.7235,Fan & Song 2022\nChevrolet Captiva,3K10409,112.8,123,Gasoline,10.4764,Fan & Song 2022\nChevrolet Captiva,3K10409,74.76,123,Gasoline,7.4129,Fan & Song 2022\nChevrolet Captiva,3K10409,12.58,123,Gasoline,1.7061,Fan & Song 2022\nChevrolet Captiva,3K10409,131.25,123,Gasoline,11.5673,Fan & Song 2022\nChevrolet Captiva,3K10409,14.62,123,Gasoline,1.8356,Fan & Song 2022\nChevrolet Captiva,3K10409,16.23,123,Gasoline,2.144,Fan & Song 2022\nChevrolet Captiva,3K10409,10.04,123,Gasoline,1.1462,Fan & Song 2022\nChevrolet Captiva,3K10409,6.91,123,Gasoline,0.7727,Fan & Song 2022\nChevrolet Captiva,3K10409,11.59,123,Gasoline,1.6978,Fan & Song 2022\nChevrolet Captiva,3K10409,104.24,123,Gasoline,9.6788,Fan & Song 2022\nChevrolet Captiva,3K10409,109.16,123,Gasoline,9.6463,Fan & Song 2022\nChevrolet Captiva,3K10409,16.65,123,Gasoline,1.7388,Fan & Song 2022\nChevrolet Captiva,3K10409,18.97,123,Gasoline,1.9784,Fan & Song 2022\nChevrolet Captiva,3K10409,50.22,123,Gasoline,4.9096,Fan & Song 2022\nChevrolet Captiva,3K10409,2.14,123,Gasoline,0.4148,Fan & Song 2022\nChevrolet Captiva,3K10409,79.3,123,Gasoline,7.5142,Fan & Song 2022\nChevrolet Captiva,3K10409,12.21,123,Gasoline,1.294,Fan & Song 2022\nChevrolet Captiva,3K10409,32.63,123,Gasoline,3.3167,Fan & Song 2022\nChevrolet Captiva,3K10409,29.34,123,Gasoline,3.4031,Fan & Song 2022\nChevrolet Captiva,3K10409,5.11,123,Gasoline,0.7589,Fan & Song 2022\nChevrolet Captiva,3K10409,33.83,123,Gasoline,3.0769,Fan & Song 2022\nChevrolet Captiva,3K10409,10.53,123,Gasoline,1.167,Fan & Song 2022\nChevrolet Captiva,3K10409,35.24,123,Gasoline,3.5812,Fan & Song 2022\nChevrolet Captiva,3K10409,126.82,123,Gasoline,11.3456,Fan & Song 2022\nChevrolet Captiva,3K10409,118.59,123,Gasoline,11.0728,Fan & Song 2022\nChevrolet Captiva,3K10409,46.47,123,Gasoline,4.497,Fan & Song 2022\nChevrolet Captiva,3K10409,10.37,123,Gasoline,1.5964,Fan & Song 2022\nChevrolet Captiva,3K10409,21.88,123,Gasoline,2.4916,Fan & Song 2022\nChevrolet Captiva,3K10409,3.57,123,Gasoline,0.4959,Fan & Song 2022\nChevrolet Captiva,3K10409,13.61,123,Gasoline,1.8876,Fan & Song 2022\nChevrolet Captiva,3K10409,49.95,123,Gasoline,4.3762,Fan & Song 2022\nChevrolet Captiva,3K10409,2.98,123,Gasoline,0.4006,Fan & Song 2022\nChevrolet Captiva,3K10409,104.27,123,Gasoline,9.6254,Fan & Song 2022\nChevrolet Captiva,3K10409,126.47,123,Gasoline,12.3434,Fan & Song 2022\nChevrolet Captiva,3K10409,153.29,123,Gasoline,14.8152,Fan & Song 2022\nChevrolet Captiva,3K10409,204.91,123,Gasoline,20.5497,Fan & Song 2022\nChevrolet Captiva,3K10409,141.79,123,Gasoline,15.0893,Fan & Song 2022\nChevrolet Captiva,3K10409,40.62,123,Gasoline,4.7925,Fan & Song 2022\nChevrolet Captiva,3K10409,117.86,123,Gasoline,10.9753,Fan & Song 2022\nChevrolet Captiva,3K10409,65.02,123,Gasoline,5.8614,Fan & Song 2022\nChevrolet Captiva,3K10409,9.51,123,Gasoline,1.2233,Fan & Song 2022\nChevrolet Captiva,3K10409,99.58,123,Gasoline,9.3326,Fan & Song 2022\nChevrolet Captiva,3K10409,310.22,123,Gasoline,31.312,Fan & Song 2022\nChevrolet Captiva,3K10409,126.3,123,Gasoline,13.8058,Fan & Song 2022\nChevrolet Captiva,3K10409,8.58,123,Gasoline,0.953,Fan & Song 2022\nChevrolet Captiva,3K10409,2.07,123,Gasoline,0.3336,Fan & Song 2022\nChevrolet Captiva,3K10409,13.07,123,Gasoline,1.5502,Fan & Song 2022\nChevrolet Captiva,3K10409,88.98,123,Gasoline,8.1466,Fan & Song 2022\nChevrolet Captiva,3K10409,109.62,123,Gasoline,9.9258,Fan & Song 2022\nChevrolet Captiva,3K10409,45.67,123,Gasoline,4.4955,Fan & Song 2022\nChevrolet Captiva,3K10409,174.48,123,Gasoline,18.3212,Fan & Song 2022\nChevrolet Captiva,3K10409,222.28,123,Gasoline,18.4166,Fan & Song 2022\nChevrolet Captiva,3K10409,48.57,123,Gasoline,5.3986,Fan & Song 2022\nChevrolet Captiva,3K10409,56.7,123,Gasoline,6.563,Fan & Song 2022\nChevrolet Captiva,3K10409,9.88,123,Gasoline,1.2471,Fan & Song 2022\nChevrolet Captiva,3K10409,2.06,123,Gasoline,0.3463,Fan & Song 2022\nChevrolet Captiva,3K10409,43.54,123,Gasoline,3.6894,Fan & Song 2022\nChevrolet Captiva,3K10409,148.62,123,Gasoline,11.665,Fan & Song 2022\nChevrolet Captiva,3K10409,75.59,123,Gasoline,6.373,Fan & Song 2022\nChevrolet Captiva,3K10409,18.56,123,Gasoline,2.3798,Fan & Song 2022\nChevrolet Captiva,3K10409,132.81,123,Gasoline,9.5083,Fan & Song 2022\nChevrolet Captiva,3K10409,5.8,123,Gasoline,0.9492,Fan & Song 2022\nChevrolet Captiva,3K10409,27.44,123,Gasoline,2.6271,Fan & Song 2022\nChevrolet Captiva,3K10409,114.73,123,Gasoline,9.6557,Fan & Song 2022\nChevrolet Captiva,3K10409,8.43,123,Gasoline,1.1348,Fan & Song 2022\nChevrolet Captiva,3K10409,2.85,123,Gasoline,0.5273,Fan & Song 2022\nChevrolet Captiva,3K10409,139.07,123,Gasoline,12.203,Fan & Song 2022\nChevrolet Captiva,3K10409,23.3,123,Gasoline,2.9753,Fan & Song 2022\nChevrolet Captiva,3K10409,4.56,123,Gasoline,0.5503,Fan & Song 2022\nChevrolet Captiva,3K10409,34.1,123,Gasoline,4.8543,Fan & Song 2022\nChevrolet Captiva,3K10409,31.68,123,Gasoline,2.8878,Fan & Song 2022\nChevrolet Captiva,3K10409,18.42,123,Gasoline,2.6863,Fan & Song 2022\nChevrolet Captiva,3K10409,13.78,123,Gasoline,1.4194,Fan & Song 2022\nChevrolet Captiva,3K10409,203.22,123,Gasoline,21.4404,Fan & Song 2022\nChevrolet Captiva,3K10409,25.82,123,Gasoline,2.9534,Fan & Song 2022\nChevrolet Captiva,3K10409,46.24,123,Gasoline,4.838,Fan & Song 2022\nChevrolet Captiva,3K10409,203.29,123,Gasoline,21.291,Fan & Song 2022\nChevrolet Captiva,3K10409,11.84,123,Gasoline,1.472,Fan & Song 2022\nChevrolet Captiva,3K10409,141.05,123,Gasoline,12.0361,Fan & Song 2022\nChevrolet Captiva,3K10409,10.42,123,Gasoline,1.0965,Fan & Song 2022\nChevrolet Captiva,3K10409,47.85,123,Gasoline,4.8997,Fan & Song 2022\nChevrolet Captiva,3K10409,31.39,123,Gasoline,3.108,Fan & Song 2022\nChevrolet Captiva,3K10409,40.57,123,Gasoline,3.5347,Fan & Song 2022\nChevrolet Captiva,3K10409,127.1,123,Gasoline,10.5851,Fan & Song 2022\nChevrolet Captiva,3K10409,182.48,123,Gasoline,15.6344,Fan & Song 2022\nChevrolet Captiva,3K10409,72.34,123,Gasoline,8.2996,Fan & Song 2022\nChevrolet Captiva,3K10409,2.78,123,Gasoline,0.5043,Fan & Song 2022\nChevrolet Captiva,3K10409,49.18,123,Gasoline,4.0029,Fan & Song 2022\nChevrolet Captiva,3K10409,4.52,123,Gasoline,0.7583,Fan & Song 2022\nChevrolet Captiva,3K10409,35.85,123,Gasoline,4.6243,Fan & Song 2022\nChevrolet Captiva,3K10409,16.67,123,Gasoline,1.4583,Fan & Song 2022\nChevrolet Captiva,3K10409,13.58,123,Gasoline,2.014,Fan & Song 2022\nChevrolet Captiva,3K10409,107.52,123,Gasoline,12.029,Fan & Song 2022\nChevrolet Captiva,3K10409,57.47,123,Gasoline,6.0792,Fan & Song 2022\nChevrolet Captiva,3K10409,38.22,123,Gasoline,6.1807,Fan & Song 2022\nChevrolet Captiva,3K10409,4.27,123,Gasoline,0.7485,Fan & Song 2022\nChevrolet Captiva,3K10409,16.62,123,Gasoline,2.0468,Fan & Song 2022\nChevrolet Captiva,3K10409,206.62,123,Gasoline,18.3862,Fan & Song 2022\nChevrolet Captiva,3K10409,48.32,123,Gasoline,4.8097,Fan & Song 2022\nChevrolet Captiva,3K10409,121.56,123,Gasoline,11.5913,Fan & Song 2022\nChevrolet Captiva,3K10409,28.08,123,Gasoline,3.7612,Fan & Song 2022\nChevrolet Captiva,3K10409,191.66,123,Gasoline,15.4623,Fan & Song 2022\nChevrolet Captiva,3K10409,109.22,123,Gasoline,9.4687,Fan & Song 2022\nChevrolet Captiva,3K10409,229.46,123,Gasoline,23.0337,Fan & Song 2022\nChevrolet Captiva,3K10409,313.11,123,Gasoline,29.5031,Fan & Song 2022\nChevrolet Captiva,3K10409,115.27,123,Gasoline,9.3383,Fan & Song 2022\nChevrolet Captiva,3K10409,3.48,123,Gasoline,0.681,Fan & Song 2022\nChevrolet Captiva,3K10409,133.14,123,Gasoline,14.3916,Fan & Song 2022\nChevrolet Captiva,3K10409,1.6,123,Gasoline,0.2673,Fan & Song 2022\nChevrolet Captiva,3K10409,8.29,123,Gasoline,1.3142,Fan & Song 2022\nChevrolet Captiva,3K10409,18.46,123,Gasoline,2.5015,Fan & Song 2022\nChevrolet Captiva,3K10409,94.92,123,Gasoline,7.5054,Fan & Song 2022\nChevrolet Captiva,3K10409,10.87,123,Gasoline,1.5242,Fan & Song 2022\nChevrolet Captiva,3K10409,3.92,123,Gasoline,0.4362,Fan & Song 2022\nChevrolet Captiva,3K10409,39.93,123,Gasoline,4.6263,Fan & Song 2022\nChevrolet Captiva,3K10409,13.7,123,Gasoline,1.5235,Fan & Song 2022\nChevrolet Captiva,3K10409,13.79,123,Gasoline,1.5826,Fan & Song 2022\nChevrolet Captiva,3K10409,9.98,123,Gasoline,1.1487,Fan & Song 2022\nChevrolet Captiva,3K10409,31.66,123,Gasoline,3.046,Fan & Song 2022\nChevrolet Captiva,3K10409,10.53,123,Gasoline,2.0292,Fan & Song 2022\nChevrolet Captiva,3K10409,103.22,123,Gasoline,10.7418,Fan & Song 2022\nChevrolet Captiva,3K10409,48.2,123,Gasoline,3.7008,Fan & Song 2022\nChevrolet Captiva,3K10409,12.28,123,Gasoline,1.856,Fan & Song 2022\nChevrolet Captiva,3K10409,5.49,123,Gasoline,0.8177,Fan & Song 2022\nChevrolet Captiva,3K10409,7.59,123,Gasoline,1.3428,Fan & Song 2022\nChevrolet Captiva,3K10409,16.3,123,Gasoline,1.9683,Fan & Song 2022\nChevrolet Captiva,3K10409,33.35,123,Gasoline,3.4155,Fan & Song 2022\nChevrolet Captiva,3K10409,8.63,123,Gasoline,1.3248,Fan & Song 2022\nChevrolet Captiva,3K10409,3.97,123,Gasoline,0.6322,Fan & Song 2022\nChevrolet Captiva,3K10409,8.28,123,Gasoline,1.6349,Fan & Song 2022\nChevrolet Captiva,3K10409,92.02,123,Gasoline,8.5947,Fan & Song 2022\nChevrolet Captiva,3K10409,41.63,123,Gasoline,5.5083,Fan & Song 2022\nChevrolet Captiva,3K10409,106.27,123,Gasoline,11.0893,Fan & Song 2022\nChevrolet Captiva,3K10409,44.26,123,Gasoline,4.2621,Fan & Song 2022\nChevrolet Captiva,3K10409,307.53,123,Gasoline,29.8627,Fan & Song 2022\nChevrolet Captiva,3K10409,10.01,123,Gasoline,1.1945,Fan & Song 2022\nChevrolet Captiva,3K10409,17.53,123,Gasoline,2.0917,Fan & Song 2022\nChevrolet Captiva,3K10409,5.31,123,Gasoline,0.8781,Fan & Song 2022\nChevrolet Captiva,3K10409,3.63,123,Gasoline,0.5363,Fan & Song 2022\nChevrolet Captiva,3K10409,37.98,123,Gasoline,4.7822,Fan & Song 2022\nChevrolet Captiva,3K10409,1.73,123,Gasoline,0.2461,Fan & Song 2022\nChevrolet Captiva,3K10409,16.62,123,Gasoline,1.6276,Fan & Song 2022\nChevrolet Captiva,3K10409,10.18,123,Gasoline,1.2062,Fan & Song 2022\nChevrolet Captiva,3K10409,184.43,123,Gasoline,19.2917,Fan & Song 2022\nChevrolet Captiva,3K10409,137.65,123,Gasoline,13.9736,Fan & Song 2022\nChevrolet Captiva,3K10409,34.95,123,Gasoline,4.5049,Fan & Song 2022\nChevrolet Captiva,3K10409,4.86,123,Gasoline,1.1859,Fan & Song 2022\nChevrolet Captiva,3K10409,40.94,123,Gasoline,4.5576,Fan & Song 2022\nChevrolet Captiva,3K10409,32.57,123,Gasoline,3.8486,Fan & Song 2022\nChevrolet Captiva,3K10409,134.44,123,Gasoline,14.533,Fan & Song 2022\nChevrolet Captiva,3K10409,107.3,123,Gasoline,9.347,Fan & Song 2022\nChevrolet Captiva,3K10409,4.38,123,Gasoline,0.7762,Fan & Song 2022\nChevrolet Captiva,3K10409,3.96,123,Gasoline,0.7622,Fan & Song 2022\nChevrolet Captiva,3K10409,2.59,123,Gasoline,0.481,Fan & Song 2022\nChevrolet Captiva,3K10409,16.68,123,Gasoline,1.5738,Fan & Song 2022\nChevrolet Captiva,3K10409,133.48,123,Gasoline,13.8852,Fan & Song 2022\nChevrolet Captiva,3K10409,24.03,123,Gasoline,4.5013,Fan & Song 2022\nChevrolet Captiva,3K10409,1.55,123,Gasoline,0.2401,Fan & Song 2022\nChevrolet Captiva,3K10409,31.99,123,Gasoline,4.228,Fan & Song 2022\nChevrolet Captiva,3K10409,31.12,123,Gasoline,2.995,Fan & Song 2022\nChevrolet Captiva,3K10409,54.76,123,Gasoline,7.5433,Fan & Song 2022\nChevrolet Captiva,3K10409,2.01,123,Gasoline,0.3857,Fan & Song 2022\nChevrolet Captiva,3K10409,22.14,123,Gasoline,2.6453,Fan & Song 2022\nChevrolet Captiva,3K10409,9.53,123,Gasoline,1.4995,Fan & Song 2022\nChevrolet Captiva,3K10409,28.78,123,Gasoline,3.6104,Fan & Song 2022\nChevrolet Captiva,3K10409,71.39,123,Gasoline,7.813,Fan & Song 2022\nChevrolet Captiva,3K10409,71.87,123,Gasoline,7.2432,Fan & Song 2022\nChevrolet Captiva,3K10409,33.98,123,Gasoline,4.3505,Fan & Song 2022\nChevrolet Captiva,3K10409,137.58,123,Gasoline,11.953,Fan & Song 2022\nChevrolet Captiva,3K10409,1.88,123,Gasoline,0.237,Fan & Song 2022\nChevrolet Captiva,3K10409,3.18,123,Gasoline,0.4979,Fan & Song 2022\nChevrolet Captiva,3K10409,8.63,123,Gasoline,1.1363,Fan & Song 2022\nChevrolet Captiva,3K10409,20.16,123,Gasoline,2.7056,Fan & Song 2022\nChevrolet Captiva,3K10409,76.1,123,Gasoline,7.0459,Fan & Song 2022\nChevrolet Captiva,3K10409,125.73,123,Gasoline,10.3489,Fan & Song 2022\nChevrolet Captiva,3K10409,234.0,123,Gasoline,27.0745,Fan & Song 2022\nChevrolet Captiva,3K10409,275.73,123,Gasoline,30.3717,Fan & Song 2022\nChevrolet Captiva,3K10409,16.01,123,Gasoline,2.091,Fan & Song 2022\nChevrolet Captiva,3K10409,21.98,123,Gasoline,2.2402,Fan & Song 2022\nChevrolet Captiva,3K10409,13.47,123,Gasoline,2.1084,Fan & Song 2022\nChevrolet Captiva,3K10409,41.98,123,Gasoline,4.0878,Fan & Song 2022\nChevrolet Captiva,3K10409,109.84,123,Gasoline,12.1433,Fan & Song 2022\nChevrolet Captiva,3K10409,51.53,123,Gasoline,5.2492,Fan & Song 2022\nChevrolet Captiva,3K10409,110.12,123,Gasoline,10.9617,Fan & Song 2022\nChevrolet Captiva,3K10409,110.48,123,Gasoline,11.1039,Fan & Song 2022\nChevrolet Captiva,3K10409,28.19,123,Gasoline,3.5948,Fan & Song 2022\nChevrolet Captiva,3K10409,30.5,123,Gasoline,3.3523,Fan & Song 2022\nChevrolet Captiva,3K10409,27.89,123,Gasoline,2.869,Fan & Song 2022\nChevrolet Captiva,3K10409,10.9,123,Gasoline,1.5615,Fan & Song 2022\nChevrolet Captiva,3K10409,141.96,123,Gasoline,12.5704,Fan & Song 2022\nChevrolet Captiva,3K10409,3.32,123,Gasoline,0.7104,Fan & Song 2022\nChevrolet Captiva,3K10409,321.59,123,Gasoline,30.2134,Fan & Song 2022\nChevrolet Captiva,3K10409,38.08,123,Gasoline,4.1624,Fan & Song 2022\nChevrolet Captiva,3K10409,12.37,123,Gasoline,1.7583,Fan & Song 2022\nChevrolet Captiva,3K10409,17.62,123,Gasoline,1.7875,Fan & Song 2022\nChevrolet Captiva,3K10409,11.82,123,Gasoline,1.6177,Fan & Song 2022\nChevrolet Captiva,3K10409,88.54,123,Gasoline,9.6343,Fan & Song 2022\nChevrolet Captiva,3K10409,169.9,123,Gasoline,17.2352,Fan & Song 2022\nChevrolet Captiva,3K10409,98.36,123,Gasoline,8.219,Fan & Song 2022\nChevrolet Captiva,3K10409,13.9,123,Gasoline,1.5187,Fan & Song 2022\nChevrolet Captiva,3K10409,194.86,123,Gasoline,19.47,Fan & Song 2022\nChevrolet Captiva,3K10409,2.17,123,Gasoline,0.5605,Fan & Song 2022\nChevrolet Captiva,3K10409,8.98,123,Gasoline,2.2205,Fan & Song 2022\nChevrolet Captiva,3K10409,46.22,123,Gasoline,4.6857,Fan & Song 2022\nChevrolet Captiva,3K10409,123.16,123,Gasoline,17.3892,Fan & Song 2022\nChevrolet Captiva,3K10409,49.03,123,Gasoline,5.8532,Fan & Song 2022\nChevrolet Captiva,3K10409,7.37,123,Gasoline,1.271,Fan & Song 2022\nChevrolet Captiva,3K10409,1.96,123,Gasoline,0.3698,Fan & Song 2022\nChevrolet Captiva,3K10409,292.47,123,Gasoline,29.2458,Fan & Song 2022\nChevrolet Captiva,3K10409,7.18,123,Gasoline,1.2548,Fan & Song 2022\nChevrolet Captiva,3K10409,8.64,123,Gasoline,1.1297,Fan & Song 2022\nChevrolet Captiva,3K10409,49.96,123,Gasoline,4.7475,Fan & Song 2022\nChevrolet Captiva,3K10409,93.5,123,Gasoline,10.1949,Fan & Song 2022\nChevrolet Captiva,3K10409,233.39,123,Gasoline,24.5563,Fan & Song 2022\nChevrolet Captiva,3K10409,339.31,123,Gasoline,31.3005,Fan & Song 2022\nChevrolet Captiva,3K10409,174.99,123,Gasoline,14.8765,Fan & Song 2022\nChevrolet Captiva,3K10409,16.92,123,Gasoline,1.6119,Fan & Song 2022\nChevrolet Captiva,3K10409,37.35,123,Gasoline,3.7755,Fan & Song 2022\nChevrolet Captiva,3K10409,129.5,123,Gasoline,12.6752,Fan & Song 2022\nChevrolet Captiva,3K10409,119.77,123,Gasoline,7.8402,Fan & Song 2022\nChevrolet Captiva,3K10409,463.38,123,Gasoline,51.1415,Fan & Song 2022\nChevrolet Captiva,3K10409,212.44,123,Gasoline,23.2294,Fan & Song 2022\nChevrolet Captiva,3K10409,2.87,123,Gasoline,0.5081,Fan & Song 2022\nChevrolet Captiva,3K10409,5.59,123,Gasoline,1.1001,Fan & Song 2022\nChevrolet Captiva,3K10409,285.68,123,Gasoline,28.5645,Fan & Song 2022\nChevrolet Captiva,3K10409,152.98,123,Gasoline,17.2097,Fan & Song 2022\nChevrolet Captiva,3K10409,221.09,123,Gasoline,23.2508,Fan & Song 2022\nChevrolet Captiva,3K10409,62.24,123,Gasoline,5.6351,Fan & Song 2022\nChevrolet Captiva,3K10409,6.66,123,Gasoline,0.8527,Fan & Song 2022\nChevrolet Captiva,3K10409,261.58,123,Gasoline,25.6718,Fan & Song 2022\nChevrolet Captiva,3K10409,9.77,123,Gasoline,1.3257,Fan & Song 2022\nChevrolet Captiva,3K10409,35.73,123,Gasoline,4.4792,Fan & Song 2022\nChevrolet Captiva,3K10409,3.5,123,Gasoline,0.4161,Fan & Song 2022\nChevrolet Captiva,3K10409,3.09,123,Gasoline,0.5401,Fan & Song 2022\nChevrolet Captiva,3K10409,11.95,123,Gasoline,1.6182,Fan & Song 2022\nChevrolet Captiva,3K10409,160.59,123,Gasoline,15.3709,Fan & Song 2022\nChevrolet Captiva,3K10409,161.51,123,Gasoline,14.6643,Fan & Song 2022\nChevrolet Captiva,3K10409,235.36,123,Gasoline,21.1088,Fan & Song 2022\nChevrolet Captiva,3K10409,12.09,123,Gasoline,1.4399,Fan & Song 2022\nChevrolet Captiva,3K10409,181.86,123,Gasoline,16.6039,Fan & Song 2022\nChevrolet Captiva,3K10409,33.98,123,Gasoline,3.3331,Fan & Song 2022\nChevrolet Captiva,3K10409,297.71,123,Gasoline,29.5297,Fan & Song 2022\nChevrolet Captiva,3K10409,19.43,123,Gasoline,1.8177,Fan & Song 2022\nChevrolet Captiva,3K10409,4.45,123,Gasoline,0.8712,Fan & Song 2022\nChevrolet Captiva,3K10409,4.69,123,Gasoline,0.7463,Fan & Song 2022\nChevrolet Captiva,3K10409,10.65,123,Gasoline,1.7805,Fan & Song 2022\nChevrolet Captiva,3K10409,138.31,123,Gasoline,13.6666,Fan & Song 2022\nChevrolet Captiva,3K10409,7.07,123,Gasoline,1.2845,Fan & Song 2022\nChevrolet Captiva,3K10409,185.73,123,Gasoline,19.9252,Fan & Song 2022\nChevrolet Captiva,3K10409,170.82,123,Gasoline,16.2215,Fan & Song 2022\nChevrolet Captiva,3K10409,23.03,123,Gasoline,3.268,Fan & Song 2022\nChevrolet Captiva,3K10409,152.86,123,Gasoline,13.1207,Fan & Song 2022\nChevrolet Captiva,3K10409,455.13,123,Gasoline,46.5934,Fan & Song 2022\nChevrolet Captiva,3K10409,186.34,123,Gasoline,19.9241,Fan & Song 2022\nChevrolet Captiva,3K10409,109.77,123,Gasoline,7.7332,Fan & Song 2022\nChevrolet Captiva,3K10409,169.88,123,Gasoline,16.6615,Fan & Song 2022\nChevrolet Captiva,3K10409,6.64,123,Gasoline,0.8229,Fan & Song 2022\nChevrolet Captiva,3K10409,7.37,123,Gasoline,1.8836,Fan & Song 2022\nChevrolet Captiva,3K10409,183.3,123,Gasoline,16.8483,Fan & Song 2022\nChevrolet Captiva,3K10409,38.16,123,Gasoline,3.5836,Fan & Song 2022\nChevrolet Captiva,3K10409,2.74,123,Gasoline,0.4619,Fan & Song 2022\nChevrolet Captiva,3K10409,36.32,123,Gasoline,3.901,Fan & Song 2022\nChevrolet Captiva,3K10409,192.71,123,Gasoline,19.5495,Fan & Song 2022\nChevrolet Captiva,3K10409,87.88,123,Gasoline,7.7645,Fan & Song 2022\nChevrolet Captiva,3K10409,17.04,123,Gasoline,2.3503,Fan & Song 2022\nChevrolet Captiva,3K10409,12.6,123,Gasoline,2.5835,Fan & Song 2022\nChevrolet Captiva,3K10409,338.34,123,Gasoline,33.707,Fan & Song 2022\nChevrolet Captiva,3K10409,141.61,123,Gasoline,14.9211,Fan & Song 2022\nChevrolet Captiva,3K10409,55.28,123,Gasoline,4.8175,Fan & Song 2022\nChevrolet Captiva,3K10409,21.13,123,Gasoline,2.8594,Fan & Song 2022\nChevrolet Captiva,3K10409,2.8,123,Gasoline,0.3876,Fan & Song 2022\nChevrolet Captiva,3K10409,272.16,123,Gasoline,22.0122,Fan & Song 2022\nChevrolet Captiva,3K10409,144.45,123,Gasoline,13.8112,Fan & Song 2022\nChevrolet Captiva,3K10409,8.97,123,Gasoline,1.2955,Fan & Song 2022\nChevrolet Captiva,3K10409,14.48,123,Gasoline,1.7928,Fan & Song 2022\nChevrolet Captiva,3K10409,218.17,123,Gasoline,17.9459,Fan & Song 2022\nChevrolet Captiva,3K10409,157.13,123,Gasoline,14.7473,Fan & Song 2022\nChevrolet Captiva,3K10409,32.47,123,Gasoline,3.0446,Fan & Song 2022\nChevrolet Captiva,3K10409,222.9,123,Gasoline,22.1603,Fan & Song 2022\nChevrolet Captiva,3K10409,57.68,123,Gasoline,6.7213,Fan & Song 2022\nChevrolet Captiva,3K10409,115.58,123,Gasoline,12.2053,Fan & Song 2022\nChevrolet Captiva,3K10409,49.25,123,Gasoline,6.3575,Fan & Song 2022\nChevrolet Captiva,3K10409,13.92,123,Gasoline,2.4593,Fan & Song 2022\nChevrolet Captiva,3K10409,218.9,123,Gasoline,20.9377,Fan & Song 2022\nChevrolet Captiva,3K10409,16.76,123,Gasoline,1.9056,Fan & Song 2022\nChevrolet Captiva,3K10409,28.64,123,Gasoline,3.0331,Fan & Song 2022\nChevrolet Captiva,3K10409,16.63,123,Gasoline,1.6903,Fan & Song 2022\nChevrolet Captiva,3K10409,35.73,123,Gasoline,3.2118,Fan & Song 2022\nChevrolet Captiva,3K10409,3.78,123,Gasoline,0.7398,Fan & Song 2022\nChevrolet Captiva,3K10409,56.6,123,Gasoline,5.0277,Fan & Song 2022\nChevrolet Captiva,3K10409,118.54,123,Gasoline,10.9399,Fan & Song 2022\nChevrolet Captiva,3K10409,22.52,123,Gasoline,2.1595,Fan & Song 2022\nChevrolet Captiva,3K10409,16.3,123,Gasoline,2.2802,Fan & Song 2022\nChevrolet Captiva,3K10409,24.81,123,Gasoline,3.0786,Fan & Song 2022\nChevrolet Captiva,3K10409,139.84,123,Gasoline,12.8727,Fan & Song 2022\nChevrolet Captiva,3K10409,102.02,123,Gasoline,10.0131,Fan & Song 2022\nChevrolet Captiva,3K10409,36.33,123,Gasoline,4.1741,Fan & Song 2022\nChevrolet Captiva,3K10409,3.46,123,Gasoline,0.6287,Fan & Song 2022\nChevrolet Captiva,3K10409,118.69,123,Gasoline,11.8868,Fan & Song 2022\nChevrolet Captiva,3K10409,20.75,123,Gasoline,2.8162,Fan & Song 2022\nChevrolet Captiva,3K10409,5.63,123,Gasoline,0.9942,Fan & Song 2022\nChevrolet Captiva,3K10409,16.42,123,Gasoline,2.2645,Fan & Song 2022\nChevrolet Captiva,3K10409,5.99,123,Gasoline,1.0443,Fan & Song 2022\nChevrolet Captiva,3K10409,31.33,123,Gasoline,3.0098,Fan & Song 2022\nChevrolet Captiva,3K10409,112.39,123,Gasoline,9.6625,Fan & Song 2022\nChevrolet Captiva,3K10409,32.43,123,Gasoline,3.3293,Fan & Song 2022\nChevrolet Captiva,3K10409,6.85,123,Gasoline,0.9282,Fan & Song 2022\nChevrolet Captiva,3K10409,17.01,123,Gasoline,2.7761,Fan & Song 2022\nChevrolet Captiva,3K10409,6.38,123,Gasoline,1.3019,Fan & Song 2022\nChevrolet Captiva,3K10409,12.4,123,Gasoline,1.1881,Fan & Song 2022\nChevrolet Captiva,3K10409,15.62,123,Gasoline,2.2863,Fan & Song 2022\nChevrolet Captiva,3K10409,7.86,123,Gasoline,1.6191,Fan & Song 2022\nChevrolet Captiva,3K10409,13.83,123,Gasoline,1.7634,Fan & Song 2022\nChevrolet Captiva,3K10409,17.07,123,Gasoline,1.6164,Fan & Song 2022\nChevrolet Captiva,3K10409,16.65,123,Gasoline,1.6381,Fan & Song 2022\nChevrolet Captiva,3K10409,54.46,123,Gasoline,5.2719,Fan & Song 2022\nChevrolet Captiva,3K10409,13.88,123,Gasoline,1.5245,Fan & Song 2022\nChevrolet Captiva,3K10409,5.05,123,Gasoline,0.8915,Fan & Song 2022\nChevrolet Captiva,3K10409,19.92,123,Gasoline,2.2082,Fan & Song 2022\nChevrolet Captiva,3K10409,57.07,123,Gasoline,7.0464,Fan & Song 2022\nChevrolet Captiva,3K10409,32.43,123,Gasoline,3.543,Fan & Song 2022\nChevrolet Captiva,3K10409,216.51,123,Gasoline,19.5839,Fan & Song 2022\nChevrolet Captiva,3K10409,169.83,123,Gasoline,13.3725,Fan & Song 2022\nChevrolet Captiva,3K10409,114.85,123,Gasoline,10.6997,Fan & Song 2022\nChevrolet Captiva,3K10409,178.79,123,Gasoline,16.3646,Fan & Song 2022\nChevrolet Captiva,3K10409,105.37,123,Gasoline,8.557,Fan & Song 2022\nChevrolet Captiva,3K10409,6.27,123,Gasoline,0.7092,Fan & Song 2022\nChevrolet Captiva,3K10409,215.26,123,Gasoline,21.8513,Fan & Song 2022\nChevrolet Captiva,3K10409,34.95,123,Gasoline,3.374,Fan & Song 2022\nChevrolet Captiva,3K10409,230.22,123,Gasoline,21.3172,Fan & Song 2022\nChevrolet Captiva,3K10409,160.87,123,Gasoline,16.5861,Fan & Song 2022\nChevrolet Captiva,3K10409,30.59,123,Gasoline,3.1082,Fan & Song 2022\nChevrolet Captiva,3K10409,26.28,123,Gasoline,2.7737,Fan & Song 2022\nChevrolet Captiva,3K10409,121.48,123,Gasoline,10.8839,Fan & Song 2022\nChevrolet Captiva,3K10409,127.31,123,Gasoline,12.6998,Fan & Song 2022\nChevrolet Captiva,3K10409,67.42,123,Gasoline,6.8896,Fan & Song 2022\nChevrolet Captiva,3K10409,25.98,123,Gasoline,2.8061,Fan & Song 2022\nChevrolet Captiva,3K10409,19.0,123,Gasoline,2.2635,Fan & Song 2022\nChevrolet Captiva,3K10409,16.96,123,Gasoline,3.1899,Fan & Song 2022\nChevrolet Captiva,3K10409,19.38,123,Gasoline,2.14,Fan & Song 2022\nChevrolet Captiva,3K10409,7.58,123,Gasoline,1.2277,Fan & Song 2022\nChevrolet Captiva,3K10409,2.39,123,Gasoline,0.4522,Fan & Song 2022\nChevrolet Captiva,3K10409,33.25,123,Gasoline,3.6736,Fan & Song 2022\nChevrolet Captiva,3K10409,4.03,123,Gasoline,0.5751,Fan & Song 2022\nChevrolet Captiva,3K10409,38.13,123,Gasoline,3.9727,Fan & Song 2022\nChevrolet Captiva,3K10409,15.67,123,Gasoline,1.6298,Fan & Song 2022\nChevrolet Captiva,3K10409,72.07,123,Gasoline,6.475,Fan & Song 2022\nChevrolet Captiva,3K10409,111.16,123,Gasoline,10.7814,Fan & Song 2022\nChevrolet Captiva,3K10409,4.32,123,Gasoline,0.7715,Fan & Song 2022\nChevrolet Captiva,3K10409,8.14,123,Gasoline,1.0884,Fan & Song 2022\nChevrolet Captiva,3K10409,4.36,123,Gasoline,0.8559,Fan & Song 2022\nChevrolet Captiva,3K10409,19.66,123,Gasoline,3.0743,Fan & Song 2022\nChevrolet Captiva,3K10409,33.23,123,Gasoline,3.8917,Fan & Song 2022\nChevrolet Captiva,3K10409,31.37,123,Gasoline,3.6909,Fan & Song 2022\nChevrolet Captiva,3K10409,3.89,123,Gasoline,0.7854,Fan & Song 2022\nChevrolet Captiva,3K10409,3.23,123,Gasoline,0.5032,Fan & Song 2022\nChevrolet Captiva,3K10409,4.44,123,Gasoline,0.6319,Fan & Song 2022\nChevrolet Captiva,3K10409,7.25,123,Gasoline,1.1598,Fan & Song 2022\nChevrolet Captiva,3K10409,50.36,123,Gasoline,5.6192,Fan & Song 2022\nChevrolet Captiva,3K10409,38.4,123,Gasoline,3.8285,Fan & Song 2022\nChevrolet Captiva,3K10409,145.91,123,Gasoline,15.635,Fan & Song 2022\nChevrolet Captiva,3K10409,47.68,123,Gasoline,4.7211,Fan & Song 2022\nChevrolet Captiva,3K10409,56.3,123,Gasoline,6.084,Fan & Song 2022\nChevrolet Captiva,3K10409,13.35,123,Gasoline,1.6788,Fan & Song 2022\nChevrolet Captiva,3K10409,35.89,123,Gasoline,3.6542,Fan & Song 2022\nChevrolet Captiva,3K10409,12.54,123,Gasoline,1.7574,Fan & Song 2022\nChevrolet Captiva,3K10409,128.87,123,Gasoline,11.4293,Fan & Song 2022\nChevrolet Captiva,3K10409,6.99,123,Gasoline,0.8932,Fan & Song 2022\nChevrolet Captiva,3K10409,54.1,123,Gasoline,5.2317,Fan & Song 2022\nChevrolet Captiva,3K10409,42.87,123,Gasoline,4.337,Fan & Song 2022\nChevrolet Captiva,3K10409,21.16,123,Gasoline,2.5112,Fan & Song 2022\nChevrolet Captiva,3K10409,64.05,123,Gasoline,6.1953,Fan & Song 2022\nChevrolet Captiva,3K10409,134.14,123,Gasoline,12.2511,Fan & Song 2022\nChevrolet Captiva,3K10409,11.72,123,Gasoline,1.6162,Fan & Song 2022\nChevrolet Captiva,3K10409,50.2,123,Gasoline,5.2302,Fan & Song 2022\nChevrolet Captiva,3K10409,21.66,123,Gasoline,2.5479,Fan & Song 2022\nChevrolet Captiva,3K10409,4.99,123,Gasoline,0.8889,Fan & Song 2022\nChevrolet Captiva,3K10409,9.28,123,Gasoline,1.4302,Fan & Song 2022\nChevrolet Captiva,3K10409,39.16,123,Gasoline,3.99,Fan & Song 2022\nChevrolet Captiva,3K10409,30.16,123,Gasoline,3.4296,Fan & Song 2022\nChevrolet Captiva,3K10409,11.19,123,Gasoline,1.47,Fan & Song 2022\nChevrolet Captiva,3K10409,12.6,123,Gasoline,1.6967,Fan & Song 2022\nChevrolet Captiva,3K10409,130.24,123,Gasoline,12.3875,Fan & Song 2022\nChevrolet Captiva,3K10409,6.81,123,Gasoline,1.1083,Fan & Song 2022\nChevrolet Captiva,3K10409,3.82,123,Gasoline,0.7615,Fan & Song 2022\nChevrolet Captiva,3K10409,6.56,123,Gasoline,1.3192,Fan & Song 2022\nChevrolet Captiva,3K10409,5.11,123,Gasoline,0.5439,Fan & Song 2022\nChevrolet Captiva,3K10409,10.76,123,Gasoline,1.4884,Fan & Song 2022\nChevrolet Captiva,3K10409,43.14,123,Gasoline,4.2457,Fan & Song 2022\nChevrolet Captiva,3K10409,16.41,123,Gasoline,1.8388,Fan & Song 2022\nChevrolet Captiva,3K10409,2.56,123,Gasoline,0.4106,Fan & Song 2022\nChevrolet Captiva,3K10409,1.56,123,Gasoline,0.2822,Fan & Song 2022\nChevrolet Captiva,3K10409,5.64,123,Gasoline,0.8366,Fan & Song 2022\nChevrolet Captiva,3K10409,1.95,123,Gasoline,0.2991,Fan & Song 2022\nChevrolet Captiva,3K10409,101.6,123,Gasoline,12.2581,Fan & Song 2022\nChevrolet Captiva,3K10409,13.64,123,Gasoline,2.1072,Fan & Song 2022\nChevrolet Captiva,3K10409,4.9,123,Gasoline,0.7515,Fan & Song 2022\nChevrolet Captiva,3K10409,2.88,123,Gasoline,0.5631,Fan & Song 2022\nChevrolet Captiva,3K10409,10.59,123,Gasoline,1.2224,Fan & Song 2022\nChevrolet Captiva,3K10409,37.95,123,Gasoline,3.939,Fan & Song 2022\nChevrolet Captiva,3K10409,30.35,123,Gasoline,2.574,Fan & Song 2022\nChevrolet Captiva,3K10409,31.07,123,Gasoline,3.2175,Fan & Song 2022\nChevrolet Captiva,3K10409,4.87,123,Gasoline,0.7535,Fan & Song 2022\nChevrolet Captiva,3K10409,2.3,123,Gasoline,0.408,Fan & Song 2022\nChevrolet Captiva,3K10409,65.81,123,Gasoline,6.9282,Fan & Song 2022\nChevrolet Captiva,3K10409,15.14,123,Gasoline,1.9166,Fan & Song 2022\nChevrolet Captiva,3K10409,167.24,123,Gasoline,15.1527,Fan & Song 2022\nChevrolet Captiva,3K10409,10.19,123,Gasoline,1.3562,Fan & Song 2022\nChevrolet Captiva,3K10409,129.28,123,Gasoline,12.6258,Fan & Song 2022\nChevrolet Captiva,3K10409,32.22,123,Gasoline,3.9483,Fan & Song 2022\nChevrolet Captiva,3K10409,33.52,123,Gasoline,3.4308,Fan & Song 2022\nChevrolet Captiva,3K10409,11.18,123,Gasoline,1.9166,Fan & Song 2022\nChevrolet Captiva,3K10409,9.78,123,Gasoline,1.3483,Fan & Song 2022\nChevrolet Captiva,3K10409,26.49,123,Gasoline,3.1707,Fan & Song 2022\nChevrolet Captiva,3K10409,3.48,123,Gasoline,0.6355,Fan & Song 2022\nChevrolet Captiva,3K10409,67.91,123,Gasoline,7.6422,Fan & Song 2022\nChevrolet Captiva,3K10409,11.44,123,Gasoline,1.4032,Fan & Song 2022\nChevrolet Captiva,3K10409,10.02,123,Gasoline,1.1061,Fan & Song 2022\nChevrolet Captiva,3K10409,17.67,123,Gasoline,2.2878,Fan & Song 2022\nChevrolet Captiva,3K10409,31.54,123,Gasoline,3.6134,Fan & Song 2022\nChevrolet Captiva,3K10409,3.11,123,Gasoline,0.8215,Fan & Song 2022\nChevrolet Captiva,3K10409,5.52,123,Gasoline,0.8069,Fan & Song 2022\nChevrolet Captiva,3K10409,12.27,123,Gasoline,1.858,Fan & Song 2022\nChevrolet Captiva,3K10409,20.24,123,Gasoline,3.6563,Fan & Song 2022\nHonda Accord,0H3082Y,126.69,114,Gasoline,10.7472,Fan & Song 2022\nHonda Accord,0H3082Y,103.0,114,Gasoline,7.3366,Fan & Song 2022\nHonda Accord,0H3082Y,8.05,114,Gasoline,0.8176,Fan & Song 2022\nHonda Accord,0H3082Y,123.5,114,Gasoline,11.893,Fan & Song 2022\nHonda Accord,0H3082Y,6.32,114,Gasoline,0.4745,Fan & Song 2022\nHonda Accord,0H3082Y,26.17,114,Gasoline,1.8839,Fan & Song 2022\nHonda Accord,0H3082Y,121.41,114,Gasoline,10.2315,Fan & Song 2022\nHonda Accord,0H3082Y,47.06,114,Gasoline,3.19,Fan & Song 2022\nHonda Accord,0H3082Y,7.51,114,Gasoline,0.5543,Fan & Song 2022\nHonda Accord,0H3082Y,4.64,114,Gasoline,0.5153,Fan & Song 2022\nHonda Accord,0H3082Y,2.52,114,Gasoline,0.3655,Fan & Song 2022\nHonda Accord,0H3082Y,36.14,114,Gasoline,2.7182,Fan & Song 2022\nHonda Accord,0H3082Y,43.76,114,Gasoline,3.7482,Fan & Song 2022\nHonda Accord,0H3082Y,155.2,114,Gasoline,12.144,Fan & Song 2022\nHonda Accord,0H3082Y,206.62,114,Gasoline,18.2041,Fan & Song 2022\nHonda Accord,0H3082Y,46.0,114,Gasoline,3.4008,Fan & Song 2022\nHonda Accord,0H3082Y,15.03,114,Gasoline,1.1318,Fan & Song 2022\nHonda Accord,0H3082Y,303.37,114,Gasoline,19.6292,Fan & Song 2022\nHonda Accord,0H3082Y,19.74,114,Gasoline,1.4406,Fan & Song 2022\nHonda Accord,0H3082Y,8.79,114,Gasoline,0.9344,Fan & Song 2022\nHonda Accord,0H3082Y,2.51,114,Gasoline,0.2911,Fan & Song 2022\nHonda Accord,0H3082Y,9.18,114,Gasoline,0.6972,Fan & Song 2022\nHonda Accord,0H3082Y,19.29,114,Gasoline,1.2391,Fan & Song 2022\nHonda Accord,0H3082Y,32.49,114,Gasoline,2.0679,Fan & Song 2022\nHonda Accord,0H3082Y,42.86,114,Gasoline,3.539,Fan & Song 2022\nHonda Accord,0H3082Y,2.09,114,Gasoline,0.3131,Fan & Song 2022\nHonda Accord,0H3082Y,10.62,114,Gasoline,0.7201,Fan & Song 2022\nHonda Accord,0H3082Y,5.51,114,Gasoline,0.4605,Fan & Song 2022\nHonda Accord,0H3082Y,121.76,114,Gasoline,10.176,Fan & Song 2022\nHonda Accord,0H3082Y,16.23,114,Gasoline,1.254,Fan & Song 2022\nHonda Accord,0H3082Y,161.61,114,Gasoline,10.1611,Fan & Song 2022\nHonda Accord,0H3082Y,12.13,114,Gasoline,0.9957,Fan & Song 2022\nHonda Accord,0H3082Y,326.36,114,Gasoline,21.2522,Fan & Song 2022\nHonda Accord,0H3082Y,221.74,114,Gasoline,19.2741,Fan & Song 2022\nHonda Accord,0H3082Y,189.59,114,Gasoline,15.2126,Fan & Song 2022\nHonda Accord,0H3082Y,24.77,114,Gasoline,2.6784,Fan & Song 2022\nHonda Accord,0H3082Y,10.44,114,Gasoline,0.9984,Fan & Song 2022\nHonda Accord,0H3082Y,5.96,114,Gasoline,0.6314,Fan & Song 2022\nHonda Accord,0H3082Y,21.41,114,Gasoline,2.2923,Fan & Song 2022\nHonda Accord,0H3082Y,16.22,114,Gasoline,1.8801,Fan & Song 2022\nHonda Accord,0H3082Y,1.95,114,Gasoline,0.3941,Fan & Song 2022\nHonda Accord,0H3082Y,4.89,114,Gasoline,0.3985,Fan & Song 2022\nHonda Accord,0H3082Y,3.04,114,Gasoline,0.2849,Fan & Song 2022\nHonda Accord,0H3082Y,29.07,114,Gasoline,3.1345,Fan & Song 2022\nHonda Accord,0H3082Y,2.53,114,Gasoline,0.304,Fan & Song 2022\nHonda Accord,0H3082Y,17.91,114,Gasoline,1.4795,Fan & Song 2022\nHonda Accord,0H3082Y,165.16,114,Gasoline,9.929,Fan & Song 2022\nHonda Accord,0H3082Y,165.28,114,Gasoline,10.6462,Fan & Song 2022\nHonda Accord,0H3082Y,26.03,114,Gasoline,1.8605,Fan & Song 2022\nHonda Accord,0H3082Y,131.18,114,Gasoline,9.6382,Fan & Song 2022\nHonda Accord,0H3082Y,351.24,114,Gasoline,22.634,Fan & Song 2022\nHonda Accord,0H3082Y,200.89,114,Gasoline,15.2151,Fan & Song 2022\nHonda Accord,0H3082Y,431.1,114,Gasoline,28.6061,Fan & Song 2022\nHonda Accord,0H3082Y,1.98,114,Gasoline,0.2788,Fan & Song 2022\nHonda Accord,0H3082Y,2.76,114,Gasoline,0.1561,Fan & Song 2022\nHonda Accord,0H3082Y,64.44,114,Gasoline,4.8158,Fan & Song 2022\nHonda Accord,0H3082Y,309.46,114,Gasoline,17.0907,Fan & Song 2022\nHonda Accord,0H3082Y,44.46,114,Gasoline,3.2716,Fan & Song 2022\nHonda Accord,0H3082Y,28.81,114,Gasoline,2.0569,Fan & Song 2022\nHonda Accord,0H3082Y,173.42,114,Gasoline,9.9978,Fan & Song 2022\nHonda Accord,0H3082Y,277.26,114,Gasoline,16.542,Fan & Song 2022\nHonda Accord,0H3082Y,2.24,114,Gasoline,0.2378,Fan & Song 2022\nHonda Accord,0H3082Y,13.15,114,Gasoline,1.5444,Fan & Song 2022\nHonda Accord,0H3082Y,34.62,114,Gasoline,2.286,Fan & Song 2022\nHonda Accord,0H3082Y,6.24,114,Gasoline,0.8199,Fan & Song 2022\nHonda Accord,0H3082Y,111.73,114,Gasoline,7.0149,Fan & Song 2022\nHonda Accord,0H3082Y,32.82,114,Gasoline,2.1394,Fan & Song 2022\nHonda Accord,0H3082Y,43.34,114,Gasoline,3.6892,Fan & Song 2022\nHonda Accord,0H3082Y,151.89,114,Gasoline,12.2208,Fan & Song 2022\nHonda Accord,0H3082Y,35.77,114,Gasoline,2.0686,Fan & Song 2022\nHonda Accord,0H3082Y,100.75,114,Gasoline,7.7988,Fan & Song 2022\nHonda Accord,0H3082Y,296.18,114,Gasoline,18.112,Fan & Song 2022\nHonda Accord,0H3082Y,172.94,114,Gasoline,9.7347,Fan & Song 2022\nHonda Accord,0H3082Y,5.74,114,Gasoline,0.6741,Fan & Song 2022\nHonda Accord,0H3082Y,147.84,114,Gasoline,12.6136,Fan & Song 2022\nHonda Accord,0H3082Y,2.63,114,Gasoline,0.3143,Fan & Song 2022\nHonda Accord,0H3082Y,4.04,114,Gasoline,0.3392,Fan & Song 2022\nHonda Accord,0H3082Y,3.55,114,Gasoline,0.3503,Fan & Song 2022\nHonda Accord,0H3082Y,54.26,114,Gasoline,3.5793,Fan & Song 2022\nHonda Accord,0H3082Y,3.51,114,Gasoline,0.4918,Fan & Song 2022\nHonda Accord,0H3082Y,156.43,114,Gasoline,11.1993,Fan & Song 2022\nHonda Accord,0H3082Y,9.12,114,Gasoline,1.0512,Fan & Song 2022\nHonda Accord,0H3082Y,31.47,114,Gasoline,2.1729,Fan & Song 2022\nHonda Accord,0H3082Y,16.3,114,Gasoline,1.2289,Fan & Song 2022\nHonda Accord,0H3082Y,3.95,114,Gasoline,0.5457,Fan & Song 2022\nHonda Accord,0H3082Y,6.35,114,Gasoline,0.4919,Fan & Song 2022\nHonda Accord,0H3082Y,130.09,114,Gasoline,5.3556,Fan & Song 2022\nHonda Accord,0H3082Y,11.96,114,Gasoline,0.8411,Fan & Song 2022\nHonda Accord,0H3082Y,2.55,114,Gasoline,0.3545,Fan & Song 2022\nHonda Accord,0H3082Y,114.62,114,Gasoline,11.567,Fan & Song 2022\nHonda Accord,0H3082Y,2.12,114,Gasoline,0.242,Fan & Song 2022\nHonda Accord,0H3082Y,10.53,114,Gasoline,0.7297,Fan & Song 2022\nHonda Accord,0H3082Y,206.32,114,Gasoline,11.9571,Fan & Song 2022\nHonda Accord,0H3082Y,8.07,114,Gasoline,0.5766,Fan & Song 2022\nHonda Accord,0H3082Y,13.12,114,Gasoline,1.3446,Fan & Song 2022\nHonda Accord,0H3082Y,131.05,114,Gasoline,10.1295,Fan & Song 2022\nHonda Accord,0H3082Y,12.83,114,Gasoline,1.0923,Fan & Song 2022\nHonda Accord,0H3082Y,35.73,114,Gasoline,2.3396,Fan & Song 2022\nHonda Accord,0H3082Y,405.89,114,Gasoline,25.3727,Fan & Song 2022\nHonda Accord,0H3082Y,31.6,114,Gasoline,1.8185,Fan & Song 2022\nHonda Accord,0H3082Y,61.68,114,Gasoline,4.0531,Fan & Song 2022\nHonda Accord,0H3082Y,104.55,114,Gasoline,9.2777,Fan & Song 2022\nHonda Accord,0H3082Y,5.31,114,Gasoline,0.5071,Fan & Song 2022\nHonda Accord,0H3082Y,1.76,114,Gasoline,0.1636,Fan & Song 2022\nHonda Accord,0H3082Y,6.9,114,Gasoline,0.735,Fan & Song 2022\nHonda Accord,0H3082Y,19.26,114,Gasoline,1.4308,Fan & Song 2022\nHonda Accord,0H3082Y,124.96,114,Gasoline,9.3866,Fan & Song 2022\nHonda Accord,0H3082Y,4.85,114,Gasoline,0.555,Fan & Song 2022\nHonda Accord,0H3082Y,56.1,114,Gasoline,5.2088,Fan & Song 2022\nHonda Accord,0H3082Y,2.46,114,Gasoline,0.239,Fan & Song 2022\nHonda Accord,0H3082Y,58.39,114,Gasoline,3.4792,Fan & Song 2022\nHonda Accord,0H3082Y,22.19,114,Gasoline,2.2659,Fan & Song 2022\nHonda Accord,0H3082Y,7.58,114,Gasoline,0.4292,Fan & Song 2022\nHonda Accord,0H3082Y,16.2,114,Gasoline,1.3962,Fan & Song 2022\nHonda Accord,0H3082Y,148.89,114,Gasoline,10.5443,Fan & Song 2022\nHonda Accord,0H3082Y,11.42,114,Gasoline,0.7972,Fan & Song 2022\nHonda Accord,0H3082Y,24.66,114,Gasoline,1.8828,Fan & Song 2022\nHonda Accord,0H3082Y,68.03,114,Gasoline,4.1892,Fan & Song 2022\nHonda Accord,0H3082Y,11.16,114,Gasoline,0.9793,Fan & Song 2022\nHonda Accord,0H3082Y,50.09,114,Gasoline,3.2956,Fan & Song 2022\nHonda Accord,0H3082Y,37.75,114,Gasoline,3.3712,Fan & Song 2022\nHonda Accord,0H3082Y,30.7,114,Gasoline,2.1641,Fan & Song 2022\nHonda Accord,0H3082Y,16.29,114,Gasoline,0.9961,Fan & Song 2022\nHonda Accord,0H3082Y,25.53,114,Gasoline,1.9849,Fan & Song 2022\nHonda Accord,0H3082Y,22.67,114,Gasoline,2.5155,Fan & Song 2022\nHonda Accord,0H3082Y,6.65,114,Gasoline,0.4816,Fan & Song 2022\nHonda Accord,0H3082Y,8.82,114,Gasoline,0.5693,Fan & Song 2022\nHonda Accord,0H3082Y,14.32,114,Gasoline,1.3229,Fan & Song 2022\nHonda Accord,0H3082Y,2.31,114,Gasoline,0.2054,Fan & Song 2022\nHonda Accord,0H3082Y,239.99,114,Gasoline,17.4576,Fan & Song 2022\nHonda Accord,0H3082Y,32.2,114,Gasoline,2.5935,Fan & Song 2022\nHonda Accord,0H3082Y,120.31,114,Gasoline,9.5795,Fan & Song 2022\nHonda Accord,0H3082Y,2.14,114,Gasoline,0.2647,Fan & Song 2022\nHonda Accord,0H3082Y,13.03,114,Gasoline,1.0497,Fan & Song 2022\nHonda Accord,0H3082Y,8.44,114,Gasoline,0.6871,Fan & Song 2022\nHonda Accord,0H3082Y,12.79,114,Gasoline,1.6917,Fan & Song 2022\nHonda Accord,0H3082Y,26.03,114,Gasoline,1.7369,Fan & Song 2022\nHonda Accord,0H3082Y,4.8,114,Gasoline,0.5069,Fan & Song 2022\nHonda Accord,0H3082Y,6.9,114,Gasoline,0.4072,Fan & Song 2022\nHonda Accord,0H3082Y,113.33,114,Gasoline,8.4509,Fan & Song 2022\nHonda Accord,0H3082Y,20.12,114,Gasoline,1.7505,Fan & Song 2022\nHonda Accord,0H3082Y,37.42,114,Gasoline,2.6365,Fan & Song 2022\nHonda Accord,0H3082Y,5.54,114,Gasoline,0.4447,Fan & Song 2022\nHonda Accord,0H3082Y,27.45,114,Gasoline,1.7492,Fan & Song 2022\nHonda Accord,0H3082Y,40.21,114,Gasoline,3.6999,Fan & Song 2022\nHonda Accord,0H3082Y,2.3,114,Gasoline,0.3044,Fan & Song 2022\nHonda Accord,0H3082Y,18.43,114,Gasoline,1.6378,Fan & Song 2022\nHonda Accord,0H3082Y,18.19,114,Gasoline,1.7344,Fan & Song 2022\nHonda Accord,0H3082Y,10.11,114,Gasoline,0.6904,Fan & Song 2022\nHonda Accord,0H3082Y,24.28,114,Gasoline,1.5732,Fan & Song 2022\nHonda Accord,0H3082Y,2.3,114,Gasoline,0.1633,Fan & Song 2022\nHonda Accord,0H3082Y,14.84,114,Gasoline,1.0351,Fan & Song 2022\nHonda Accord,0H3082Y,28.65,114,Gasoline,3.5148,Fan & Song 2022\nHonda Accord,0H3082Y,114.78,114,Gasoline,8.1806,Fan & Song 2022\nHonda Accord,0H3082Y,2.15,114,Gasoline,0.2538,Fan & Song 2022\nHonda Accord,0H3082Y,54.25,114,Gasoline,5.3751,Fan & Song 2022\nHonda Accord,0H3082Y,88.31,114,Gasoline,5.7854,Fan & Song 2022\nHonda Accord,0H3082Y,1.7,114,Gasoline,0.1154,Fan & Song 2022\nHonda Accord,0H3082Y,16.21,114,Gasoline,0.985,Fan & Song 2022\nHonda Accord,0H3082Y,11.37,114,Gasoline,0.7388,Fan & Song 2022\nHonda Accord,0H3082Y,249.96,114,Gasoline,14.7882,Fan & Song 2022\nHonda Accord,0H3082Y,144.36,114,Gasoline,9.602,Fan & Song 2022\nHonda Accord,0H3082Y,185.08,114,Gasoline,8.6707,Fan & Song 2022\nHonda Accord,0H3082Y,5.62,114,Gasoline,0.4222,Fan & Song 2022\nHonda Accord,0H3082Y,21.55,114,Gasoline,1.7487,Fan & Song 2022\nHonda Accord,0H3082Y,4.17,114,Gasoline,0.3854,Fan & Song 2022\nHonda Accord,0H3082Y,6.78,114,Gasoline,0.5105,Fan & Song 2022\nHonda Accord,0H3082Y,64.34,114,Gasoline,4.7831,Fan & Song 2022\nHonda Accord,0H3082Y,22.1,114,Gasoline,2.131,Fan & Song 2022\nHonda Accord,0H3082Y,64.33,114,Gasoline,5.778,Fan & Song 2022\nHonda Accord,0H3082Y,17.58,114,Gasoline,0.9962,Fan & Song 2022\nHonda Accord,0H3082Y,16.48,114,Gasoline,1.5913,Fan & Song 2022\nHonda Accord,0H3082Y,5.47,114,Gasoline,0.3402,Fan & Song 2022\nHonda Accord,0H3082Y,51.77,114,Gasoline,5.0938,Fan & Song 2022\nHonda Accord,0H3082Y,11.98,114,Gasoline,1.1728,Fan & Song 2022\nHonda Accord,0H3082Y,148.97,114,Gasoline,11.3004,Fan & Song 2022\nHonda Accord,0H3082Y,11.46,114,Gasoline,0.9701,Fan & Song 2022\nHonda Accord,0H3082Y,14.44,114,Gasoline,1.1751,Fan & Song 2022\nHonda Accord,0H3082Y,5.08,114,Gasoline,0.6865,Fan & Song 2022\nHonda Accord,0H3082Y,2.21,114,Gasoline,0.2403,Fan & Song 2022\nHonda Accord,0H3082Y,65.56,114,Gasoline,6.4528,Fan & Song 2022\nHonda Accord,0H3082Y,79.17,114,Gasoline,7.4744,Fan & Song 2022\nHonda Accord,0H3082Y,102.83,114,Gasoline,8.7417,Fan & Song 2022\nHonda Accord,0H3082Y,112.69,114,Gasoline,8.3314,Fan & Song 2022\nHonda Accord,0H3082Y,13.34,114,Gasoline,0.9466,Fan & Song 2022\nHonda Accord,0H3082Y,63.62,114,Gasoline,5.3957,Fan & Song 2022\nHonda Accord,0H3082Y,54.44,114,Gasoline,5.2902,Fan & Song 2022\nHonda Accord,0H3082Y,249.0,114,Gasoline,14.8126,Fan & Song 2022\nHonda Accord,0H3082Y,125.31,114,Gasoline,10.3225,Fan & Song 2022\nHonda Accord,0H3082Y,11.92,114,Gasoline,0.7366,Fan & Song 2022\nHonda Accord,0H3082Y,30.57,114,Gasoline,2.126,Fan & Song 2022\nHonda Accord,0H3082Y,4.43,114,Gasoline,0.5678,Fan & Song 2022\nHonda Accord,0H3082Y,170.46,114,Gasoline,10.0581,Fan & Song 2022\nHonda Accord,0H3082Y,28.6,114,Gasoline,1.7114,Fan & Song 2022\nHonda Accord,0H3082Y,2.25,114,Gasoline,0.1781,Fan & Song 2022\nHonda Accord,0H3082Y,34.63,114,Gasoline,3.1559,Fan & Song 2022\nHonda Accord,0H3082Y,84.32,114,Gasoline,7.2177,Fan & Song 2022\nHonda Accord,0H3082Y,4.24,114,Gasoline,0.4546,Fan & Song 2022\nHonda Accord,0H3082Y,12.96,114,Gasoline,1.3254,Fan & Song 2022\nHonda Accord,0H3082Y,19.08,114,Gasoline,1.4914,Fan & Song 2022\nHonda Accord,0H3082Y,71.94,114,Gasoline,5.5901,Fan & Song 2022\nHonda Accord,0H3082Y,2.15,114,Gasoline,0.2298,Fan & Song 2022\nHonda Accord,0H3082Y,1.61,114,Gasoline,0.1398,Fan & Song 2022\nHonda Accord,0H3082Y,161.44,114,Gasoline,12.8016,Fan & Song 2022\nHonda Accord,0H3082Y,16.51,114,Gasoline,1.1233,Fan & Song 2022\nHonda Accord,0H3082Y,84.77,114,Gasoline,6.7946,Fan & Song 2022\nHonda Accord,0H3082Y,27.7,114,Gasoline,2.7644,Fan & Song 2022\nHonda Accord,0H3082Y,125.12,114,Gasoline,11.124,Fan & Song 2022\nHonda Accord,0H3082Y,2.6,114,Gasoline,0.2747,Fan & Song 2022\nHonda Accord,0H3082Y,100.26,114,Gasoline,10.0376,Fan & Song 2022\nHonda Accord,0H3082Y,25.62,114,Gasoline,1.852,Fan & Song 2022\nHonda Accord,0H3082Y,4.3,114,Gasoline,0.2787,Fan & Song 2022\nHonda Accord,0H3082Y,76.82,114,Gasoline,7.4389,Fan & Song 2022\nHonda Accord,0H3082Y,76.96,114,Gasoline,4.8741,Fan & Song 2022\nHonda Accord,0H3082Y,36.54,114,Gasoline,2.5588,Fan & Song 2022\nHonda Accord,0H3082Y,12.32,114,Gasoline,0.8693,Fan & Song 2022\nHonda Accord,0H3082Y,34.83,114,Gasoline,2.5324,Fan & Song 2022\nHonda Accord,0H3082Y,52.84,114,Gasoline,4.6519,Fan & Song 2022\nHonda Accord,0H3082Y,29.84,114,Gasoline,1.9457,Fan & Song 2022\nHonda Accord,0H3082Y,104.96,114,Gasoline,10.6915,Fan & Song 2022\nHonda Accord,0H3082Y,6.22,114,Gasoline,0.4685,Fan & Song 2022\nHonda Accord,0H3082Y,2.35,114,Gasoline,0.4132,Fan & Song 2022\nHonda Accord,0H3082Y,27.72,114,Gasoline,2.7789,Fan & Song 2022\nHonda Accord,0H3082Y,28.12,114,Gasoline,2.1367,Fan & Song 2022\nHonda Accord,0H3082Y,5.46,114,Gasoline,0.3904,Fan & Song 2022\nHonda Accord,0H3082Y,230.38,114,Gasoline,14.4926,Fan & Song 2022\nHonda Accord,0H3082Y,4.56,114,Gasoline,0.5226,Fan & Song 2022\nHonda Accord,0H3082Y,118.35,114,Gasoline,9.4794,Fan & Song 2022\nHonda Accord,0H3082Y,8.6,114,Gasoline,0.9939,Fan & Song 2022\nHonda Accord,0H3082Y,10.72,114,Gasoline,1.5494,Fan & Song 2022\nHonda Accord,0H3082Y,8.78,114,Gasoline,0.6412,Fan & Song 2022\nHonda Accord,0H3082Y,7.27,114,Gasoline,0.4369,Fan & Song 2022\nHonda Accord,0H3082Y,165.96,114,Gasoline,10.183,Fan & Song 2022\nHonda Accord,0H3082Y,2.82,114,Gasoline,0.3868,Fan & Song 2022\nHonda Accord,0H3082Y,54.62,114,Gasoline,4.0466,Fan & Song 2022\nHonda Accord,0H3082Y,62.75,114,Gasoline,3.9244,Fan & Song 2022\nHonda Accord,0H3082Y,264.6,114,Gasoline,18.5139,Fan & Song 2022\nHonda Accord,0H3082Y,17.16,114,Gasoline,1.3281,Fan & Song 2022\nHonda Accord,0H3082Y,21.74,114,Gasoline,1.4738,Fan & Song 2022\nHonda Accord,0H3082Y,101.49,114,Gasoline,6.4951,Fan & Song 2022\nHonda Accord,0H3082Y,146.87,114,Gasoline,9.6251,Fan & Song 2022\nHonda Accord,0H3082Y,318.93,114,Gasoline,20.6948,Fan & Song 2022\nHonda Accord,0H3082Y,125.79,114,Gasoline,10.6784,Fan & Song 2022\nHonda Accord,0H3082Y,3.08,114,Gasoline,0.2619,Fan & Song 2022\nHonda Accord,0H3082Y,27.48,114,Gasoline,1.7337,Fan & Song 2022\nHonda Accord,0H3082Y,3.36,114,Gasoline,0.2564,Fan & Song 2022\nHonda Accord,0H3082Y,22.06,114,Gasoline,1.959,Fan & Song 2022\nHonda Accord,0H3082Y,105.1,114,Gasoline,9.2413,Fan & Song 2022\nHonda Accord,0H3082Y,25.35,114,Gasoline,2.0109,Fan & Song 2022\nHonda Accord,0H3082Y,35.75,114,Gasoline,3.2385,Fan & Song 2022\nHonda Accord,0H3082Y,11.7,114,Gasoline,0.7819,Fan & Song 2022\nHonda Accord,0H3082Y,67.08,114,Gasoline,4.8359,Fan & Song 2022\nHonda Accord,0H3082Y,3.53,114,Gasoline,0.2243,Fan & Song 2022\nHonda Accord,0H3082Y,167.67,114,Gasoline,16.9898,Fan & Song 2022\nHonda Accord,0H3082Y,46.29,114,Gasoline,3.6524,Fan & Song 2022\nHonda Accord,0H3082Y,31.93,114,Gasoline,2.4874,Fan & Song 2022\nHonda Accord,0H3082Y,39.34,114,Gasoline,2.6776,Fan & Song 2022\nHonda Accord,0H3082Y,36.64,114,Gasoline,3.6667,Fan & Song 2022\nHonda Accord,0H3082Y,9.01,114,Gasoline,0.9564,Fan & Song 2022\nHonda Accord,0H3082Y,111.75,114,Gasoline,8.6974,Fan & Song 2022\nHonda Accord,0H3082Y,28.73,114,Gasoline,2.07,Fan & Song 2022\nHonda Accord,0H3082Y,19.75,114,Gasoline,1.2,Fan & Song 2022\nHonda Accord,0H3082Y,4.21,114,Gasoline,0.4008,Fan & Song 2022\nHonda Accord,0H3082Y,2.09,114,Gasoline,0.2995,Fan & Song 2022\nHonda Accord,0H3082Y,26.5,114,Gasoline,2.4764,Fan & Song 2022\nHonda Accord,0H3082Y,5.08,114,Gasoline,0.8317,Fan & Song 2022\nHonda Accord,0H3082Y,144.37,114,Gasoline,9.3813,Fan & Song 2022\nHonda Accord,0H3082Y,164.26,114,Gasoline,15.2084,Fan & Song 2022\nHonda Accord,0H3082Y,57.36,114,Gasoline,3.203,Fan & Song 2022\nHonda Accord,0H3082Y,113.56,114,Gasoline,7.3663,Fan & Song 2022\nHonda Accord,0H3082Y,108.89,114,Gasoline,7.3971,Fan & Song 2022\nHonda Accord,0H3082Y,22.97,114,Gasoline,1.5723,Fan & Song 2022\nHonda Accord,0H3082Y,45.01,114,Gasoline,4.276,Fan & Song 2022\nHonda Accord,0H3082Y,26.7,114,Gasoline,2.0067,Fan & Song 2022\nHonda Accord,0H3082Y,101.33,114,Gasoline,6.2493,Fan & Song 2022\nHonda Accord,0H3082Y,5.12,114,Gasoline,0.3612,Fan & Song 2022\nHonda Accord,0H3082Y,11.26,114,Gasoline,1.1374,Fan & Song 2022\nHonda Accord,0H3082Y,72.17,114,Gasoline,5.4204,Fan & Song 2022\nHonda Accord,0H3082Y,42.58,114,Gasoline,2.9234,Fan & Song 2022\nHonda Accord,0H3082Y,34.69,114,Gasoline,3.3437,Fan & Song 2022\nHonda Accord,0H3082Y,31.41,114,Gasoline,3.4533,Fan & Song 2022\nHonda Accord,0H3082Y,74.1,114,Gasoline,5.8426,Fan & Song 2022\nHonda Accord,0H3082Y,108.22,114,Gasoline,9.4706,Fan & Song 2022\nHonda Accord,0H3082Y,2.51,114,Gasoline,0.327,Fan & Song 2022\nHonda Accord,0H3082Y,144.85,114,Gasoline,10.1372,Fan & Song 2022\nHonda Accord,0H3082Y,271.02,114,Gasoline,25.069,Fan & Song 2022\nHonda Accord,0H3082Y,27.02,114,Gasoline,1.8118,Fan & Song 2022\nHonda Accord,0H3082Y,34.78,114,Gasoline,1.8466,Fan & Song 2022\nHonda Accord,0H3082Y,18.27,114,Gasoline,2.4115,Fan & Song 2022\nHonda Accord,0H3082Y,6.98,114,Gasoline,0.7854,Fan & Song 2022\nHonda Accord,0H3082Y,3.36,114,Gasoline,0.2359,Fan & Song 2022\nHonda Accord,0H3082Y,3.58,114,Gasoline,0.4481,Fan & Song 2022\nHonda Accord,0H3082Y,10.85,114,Gasoline,0.8344,Fan & Song 2022\nHonda Accord,0H3082Y,140.84,114,Gasoline,12.4347,Fan & Song 2022\nHonda Accord,0H3082Y,21.07,114,Gasoline,1.6075,Fan & Song 2022\nHonda Accord,0H3082Y,105.26,114,Gasoline,10.9901,Fan & Song 2022\nHonda Accord,0H3082Y,78.96,114,Gasoline,8.7754,Fan & Song 2022\nHonda Accord,0H3082Y,189.07,114,Gasoline,12.3251,Fan & Song 2022\nHonda Accord,0H3082Y,21.58,114,Gasoline,3.3215,Fan & Song 2022\nHonda Accord,0H3082Y,47.62,114,Gasoline,5.7381,Fan & Song 2022\nHonda Accord,0H3082Y,27.55,114,Gasoline,2.6225,Fan & Song 2022\nHonda Accord,0H3082Y,348.92,114,Gasoline,23.384,Fan & Song 2022\nHonda Accord,0H3082Y,102.07,114,Gasoline,6.0364,Fan & Song 2022\nHonda Accord,0H3082Y,3.24,114,Gasoline,0.3524,Fan & Song 2022\nHonda Accord,0H3082Y,4.9,114,Gasoline,0.3151,Fan & Song 2022\nHonda Accord,0H3082Y,2.53,114,Gasoline,0.1753,Fan & Song 2022\nHonda Accord,0H3082Y,7.23,114,Gasoline,0.8527,Fan & Song 2022\nHonda Accord,0H3082Y,1.56,114,Gasoline,0.1125,Fan & Song 2022\nHonda Accord,0H3082Y,30.68,114,Gasoline,1.7166,Fan & Song 2022\nHonda Accord,0H3082Y,12.13,114,Gasoline,1.5409,Fan & Song 2022\nHonda Accord,0H3082Y,10.73,114,Gasoline,0.8443,Fan & Song 2022\nHonda Accord,0H3082Y,19.64,114,Gasoline,1.2626,Fan & Song 2022\nHonda Accord,0H3082Y,16.21,114,Gasoline,1.5185,Fan & Song 2022\nHonda Accord,0H3082Y,36.53,114,Gasoline,2.9136,Fan & Song 2022\nHonda Accord,0H3082Y,114.21,114,Gasoline,9.8122,Fan & Song 2022\nHonda Accord,0H3082Y,34.13,114,Gasoline,3.9957,Fan & Song 2022\nHonda Accord,0H3082Y,4.52,114,Gasoline,0.5416,Fan & Song 2022\nHonda Accord,0H3082Y,28.08,114,Gasoline,2.8182,Fan & Song 2022\nHonda Accord,0H3082Y,252.02,114,Gasoline,16.1745,Fan & Song 2022\nHonda Accord,0H3082Y,35.4,114,Gasoline,2.5594,Fan & Song 2022\nHonda Accord,0H3082Y,155.86,114,Gasoline,14.7055,Fan & Song 2022\nHonda Accord,0H3082Y,123.87,114,Gasoline,9.366,Fan & Song 2022\nHonda Accord,0H3082Y,3.64,114,Gasoline,0.4059,Fan & Song 2022\nHonda Accord,0H3082Y,148.96,114,Gasoline,14.0851,Fan & Song 2022\nHonda Accord,0H3082Y,3.06,114,Gasoline,0.3482,Fan & Song 2022\nHonda Accord,0H3082Y,2.65,114,Gasoline,0.3476,Fan & Song 2022\nHonda Accord,0H3082Y,103.12,114,Gasoline,8.679,Fan & Song 2022\nHonda Accord,0H3082Y,91.43,114,Gasoline,6.6491,Fan & Song 2022\nHonda Accord,0H3082Y,34.36,114,Gasoline,2.7575,Fan & Song 2022\nHonda Accord,0H3082Y,14.44,114,Gasoline,1.2629,Fan & Song 2022\nHonda Accord,0H3082Y,27.55,114,Gasoline,2.9593,Fan & Song 2022\nHonda Accord,0H3082Y,38.19,114,Gasoline,2.9692,Fan & Song 2022\nHonda Accord,0H3082Y,14.0,114,Gasoline,1.0217,Fan & Song 2022\nHonda Accord,0H3082Y,24.88,114,Gasoline,1.9972,Fan & Song 2022\nHonda Accord,0H3082Y,36.21,114,Gasoline,2.2715,Fan & Song 2022\nHonda Accord,0H3082Y,15.2,114,Gasoline,1.4911,Fan & Song 2022\nHonda Accord,0H3082Y,7.26,114,Gasoline,0.9703,Fan & Song 2022\nHonda Accord,0H3082Y,113.61,114,Gasoline,9.9355,Fan & Song 2022\nHonda Accord,0H3082Y,9.94,114,Gasoline,0.7158,Fan & Song 2022\nHonda Accord,0H3082Y,101.17,114,Gasoline,10.6519,Fan & Song 2022\nHonda Accord,0H3082Y,104.07,114,Gasoline,8.263,Fan & Song 2022\nHonda Accord,0H3082Y,14.28,114,Gasoline,1.5742,Fan & Song 2022\nHonda Accord,0H3082Y,21.22,114,Gasoline,2.8019,Fan & Song 2022\nHonda Accord,0H3082Y,27.2,114,Gasoline,4.1126,Fan & Song 2022\nHonda Accord,0H3082Y,39.0,114,Gasoline,5.6375,Fan & Song 2022\nHonda Accord,0H3082Y,10.49,114,Gasoline,1.3331,Fan & Song 2022\nHonda Accord,0H3082Y,33.8,114,Gasoline,2.4652,Fan & Song 2022\nHonda Accord,0H3082Y,1.58,114,Gasoline,0.2595,Fan & Song 2022\nHonda Accord,0H3082Y,1.65,114,Gasoline,0.21,Fan & Song 2022\nHonda Accord,0H3082Y,1.79,114,Gasoline,0.2232,Fan & Song 2022\nHonda Accord,0H3082Y,26.02,114,Gasoline,1.6103,Fan & Song 2022\nHonda Accord,0H3082Y,13.85,114,Gasoline,1.1774,Fan & Song 2022\nHonda Accord,0H3082Y,2.87,114,Gasoline,0.1699,Fan & Song 2022\nHonda Accord,0H3082Y,116.85,114,Gasoline,10.4722,Fan & Song 2022\nHonda Accord,0H3082Y,38.38,114,Gasoline,2.5729,Fan & Song 2022\nHonda Accord,0H3082Y,115.58,114,Gasoline,12.1391,Fan & Song 2022\nHonda Accord,0H3082Y,134.77,114,Gasoline,8.7206,Fan & Song 2022\nHonda Accord,0H3082Y,41.46,114,Gasoline,2.6033,Fan & Song 2022\nHonda Accord,0H3082Y,7.54,114,Gasoline,0.8566,Fan & Song 2022\nHonda Accord,0H3082Y,142.0,114,Gasoline,9.4341,Fan & Song 2022\nHonda Accord,0H3082Y,19.84,114,Gasoline,1.488,Fan & Song 2022\nHonda Accord,0H3082Y,1.75,114,Gasoline,0.1293,Fan & Song 2022\nHonda Accord,0H3082Y,6.14,114,Gasoline,0.7663,Fan & Song 2022\nHonda Accord,0H3082Y,5.94,114,Gasoline,0.4181,Fan & Song 2022\nHonda Accord,0H3082Y,42.68,114,Gasoline,3.4983,Fan & Song 2022\nHonda Accord,0H3082Y,25.56,114,Gasoline,4.0155,Fan & Song 2022\nHonda Accord,0H3082Y,103.64,114,Gasoline,7.4207,Fan & Song 2022\nHonda Accord,0H3082Y,29.41,114,Gasoline,1.7203,Fan & Song 2022\nHonda Accord,0H3082Y,36.5,114,Gasoline,3.9008,Fan & Song 2022\nHonda Accord,0H3082Y,273.74,114,Gasoline,20.213,Fan & Song 2022\nHonda Accord,0H3082Y,53.12,114,Gasoline,3.357,Fan & Song 2022\nHonda Accord,0H3082Y,85.52,114,Gasoline,5.7584,Fan & Song 2022\nHonda Accord,0H3082Y,3.16,114,Gasoline,0.367,Fan & Song 2022\nHonda Accord,0H3082Y,1.58,114,Gasoline,0.2169,Fan & Song 2022\nHonda Accord,0H3082Y,10.54,114,Gasoline,0.9592,Fan & Song 2022\nHonda Accord,0H3082Y,6.41,114,Gasoline,0.8967,Fan & Song 2022\nHonda Accord,0H3082Y,10.48,114,Gasoline,0.7493,Fan & Song 2022\nHonda Accord,0H3082Y,16.82,114,Gasoline,1.1163,Fan & Song 2022\nHonda Accord,0H3082Y,128.58,114,Gasoline,10.166,Fan & Song 2022\nHonda Accord,0H3082Y,76.35,114,Gasoline,4.1383,Fan & Song 2022\nHonda Accord,0H3082Y,1.99,114,Gasoline,0.2758,Fan & Song 2022\nHonda Accord,0H3082Y,1.54,114,Gasoline,0.1955,Fan & Song 2022\nHonda Accord,0H3082Y,13.29,114,Gasoline,0.8582,Fan & Song 2022\nHonda Accord,0H3082Y,32.32,114,Gasoline,2.5121,Fan & Song 2022\nHonda Accord,0H3082Y,3.58,114,Gasoline,0.239,Fan & Song 2022\nHonda Accord,0H3082Y,2.71,114,Gasoline,0.3083,Fan & Song 2022\nHonda Accord,0H3082Y,9.11,114,Gasoline,0.8214,Fan & Song 2022\nHonda Accord,0H3082Y,108.57,114,Gasoline,7.3164,Fan & Song 2022\nHonda Accord,0H3082Y,123.89,114,Gasoline,7.9915,Fan & Song 2022\nHonda Accord,0H3082Y,108.66,114,Gasoline,8.0593,Fan & Song 2022\nHonda Accord,0H3082Y,51.99,114,Gasoline,5.5889,Fan & Song 2022\nHonda Accord,0H3082Y,109.94,114,Gasoline,6.7082,Fan & Song 2022\nHonda Accord,0H3082Y,121.72,114,Gasoline,9.1965,Fan & Song 2022\nHonda Accord,0H3082Y,59.7,114,Gasoline,3.927,Fan & Song 2022\nHonda Accord,0H3082Y,57.31,114,Gasoline,3.4321,Fan & Song 2022\nHonda Accord,0H3082Y,153.66,114,Gasoline,12.1771,Fan & Song 2022\nHonda Accord,0H3082Y,50.89,114,Gasoline,6.5853,Fan & Song 2022\nHonda Accord,0H3082Y,54.5,114,Gasoline,5.9134,Fan & Song 2022\nHonda Accord,0H3082Y,164.31,114,Gasoline,13.9812,Fan & Song 2022\nHonda Accord,0H3082Y,130.53,114,Gasoline,9.8177,Fan & Song 2022\nHonda Accord,0H3082Y,44.1,114,Gasoline,5.2009,Fan & Song 2022\nHonda Accord,0H3082Y,1.72,114,Gasoline,0.1488,Fan & Song 2022\nHonda Accord,0H3082Y,141.23,114,Gasoline,12.9263,Fan & Song 2022\nHonda Accord,0H3082Y,2.71,114,Gasoline,0.2047,Fan & Song 2022\nHonda Accord,0H3082Y,34.77,114,Gasoline,2.7674,Fan & Song 2022\nHonda Accord,0H3082Y,18.04,114,Gasoline,1.481,Fan & Song 2022\nHonda Accord,0H3082Y,45.83,114,Gasoline,3.5845,Fan & Song 2022\nHonda Accord,0H3082Y,19.19,114,Gasoline,1.4086,Fan & Song 2022\nHonda Accord,0H3082Y,26.55,114,Gasoline,2.2198,Fan & Song 2022\nHonda Accord,0H3082Y,3.66,114,Gasoline,0.3842,Fan & Song 2022\nHonda Accord,0H3082Y,27.98,114,Gasoline,1.5659,Fan & Song 2022\nHonda Accord,0H3082Y,102.08,114,Gasoline,6.8584,Fan & Song 2022\nHonda Accord,0H3082Y,2.97,114,Gasoline,0.1823,Fan & Song 2022\nHonda Accord,0H3082Y,110.44,114,Gasoline,7.3407,Fan & Song 2022\nHonda Accord,0H3082Y,15.47,114,Gasoline,1.0772,Fan & Song 2022\nHonda Accord,0H3082Y,1.51,114,Gasoline,0.1902,Fan & Song 2022\nHonda Accord,0H3082Y,14.81,114,Gasoline,1.1425,Fan & Song 2022\nHonda Accord,0H3082Y,54.53,114,Gasoline,3.2893,Fan & Song 2022\nHonda Accord,0H3082Y,19.76,114,Gasoline,1.13,Fan & Song 2022\nHonda Accord,0H3082Y,43.44,114,Gasoline,2.9204,Fan & Song 2022\nHonda Accord,0H3082Y,142.02,114,Gasoline,10.0337,Fan & Song 2022\nHonda Accord,0H3082Y,43.35,114,Gasoline,4.0246,Fan & Song 2022\nHonda Accord,0H3082Y,15.43,114,Gasoline,1.4653,Fan & Song 2022\nHonda Accord,0H3082Y,16.28,114,Gasoline,1.442,Fan & Song 2022\nHonda Accord,0H3082Y,2.42,114,Gasoline,0.1753,Fan & Song 2022\nHonda Accord,0H3082Y,31.84,114,Gasoline,2.0137,Fan & Song 2022\nHonda Accord,0H3082Y,40.71,114,Gasoline,2.4153,Fan & Song 2022\nHonda Accord,0H3082Y,7.82,114,Gasoline,1.1144,Fan & Song 2022\nHonda Accord,0H3082Y,22.55,114,Gasoline,1.3795,Fan & Song 2022\nHonda Accord,0H3082Y,105.32,114,Gasoline,8.5542,Fan & Song 2022\nHonda Accord,0H3082Y,66.63,114,Gasoline,4.6428,Fan & Song 2022\nHonda Accord,0H3082Y,4.6,114,Gasoline,0.4632,Fan & Song 2022\nHonda Accord,0H3082Y,191.96,114,Gasoline,11.7049,Fan & Song 2022\nHonda Accord,0H3082Y,5.0,114,Gasoline,0.5137,Fan & Song 2022\nHonda Accord,0H3082Y,73.55,114,Gasoline,4.694,Fan & Song 2022\nHonda Accord,0H3082Y,264.93,114,Gasoline,17.0869,Fan & Song 2022\nHonda Accord,0H3082Y,129.92,114,Gasoline,9.7388,Fan & Song 2022\nHonda Accord,0H3082Y,149.88,114,Gasoline,8.787,Fan & Song 2022\nHonda Accord,0H3082Y,42.92,114,Gasoline,3.2199,Fan & Song 2022\nHonda Accord,0H3082Y,274.73,114,Gasoline,15.5051,Fan & Song 2022\nHonda Accord,0H3082Y,251.42,114,Gasoline,13.2333,Fan & Song 2022\nHonda Accord,0H3082Y,77.28,114,Gasoline,4.9934,Fan & Song 2022\nHonda Accord,0H3082Y,5.19,114,Gasoline,0.3508,Fan & Song 2022\nHonda Accord,0H3082Y,28.11,114,Gasoline,2.1428,Fan & Song 2022\nHonda Accord,0H3082Y,17.48,114,Gasoline,1.2751,Fan & Song 2022\nHonda Accord,0H3082Y,2.17,114,Gasoline,0.3652,Fan & Song 2022\nHonda Accord,0H3082Y,67.63,114,Gasoline,4.4125,Fan & Song 2022\nHonda Accord,0H3082Y,123.19,114,Gasoline,10.21,Fan & Song 2022\nHonda Accord,0H3082Y,15.1,114,Gasoline,0.972,Fan & Song 2022\nHonda Accord,0H3082Y,10.33,114,Gasoline,1.0841,Fan & Song 2022\nHonda Accord,0H3082Y,15.14,114,Gasoline,0.9499,Fan & Song 2022\nHonda Accord,0H3082Y,41.67,114,Gasoline,2.9436,Fan & Song 2022\nHonda Accord,0H3082Y,1.65,114,Gasoline,0.2547,Fan & Song 2022\nHonda Accord,0H3082Y,83.76,114,Gasoline,4.7594,Fan & Song 2022\nHonda Accord,0H3082Y,386.22,114,Gasoline,24.4283,Fan & Song 2022\nHonda Accord,0H3082Y,46.05,114,Gasoline,3.6159,Fan & Song 2022\nHonda Accord,0H3082Y,75.73,114,Gasoline,4.2734,Fan & Song 2022\nHonda Accord,0H3082Y,12.79,114,Gasoline,1.1874,Fan & Song 2022\nHonda Accord,0H3082Y,12.84,114,Gasoline,1.293,Fan & Song 2022\nHonda Accord,0H3082Y,1.72,114,Gasoline,0.1268,Fan & Song 2022\nHonda Accord,0H3082Y,19.07,114,Gasoline,1.8778,Fan & Song 2022\nHonda Accord,0H3082Y,31.08,114,Gasoline,1.8842,Fan & Song 2022\nHonda Accord,0H3082Y,128.53,114,Gasoline,8.4488,Fan & Song 2022\nHonda Accord,0H3082Y,32.59,114,Gasoline,2.0427,Fan & Song 2022\nHonda Accord,0H3082Y,101.31,114,Gasoline,9.7665,Fan & Song 2022\nHonda Accord,0H3082Y,68.09,114,Gasoline,4.8888,Fan & Song 2022\nHonda Accord,0H3082Y,136.44,114,Gasoline,12.8628,Fan & Song 2022\nHonda Accord,0H3082Y,3.55,114,Gasoline,0.3507,Fan & Song 2022\nHonda Accord,0H3082Y,5.43,114,Gasoline,0.3498,Fan & Song 2022\nHonda Accord,0H3082Y,55.33,114,Gasoline,4.9158,Fan & Song 2022\nHonda Accord,0H3082Y,14.96,114,Gasoline,1.5538,Fan & Song 2022\nHonda Accord,0H3082Y,146.17,114,Gasoline,8.8556,Fan & Song 2022\nHonda Accord,0H3082Y,22.02,114,Gasoline,1.7878,Fan & Song 2022\nHonda Accord,0H3082Y,2.48,114,Gasoline,0.2502,Fan & Song 2022\nHonda Accord,0H3082Y,10.14,114,Gasoline,0.9578,Fan & Song 2022\nHonda Accord,0H3082Y,33.8,114,Gasoline,2.5756,Fan & Song 2022\nHonda Accord,0H3082Y,48.64,114,Gasoline,3.113,Fan & Song 2022\nHonda Accord,0H3082Y,12.21,114,Gasoline,1.1347,Fan & Song 2022\nHonda Accord,0H3082Y,5.9,114,Gasoline,0.5564,Fan & Song 2022\nHonda Accord,0H3082Y,113.33,114,Gasoline,6.7171,Fan & Song 2022\nHonda Accord,0H3082Y,35.86,114,Gasoline,2.6331,Fan & Song 2022\nHonda Accord,0H3082Y,1.81,114,Gasoline,0.2341,Fan & Song 2022\nHonda Accord,0H3082Y,136.54,114,Gasoline,10.8219,Fan & Song 2022\nHonda Accord,0H3082Y,114.96,114,Gasoline,9.8732,Fan & Song 2022\nHonda Accord,0H3082Y,5.71,114,Gasoline,0.5774,Fan & Song 2022\nHonda Accord,0H3082Y,114.69,114,Gasoline,7.5924,Fan & Song 2022\nHonda Accord,0H3082Y,21.94,114,Gasoline,1.7898,Fan & Song 2022\nHonda Accord,0H3082Y,40.05,114,Gasoline,2.937,Fan & Song 2022\nHonda Accord,0H3082Y,10.89,114,Gasoline,0.9368,Fan & Song 2022\nHonda Accord,0H3082Y,8.82,114,Gasoline,0.7472,Fan & Song 2022\nHonda Accord,0H3082Y,111.43,114,Gasoline,6.2384,Fan & Song 2022\nHonda Accord,0H3082Y,4.9,114,Gasoline,0.3055,Fan & Song 2022\nHonda Accord,0H3082Y,14.3,114,Gasoline,0.938,Fan & Song 2022\nHonda Accord,0H3082Y,4.24,114,Gasoline,0.4321,Fan & Song 2022\nHonda Accord,0H3082Y,69.56,114,Gasoline,5.7754,Fan & Song 2022\nHonda Accord,0H3082Y,51.14,114,Gasoline,5.7569,Fan & Song 2022\nHonda Accord,0H3082Y,34.03,114,Gasoline,2.2006,Fan & Song 2022\nHonda Accord,0H3082Y,53.98,114,Gasoline,4.5324,Fan & Song 2022\nHonda Accord,0H3082Y,36.06,114,Gasoline,2.2437,Fan & Song 2022\nHonda Accord,0H3082Y,80.21,114,Gasoline,5.535,Fan & Song 2022\nHonda Accord,0H3082Y,3.2,114,Gasoline,0.2643,Fan & Song 2022\nHonda Accord,0H3082Y,147.36,114,Gasoline,8.2835,Fan & Song 2022\nHonda Accord,0H3082Y,136.07,114,Gasoline,9.5258,Fan & Song 2022\nHonda Accord,0H3082Y,13.69,114,Gasoline,0.7893,Fan & Song 2022\nHonda Accord,0H3082Y,73.46,114,Gasoline,5.4006,Fan & Song 2022\nHonda Accord,0H3082Y,3.36,114,Gasoline,0.2678,Fan & Song 2022\nHonda Accord,0H3082Y,2.28,114,Gasoline,0.2226,Fan & Song 2022\nHonda Accord,0H3082Y,70.79,114,Gasoline,5.4676,Fan & Song 2022\nHonda Accord,0H3082Y,100.59,114,Gasoline,8.3924,Fan & Song 2022\nHonda Accord,0H3082Y,45.99,114,Gasoline,5.3919,Fan & Song 2022\nHonda Accord,0H3082Y,2.26,114,Gasoline,0.1687,Fan & Song 2022\nHonda Accord,0H3082Y,2.25,114,Gasoline,0.1979,Fan & Song 2022\nHonda Accord,0H3082Y,34.9,114,Gasoline,3.0369,Fan & Song 2022\nHonda Accord,0H3082Y,79.64,114,Gasoline,5.7446,Fan & Song 2022\nHonda Accord,0H3082Y,10.58,114,Gasoline,0.9962,Fan & Song 2022\nHonda Accord,0H3082Y,101.66,114,Gasoline,6.447,Fan & Song 2022\nHonda Accord,0H3082Y,10.07,114,Gasoline,0.8747,Fan & Song 2022\nHonda Accord,0H3082Y,6.44,114,Gasoline,0.5615,Fan & Song 2022\nHonda Accord,0H3082Y,42.68,114,Gasoline,3.763,Fan & Song 2022\nHonda Accord,0H3082Y,57.42,114,Gasoline,4.279,Fan & Song 2022\nHonda Accord,0H3082Y,146.48,114,Gasoline,11.6388,Fan & Song 2022\nHonda Accord,0H3082Y,20.01,114,Gasoline,1.2052,Fan & Song 2022\nHonda Accord,0H3082Y,230.13,114,Gasoline,13.3249,Fan & Song 2022\nHonda Accord,0H3082Y,102.81,114,Gasoline,8.3643,Fan & Song 2022\nHonda Accord,0H3082Y,13.14,114,Gasoline,1.0052,Fan & Song 2022\nHonda Accord,0H3082Y,26.27,114,Gasoline,2.9345,Fan & Song 2022\nHonda Accord,0H3082Y,71.96,114,Gasoline,5.0561,Fan & Song 2022\nHonda Accord,0H3082Y,41.25,114,Gasoline,3.7481,Fan & Song 2022\nHonda Accord,0H3082Y,4.5,114,Gasoline,0.4438,Fan & Song 2022\nHonda Accord,0H3082Y,17.71,114,Gasoline,1.3259,Fan & Song 2022\nHonda Accord,0H3082Y,197.74,114,Gasoline,14.429,Fan & Song 2022\nHonda Accord,0H3082Y,4.61,114,Gasoline,0.5265,Fan & Song 2022\nHonda Accord,0H3082Y,112.54,114,Gasoline,9.4966,Fan & Song 2022\nHonda Accord,0H3082Y,153.95,114,Gasoline,11.231,Fan & Song 2022\nHonda Accord,0H3082Y,1.79,114,Gasoline,0.2307,Fan & Song 2022\nHonda Accord,0H3082Y,7.01,114,Gasoline,0.61,Fan & Song 2022\nHonda Accord,0H3082Y,17.41,114,Gasoline,1.9776,Fan & Song 2022\nHonda Accord,0H3082Y,35.2,114,Gasoline,3.3741,Fan & Song 2022\nHonda Accord,0H3082Y,30.98,114,Gasoline,1.7797,Fan & Song 2022\nHonda Accord,0H3082Y,232.45,114,Gasoline,13.8328,Fan & Song 2022\nHonda Accord,0H3082Y,102.24,114,Gasoline,7.8598,Fan & Song 2022\nHonda Accord,0H3082Y,9.21,114,Gasoline,0.5991,Fan & Song 2022\nHonda Accord,0H3082Y,44.03,114,Gasoline,2.63,Fan & Song 2022\nHonda Accord,0H3082Y,128.69,114,Gasoline,9.7128,Fan & Song 2022\nHonda Accord,0H3082Y,108.12,114,Gasoline,4.5423,Fan & Song 2022\nHonda Accord,0H3082Y,45.7,114,Gasoline,3.2797,Fan & Song 2022\nHonda Accord,0H3082Y,45.95,114,Gasoline,3.1547,Fan & Song 2022\nHonda Accord,0H3082Y,17.24,114,Gasoline,1.6311,Fan & Song 2022\nHonda Accord,0H3082Y,31.85,114,Gasoline,3.2171,Fan & Song 2022\nHonda Accord,0H3082Y,102.84,114,Gasoline,9.3235,Fan & Song 2022\nHonda Accord,0H3082Y,108.56,114,Gasoline,6.9276,Fan & Song 2022\nHonda Accord,0H3082Y,19.29,114,Gasoline,2.4384,Fan & Song 2022\nHonda Accord,0H3082Y,4.84,114,Gasoline,0.5192,Fan & Song 2022\nHonda Accord,0H3082Y,62.26,114,Gasoline,4.2844,Fan & Song 2022\nHonda Accord,0H3082Y,2.0,114,Gasoline,0.3245,Fan & Song 2022\nHonda Accord,0H3082Y,37.46,114,Gasoline,2.865,Fan & Song 2022\nHonda Accord,0H3082Y,9.96,114,Gasoline,0.7733,Fan & Song 2022\nHonda Accord,0H3082Y,37.84,114,Gasoline,3.7164,Fan & Song 2022\nHonda Accord,0H3082Y,14.92,114,Gasoline,1.9675,Fan & Song 2022\nHonda Accord,0H3082Y,10.38,114,Gasoline,1.1089,Fan & Song 2022\nHonda Accord,0H3082Y,41.62,114,Gasoline,3.3208,Fan & Song 2022\nHonda Accord,0H3082Y,44.99,114,Gasoline,3.5033,Fan & Song 2022\nHonda Accord,0H3082Y,1.7,114,Gasoline,0.2809,Fan & Song 2022\nHonda Accord,0H3082Y,35.59,114,Gasoline,3.436,Fan & Song 2022\nHonda Accord,0H3082Y,105.07,114,Gasoline,9.0047,Fan & Song 2022\nHonda Accord,0H3082Y,129.6,114,Gasoline,8.5539,Fan & Song 2022\nHonda Accord,0H3082Y,36.43,114,Gasoline,4.4005,Fan & Song 2022\nHonda Accord,0H3082Y,101.31,114,Gasoline,9.1514,Fan & Song 2022\nHonda Accord,0H3082Y,5.16,114,Gasoline,0.5624,Fan & Song 2022\nHonda Accord,0H3082Y,121.34,114,Gasoline,8.5015,Fan & Song 2022\nHonda Accord,0H3082Y,15.32,114,Gasoline,1.4291,Fan & Song 2022\nHonda Accord,0H3082Y,13.03,114,Gasoline,1.0918,Fan & Song 2022\nHonda Accord,0H3082Y,48.27,114,Gasoline,4.0371,Fan & Song 2022\nHonda Accord,0H3082Y,13.21,114,Gasoline,1.2198,Fan & Song 2022\nHonda Accord,0H3082Y,34.37,114,Gasoline,2.3782,Fan & Song 2022\nHonda Accord,0H3082Y,33.76,114,Gasoline,2.241,Fan & Song 2022\nHonda Accord,0H3082Y,32.15,114,Gasoline,2.5539,Fan & Song 2022\nHonda Accord,0H3082Y,41.57,114,Gasoline,4.7835,Fan & Song 2022\nHonda Accord,0H3082Y,3.66,114,Gasoline,0.3648,Fan & Song 2022\nHonda Accord,0H3082Y,57.29,114,Gasoline,3.256,Fan & Song 2022\nHonda Accord,0H3082Y,62.16,114,Gasoline,6.5702,Fan & Song 2022\nHonda Accord,0H3082Y,4.38,114,Gasoline,0.3587,Fan & Song 2022\nHonda Accord,0H3082Y,10.16,114,Gasoline,0.7661,Fan & Song 2022\nHonda Accord,0H3082Y,82.34,114,Gasoline,6.3253,Fan & Song 2022\nHonda Accord,0H3082Y,275.88,114,Gasoline,20.4397,Fan & Song 2022\nHonda Accord,0H3082Y,76.53,114,Gasoline,5.3286,Fan & Song 2022\nHonda Accord,0H3082Y,11.11,114,Gasoline,0.8762,Fan & Song 2022\nHonda Accord,0H3082Y,11.71,114,Gasoline,0.902,Fan & Song 2022\nHonda Accord,0H3082Y,22.83,114,Gasoline,1.56,Fan & Song 2022\nHonda Accord,0H3082Y,51.85,114,Gasoline,3.166,Fan & Song 2022\nHonda Accord,0H3082Y,46.03,114,Gasoline,3.3603,Fan & Song 2022\nHonda Accord,0H3082Y,16.0,114,Gasoline,1.5682,Fan & Song 2022\nHonda Accord,0H3082Y,6.18,114,Gasoline,0.5528,Fan & Song 2022\nHonda Accord,0H3082Y,112.91,114,Gasoline,7.772,Fan & Song 2022\nHonda Accord,0H3082Y,163.65,114,Gasoline,10.9854,Fan & Song 2022\nHonda Accord,0H3082Y,110.12,114,Gasoline,8.1586,Fan & Song 2022\nHonda Accord,0H3082Y,3.3,114,Gasoline,0.3828,Fan & Song 2022\nHonda Accord,0H3082Y,112.88,114,Gasoline,11.3511,Fan & Song 2022\nHonda Accord,0H3082Y,116.22,114,Gasoline,12.1599,Fan & Song 2022\nHonda Accord,0H3082Y,138.18,114,Gasoline,11.2393,Fan & Song 2022\nHonda Odyssey,0H3011H,48.91,137,Gasoline,4.2577,Fan & Song 2022\nHonda Odyssey,0H3011H,51.59,137,Gasoline,3.8814,Fan & Song 2022\nHonda Odyssey,0H3011H,12.58,137,Gasoline,1.1345,Fan & Song 2022\nHonda Odyssey,0H3011H,115.1,137,Gasoline,8.7183,Fan & Song 2022\nHonda Odyssey,0H3011H,15.81,137,Gasoline,1.7756,Fan & Song 2022\nHonda Odyssey,0H3011H,11.28,137,Gasoline,1.4381,Fan & Song 2022\nHonda Odyssey,0H3011H,3.41,137,Gasoline,0.4677,Fan & Song 2022\nHonda Odyssey,0H3011H,23.58,137,Gasoline,2.3036,Fan & Song 2022\nHonda Odyssey,0H3011H,34.17,137,Gasoline,2.8238,Fan & Song 2022\nHonda Odyssey,0H3011H,39.71,137,Gasoline,3.8776,Fan & Song 2022\nHonda Odyssey,0H3011H,17.46,137,Gasoline,1.4189,Fan & Song 2022\nHonda Odyssey,0H3011H,47.53,137,Gasoline,4.2374,Fan & Song 2022\nHonda Odyssey,0H3011H,51.46,137,Gasoline,4.5301,Fan & Song 2022\nHonda Odyssey,0H3011H,6.0,137,Gasoline,0.6551,Fan & Song 2022\nHonda Odyssey,0H3011H,36.39,137,Gasoline,2.5869,Fan & Song 2022\nHonda Odyssey,0H3011H,6.22,137,Gasoline,0.5229,Fan & Song 2022\nHonda Odyssey,0H3011H,60.2,137,Gasoline,4.9794,Fan & Song 2022\nHonda Odyssey,0H3011H,11.88,137,Gasoline,1.0518,Fan & Song 2022\nHonda Odyssey,0H3011H,13.55,137,Gasoline,1.4711,Fan & Song 2022\nHonda Odyssey,0H3011H,88.3,137,Gasoline,7.0778,Fan & Song 2022\nHonda Odyssey,0H3011H,10.49,137,Gasoline,1.2861,Fan & Song 2022\nHonda Odyssey,0H3011H,123.69,137,Gasoline,9.8957,Fan & Song 2022\nHonda Odyssey,0H3011H,130.04,137,Gasoline,10.3582,Fan & Song 2022\nHonda Odyssey,0H3011H,114.52,137,Gasoline,8.713,Fan & Song 2022\nHonda Odyssey,0H3011H,177.26,137,Gasoline,11.0755,Fan & Song 2022\nHonda Odyssey,0H3011H,19.1,137,Gasoline,2.2288,Fan & Song 2022\nHonda Odyssey,0H3011H,35.32,137,Gasoline,2.4602,Fan & Song 2022\nHonda Odyssey,0H3011H,32.36,137,Gasoline,2.5351,Fan & Song 2022\nHonda Odyssey,0H3011H,43.97,137,Gasoline,3.7978,Fan & Song 2022\nHonda Odyssey,0H3011H,14.89,137,Gasoline,1.9137,Fan & Song 2022\nHonda Odyssey,0H3011H,24.6,137,Gasoline,2.1703,Fan & Song 2022\nHonda Odyssey,0H3011H,32.81,137,Gasoline,3.2096,Fan & Song 2022\nHonda Odyssey,0H3011H,50.1,137,Gasoline,3.3945,Fan & Song 2022\nHonda Odyssey,0H3011H,12.84,137,Gasoline,1.4362,Fan & Song 2022\nHonda Odyssey,0H3011H,4.21,137,Gasoline,0.4241,Fan & Song 2022\nHonda Odyssey,0H3011H,46.8,137,Gasoline,3.9078,Fan & Song 2022\nHonda Odyssey,0H3011H,4.65,137,Gasoline,0.64,Fan & Song 2022\nHonda Odyssey,0H3011H,39.9,137,Gasoline,3.1936,Fan & Song 2022\nHonda Odyssey,0H3011H,146.32,137,Gasoline,12.8464,Fan & Song 2022\nHonda Odyssey,0H3011H,228.19,137,Gasoline,18.0511,Fan & Song 2022\nHonda Odyssey,0H3011H,387.86,137,Gasoline,31.7825,Fan & Song 2022\nHonda Odyssey,0H3011H,375.6,137,Gasoline,29.2246,Fan & Song 2022\nHonda Odyssey,0H3011H,2.55,137,Gasoline,0.3441,Fan & Song 2022\nHonda Odyssey,0H3011H,1.58,137,Gasoline,0.2587,Fan & Song 2022\nHonda Odyssey,0H3011H,106.69,137,Gasoline,9.4983,Fan & Song 2022\nHonda Odyssey,0H3011H,15.52,137,Gasoline,1.312,Fan & Song 2022\nHonda Odyssey,0H3011H,164.95,137,Gasoline,11.015,Fan & Song 2022\nHonda Odyssey,0H3011H,39.03,137,Gasoline,4.576,Fan & Song 2022\nHonda Odyssey,0H3011H,69.59,137,Gasoline,4.9364,Fan & Song 2022\nHonda Odyssey,0H3011H,19.09,137,Gasoline,2.2277,Fan & Song 2022\nHonda Odyssey,0H3011H,105.39,137,Gasoline,7.1858,Fan & Song 2022\nHonda Odyssey,0H3011H,278.25,137,Gasoline,19.5034,Fan & Song 2022\nHonda Odyssey,0H3011H,41.1,137,Gasoline,3.015,Fan & Song 2022\nHonda Odyssey,0H3011H,26.93,137,Gasoline,2.0176,Fan & Song 2022\nHonda Odyssey,0H3011H,145.62,137,Gasoline,11.7365,Fan & Song 2022\nHonda Odyssey,0H3011H,102.21,137,Gasoline,8.304,Fan & Song 2022\nHonda Odyssey,0H3011H,127.39,137,Gasoline,9.2439,Fan & Song 2022\nHonda Odyssey,0H3011H,63.95,137,Gasoline,4.8904,Fan & Song 2022\nHonda Odyssey,0H3011H,21.14,137,Gasoline,1.8045,Fan & Song 2022\nHonda Odyssey,0H3011H,5.32,137,Gasoline,0.7971,Fan & Song 2022\nHonda Odyssey,0H3011H,22.51,137,Gasoline,1.8497,Fan & Song 2022\nHonda Odyssey,0H3011H,17.65,137,Gasoline,1.9897,Fan & Song 2022\nHonda Odyssey,0H3011H,3.24,137,Gasoline,0.5189,Fan & Song 2022\nHonda Odyssey,0H3011H,27.5,137,Gasoline,2.0988,Fan & Song 2022\nHonda Odyssey,0H3011H,18.89,137,Gasoline,1.4778,Fan & Song 2022\nHonda Odyssey,0H3011H,3.16,137,Gasoline,0.3355,Fan & Song 2022\nHonda Odyssey,0H3011H,147.29,137,Gasoline,11.0569,Fan & Song 2022\nHonda Odyssey,0H3011H,23.95,137,Gasoline,1.8317,Fan & Song 2022\nHonda Odyssey,0H3011H,110.52,137,Gasoline,7.4563,Fan & Song 2022\nHonda Odyssey,0H3011H,54.02,137,Gasoline,4.8191,Fan & Song 2022\nHonda Odyssey,0H3011H,2.02,137,Gasoline,0.2389,Fan & Song 2022\nHonda Odyssey,0H3011H,102.49,137,Gasoline,8.3056,Fan & Song 2022\nHonda Odyssey,0H3011H,144.61,137,Gasoline,10.0997,Fan & Song 2022\nHonda Odyssey,0H3011H,16.92,137,Gasoline,1.4613,Fan & Song 2022\nHonda Odyssey,0H3011H,198.58,137,Gasoline,14.7885,Fan & Song 2022\nHonda Odyssey,0H3011H,56.0,137,Gasoline,4.2483,Fan & Song 2022\nHonda Odyssey,0H3011H,401.81,137,Gasoline,31.8096,Fan & Song 2022\nHonda Odyssey,0H3011H,4.74,137,Gasoline,0.4409,Fan & Song 2022\nHonda Odyssey,0H3011H,7.48,137,Gasoline,1.1359,Fan & Song 2022\nHonda Odyssey,0H3011H,471.86,137,Gasoline,39.3466,Fan & Song 2022\nHonda Odyssey,0H3011H,249.65,137,Gasoline,18.1018,Fan & Song 2022\nHonda Odyssey,0H3011H,510.69,137,Gasoline,37.7557,Fan & Song 2022\nHonda Odyssey,0H3011H,151.43,137,Gasoline,10.8698,Fan & Song 2022\nHonda Odyssey,0H3011H,24.85,137,Gasoline,1.9934,Fan & Song 2022\nHonda Odyssey,0H3011H,12.84,137,Gasoline,1.2294,Fan & Song 2022\nHonda Odyssey,0H3011H,12.43,137,Gasoline,1.45,Fan & Song 2022\nHonda Odyssey,0H3011H,29.53,137,Gasoline,1.9968,Fan & Song 2022\nHonda Odyssey,0H3011H,24.21,137,Gasoline,2.0801,Fan & Song 2022\nHonda Odyssey,0H3011H,4.47,137,Gasoline,0.5445,Fan & Song 2022\nHonda Odyssey,0H3011H,90.51,137,Gasoline,8.3325,Fan & Song 2022\nHonda Odyssey,0H3011H,4.16,137,Gasoline,0.4167,Fan & Song 2022\nHonda Odyssey,0H3011H,36.29,137,Gasoline,2.1977,Fan & Song 2022\nHonda Odyssey,0H3011H,60.35,137,Gasoline,4.0113,Fan & Song 2022\nHonda Odyssey,0H3011H,6.5,137,Gasoline,0.7689,Fan & Song 2022\nHonda Odyssey,0H3011H,4.84,137,Gasoline,0.7339,Fan & Song 2022\nHonda Odyssey,0H3011H,2.88,137,Gasoline,0.5038,Fan & Song 2022\nHonda Odyssey,0H3011H,38.12,137,Gasoline,3.0362,Fan & Song 2022\nHonda Odyssey,0H3011H,28.14,137,Gasoline,2.6443,Fan & Song 2022\nHonda Odyssey,0H3011H,3.92,137,Gasoline,0.479,Fan & Song 2022\nHonda Odyssey,0H3011H,118.95,137,Gasoline,12.4106,Fan & Song 2022\nHonda Odyssey,0H3011H,49.75,137,Gasoline,3.7825,Fan & Song 2022\nHonda Odyssey,0H3011H,1.55,137,Gasoline,0.1406,Fan & Song 2022\nHonda Odyssey,0H3011H,5.02,137,Gasoline,0.5607,Fan & Song 2022\nHonda Odyssey,0H3011H,36.34,137,Gasoline,2.9417,Fan & Song 2022\nHonda Odyssey,0H3011H,5.33,137,Gasoline,0.6385,Fan & Song 2022\nHonda Odyssey,0H3011H,48.09,137,Gasoline,5.0461,Fan & Song 2022\nHonda Odyssey,0H3011H,5.14,137,Gasoline,0.5977,Fan & Song 2022\nHonda Odyssey,0H3011H,15.35,137,Gasoline,1.392,Fan & Song 2022\nHonda Odyssey,0H3011H,66.92,137,Gasoline,6.7792,Fan & Song 2022\nHonda Odyssey,0H3011H,23.95,137,Gasoline,2.6881,Fan & Song 2022\nHonda Odyssey,0H3011H,35.15,137,Gasoline,3.3441,Fan & Song 2022\nHonda Odyssey,0H3011H,54.18,137,Gasoline,3.9436,Fan & Song 2022\nHonda Odyssey,0H3011H,6.96,137,Gasoline,1.0262,Fan & Song 2022\nHonda Odyssey,0H3011H,27.96,137,Gasoline,2.4413,Fan & Song 2022\nHonda Odyssey,0H3011H,103.28,137,Gasoline,10.2664,Fan & Song 2022\nHonda Odyssey,0H3011H,49.73,137,Gasoline,3.8918,Fan & Song 2022\nHonda Odyssey,0H3011H,34.6,137,Gasoline,2.9305,Fan & Song 2022\nHonda Odyssey,0H3011H,9.03,137,Gasoline,1.1912,Fan & Song 2022\nHonda Odyssey,0H3011H,62.82,137,Gasoline,5.9701,Fan & Song 2022\nHonda Odyssey,0H3011H,7.53,137,Gasoline,0.7231,Fan & Song 2022\nHonda Odyssey,0H3011H,5.74,137,Gasoline,0.5373,Fan & Song 2022\nHonda Odyssey,0H3011H,119.22,137,Gasoline,9.672,Fan & Song 2022\nHonda Odyssey,0H3011H,27.2,137,Gasoline,2.4428,Fan & Song 2022\nHonda Odyssey,0H3011H,6.7,137,Gasoline,0.5859,Fan & Song 2022\nHonda Odyssey,0H3011H,12.25,137,Gasoline,1.0563,Fan & Song 2022\nHonda Odyssey,0H3011H,1.64,137,Gasoline,0.2352,Fan & Song 2022\nHonda Odyssey,0H3011H,70.22,137,Gasoline,5.9948,Fan & Song 2022\nHonda Odyssey,0H3011H,61.73,137,Gasoline,5.334,Fan & Song 2022\nHonda Odyssey,0H3011H,29.17,137,Gasoline,2.1345,Fan & Song 2022\nHonda Odyssey,0H3011H,347.78,137,Gasoline,26.6494,Fan & Song 2022\nHonda Odyssey,0H3011H,75.35,137,Gasoline,5.7664,Fan & Song 2022\nHonda Odyssey,0H3011H,4.39,137,Gasoline,0.431,Fan & Song 2022\nHonda Odyssey,0H3011H,14.82,137,Gasoline,1.9848,Fan & Song 2022\nHonda Odyssey,0H3011H,2.87,137,Gasoline,0.3699,Fan & Song 2022\nHonda Odyssey,0H3011H,7.34,137,Gasoline,0.75,Fan & Song 2022\nHonda Odyssey,0H3011H,35.82,137,Gasoline,5.2277,Fan & Song 2022\nHonda Odyssey,0H3011H,4.33,137,Gasoline,0.4309,Fan & Song 2022\nHonda Odyssey,0H3011H,170.61,137,Gasoline,13.4806,Fan & Song 2022\nHonda Odyssey,0H3011H,318.99,137,Gasoline,23.5672,Fan & Song 2022\nHonda Odyssey,0H3011H,225.17,137,Gasoline,18.3513,Fan & Song 2022\nHonda Odyssey,0H3011H,13.08,137,Gasoline,1.2294,Fan & Song 2022\nHonda Odyssey,0H3011H,9.25,137,Gasoline,0.9641,Fan & Song 2022\nHonda Odyssey,0H3011H,25.13,137,Gasoline,2.13,Fan & Song 2022\nHonda Odyssey,0H3011H,32.82,137,Gasoline,3.2998,Fan & Song 2022\nHonda Odyssey,0H3011H,30.79,137,Gasoline,2.505,Fan & Song 2022\nHonda Odyssey,0H3011H,122.74,137,Gasoline,10.271,Fan & Song 2022\nHonda Odyssey,0H3011H,31.35,137,Gasoline,2.8968,Fan & Song 2022\nHonda Odyssey,0H3011H,4.06,137,Gasoline,0.5177,Fan & Song 2022\nHonda Odyssey,0H3011H,45.0,137,Gasoline,3.6242,Fan & Song 2022\nHonda Odyssey,0H3011H,1.66,137,Gasoline,0.2529,Fan & Song 2022\nHonda Odyssey,0H3011H,35.02,137,Gasoline,4.253,Fan & Song 2022\nHonda Odyssey,0H3011H,7.56,137,Gasoline,0.6306,Fan & Song 2022\nHonda Odyssey,0H3011H,44.26,137,Gasoline,3.0822,Fan & Song 2022\nHonda Odyssey,0H3011H,170.61,137,Gasoline,10.9203,Fan & Song 2022\nHonda Odyssey,0H3011H,18.9,137,Gasoline,2.0051,Fan & Song 2022\nHonda Odyssey,0H3011H,62.42,137,Gasoline,4.4755,Fan & Song 2022\nHonda Odyssey,0H3011H,117.59,137,Gasoline,7.9632,Fan & Song 2022\nHonda Odyssey,0H3011H,202.42,137,Gasoline,14.7986,Fan & Song 2022\nHonda Odyssey,0H3011H,48.29,137,Gasoline,3.9295,Fan & Song 2022\nHonda Odyssey,0H3011H,49.11,137,Gasoline,4.1633,Fan & Song 2022\nHonda Odyssey,0H3011H,33.7,137,Gasoline,2.3605,Fan & Song 2022\nHonda Odyssey,0H3011H,4.64,137,Gasoline,0.651,Fan & Song 2022\nHonda Odyssey,0H3011H,173.46,137,Gasoline,17.433,Fan & Song 2022\nHonda Odyssey,0H3011H,139.19,137,Gasoline,9.019,Fan & Song 2022\nHonda Odyssey,0H3011H,132.75,137,Gasoline,9.6542,Fan & Song 2022\nHonda Odyssey,0H3011H,10.94,137,Gasoline,1.1132,Fan & Song 2022\nHonda Odyssey,0H3011H,217.54,137,Gasoline,18.4449,Fan & Song 2022\nHonda Odyssey,0H3011H,206.8,137,Gasoline,14.654,Fan & Song 2022\nHonda Odyssey,0H3011H,136.8,137,Gasoline,9.3094,Fan & Song 2022\nHonda Odyssey,0H3011H,22.39,137,Gasoline,2.0206,Fan & Song 2022\nHonda Odyssey,0H3011H,15.53,137,Gasoline,1.3103,Fan & Song 2022\nHonda Odyssey,0H3011H,29.12,137,Gasoline,2.1709,Fan & Song 2022\nHonda Odyssey,0H3011H,42.73,137,Gasoline,3.7828,Fan & Song 2022\nHonda Odyssey,0H3011H,327.61,137,Gasoline,24.2619,Fan & Song 2022\nHonda Odyssey,0H3011H,42.37,137,Gasoline,3.9356,Fan & Song 2022\nHonda Odyssey,0H3011H,25.71,137,Gasoline,1.9331,Fan & Song 2022\nHonda Odyssey,0H3011H,57.51,137,Gasoline,5.7616,Fan & Song 2022\nHonda Odyssey,0H3011H,49.25,137,Gasoline,5.8316,Fan & Song 2022\nHonda Odyssey,0H3011H,22.51,137,Gasoline,2.2931,Fan & Song 2022\nHonda Odyssey,0H3011H,34.23,137,Gasoline,3.1891,Fan & Song 2022\nHonda Odyssey,0H3011H,23.05,137,Gasoline,2.338,Fan & Song 2022\nHonda Odyssey,0H3011H,8.5,137,Gasoline,1.1413,Fan & Song 2022\nHonda Odyssey,0H3011H,14.38,137,Gasoline,1.6369,Fan & Song 2022\nHonda Odyssey,0H3011H,5.41,137,Gasoline,0.5073,Fan & Song 2022\nHonda Odyssey,0H3011H,1.51,137,Gasoline,0.2285,Fan & Song 2022\nHonda Odyssey,0H3011H,54.71,137,Gasoline,6.573,Fan & Song 2022\nHonda Odyssey,0H3011H,15.77,137,Gasoline,1.4553,Fan & Song 2022\nHonda Odyssey,0H3011H,6.21,137,Gasoline,0.5109,Fan & Song 2022\nHonda Odyssey,0H3011H,8.27,137,Gasoline,1.172,Fan & Song 2022\nHonda Odyssey,0H3011H,4.47,137,Gasoline,0.4736,Fan & Song 2022\nHonda Odyssey,0H3011H,8.05,137,Gasoline,0.6031,Fan & Song 2022\nHonda Odyssey,0H3011H,189.87,137,Gasoline,15.5509,Fan & Song 2022\nHonda Odyssey,0H3011H,3.92,137,Gasoline,0.5579,Fan & Song 2022\nHonda Odyssey,0H3011H,7.33,137,Gasoline,0.4898,Fan & Song 2022\nHonda Odyssey,0H3011H,11.98,137,Gasoline,1.4939,Fan & Song 2022\nHonda Odyssey,0H3011H,4.7,137,Gasoline,0.372,Fan & Song 2022\nHonda Odyssey,0H3011H,46.01,137,Gasoline,3.6358,Fan & Song 2022\nHonda Odyssey,0H3011H,19.22,137,Gasoline,1.5783,Fan & Song 2022\nHonda Odyssey,0H3011H,35.97,137,Gasoline,2.7211,Fan & Song 2022\nHonda Odyssey,0H3011H,1.55,137,Gasoline,0.195,Fan & Song 2022\nHonda Odyssey,0H3011H,9.12,137,Gasoline,1.2936,Fan & Song 2022\nHonda Odyssey,0H3011H,6.11,137,Gasoline,1.0558,Fan & Song 2022\nHonda Odyssey,0H3011H,3.8,137,Gasoline,0.5419,Fan & Song 2022\nHonda Odyssey,0H3011H,35.23,137,Gasoline,3.5275,Fan & Song 2022\nHonda Odyssey,0H3011H,1.66,137,Gasoline,0.2108,Fan & Song 2022\nHonda Odyssey,0H3011H,6.22,137,Gasoline,0.7047,Fan & Song 2022\nHonda Odyssey,0H3011H,11.96,137,Gasoline,1.221,Fan & Song 2022\nHonda Odyssey,0H3011H,21.52,137,Gasoline,2.0698,Fan & Song 2022\nHonda Odyssey,0H3011H,4.1,137,Gasoline,0.5385,Fan & Song 2022\nHonda Odyssey,0H3011H,1.55,137,Gasoline,0.1786,Fan & Song 2022\nHonda Odyssey,0H3011H,51.05,137,Gasoline,3.6299,Fan & Song 2022\nHonda Odyssey,0H3011H,9.47,137,Gasoline,1.2253,Fan & Song 2022\nHonda Odyssey,0H3011H,56.16,137,Gasoline,5.0786,Fan & Song 2022\nHonda Odyssey,0H3011H,64.94,137,Gasoline,4.7441,Fan & Song 2022\nHonda Odyssey,0H3011H,21.6,137,Gasoline,2.0979,Fan & Song 2022\nHonda Odyssey,0H3011H,7.11,137,Gasoline,0.623,Fan & Song 2022\nHonda Odyssey,0H3011H,2.87,137,Gasoline,0.4226,Fan & Song 2022\nHonda Odyssey,0H3011H,37.48,137,Gasoline,2.8286,Fan & Song 2022\nHonda Odyssey,0H3011H,2.76,137,Gasoline,0.3344,Fan & Song 2022\nHonda Odyssey,0H3011H,92.18,137,Gasoline,10.9088,Fan & Song 2022\nHonda Odyssey,0H3011H,3.79,137,Gasoline,0.4541,Fan & Song 2022\nHonda Odyssey,0H3011H,29.94,137,Gasoline,3.2481,Fan & Song 2022\nHonda Odyssey,0H3011H,131.1,137,Gasoline,11.6027,Fan & Song 2022\nHonda Odyssey,0H3011H,4.41,137,Gasoline,0.4425,Fan & Song 2022\nHonda Odyssey,0H3011H,19.93,137,Gasoline,1.7668,Fan & Song 2022\nHonda Odyssey,0H3011H,33.04,137,Gasoline,3.4476,Fan & Song 2022\nHonda Odyssey,0H3011H,12.36,137,Gasoline,1.0598,Fan & Song 2022\nHonda Odyssey,0H3011H,13.42,137,Gasoline,1.1299,Fan & Song 2022\nHonda Odyssey,0H3011H,120.07,137,Gasoline,9.3884,Fan & Song 2022\nHonda Odyssey,0H3011H,13.65,137,Gasoline,1.633,Fan & Song 2022\nHonda Odyssey,0H3011H,3.49,137,Gasoline,0.5008,Fan & Song 2022\nHonda Odyssey,0H3011H,149.31,137,Gasoline,11.6478,Fan & Song 2022\nHonda Odyssey,0H3011H,26.58,137,Gasoline,2.4725,Fan & Song 2022\nHonda Odyssey,0H3011H,111.09,137,Gasoline,8.5174,Fan & Song 2022\nHonda Odyssey,0H3011H,21.29,137,Gasoline,1.9382,Fan & Song 2022\nHonda Odyssey,0H3011H,10.2,137,Gasoline,0.9747,Fan & Song 2022\nHonda Odyssey,0H3011H,2.64,137,Gasoline,0.3635,Fan & Song 2022\nHonda Odyssey,0H3011H,51.46,137,Gasoline,3.3993,Fan & Song 2022\nHonda Odyssey,0H3011H,49.39,137,Gasoline,4.1624,Fan & Song 2022\nHonda Odyssey,0H3011H,27.74,137,Gasoline,2.9126,Fan & Song 2022\nHonda Odyssey,0H3011H,4.35,137,Gasoline,0.4776,Fan & Song 2022\nHonda Odyssey,0H3011H,173.54,137,Gasoline,13.6937,Fan & Song 2022\nHonda Odyssey,0H3011H,54.65,137,Gasoline,3.9787,Fan & Song 2022\nHonda Odyssey,0H3011H,33.53,137,Gasoline,2.8825,Fan & Song 2022\nHonda Odyssey,0H3011H,15.05,137,Gasoline,1.1834,Fan & Song 2022\nHonda Odyssey,0H3011H,191.39,137,Gasoline,15.2125,Fan & Song 2022\nHonda Odyssey,0H3011H,85.39,137,Gasoline,6.5157,Fan & Song 2022\nHonda Odyssey,0H3011H,47.12,137,Gasoline,3.607,Fan & Song 2022\nHonda Odyssey,0H3011H,14.55,137,Gasoline,1.0843,Fan & Song 2022\nHonda Odyssey,0H3011H,11.44,137,Gasoline,1.1026,Fan & Song 2022\nHonda Odyssey,0H3011H,49.37,137,Gasoline,3.5217,Fan & Song 2022\nHonda Odyssey,0H3011H,12.66,137,Gasoline,1.5368,Fan & Song 2022\nHonda Odyssey,0H3011H,36.67,137,Gasoline,3.0588,Fan & Song 2022\nHonda Odyssey,0H3011H,21.93,137,Gasoline,2.0809,Fan & Song 2022\nHonda Odyssey,0H3011H,5.05,137,Gasoline,0.5618,Fan & Song 2022\nHonda Odyssey,0H3011H,1.57,137,Gasoline,0.231,Fan & Song 2022\nHonda Odyssey,0H3011H,9.86,137,Gasoline,1.1915,Fan & Song 2022\nHonda Odyssey,0H3011H,6.97,137,Gasoline,0.6739,Fan & Song 2022\nHonda Odyssey,0H3011H,23.06,137,Gasoline,1.9514,Fan & Song 2022\nHonda Odyssey,0H3011H,117.61,137,Gasoline,9.4727,Fan & Song 2022\nHonda Odyssey,0H3011H,7.91,137,Gasoline,0.6955,Fan & Song 2022\nHonda Odyssey,0H3011H,4.05,137,Gasoline,0.6278,Fan & Song 2022\nHonda Odyssey,0H3011H,3.89,137,Gasoline,0.4304,Fan & Song 2022\nHonda Odyssey,0H3011H,28.4,137,Gasoline,2.1984,Fan & Song 2022\nHonda Odyssey,0H3011H,8.33,137,Gasoline,0.974,Fan & Song 2022\nHonda Odyssey,0H3011H,22.64,137,Gasoline,2.0879,Fan & Song 2022\nHonda Odyssey,0H3011H,119.45,137,Gasoline,10.7545,Fan & Song 2022\nHonda Odyssey,0H3011H,47.52,137,Gasoline,4.5298,Fan & Song 2022\nHonda Odyssey,0H3011H,2.76,137,Gasoline,0.3895,Fan & Song 2022\nHonda Odyssey,0H3011H,45.31,137,Gasoline,3.3896,Fan & Song 2022\nHonda Odyssey,0H3011H,24.45,137,Gasoline,1.7332,Fan & Song 2022\nHonda Odyssey,0H3011H,1.98,137,Gasoline,0.2446,Fan & Song 2022\nHonda Odyssey,0H3011H,30.35,137,Gasoline,2.6903,Fan & Song 2022\nHonda Odyssey,0H3011H,4.93,137,Gasoline,0.593,Fan & Song 2022\nHonda Odyssey,0H3011H,86.72,137,Gasoline,10.2923,Fan & Song 2022\nHonda Odyssey,0H3011H,3.87,137,Gasoline,0.5439,Fan & Song 2022\nHonda Odyssey,0H3011H,34.1,137,Gasoline,4.0851,Fan & Song 2022\nHonda Odyssey,0H3011H,41.6,137,Gasoline,3.9027,Fan & Song 2022\nHonda Odyssey,0H3011H,10.66,137,Gasoline,0.8979,Fan & Song 2022\nHonda Odyssey,0H3011H,18.94,137,Gasoline,2.0351,Fan & Song 2022\nHonda Odyssey,0H3011H,30.31,137,Gasoline,3.1714,Fan & Song 2022\nHonda Odyssey,0H3011H,2.19,137,Gasoline,0.226,Fan & Song 2022\nHonda Odyssey,0H3011H,102.8,137,Gasoline,6.705,Fan & Song 2022\nHonda Odyssey,0H3011H,104.05,137,Gasoline,7.2764,Fan & Song 2022\nHonda Odyssey,0H3011H,36.55,137,Gasoline,3.8491,Fan & Song 2022\nHonda Odyssey,0H3011H,27.44,137,Gasoline,2.3275,Fan & Song 2022\nHonda Odyssey,0H3011H,76.83,137,Gasoline,5.7303,Fan & Song 2022\nHonda Odyssey,0H3011H,43.69,137,Gasoline,4.3312,Fan & Song 2022\nHonda Odyssey,0H3011H,26.86,137,Gasoline,2.4257,Fan & Song 2022\nHonda Odyssey,0H3011H,11.48,137,Gasoline,1.1832,Fan & Song 2022\nHonda Odyssey,0H3011H,52.64,137,Gasoline,3.9003,Fan & Song 2022\nHonda Odyssey,0H3011H,37.82,137,Gasoline,3.459,Fan & Song 2022\nHonda Odyssey,0H3011H,22.24,137,Gasoline,2.9785,Fan & Song 2022\nHonda Odyssey,0H3011H,10.4,137,Gasoline,1.8049,Fan & Song 2022\nHonda Odyssey,0H3011H,11.86,137,Gasoline,1.4988,Fan & Song 2022\nHonda Odyssey,0H3011H,210.84,137,Gasoline,16.3089,Fan & Song 2022\nHonda Odyssey,0H3011H,126.9,137,Gasoline,10.1625,Fan & Song 2022\nHonda Odyssey,0H3011H,84.14,137,Gasoline,6.9591,Fan & Song 2022\nHonda Odyssey,0H3011H,5.98,137,Gasoline,0.6969,Fan & Song 2022\nHonda Odyssey,0H3011H,49.45,137,Gasoline,4.1623,Fan & Song 2022\nHonda Odyssey,0H3011H,242.97,137,Gasoline,18.2999,Fan & Song 2022\nHonda Odyssey,0H3011H,96.29,137,Gasoline,6.6158,Fan & Song 2022\nHonda Odyssey,0H3011H,134.64,137,Gasoline,10.1986,Fan & Song 2022\nHonda Odyssey,0H3011H,1.9,137,Gasoline,0.3094,Fan & Song 2022\nHonda Odyssey,0H3011H,21.11,137,Gasoline,2.3247,Fan & Song 2022\nHonda Odyssey,0H3011H,272.26,137,Gasoline,18.7677,Fan & Song 2022\nHonda Odyssey,0H3011H,129.73,137,Gasoline,9.7322,Fan & Song 2022\nHonda Odyssey,0H3011H,10.35,137,Gasoline,1.1611,Fan & Song 2022\nHonda Odyssey,0H3011H,154.43,137,Gasoline,12.0948,Fan & Song 2022\nHonda Odyssey,0H3011H,264.51,137,Gasoline,23.5779,Fan & Song 2022\nHonda Odyssey,0H3011H,3.77,137,Gasoline,0.4683,Fan & Song 2022\nHonda Odyssey,0H3011H,299.61,137,Gasoline,24.5604,Fan & Song 2022\nHonda Odyssey,0H3011H,215.21,137,Gasoline,15.1431,Fan & Song 2022\nHonda Odyssey,0H3011H,167.73,137,Gasoline,13.8579,Fan & Song 2022\nHonda Odyssey,0H3011H,156.01,137,Gasoline,11.8038,Fan & Song 2022\nHonda Odyssey,0H3011H,34.06,137,Gasoline,2.7329,Fan & Song 2022\nHonda Odyssey,0H3011H,10.07,137,Gasoline,1.0578,Fan & Song 2022\nHonda Odyssey,0H3011H,199.58,137,Gasoline,13.8031,Fan & Song 2022\nHonda Odyssey,0H3011H,273.28,137,Gasoline,18.6922,Fan & Song 2022\nHonda Odyssey,0H3011H,121.82,137,Gasoline,12.1205,Fan & Song 2022\nHonda Odyssey,0H3011H,21.07,137,Gasoline,2.3706,Fan & Song 2022\nHonda Odyssey,0H3011H,140.94,137,Gasoline,12.8862,Fan & Song 2022\nHonda Odyssey,0H3011H,3.48,137,Gasoline,0.4011,Fan & Song 2022\nHonda Odyssey,0H3011H,126.46,137,Gasoline,9.4583,Fan & Song 2022\nHonda Odyssey,0H3011H,158.51,137,Gasoline,10.1705,Fan & Song 2022\nHonda Odyssey,0H3011H,158.6,137,Gasoline,10.9635,Fan & Song 2022\nHonda Odyssey,0H3011H,32.89,137,Gasoline,2.0226,Fan & Song 2022\nHonda Odyssey,0H3011H,158.83,137,Gasoline,9.9076,Fan & Song 2022\nHonda Odyssey,0H3011H,118.99,137,Gasoline,8.9546,Fan & Song 2022\nHonda Odyssey,0H3011H,2.33,137,Gasoline,0.3023,Fan & Song 2022\nHonda Odyssey,0H3011H,306.48,137,Gasoline,24.4943,Fan & Song 2022\nHonda Odyssey,0H3011H,242.32,137,Gasoline,19.4884,Fan & Song 2022\nHonda Odyssey,0H3011H,8.95,137,Gasoline,1.0297,Fan & Song 2022\nHonda Odyssey,0H3011H,17.79,137,Gasoline,1.4085,Fan & Song 2022\nHonda Odyssey,0H3011H,107.01,137,Gasoline,8.0376,Fan & Song 2022\nHonda Odyssey,0H3011H,70.24,137,Gasoline,4.3126,Fan & Song 2022\nHonda Odyssey,0H3011H,12.71,137,Gasoline,1.1836,Fan & Song 2022\nHonda Odyssey,0H3011H,107.6,137,Gasoline,7.5791,Fan & Song 2022\nHonda Odyssey,0H3011H,6.24,137,Gasoline,0.6612,Fan & Song 2022\nHonda Odyssey,0H3011H,17.68,137,Gasoline,1.1441,Fan & Song 2022\nHonda Odyssey,0H3011H,251.26,137,Gasoline,16.8258,Fan & Song 2022\nHonda Odyssey,0H3011H,55.75,137,Gasoline,5.1451,Fan & Song 2022\nHonda Odyssey,0H3011H,215.71,137,Gasoline,16.9099,Fan & Song 2022\nHonda Odyssey,0H3011H,44.17,137,Gasoline,3.3927,Fan & Song 2022\nHonda Odyssey,0H3011H,5.74,137,Gasoline,0.5998,Fan & Song 2022\nHonda Odyssey,0H3011H,14.25,137,Gasoline,1.3857,Fan & Song 2022\nHonda Odyssey,0H3011H,100.48,137,Gasoline,7.2079,Fan & Song 2022\nHonda Odyssey,0H3011H,53.17,137,Gasoline,4.2488,Fan & Song 2022\nHonda Odyssey,0H3011H,103.09,137,Gasoline,7.1061,Fan & Song 2022\nHonda Odyssey,0H3011H,91.6,137,Gasoline,7.2747,Fan & Song 2022\nHonda Odyssey,0H3011H,98.7,137,Gasoline,8.7812,Fan & Song 2022\nHonda Odyssey,0H3011H,39.61,137,Gasoline,3.531,Fan & Song 2022\nHonda Odyssey,0H3011H,52.48,137,Gasoline,5.0056,Fan & Song 2022\nHonda Odyssey,0H3011H,150.81,137,Gasoline,13.5356,Fan & Song 2022\nHonda Odyssey,0H3011H,15.8,137,Gasoline,1.3026,Fan & Song 2022\nHonda Odyssey,0H3011H,28.89,137,Gasoline,2.4753,Fan & Song 2022\nHonda Odyssey,0H3011H,42.35,137,Gasoline,4.0763,Fan & Song 2022\nHonda Odyssey,0H3011H,102.56,137,Gasoline,8.2297,Fan & Song 2022\nHonda Odyssey,0H3011H,160.6,137,Gasoline,13.5357,Fan & Song 2022\nHonda Odyssey,0H3011H,1.94,137,Gasoline,0.2732,Fan & Song 2022\nHonda Odyssey,0H3011H,55.91,137,Gasoline,5.6608,Fan & Song 2022\nHonda Odyssey,0H3011H,5.6,137,Gasoline,0.6303,Fan & Song 2022\nHonda Odyssey,0H3011H,3.14,137,Gasoline,0.325,Fan & Song 2022\nHonda Odyssey,0H3011H,46.78,137,Gasoline,3.7886,Fan & Song 2022\nHonda Odyssey,0H3011H,12.06,137,Gasoline,1.1354,Fan & Song 2022\nHonda Odyssey,0H3011H,146.18,137,Gasoline,11.8633,Fan & Song 2022\nHonda Odyssey,0H3011H,207.01,137,Gasoline,18.7627,Fan & Song 2022\nHonda Odyssey,0H3011H,22.69,137,Gasoline,1.634,Fan & Song 2022\nHonda Odyssey,0H3011H,19.41,137,Gasoline,2.6853,Fan & Song 2022\nHonda Odyssey,0H3011H,3.07,137,Gasoline,0.3688,Fan & Song 2022\nHonda Odyssey,0H3011H,104.39,137,Gasoline,9.4377,Fan & Song 2022\nHonda Odyssey,0H3011H,4.84,137,Gasoline,0.4288,Fan & Song 2022\nHonda Odyssey,0H3011H,481.71,137,Gasoline,35.7891,Fan & Song 2022\nHonda Odyssey,0H3011H,13.18,137,Gasoline,1.9973,Fan & Song 2022\nHonda Odyssey,0H3011H,156.22,137,Gasoline,14.4919,Fan & Song 2022\nHonda Odyssey,0H3011H,28.05,137,Gasoline,3.014,Fan & Song 2022\nHonda Odyssey,0H3011H,42.4,137,Gasoline,3.0409,Fan & Song 2022\nHonda Odyssey,0H3011H,115.41,137,Gasoline,11.8808,Fan & Song 2022\nHonda Odyssey,0H3011H,2.24,137,Gasoline,0.2845,Fan & Song 2022\nHonda Odyssey,0H3011H,34.19,137,Gasoline,2.4499,Fan & Song 2022\nHonda Odyssey,0H3011H,6.02,137,Gasoline,0.6598,Fan & Song 2022\nHonda Odyssey,0H3011H,3.96,137,Gasoline,0.4521,Fan & Song 2022\nHonda Odyssey,0H3011H,13.68,137,Gasoline,1.2637,Fan & Song 2022\nHonda Odyssey,0H3011H,52.15,137,Gasoline,3.8874,Fan & Song 2022\nHonda Odyssey,0H3011H,26.33,137,Gasoline,2.5501,Fan & Song 2022\nHonda Odyssey,0H3011H,12.52,137,Gasoline,1.3439,Fan & Song 2022\nHonda Odyssey,0H3011H,6.07,137,Gasoline,0.6256,Fan & Song 2022\nHonda Odyssey,0H3011H,9.52,137,Gasoline,0.7833,Fan & Song 2022\nHonda Odyssey,0H3011H,41.91,137,Gasoline,3.7007,Fan & Song 2022\nHonda Odyssey,0H3011H,1.55,137,Gasoline,0.2282,Fan & Song 2022\nHonda Odyssey,0H3011H,33.98,137,Gasoline,3.1849,Fan & Song 2022\nHonda Odyssey,0H3011H,27.34,137,Gasoline,2.3689,Fan & Song 2022\nHonda Odyssey,0H3011H,155.92,137,Gasoline,11.9988,Fan & Song 2022\nHonda Odyssey,0H3011H,12.26,137,Gasoline,1.2013,Fan & Song 2022\nHonda Odyssey,0H3011H,2.95,137,Gasoline,0.4136,Fan & Song 2022\nHonda Odyssey,0H3011H,56.34,137,Gasoline,3.7631,Fan & Song 2022\nHonda Odyssey,0H3011H,9.67,137,Gasoline,0.9156,Fan & Song 2022\nHonda Odyssey,0H3011H,38.35,137,Gasoline,3.2388,Fan & Song 2022\nHonda Odyssey,0H3011H,18.14,137,Gasoline,1.9629,Fan & Song 2022\nHonda Odyssey,0H3011H,141.75,137,Gasoline,11.431,Fan & Song 2022\nHonda Odyssey,0H3011H,10.06,137,Gasoline,1.038,Fan & Song 2022\nHonda Odyssey,0H3011H,53.04,137,Gasoline,5.2008,Fan & Song 2022\nHonda Odyssey,0H3011H,39.38,137,Gasoline,3.4259,Fan & Song 2022\nHonda Odyssey,0H3011H,24.96,137,Gasoline,1.7064,Fan & Song 2022\nHonda Odyssey,0H3011H,7.17,137,Gasoline,1.2257,Fan & Song 2022\nHonda Odyssey,0H3011H,16.65,137,Gasoline,2.2491,Fan & Song 2022\nHonda Odyssey,0H3011H,5.1,137,Gasoline,0.52,Fan & Song 2022\nHonda Odyssey,0H3011H,108.72,137,Gasoline,6.9055,Fan & Song 2022\nHonda Odyssey,0H3011H,3.94,137,Gasoline,0.6331,Fan & Song 2022\nHonda Odyssey,0H3011H,64.7,137,Gasoline,4.7271,Fan & Song 2022\nHonda Odyssey,0H3011H,36.19,137,Gasoline,3.228,Fan & Song 2022\nHonda Odyssey,0H3011H,19.42,137,Gasoline,2.5222,Fan & Song 2022\nHonda Odyssey,0H3011H,15.96,137,Gasoline,1.5195,Fan & Song 2022\nHonda Odyssey,0H3011H,79.43,137,Gasoline,5.1025,Fan & Song 2022\nHonda Odyssey,0H3011H,121.67,137,Gasoline,8.4466,Fan & Song 2022\nHonda Odyssey,0H3011H,17.35,137,Gasoline,1.7156,Fan & Song 2022\nHonda Odyssey,0H3011H,4.51,137,Gasoline,0.4933,Fan & Song 2022\nHonda Odyssey,0H3011H,7.28,137,Gasoline,0.6995,Fan & Song 2022\nHonda Odyssey,0H3011H,42.71,137,Gasoline,3.4944,Fan & Song 2022\nHonda Odyssey,0H3011H,36.33,137,Gasoline,2.7759,Fan & Song 2022\nHonda Odyssey,0H3011H,11.92,137,Gasoline,1.4441,Fan & Song 2022\nHonda Odyssey,0H3011H,45.88,137,Gasoline,4.3513,Fan & Song 2022\nHonda Odyssey,0H3011H,35.96,137,Gasoline,3.2257,Fan & Song 2022\nHonda Odyssey,0H3011H,3.31,137,Gasoline,0.3204,Fan & Song 2022\nHonda Odyssey,0H3011H,15.53,137,Gasoline,1.216,Fan & Song 2022\nHonda Odyssey,0H3011H,2.19,137,Gasoline,0.2831,Fan & Song 2022\nHonda Odyssey,0H3011H,14.41,137,Gasoline,1.2389,Fan & Song 2022\nHonda Odyssey,0H3011H,3.77,137,Gasoline,0.3727,Fan & Song 2022\nHonda Odyssey,0H3011H,45.86,137,Gasoline,4.3397,Fan & Song 2022\nHonda Odyssey,0H3011H,136.73,137,Gasoline,11.7197,Fan & Song 2022\nHonda Odyssey,0H3011H,20.77,137,Gasoline,2.2365,Fan & Song 2022\nHonda Odyssey,0H3011H,32.41,137,Gasoline,2.3511,Fan & Song 2022\nHonda Odyssey,0H3011H,130.15,137,Gasoline,10.2483,Fan & Song 2022\nHonda Odyssey,0H3011H,185.41,137,Gasoline,15.1965,Fan & Song 2022\nHonda Odyssey,0H3011H,48.62,137,Gasoline,4.1221,Fan & Song 2022\nHonda Odyssey,0H3011H,50.79,137,Gasoline,4.7496,Fan & Song 2022\nHonda Odyssey,0H3011H,241.17,137,Gasoline,18.1769,Fan & Song 2022\nHonda Odyssey,0H3011H,7.57,137,Gasoline,0.8472,Fan & Song 2022\nHonda Odyssey,0H3011H,10.6,137,Gasoline,0.9883,Fan & Song 2022\nHonda Odyssey,0H3011H,128.0,137,Gasoline,9.8464,Fan & Song 2022\nHonda Odyssey,0H3011H,188.68,137,Gasoline,13.4929,Fan & Song 2022\nHonda Odyssey,0H3011H,46.76,137,Gasoline,4.2942,Fan & Song 2022\nHonda Odyssey,0H3011H,2.63,137,Gasoline,0.355,Fan & Song 2022\nHonda Odyssey,0H3011H,12.15,137,Gasoline,1.1969,Fan & Song 2022\nHonda Odyssey,0H3011H,76.79,137,Gasoline,6.3892,Fan & Song 2022\nHonda Odyssey,0H3011H,191.35,137,Gasoline,14.1175,Fan & Song 2022\nHonda Odyssey,0H3011H,276.52,137,Gasoline,21.4603,Fan & Song 2022\nHonda Odyssey,0H3011H,118.69,137,Gasoline,10.3087,Fan & Song 2022\nHonda Odyssey,0H3011H,134.46,137,Gasoline,8.6845,Fan & Song 2022\nHonda Odyssey,0H3011H,3.05,137,Gasoline,0.2283,Fan & Song 2022\nHonda Odyssey,0H3011H,147.44,137,Gasoline,9.5454,Fan & Song 2022\nHonda Odyssey,0H3011H,64.31,137,Gasoline,4.9862,Fan & Song 2022\nHonda Odyssey,0H3011H,315.01,137,Gasoline,23.4081,Fan & Song 2022\nHonda Odyssey,0H3011H,5.86,137,Gasoline,0.8248,Fan & Song 2022\nHonda Odyssey,0H3011H,56.05,137,Gasoline,3.8189,Fan & Song 2022\nHonda Odyssey,0H3011H,6.96,137,Gasoline,0.7881,Fan & Song 2022\nHonda Odyssey,0H3011H,191.12,137,Gasoline,13.3944,Fan & Song 2022\nHonda Odyssey,0H3011H,38.53,137,Gasoline,2.9625,Fan & Song 2022\nHonda Odyssey,0H3011H,104.48,137,Gasoline,6.3841,Fan & Song 2022\nHonda Odyssey,0H3011H,120.79,137,Gasoline,9.6755,Fan & Song 2022\nHonda Odyssey,0H3011H,122.79,137,Gasoline,9.9457,Fan & Song 2022\nHonda Odyssey,0H3011H,143.1,137,Gasoline,10.3224,Fan & Song 2022\nHonda Odyssey,0H3011H,50.55,137,Gasoline,3.5101,Fan & Song 2022\nHonda Odyssey,0H3011H,17.88,137,Gasoline,1.7466,Fan & Song 2022\nHonda Odyssey,0H3011H,17.56,137,Gasoline,2.0358,Fan & Song 2022\nHonda Odyssey,0H3011H,1.98,137,Gasoline,0.2749,Fan & Song 2022\nHonda Odyssey,0H3011H,265.97,137,Gasoline,19.1147,Fan & Song 2022\nHonda Odyssey,0H3011H,24.9,137,Gasoline,1.9942,Fan & Song 2022\nHonda Odyssey,0H3011H,156.66,137,Gasoline,13.4022,Fan & Song 2022\nHonda Odyssey,0H3011H,29.22,137,Gasoline,2.2755,Fan & Song 2022\nHonda Odyssey,0H3011H,314.89,137,Gasoline,23.9525,Fan & Song 2022\nHonda Odyssey,0H3011H,160.6,137,Gasoline,15.839,Fan & Song 2022\nHonda Odyssey,0H3011H,10.71,137,Gasoline,1.0355,Fan & Song 2022\nHonda Odyssey,0H3011H,100.46,137,Gasoline,8.3094,Fan & Song 2022\nHonda Odyssey,0H3011H,330.18,137,Gasoline,24.5485,Fan & Song 2022\nHonda Odyssey,0H3011H,253.37,137,Gasoline,17.939,Fan & Song 2022\nHonda Odyssey,0H3011H,186.39,137,Gasoline,14.1958,Fan & Song 2022\nHonda Odyssey,0H3011H,186.61,137,Gasoline,12.829,Fan & Song 2022\nHonda Odyssey,0H3011H,60.24,137,Gasoline,4.5929,Fan & Song 2022\nHonda Odyssey,0H3011H,162.25,137,Gasoline,12.1997,Fan & Song 2022\nHonda Odyssey,0H3011H,33.98,137,Gasoline,3.1789,Fan & Song 2022\nHonda Odyssey,0H3011H,6.0,137,Gasoline,0.6421,Fan & Song 2022\nHonda Odyssey,0H3011H,111.9,137,Gasoline,9.0315,Fan & Song 2022\nHonda Odyssey,0H3011H,271.29,137,Gasoline,19.2051,Fan & Song 2022\nHonda Odyssey,0H3011H,16.4,137,Gasoline,1.1608,Fan & Song 2022\nHonda Odyssey,0H3011H,198.33,137,Gasoline,15.0602,Fan & Song 2022\nHonda Odyssey,0H3011H,7.26,137,Gasoline,0.6065,Fan & Song 2022\nHonda Odyssey,0H3011H,180.67,137,Gasoline,13.2684,Fan & Song 2022\nHonda Odyssey,0H3011H,181.02,137,Gasoline,11.9241,Fan & Song 2022\nHonda Odyssey,0H3011H,285.46,137,Gasoline,22.591,Fan & Song 2022\nHonda Odyssey,0H3011H,102.31,137,Gasoline,7.9065,Fan & Song 2022\nHonda Odyssey,0H3011H,10.87,137,Gasoline,1.2529,Fan & Song 2022\nHonda Odyssey,0H3011H,42.34,137,Gasoline,4.4434,Fan & Song 2022\nHonda Odyssey,0H3011H,232.49,137,Gasoline,17.7823,Fan & Song 2022\nHonda Odyssey,0H3011H,3.36,137,Gasoline,0.3868,Fan & Song 2022\nHonda Odyssey,0H3011H,15.6,137,Gasoline,1.2794,Fan & Song 2022\nHonda Odyssey,0H3011H,69.05,137,Gasoline,5.4155,Fan & Song 2022\nHonda Odyssey,0H3011H,33.09,137,Gasoline,3.2246,Fan & Song 2022\nHonda Odyssey,0H3011H,8.08,137,Gasoline,0.8424,Fan & Song 2022\nHonda Odyssey,0H3011H,21.07,137,Gasoline,2.0276,Fan & Song 2022\nHonda Odyssey,0H3011H,2.37,137,Gasoline,0.2741,Fan & Song 2022\nHonda Odyssey,0H3011H,21.02,137,Gasoline,1.7757,Fan & Song 2022\nHonda Odyssey,0H3011H,25.96,137,Gasoline,2.3485,Fan & Song 2022\nHonda Odyssey,0H3011H,38.77,137,Gasoline,4.2273,Fan & Song 2022\nHonda Odyssey,0H3011H,27.13,137,Gasoline,3.8131,Fan & Song 2022\nHonda Odyssey,0H3011H,306.69,137,Gasoline,23.5438,Fan & Song 2022\nHonda Odyssey,0H3011H,101.24,137,Gasoline,7.5283,Fan & Song 2022\nHonda Odyssey,0H3011H,254.41,137,Gasoline,18.189,Fan & Song 2022\nHonda Odyssey,0H3011H,185.94,137,Gasoline,13.5548,Fan & Song 2022\nHonda Odyssey,0H3011H,57.93,137,Gasoline,4.6843,Fan & Song 2022\nHonda Odyssey,0H3011H,82.33,137,Gasoline,5.8891,Fan & Song 2022\nHonda Odyssey,0H3011H,4.24,137,Gasoline,0.553,Fan & Song 2022\nHonda Odyssey,0H3011H,8.83,137,Gasoline,1.1329,Fan & Song 2022\nHonda Odyssey,0H3011H,11.73,137,Gasoline,1.2282,Fan & Song 2022\nHonda Odyssey,0H3011H,7.02,137,Gasoline,0.6867,Fan & Song 2022\nHonda Odyssey,0H3011H,23.27,137,Gasoline,1.8119,Fan & Song 2022\nHonda Odyssey,0H3011H,2.11,137,Gasoline,0.2099,Fan & Song 2022\nHonda Odyssey,0H3011H,15.81,137,Gasoline,1.3392,Fan & Song 2022\nHonda Odyssey,0H3011H,2.77,137,Gasoline,0.3789,Fan & Song 2022\nHonda Odyssey,0H3011H,50.82,137,Gasoline,4.0228,Fan & Song 2022\nHonda Odyssey,0H3011H,27.25,137,Gasoline,2.2165,Fan & Song 2022\nHonda Odyssey,0H3011H,8.78,137,Gasoline,1.1586,Fan & Song 2022\nHonda Odyssey,0H3011H,148.04,137,Gasoline,9.1034,Fan & Song 2022\nHonda Odyssey,0H3011H,10.34,137,Gasoline,1.221,Fan & Song 2022\nHonda Odyssey,0H3011H,15.83,137,Gasoline,1.5654,Fan & Song 2022\nHonda Odyssey,0H3011H,1.57,137,Gasoline,0.1948,Fan & Song 2022\nHonda Odyssey,0H3011H,20.73,137,Gasoline,2.3836,Fan & Song 2022\nHonda Odyssey,0H3011H,33.49,137,Gasoline,2.9789,Fan & Song 2022\nHonda Odyssey,0H3011H,121.21,137,Gasoline,8.7759,Fan & Song 2022\nHonda Odyssey,0H3011H,1.78,137,Gasoline,0.3241,Fan & Song 2022\nHonda Odyssey,0H3011H,255.74,137,Gasoline,19.9507,Fan & Song 2022\nHonda Odyssey,0H3011H,2.9,137,Gasoline,0.4028,Fan & Song 2022\nHonda Odyssey,0H3011H,33.78,137,Gasoline,3.3893,Fan & Song 2022\nHonda Odyssey,0H3011H,41.1,137,Gasoline,3.313,Fan & Song 2022\nHonda Odyssey,0H3011H,104.69,137,Gasoline,8.6343,Fan & Song 2022\nHonda Odyssey,0H3011H,17.95,137,Gasoline,1.6987,Fan & Song 2022\nHonda Odyssey,0H3011H,4.28,137,Gasoline,0.6236,Fan & Song 2022\nHonda Odyssey,0H3011H,9.55,137,Gasoline,1.1815,Fan & Song 2022\nHonda Odyssey,0H3011H,2.58,137,Gasoline,0.4181,Fan & Song 2022\nHonda Odyssey,0H3011H,141.03,137,Gasoline,11.16,Fan & Song 2022\nHonda Odyssey,0H3011H,10.1,137,Gasoline,0.9193,Fan & Song 2022\nHonda Odyssey,0H3011H,29.5,137,Gasoline,2.7666,Fan & Song 2022\nHonda Odyssey,0H3011H,61.02,137,Gasoline,6.3266,Fan & Song 2022\nHonda Odyssey,0H3011H,178.05,137,Gasoline,14.7664,Fan & Song 2022\nHonda Odyssey,0H3011H,2.55,137,Gasoline,0.3937,Fan & Song 2022\nHonda Odyssey,0H3011H,5.52,137,Gasoline,0.5656,Fan & Song 2022\nHonda Odyssey,0H3011H,11.54,137,Gasoline,1.7085,Fan & Song 2022\nHonda Odyssey,0H3011H,13.48,137,Gasoline,1.213,Fan & Song 2022\nHonda Odyssey,0H3011H,6.79,137,Gasoline,0.7328,Fan & Song 2022\nHonda Odyssey,0H3011H,3.76,137,Gasoline,0.3798,Fan & Song 2022\nHonda Odyssey,0H3011H,29.02,137,Gasoline,2.6763,Fan & Song 2022\nHonda Odyssey,0H3011H,31.97,137,Gasoline,2.7197,Fan & Song 2022\nHonda Odyssey,0H3011H,120.91,137,Gasoline,9.3592,Fan & Song 2022\nHonda Odyssey,0H3011H,21.82,137,Gasoline,2.3164,Fan & Song 2022\nHonda Odyssey,0H3011H,9.18,137,Gasoline,0.9946,Fan & Song 2022\nHonda Odyssey,0H3011H,1.59,137,Gasoline,0.2099,Fan & Song 2022\nHonda Odyssey,0H3011H,2.06,137,Gasoline,0.387,Fan & Song 2022\nHonda Odyssey,0H3011H,167.57,137,Gasoline,10.5295,Fan & Song 2022\nHonda Odyssey,0H3011H,141.42,137,Gasoline,11.0597,Fan & Song 2022\nHonda Odyssey,0H3011H,1.79,137,Gasoline,0.3713,Fan & Song 2022\nHonda Odyssey,0H3011H,4.09,137,Gasoline,0.6901,Fan & Song 2022\nHonda Odyssey,0H3011H,10.74,137,Gasoline,1.3025,Fan & Song 2022\nHonda Odyssey,0H3011H,2.75,137,Gasoline,0.3162,Fan & Song 2022\nHonda Odyssey,0H3011H,28.72,137,Gasoline,2.5226,Fan & Song 2022\nHonda Odyssey,0H3011H,235.19,137,Gasoline,18.6512,Fan & Song 2022\nHonda Odyssey,0H3011H,332.7,137,Gasoline,24.8299,Fan & Song 2022\nHonda Odyssey,0H3011H,61.39,137,Gasoline,5.7404,Fan & Song 2022\nHonda Odyssey,0H3011H,14.13,137,Gasoline,1.3588,Fan & Song 2022\nHonda Odyssey,0H3011H,4.24,137,Gasoline,0.62,Fan & Song 2022\nHonda Odyssey,0H3011H,2.29,137,Gasoline,0.2552,Fan & Song 2022\nHonda Odyssey,0H3011H,1.61,137,Gasoline,0.1965,Fan & Song 2022\nHonda Odyssey,0H3011H,36.19,137,Gasoline,4.1677,Fan & Song 2022\nHonda Odyssey,0H3011H,55.03,137,Gasoline,4.4259,Fan & Song 2022\nHonda Odyssey,0H3011H,103.09,137,Gasoline,9.7298,Fan & Song 2022\nHonda Odyssey,0H3011H,7.82,137,Gasoline,0.9367,Fan & Song 2022\nHonda Odyssey,0H3011H,56.4,137,Gasoline,5.2403,Fan & Song 2022\nHonda Odyssey,0H3011H,34.93,137,Gasoline,3.8579,Fan & Song 2022\nHonda Odyssey,0H3011H,10.48,137,Gasoline,1.148,Fan & Song 2022\nHonda Odyssey,0H3011H,158.07,137,Gasoline,16.445,Fan & Song 2022\nHonda Odyssey,0H3011H,30.85,137,Gasoline,2.5701,Fan & Song 2022\nHonda Odyssey,0H3011H,108.6,137,Gasoline,11.3489,Fan & Song 2022\nHonda Odyssey,0H3011H,6.14,137,Gasoline,0.9349,Fan & Song 2022\nHonda Odyssey,0H3011H,47.14,137,Gasoline,3.7276,Fan & Song 2022\nHonda Odyssey,0H3011H,22.36,137,Gasoline,2.5111,Fan & Song 2022\nHonda Odyssey,0H3011H,23.46,137,Gasoline,2.0023,Fan & Song 2022\nHonda Odyssey,0H3011H,54.96,137,Gasoline,3.9928,Fan & Song 2022\nHonda Odyssey,0H3011H,23.52,137,Gasoline,1.9083,Fan & Song 2022\nHonda Odyssey,0H3011H,13.9,137,Gasoline,1.3678,Fan & Song 2022\nHonda Odyssey,0H3011H,2.86,137,Gasoline,0.4319,Fan & Song 2022\nHonda Odyssey,0H3011H,1.81,137,Gasoline,0.2224,Fan & Song 2022\nHonda Odyssey,0H3011H,10.37,137,Gasoline,1.5674,Fan & Song 2022\nHonda Odyssey,0H3011H,42.36,137,Gasoline,3.9075,Fan & Song 2022\nHonda Odyssey,0H3011H,14.92,137,Gasoline,2.3143,Fan & Song 2022\nHonda Odyssey,0H3011H,3.5,137,Gasoline,0.3241,Fan & Song 2022\nHonda Odyssey,0H3011H,34.72,137,Gasoline,2.9565,Fan & Song 2022\nHonda Odyssey,0H3011H,6.6,137,Gasoline,0.927,Fan & Song 2022\nHonda Odyssey,0H3011H,12.9,137,Gasoline,1.5642,Fan & Song 2022\nHonda Odyssey,0H3011H,7.64,137,Gasoline,0.753,Fan & Song 2022\nHonda Odyssey,0H3011H,16.31,137,Gasoline,1.2783,Fan & Song 2022\nHonda Odyssey,0H3011H,17.86,137,Gasoline,1.7082,Fan & Song 2022\nHyundai ix35,3J1020O,22.31,121,Gasoline,2.2556,Fan & Song 2022\nHyundai ix35,3J1020O,6.61,121,Gasoline,0.7198,Fan & Song 2022\nHyundai ix35,3J1020O,51.82,121,Gasoline,5.0061,Fan & Song 2022\nHyundai ix35,3J1020O,47.98,121,Gasoline,4.7109,Fan & Song 2022\nHyundai ix35,3J1020O,4.3,121,Gasoline,0.7716,Fan & Song 2022\nHyundai ix35,3J1020O,2.71,121,Gasoline,0.4188,Fan & Song 2022\nHyundai ix35,3J1020O,3.02,121,Gasoline,0.5953,Fan & Song 2022\nHyundai ix35,3J1020O,4.72,121,Gasoline,0.5621,Fan & Song 2022\nHyundai ix35,3J1020O,15.61,121,Gasoline,1.8238,Fan & Song 2022\nHyundai ix35,3J1020O,11.94,121,Gasoline,1.4322,Fan & Song 2022\nHyundai ix35,3J1020O,18.05,121,Gasoline,1.7655,Fan & Song 2022\nHyundai ix35,3J1020O,18.77,121,Gasoline,1.4687,Fan & Song 2022\nHyundai ix35,3J1020O,4.17,121,Gasoline,0.6639,Fan & Song 2022\nHyundai ix35,3J1020O,12.21,121,Gasoline,1.3923,Fan & Song 2022\nHyundai ix35,3J1020O,12.25,121,Gasoline,1.2135,Fan & Song 2022\nHyundai ix35,3J1020O,27.71,121,Gasoline,3.2088,Fan & Song 2022\nHyundai ix35,3J1020O,1.76,121,Gasoline,0.2993,Fan & Song 2022\nHyundai ix35,3J1020O,2.76,121,Gasoline,0.318,Fan & Song 2022\nHyundai ix35,3J1020O,24.67,121,Gasoline,2.2686,Fan & Song 2022\nHyundai ix35,3J1020O,9.03,121,Gasoline,1.1173,Fan & Song 2022\nHyundai ix35,3J1020O,13.53,121,Gasoline,1.1405,Fan & Song 2022\nHyundai ix35,3J1020O,46.76,121,Gasoline,3.669,Fan & Song 2022\nHyundai ix35,3J1020O,7.41,121,Gasoline,0.9047,Fan & Song 2022\nHyundai ix35,3J1020O,8.72,121,Gasoline,0.8993,Fan & Song 2022\nHyundai ix35,3J1020O,5.6,121,Gasoline,0.5355,Fan & Song 2022\nHyundai ix35,3J1020O,68.9,121,Gasoline,6.1093,Fan & Song 2022\nHyundai ix35,3J1020O,15.04,121,Gasoline,1.4554,Fan & Song 2022\nHyundai ix35,3J1020O,59.13,121,Gasoline,4.625,Fan & Song 2022\nHyundai ix35,3J1020O,4.85,121,Gasoline,0.521,Fan & Song 2022\nHyundai ix35,3J1020O,13.23,121,Gasoline,1.701,Fan & Song 2022\nHyundai ix35,3J1020O,37.37,121,Gasoline,2.8855,Fan & Song 2022\nHyundai ix35,3J1020O,5.32,121,Gasoline,0.6846,Fan & Song 2022\nHyundai ix35,3J1020O,6.46,121,Gasoline,0.9825,Fan & Song 2022\nHyundai ix35,3J1020O,13.6,121,Gasoline,1.0798,Fan & Song 2022\nHyundai ix35,3J1020O,238.18,121,Gasoline,19.6107,Fan & Song 2022\nHyundai ix35,3J1020O,4.17,121,Gasoline,0.6429,Fan & Song 2022\nHyundai ix35,3J1020O,612.58,121,Gasoline,54.3232,Fan & Song 2022\nHyundai ix35,3J1020O,2.21,121,Gasoline,0.3797,Fan & Song 2022\nHyundai ix35,3J1020O,9.99,121,Gasoline,1.1228,Fan & Song 2022\nHyundai ix35,3J1020O,1.86,121,Gasoline,0.378,Fan & Song 2022\nHyundai ix35,3J1020O,94.46,121,Gasoline,9.3972,Fan & Song 2022\nHyundai ix35,3J1020O,39.28,121,Gasoline,3.5677,Fan & Song 2022\nHyundai ix35,3J1020O,17.08,121,Gasoline,1.5554,Fan & Song 2022\nHyundai ix35,3J1020O,3.0,121,Gasoline,0.3599,Fan & Song 2022\nHyundai ix35,3J1020O,34.05,121,Gasoline,2.816,Fan & Song 2022\nHyundai ix35,3J1020O,1.9,121,Gasoline,0.2403,Fan & Song 2022\nHyundai ix35,3J1020O,17.38,121,Gasoline,1.545,Fan & Song 2022\nHyundai ix35,3J1020O,63.14,121,Gasoline,4.839,Fan & Song 2022\nHyundai ix35,3J1020O,45.69,121,Gasoline,3.9351,Fan & Song 2022\nHyundai ix35,3J1020O,27.19,121,Gasoline,2.895,Fan & Song 2022\nHyundai ix35,3J1020O,46.73,121,Gasoline,4.7861,Fan & Song 2022\nHyundai ix35,3J1020O,227.83,121,Gasoline,21.4878,Fan & Song 2022\nHyundai ix35,3J1020O,151.65,121,Gasoline,15.1964,Fan & Song 2022\nHyundai ix35,3J1020O,124.24,121,Gasoline,11.8014,Fan & Song 2022\nHyundai ix35,3J1020O,57.33,121,Gasoline,5.8481,Fan & Song 2022\nHyundai ix35,3J1020O,114.7,121,Gasoline,10.945,Fan & Song 2022\nHyundai ix35,3J1020O,232.36,121,Gasoline,15.4063,Fan & Song 2022\nHyundai ix35,3J1020O,7.68,121,Gasoline,0.8368,Fan & Song 2022\nHyundai ix35,3J1020O,124.38,121,Gasoline,11.1704,Fan & Song 2022\nHyundai ix35,3J1020O,23.8,121,Gasoline,1.9506,Fan & Song 2022\nHyundai ix35,3J1020O,133.65,121,Gasoline,9.6868,Fan & Song 2022\nHyundai ix35,3J1020O,87.73,121,Gasoline,8.1607,Fan & Song 2022\nHyundai ix35,3J1020O,14.78,121,Gasoline,1.5056,Fan & Song 2022\nHyundai ix35,3J1020O,8.17,121,Gasoline,0.8678,Fan & Song 2022\nHyundai ix35,3J1020O,6.48,121,Gasoline,0.6593,Fan & Song 2022\nHyundai ix35,3J1020O,6.88,121,Gasoline,0.898,Fan & Song 2022\nHyundai ix35,3J1020O,612.65,121,Gasoline,52.1834,Fan & Song 2022\nHyundai ix35,3J1020O,156.66,121,Gasoline,13.9236,Fan & Song 2022\nHyundai ix35,3J1020O,106.69,121,Gasoline,9.2914,Fan & Song 2022\nHyundai ix35,3J1020O,39.58,121,Gasoline,2.8921,Fan & Song 2022\nHyundai ix35,3J1020O,11.04,121,Gasoline,1.4995,Fan & Song 2022\nHyundai ix35,3J1020O,12.01,121,Gasoline,1.3473,Fan & Song 2022\nHyundai ix35,3J1020O,9.57,121,Gasoline,1.3864,Fan & Song 2022\nHyundai ix35,3J1020O,13.17,121,Gasoline,1.5398,Fan & Song 2022\nHyundai ix35,3J1020O,359.63,121,Gasoline,34.2248,Fan & Song 2022\nHyundai ix35,3J1020O,231.07,121,Gasoline,20.2596,Fan & Song 2022\nHyundai ix35,3J1020O,14.8,121,Gasoline,1.3858,Fan & Song 2022\nHyundai ix35,3J1020O,31.87,121,Gasoline,2.5481,Fan & Song 2022\nHyundai ix35,3J1020O,47.22,121,Gasoline,4.1133,Fan & Song 2022\nHyundai ix35,3J1020O,5.0,121,Gasoline,0.568,Fan & Song 2022\nHyundai ix35,3J1020O,5.07,121,Gasoline,0.6657,Fan & Song 2022\nHyundai ix35,3J1020O,15.6,121,Gasoline,1.6908,Fan & Song 2022\nHyundai ix35,3J1020O,72.23,121,Gasoline,5.1381,Fan & Song 2022\nHyundai ix35,3J1020O,145.69,121,Gasoline,12.5644,Fan & Song 2022\nHyundai ix35,3J1020O,2.79,121,Gasoline,0.2914,Fan & Song 2022\nHyundai ix35,3J1020O,116.99,121,Gasoline,11.2903,Fan & Song 2022\nHyundai ix35,3J1020O,136.35,121,Gasoline,11.3946,Fan & Song 2022\nHyundai ix35,3J1020O,75.71,121,Gasoline,6.8158,Fan & Song 2022\nHyundai ix35,3J1020O,257.74,121,Gasoline,19.7827,Fan & Song 2022\nHyundai ix35,3J1020O,4.54,121,Gasoline,0.5051,Fan & Song 2022\nHyundai ix35,3J1020O,10.72,121,Gasoline,0.9803,Fan & Song 2022\nHyundai ix35,3J1020O,5.28,121,Gasoline,0.718,Fan & Song 2022\nHyundai ix35,3J1020O,24.03,121,Gasoline,2.0642,Fan & Song 2022\nHyundai ix35,3J1020O,3.5,121,Gasoline,0.3912,Fan & Song 2022\nHyundai ix35,3J1020O,107.0,121,Gasoline,9.2243,Fan & Song 2022\nHyundai ix35,3J1020O,25.81,121,Gasoline,2.2745,Fan & Song 2022\nHyundai ix35,3J1020O,57.22,121,Gasoline,4.46,Fan & Song 2022\nHyundai ix35,3J1020O,18.86,121,Gasoline,1.8849,Fan & Song 2022\nHyundai ix35,3J1020O,17.84,121,Gasoline,1.7054,Fan & Song 2022\nHyundai ix35,3J1020O,30.13,121,Gasoline,2.8755,Fan & Song 2022\nHyundai ix35,3J1020O,18.3,121,Gasoline,1.8643,Fan & Song 2022\nHyundai ix35,3J1020O,15.96,121,Gasoline,1.8144,Fan & Song 2022\nHyundai ix35,3J1020O,28.64,121,Gasoline,2.3849,Fan & Song 2022\nHyundai ix35,3J1020O,35.89,121,Gasoline,2.7521,Fan & Song 2022\nHyundai ix35,3J1020O,35.0,121,Gasoline,3.0176,Fan & Song 2022\nHyundai ix35,3J1020O,12.56,121,Gasoline,1.4766,Fan & Song 2022\nHyundai ix35,3J1020O,2.54,121,Gasoline,0.271,Fan & Song 2022\nHyundai ix35,3J1020O,3.47,121,Gasoline,0.4605,Fan & Song 2022\nHyundai ix35,3J1020O,11.11,121,Gasoline,1.15,Fan & Song 2022\nHyundai ix35,3J1020O,213.03,121,Gasoline,16.9291,Fan & Song 2022\nHyundai ix35,3J1020O,48.32,121,Gasoline,3.8201,Fan & Song 2022\nHyundai ix35,3J1020O,198.49,121,Gasoline,14.8457,Fan & Song 2022\nHyundai ix35,3J1020O,1.93,121,Gasoline,0.1995,Fan & Song 2022\nHyundai ix35,3J1020O,51.16,121,Gasoline,4.125,Fan & Song 2022\nHyundai ix35,3J1020O,43.37,121,Gasoline,3.4481,Fan & Song 2022\nHyundai ix35,3J1020O,38.08,121,Gasoline,3.4371,Fan & Song 2022\nHyundai ix35,3J1020O,38.52,121,Gasoline,3.5434,Fan & Song 2022\nHyundai ix35,3J1020O,77.12,121,Gasoline,6.1799,Fan & Song 2022\nHyundai ix35,3J1020O,48.63,121,Gasoline,3.5019,Fan & Song 2022\nHyundai ix35,3J1020O,3.38,121,Gasoline,0.3095,Fan & Song 2022\nHyundai ix35,3J1020O,15.54,121,Gasoline,1.3567,Fan & Song 2022\nHyundai ix35,3J1020O,117.52,121,Gasoline,10.6032,Fan & Song 2022\nHyundai ix35,3J1020O,3.08,121,Gasoline,0.4176,Fan & Song 2022\nHyundai ix35,3J1020O,113.28,121,Gasoline,9.7085,Fan & Song 2022\nHyundai ix35,3J1020O,36.04,121,Gasoline,3.0467,Fan & Song 2022\nHyundai ix35,3J1020O,4.41,121,Gasoline,0.4319,Fan & Song 2022\nHyundai ix35,3J1020O,2.52,121,Gasoline,0.255,Fan & Song 2022\nHyundai ix35,3J1020O,30.48,121,Gasoline,3.123,Fan & Song 2022\nHyundai ix35,3J1020O,14.97,121,Gasoline,1.4133,Fan & Song 2022\nHyundai ix35,3J1020O,51.07,121,Gasoline,4.1295,Fan & Song 2022\nHyundai ix35,3J1020O,18.51,121,Gasoline,1.5612,Fan & Song 2022\nHyundai ix35,3J1020O,2.91,121,Gasoline,0.461,Fan & Song 2022\nHyundai ix35,3J1020O,15.35,121,Gasoline,1.4228,Fan & Song 2022\nHyundai ix35,3J1020O,37.38,121,Gasoline,2.9752,Fan & Song 2022\nHyundai ix35,3J1020O,26.0,121,Gasoline,2.5063,Fan & Song 2022\nHyundai ix35,3J1020O,49.46,121,Gasoline,3.6811,Fan & Song 2022\nHyundai ix35,3J1020O,15.12,121,Gasoline,1.569,Fan & Song 2022\nHyundai ix35,3J1020O,38.05,121,Gasoline,3.7995,Fan & Song 2022\nHyundai ix35,3J1020O,3.76,121,Gasoline,0.4339,Fan & Song 2022\nHyundai ix35,3J1020O,5.01,121,Gasoline,0.5502,Fan & Song 2022\nHyundai ix35,3J1020O,2.54,121,Gasoline,0.3219,Fan & Song 2022\nHyundai ix35,3J1020O,27.56,121,Gasoline,2.7754,Fan & Song 2022\nHyundai ix35,3J1020O,17.15,121,Gasoline,2.1684,Fan & Song 2022\nHyundai ix35,3J1020O,44.51,121,Gasoline,4.5932,Fan & Song 2022\nHyundai ix35,3J1020O,39.22,121,Gasoline,2.8186,Fan & Song 2022\nHyundai ix35,3J1020O,45.6,121,Gasoline,3.4045,Fan & Song 2022\nHyundai ix35,3J1020O,5.2,121,Gasoline,0.5871,Fan & Song 2022\nHyundai ix35,3J1020O,152.16,121,Gasoline,10.7311,Fan & Song 2022\nHyundai ix35,3J1020O,22.81,121,Gasoline,2.8907,Fan & Song 2022\nHyundai ix35,3J1020O,183.3,121,Gasoline,15.1808,Fan & Song 2022\nHyundai ix35,3J1020O,35.44,121,Gasoline,2.8241,Fan & Song 2022\nHyundai ix35,3J1020O,2.23,121,Gasoline,0.2715,Fan & Song 2022\nHyundai ix35,3J1020O,195.25,121,Gasoline,15.1508,Fan & Song 2022\nHyundai ix35,3J1020O,49.2,121,Gasoline,4.0047,Fan & Song 2022\nHyundai ix35,3J1020O,286.0,121,Gasoline,20.5465,Fan & Song 2022\nHyundai ix35,3J1020O,34.22,121,Gasoline,2.7734,Fan & Song 2022\nHyundai ix35,3J1020O,40.83,121,Gasoline,3.4206,Fan & Song 2022\nHyundai ix35,3J1020O,78.94,121,Gasoline,6.4407,Fan & Song 2022\nHyundai ix35,3J1020O,7.58,121,Gasoline,0.6374,Fan & Song 2022\nHyundai ix35,3J1020O,26.93,121,Gasoline,1.702,Fan & Song 2022\nHyundai ix35,3J1020O,38.62,121,Gasoline,4.0429,Fan & Song 2022\nHyundai ix35,3J1020O,3.23,121,Gasoline,0.3683,Fan & Song 2022\nHyundai ix35,3J1020O,22.18,121,Gasoline,1.7887,Fan & Song 2022\nHyundai ix35,3J1020O,3.84,121,Gasoline,0.5292,Fan & Song 2022\nHyundai ix35,3J1020O,23.18,121,Gasoline,2.0351,Fan & Song 2022\nHyundai ix35,3J1020O,23.77,121,Gasoline,2.4778,Fan & Song 2022\nHyundai ix35,3J1020O,102.29,121,Gasoline,9.223,Fan & Song 2022\nHyundai ix35,3J1020O,27.05,121,Gasoline,2.5966,Fan & Song 2022\nHyundai ix35,3J1020O,31.05,121,Gasoline,2.7376,Fan & Song 2022\nHyundai ix35,3J1020O,15.26,121,Gasoline,1.664,Fan & Song 2022\nHyundai ix35,3J1020O,17.17,121,Gasoline,1.8654,Fan & Song 2022\nHyundai ix35,3J1020O,6.58,121,Gasoline,0.5945,Fan & Song 2022\nHyundai ix35,3J1020O,2.21,121,Gasoline,0.2664,Fan & Song 2022\nHyundai ix35,3J1020O,44.01,121,Gasoline,3.3115,Fan & Song 2022\nHyundai ix35,3J1020O,52.72,121,Gasoline,4.0483,Fan & Song 2022\nHyundai ix35,3J1020O,14.91,121,Gasoline,1.4957,Fan & Song 2022\nHyundai ix35,3J1020O,7.68,121,Gasoline,0.8306,Fan & Song 2022\nHyundai ix35,3J1020O,9.16,121,Gasoline,0.88,Fan & Song 2022\nHyundai ix35,3J1020O,14.79,121,Gasoline,1.2087,Fan & Song 2022\nHyundai ix35,3J1020O,15.73,121,Gasoline,1.2148,Fan & Song 2022\nHyundai ix35,3J1020O,4.7,121,Gasoline,0.5513,Fan & Song 2022\nHyundai ix35,3J1020O,19.08,121,Gasoline,2.2128,Fan & Song 2022\nHyundai ix35,3J1020O,6.26,121,Gasoline,0.8039,Fan & Song 2022\nHyundai ix35,3J1020O,24.16,121,Gasoline,1.9409,Fan & Song 2022\nHyundai ix35,3J1020O,15.35,121,Gasoline,1.4898,Fan & Song 2022\nHyundai ix35,3J1020O,5.28,121,Gasoline,0.7016,Fan & Song 2022\nHyundai ix35,3J1020O,30.74,121,Gasoline,2.9649,Fan & Song 2022\nHyundai ix35,3J1020O,13.82,121,Gasoline,1.235,Fan & Song 2022\nHyundai ix35,3J1020O,156.29,121,Gasoline,13.3169,Fan & Song 2022\nHyundai ix35,3J1020O,148.94,121,Gasoline,12.8479,Fan & Song 2022\nHyundai ix35,3J1020O,2.74,121,Gasoline,0.3975,Fan & Song 2022\nHyundai ix35,3J1020O,149.42,121,Gasoline,13.011,Fan & Song 2022\nHyundai ix35,3J1020O,32.01,121,Gasoline,2.7051,Fan & Song 2022\nHyundai ix35,3J1020O,103.3,121,Gasoline,8.911,Fan & Song 2022\nHyundai ix35,3J1020O,119.67,121,Gasoline,9.5458,Fan & Song 2022\nHyundai ix35,3J1020O,238.07,121,Gasoline,17.6533,Fan & Song 2022\nHyundai ix35,3J1020O,33.7,121,Gasoline,2.4813,Fan & Song 2022\nHyundai ix35,3J1020O,3.52,121,Gasoline,0.5116,Fan & Song 2022\nHyundai ix35,3J1020O,9.0,121,Gasoline,0.6844,Fan & Song 2022\nHyundai ix35,3J1020O,30.7,121,Gasoline,2.5958,Fan & Song 2022\nHyundai ix35,3J1020O,2.08,121,Gasoline,0.2578,Fan & Song 2022\nHyundai ix35,3J1020O,7.19,121,Gasoline,0.8348,Fan & Song 2022\nHyundai ix35,3J1020O,109.92,121,Gasoline,9.5915,Fan & Song 2022\nHyundai ix35,3J1020O,54.58,121,Gasoline,4.2214,Fan & Song 2022\nHyundai ix35,3J1020O,176.5,121,Gasoline,14.9044,Fan & Song 2022\nHyundai ix35,3J1020O,50.84,121,Gasoline,3.831,Fan & Song 2022\nHyundai ix35,3J1020O,38.38,121,Gasoline,3.175,Fan & Song 2022\nHyundai ix35,3J1020O,68.55,121,Gasoline,5.6283,Fan & Song 2022\nHyundai ix35,3J1020O,110.29,121,Gasoline,8.9859,Fan & Song 2022\nHyundai ix35,3J1020O,26.76,121,Gasoline,2.4281,Fan & Song 2022\nHyundai ix35,3J1020O,4.46,121,Gasoline,0.3864,Fan & Song 2022\nHyundai ix35,3J1020O,95.29,121,Gasoline,8.0622,Fan & Song 2022\nHyundai ix35,3J1020O,121.87,121,Gasoline,9.3858,Fan & Song 2022\nHyundai ix35,3J1020O,179.72,121,Gasoline,12.4657,Fan & Song 2022\nHyundai ix35,3J1020O,6.09,121,Gasoline,0.7912,Fan & Song 2022\nHyundai ix35,3J1020O,15.81,121,Gasoline,1.5232,Fan & Song 2022\nHyundai ix35,3J1020O,4.23,121,Gasoline,0.471,Fan & Song 2022\nHyundai ix35,3J1020O,7.97,121,Gasoline,0.813,Fan & Song 2022\nHyundai ix35,3J1020O,10.18,121,Gasoline,1.0575,Fan & Song 2022\nHyundai ix35,3J1020O,32.54,121,Gasoline,2.5309,Fan & Song 2022\nHyundai ix35,3J1020O,4.02,121,Gasoline,0.5799,Fan & Song 2022\nHyundai ix35,3J1020O,3.26,121,Gasoline,0.3893,Fan & Song 2022\nHyundai ix35,3J1020O,3.86,121,Gasoline,0.4745,Fan & Song 2022\nHyundai ix35,3J1020O,88.83,121,Gasoline,7.8857,Fan & Song 2022\nHyundai ix35,3J1020O,6.04,121,Gasoline,0.4933,Fan & Song 2022\nHyundai ix35,3J1020O,13.2,121,Gasoline,1.1717,Fan & Song 2022\nHyundai ix35,3J1020O,33.25,121,Gasoline,3.3127,Fan & Song 2022\nHyundai ix35,3J1020O,4.7,121,Gasoline,0.4919,Fan & Song 2022\nHyundai ix35,3J1020O,5.31,121,Gasoline,0.7054,Fan & Song 2022\nHyundai ix35,3J1020O,46.78,121,Gasoline,3.4326,Fan & Song 2022\nHyundai ix35,3J1020O,4.3,121,Gasoline,0.5659,Fan & Song 2022\nHyundai ix35,3J1020O,41.56,121,Gasoline,4.0622,Fan & Song 2022\nHyundai ix35,3J1020O,3.72,121,Gasoline,0.5347,Fan & Song 2022\nHyundai ix35,3J1020O,22.72,121,Gasoline,2.3646,Fan & Song 2022\nHyundai ix35,3J1020O,21.78,121,Gasoline,1.7506,Fan & Song 2022\nHyundai ix35,3J1020O,25.05,121,Gasoline,1.8563,Fan & Song 2022\nHyundai ix35,3J1020O,13.36,121,Gasoline,1.344,Fan & Song 2022\nHyundai ix35,3J1020O,15.38,121,Gasoline,1.0943,Fan & Song 2022\nHyundai ix35,3J1020O,4.82,121,Gasoline,0.5932,Fan & Song 2022\nHyundai ix35,3J1020O,4.82,121,Gasoline,0.6114,Fan & Song 2022\nHyundai ix35,3J1020O,59.73,121,Gasoline,5.12,Fan & Song 2022\nHyundai ix35,3J1020O,45.12,121,Gasoline,3.7177,Fan & Song 2022\nHyundai ix35,3J1020O,5.14,121,Gasoline,0.6629,Fan & Song 2022\nHyundai ix35,3J1020O,43.71,121,Gasoline,3.4704,Fan & Song 2022\nHyundai ix35,3J1020O,34.0,121,Gasoline,2.8047,Fan & Song 2022\nHyundai ix35,3J1020O,5.95,121,Gasoline,0.5968,Fan & Song 2022\nHyundai ix35,3J1020O,22.35,121,Gasoline,1.8098,Fan & Song 2022\nHyundai ix35,3J1020O,5.31,121,Gasoline,0.6328,Fan & Song 2022\nHyundai ix35,3J1020O,15.29,121,Gasoline,1.4877,Fan & Song 2022\nHyundai ix35,3J1020O,6.89,121,Gasoline,0.8708,Fan & Song 2022\nHyundai ix35,3J1020O,4.43,121,Gasoline,0.6457,Fan & Song 2022\nHyundai ix35,3J1020O,137.62,121,Gasoline,10.6713,Fan & Song 2022\nHyundai ix35,3J1020O,1.84,121,Gasoline,0.2372,Fan & Song 2022\nHyundai ix35,3J1020O,76.39,121,Gasoline,6.4598,Fan & Song 2022\nHyundai ix35,3J1020O,64.95,121,Gasoline,5.7835,Fan & Song 2022\nHyundai ix35,3J1020O,23.23,121,Gasoline,2.1942,Fan & Song 2022\nHyundai ix35,3J1020O,38.28,121,Gasoline,2.9922,Fan & Song 2022\nHyundai ix35,3J1020O,3.11,121,Gasoline,0.4502,Fan & Song 2022\nHyundai ix35,3J1020O,31.35,121,Gasoline,2.3879,Fan & Song 2022\nHyundai ix35,3J1020O,205.21,121,Gasoline,14.6456,Fan & Song 2022\nHyundai ix35,3J1020O,153.37,121,Gasoline,12.9457,Fan & Song 2022\nHyundai ix35,3J1020O,8.95,121,Gasoline,1.0556,Fan & Song 2022\nHyundai ix35,3J1020O,6.17,121,Gasoline,0.9104,Fan & Song 2022\nHyundai ix35,3J1020O,145.9,121,Gasoline,13.2126,Fan & Song 2022\nHyundai ix35,3J1020O,87.15,121,Gasoline,7.6769,Fan & Song 2022\nHyundai ix35,3J1020O,84.04,121,Gasoline,7.7693,Fan & Song 2022\nHyundai ix35,3J1020O,3.37,121,Gasoline,0.4413,Fan & Song 2022\nHyundai ix35,3J1020O,105.06,121,Gasoline,8.119,Fan & Song 2022\nHyundai ix35,3J1020O,104.77,121,Gasoline,7.825,Fan & Song 2022\nHyundai ix35,3J1020O,45.51,121,Gasoline,3.4847,Fan & Song 2022\nHyundai ix35,3J1020O,3.44,121,Gasoline,0.5901,Fan & Song 2022\nHyundai ix35,3J1020O,30.08,121,Gasoline,3.0565,Fan & Song 2022\nHyundai ix35,3J1020O,15.23,121,Gasoline,1.5324,Fan & Song 2022\nHyundai ix35,3J1020O,2.22,121,Gasoline,0.4007,Fan & Song 2022\nHyundai ix35,3J1020O,14.86,121,Gasoline,1.281,Fan & Song 2022\nHyundai ix35,3J1020O,50.98,121,Gasoline,4.1469,Fan & Song 2022\nHyundai ix35,3J1020O,49.26,121,Gasoline,4.1758,Fan & Song 2022\nHyundai ix35,3J1020O,16.2,121,Gasoline,1.852,Fan & Song 2022\nHyundai ix35,3J1020O,43.59,121,Gasoline,3.3277,Fan & Song 2022\nHyundai ix35,3J1020O,50.19,121,Gasoline,3.4828,Fan & Song 2022\nHyundai ix35,3J1020O,4.94,121,Gasoline,0.6084,Fan & Song 2022\nHyundai ix35,3J1020O,3.74,121,Gasoline,0.7383,Fan & Song 2022\nHyundai ix35,3J1020O,2.27,121,Gasoline,0.3003,Fan & Song 2022\nHyundai ix35,3J1020O,48.94,121,Gasoline,3.9783,Fan & Song 2022\nHyundai ix35,3J1020O,31.73,121,Gasoline,2.4366,Fan & Song 2022\nHyundai ix35,3J1020O,117.21,121,Gasoline,9.9568,Fan & Song 2022\nHyundai ix35,3J1020O,159.21,121,Gasoline,14.4043,Fan & Song 2022\nHyundai ix35,3J1020O,5.19,121,Gasoline,0.7525,Fan & Song 2022\nHyundai ix35,3J1020O,31.61,121,Gasoline,2.8845,Fan & Song 2022\nHyundai ix35,3J1020O,163.69,121,Gasoline,14.3976,Fan & Song 2022\nHyundai ix35,3J1020O,57.59,121,Gasoline,4.8175,Fan & Song 2022\nHyundai ix35,3J1020O,3.05,121,Gasoline,0.2991,Fan & Song 2022\nHyundai ix35,3J1020O,162.37,121,Gasoline,12.735,Fan & Song 2022\nHyundai ix35,3J1020O,249.12,121,Gasoline,19.2843,Fan & Song 2022\nHyundai ix35,3J1020O,193.46,121,Gasoline,15.5408,Fan & Song 2022\nHyundai ix35,3J1020O,5.8,121,Gasoline,0.7039,Fan & Song 2022\nHyundai ix35,3J1020O,231.95,121,Gasoline,19.688,Fan & Song 2022\nHyundai ix35,3J1020O,209.2,121,Gasoline,17.244,Fan & Song 2022\nHyundai ix35,3J1020O,280.97,121,Gasoline,22.7751,Fan & Song 2022\nHyundai ix35,3J1020O,46.19,121,Gasoline,3.564,Fan & Song 2022\nHyundai ix35,3J1020O,15.14,121,Gasoline,2.0072,Fan & Song 2022\nHyundai ix35,3J1020O,17.95,121,Gasoline,1.7235,Fan & Song 2022\nHyundai ix35,3J1020O,3.71,121,Gasoline,0.4627,Fan & Song 2022\nHyundai ix35,3J1020O,6.0,121,Gasoline,0.7967,Fan & Song 2022\nHyundai ix35,3J1020O,115.5,121,Gasoline,9.2385,Fan & Song 2022\nHyundai ix35,3J1020O,108.24,121,Gasoline,9.7288,Fan & Song 2022\nHyundai ix35,3J1020O,46.86,121,Gasoline,3.9632,Fan & Song 2022\nHyundai ix35,3J1020O,136.72,121,Gasoline,11.6225,Fan & Song 2022\nHyundai ix35,3J1020O,139.27,121,Gasoline,10.2304,Fan & Song 2022\nHyundai ix35,3J1020O,7.06,121,Gasoline,1.1,Fan & Song 2022\nHyundai ix35,3J1020O,120.66,121,Gasoline,10.4874,Fan & Song 2022\nHyundai ix35,3J1020O,146.7,121,Gasoline,12.7382,Fan & Song 2022\nHyundai ix35,3J1020O,10.28,121,Gasoline,1.3436,Fan & Song 2022\nHyundai ix35,3J1020O,9.74,121,Gasoline,1.204,Fan & Song 2022\nHyundai ix35,3J1020O,200.66,121,Gasoline,18.5976,Fan & Song 2022\nHyundai ix35,3J1020O,16.06,121,Gasoline,1.5371,Fan & Song 2022\nHyundai ix35,3J1020O,237.93,121,Gasoline,21.0145,Fan & Song 2022\nHyundai ix35,3J1020O,18.09,121,Gasoline,2.0143,Fan & Song 2022\nHyundai ix35,3J1020O,5.24,121,Gasoline,0.7567,Fan & Song 2022\nHyundai ix35,3J1020O,8.09,121,Gasoline,0.8352,Fan & Song 2022\nHyundai ix35,3J1020O,51.4,121,Gasoline,4.6692,Fan & Song 2022\nHyundai ix35,3J1020O,14.7,121,Gasoline,1.1086,Fan & Song 2022\nHyundai ix35,3J1020O,27.96,121,Gasoline,2.1923,Fan & Song 2022\nHyundai ix35,3J1020O,147.0,121,Gasoline,12.923,Fan & Song 2022\nHyundai ix35,3J1020O,154.97,121,Gasoline,13.4486,Fan & Song 2022\nHyundai ix35,3J1020O,6.02,121,Gasoline,0.8112,Fan & Song 2022\nHyundai ix35,3J1020O,282.43,121,Gasoline,22.5423,Fan & Song 2022\nHyundai ix35,3J1020O,49.28,121,Gasoline,4.1992,Fan & Song 2022\nHyundai ix35,3J1020O,5.39,121,Gasoline,0.629,Fan & Song 2022\nHyundai ix35,3J1020O,195.48,121,Gasoline,16.2912,Fan & Song 2022\nHyundai ix35,3J1020O,31.0,121,Gasoline,3.1265,Fan & Song 2022\nHyundai ix35,3J1020O,45.34,121,Gasoline,3.9639,Fan & Song 2022\nHyundai ix35,3J1020O,5.81,121,Gasoline,0.6315,Fan & Song 2022\nHyundai ix35,3J1020O,11.45,121,Gasoline,0.9134,Fan & Song 2022\nHyundai ix35,3J1020O,20.06,121,Gasoline,2.1814,Fan & Song 2022\nHyundai ix35,3J1020O,6.74,121,Gasoline,0.6544,Fan & Song 2022\nHyundai ix35,3J1020O,39.48,121,Gasoline,4.7622,Fan & Song 2022\nHyundai ix35,3J1020O,15.68,121,Gasoline,1.6177,Fan & Song 2022\nHyundai ix35,3J1020O,3.26,121,Gasoline,0.4809,Fan & Song 2022\nHyundai ix35,3J1020O,23.19,121,Gasoline,2.2966,Fan & Song 2022\nHyundai ix35,3J1020O,148.15,121,Gasoline,13.4098,Fan & Song 2022\nHyundai ix35,3J1020O,90.34,121,Gasoline,7.3386,Fan & Song 2022\nHyundai ix35,3J1020O,316.99,121,Gasoline,32.4718,Fan & Song 2022\nHyundai ix35,3J1020O,13.56,121,Gasoline,1.3377,Fan & Song 2022\nHyundai ix35,3J1020O,223.14,121,Gasoline,19.0603,Fan & Song 2022\nHyundai ix35,3J1020O,186.26,121,Gasoline,16.579,Fan & Song 2022\nHyundai ix35,3J1020O,37.77,121,Gasoline,2.7126,Fan & Song 2022\nHyundai ix35,3J1020O,4.32,121,Gasoline,0.7076,Fan & Song 2022\nHyundai ix35,3J1020O,270.96,121,Gasoline,21.6879,Fan & Song 2022\nHyundai ix35,3J1020O,49.9,121,Gasoline,4.4286,Fan & Song 2022\nHyundai ix35,3J1020O,190.53,121,Gasoline,17.005,Fan & Song 2022\nHyundai ix35,3J1020O,310.4,121,Gasoline,22.2454,Fan & Song 2022\nHyundai ix35,3J1020O,137.24,121,Gasoline,12.7824,Fan & Song 2022\nHyundai ix35,3J1020O,135.3,121,Gasoline,13.4387,Fan & Song 2022\nHyundai ix35,3J1020O,45.32,121,Gasoline,4.3654,Fan & Song 2022\nHyundai ix35,3J1020O,16.57,121,Gasoline,1.4544,Fan & Song 2022\nHyundai ix35,3J1020O,86.42,121,Gasoline,6.8607,Fan & Song 2022\nHyundai ix35,3J1020O,226.2,121,Gasoline,19.8782,Fan & Song 2022\nHyundai ix35,3J1020O,414.73,121,Gasoline,34.7721,Fan & Song 2022\nHyundai ix35,3J1020O,3.92,121,Gasoline,0.775,Fan & Song 2022\nHyundai ix35,3J1020O,15.65,121,Gasoline,1.3506,Fan & Song 2022\nHyundai ix35,3J1020O,46.06,121,Gasoline,4.1878,Fan & Song 2022\nHyundai ix35,3J1020O,10.74,121,Gasoline,1.5613,Fan & Song 2022\nHyundai ix35,3J1020O,18.2,121,Gasoline,1.5913,Fan & Song 2022\nHyundai ix35,3J1020O,15.56,121,Gasoline,1.4745,Fan & Song 2022\nHyundai ix35,3J1020O,31.41,121,Gasoline,2.9664,Fan & Song 2022\nHyundai ix35,3J1020O,28.44,121,Gasoline,2.8419,Fan & Song 2022\nHyundai ix35,3J1020O,5.72,121,Gasoline,0.8609,Fan & Song 2022\nHyundai ix35,3J1020O,5.3,121,Gasoline,0.6666,Fan & Song 2022\nHyundai ix35,3J1020O,4.38,121,Gasoline,0.6117,Fan & Song 2022\nHyundai ix35,3J1020O,34.75,121,Gasoline,3.4221,Fan & Song 2022\nHyundai ix35,3J1020O,2.64,121,Gasoline,0.3617,Fan & Song 2022\nHyundai ix35,3J1020O,54.43,121,Gasoline,4.5748,Fan & Song 2022\nHyundai ix35,3J1020O,8.89,121,Gasoline,0.7463,Fan & Song 2022\nHyundai ix35,3J1020O,15.24,121,Gasoline,1.4342,Fan & Song 2022\nHyundai ix35,3J1020O,273.74,121,Gasoline,24.2325,Fan & Song 2022\nHyundai ix35,3J1020O,121.39,121,Gasoline,12.758,Fan & Song 2022\nHyundai ix35,3J1020O,170.98,121,Gasoline,12.2685,Fan & Song 2022\nHyundai ix35,3J1020O,35.47,121,Gasoline,2.342,Fan & Song 2022\nHyundai ix35,3J1020O,148.56,121,Gasoline,11.359,Fan & Song 2022\nHyundai ix35,3J1020O,58.0,121,Gasoline,4.2336,Fan & Song 2022\nHyundai ix35,3J1020O,190.13,121,Gasoline,15.4043,Fan & Song 2022\nHyundai ix35,3J1020O,44.81,121,Gasoline,3.8231,Fan & Song 2022\nHyundai ix35,3J1020O,3.99,121,Gasoline,0.5071,Fan & Song 2022\nHyundai ix35,3J1020O,14.66,121,Gasoline,1.1488,Fan & Song 2022\nHyundai ix35,3J1020O,7.79,121,Gasoline,1.1271,Fan & Song 2022\nHyundai ix35,3J1020O,259.08,121,Gasoline,22.433,Fan & Song 2022\nHyundai ix35,3J1020O,19.82,121,Gasoline,1.7445,Fan & Song 2022\nHyundai ix35,3J1020O,115.68,121,Gasoline,8.8001,Fan & Song 2022\nHyundai ix35,3J1020O,38.89,121,Gasoline,3.0974,Fan & Song 2022\nHyundai ix35,3J1020O,116.72,121,Gasoline,9.4337,Fan & Song 2022\nHyundai ix35,3J1020O,110.49,121,Gasoline,8.1404,Fan & Song 2022\nHyundai ix35,3J1020O,126.39,121,Gasoline,10.6214,Fan & Song 2022\nHyundai ix35,3J1020O,23.65,121,Gasoline,2.4713,Fan & Song 2022\nHyundai ix35,3J1020O,42.28,121,Gasoline,4.4847,Fan & Song 2022\nHyundai ix35,3J1020O,8.05,121,Gasoline,1.1594,Fan & Song 2022\nHyundai ix35,3J1020O,35.32,121,Gasoline,2.4147,Fan & Song 2022\nHyundai ix35,3J1020O,195.05,121,Gasoline,14.6164,Fan & Song 2022\nHyundai ix35,3J1020O,5.57,121,Gasoline,0.823,Fan & Song 2022\nHyundai ix35,3J1020O,187.36,121,Gasoline,17.4302,Fan & Song 2022\nHyundai ix35,3J1020O,16.96,121,Gasoline,2.3881,Fan & Song 2022\nHyundai ix35,3J1020O,200.09,121,Gasoline,13.7864,Fan & Song 2022\nHyundai ix35,3J1020O,175.83,121,Gasoline,14.9879,Fan & Song 2022\nHyundai ix35,3J1020O,121.96,121,Gasoline,11.1336,Fan & Song 2022\nHyundai ix35,3J1020O,2.42,121,Gasoline,0.3619,Fan & Song 2022\nHyundai ix35,3J1020O,3.29,121,Gasoline,0.4878,Fan & Song 2022\nHyundai ix35,3J1020O,15.11,121,Gasoline,1.2548,Fan & Song 2022\nHyundai ix35,3J1020O,39.12,121,Gasoline,3.7156,Fan & Song 2022\nHyundai ix35,3J1020O,19.76,121,Gasoline,1.974,Fan & Song 2022\nHyundai ix35,3J1020O,183.08,121,Gasoline,15.6532,Fan & Song 2022\nHyundai ix35,3J1020O,136.44,121,Gasoline,12.2857,Fan & Song 2022\nHyundai ix35,3J1020O,4.86,121,Gasoline,0.54,Fan & Song 2022\nHyundai ix35,3J1020O,131.6,121,Gasoline,12.3489,Fan & Song 2022\nHyundai ix35,3J1020O,54.27,121,Gasoline,4.966,Fan & Song 2022\nHyundai ix35,3J1020O,236.97,121,Gasoline,20.2004,Fan & Song 2022\nHyundai ix35,3J1020O,138.44,121,Gasoline,11.9209,Fan & Song 2022\nHyundai ix35,3J1020O,12.54,121,Gasoline,1.7343,Fan & Song 2022\nHyundai ix35,3J1020O,31.54,121,Gasoline,5.3039,Fan & Song 2022\nHyundai ix35,3J1020O,212.88,121,Gasoline,17.9952,Fan & Song 2022\nHyundai ix35,3J1020O,111.24,121,Gasoline,8.5181,Fan & Song 2022\nHyundai ix35,3J1020O,16.37,121,Gasoline,1.7705,Fan & Song 2022\nHyundai ix35,3J1020O,32.4,121,Gasoline,2.7943,Fan & Song 2022\nHyundai ix35,3J1020O,17.31,121,Gasoline,1.4666,Fan & Song 2022\nHyundai ix35,3J1020O,183.49,121,Gasoline,15.6373,Fan & Song 2022\nHyundai ix35,3J1020O,83.17,121,Gasoline,7.211,Fan & Song 2022\nHyundai ix35,3J1020O,14.65,121,Gasoline,1.3008,Fan & Song 2022\nHyundai ix35,3J1020O,18.34,121,Gasoline,2.0355,Fan & Song 2022\nHyundai ix35,3J1020O,39.96,121,Gasoline,3.7856,Fan & Song 2022\nHyundai ix35,3J1020O,2.64,121,Gasoline,0.3079,Fan & Song 2022\nHyundai ix35,3J1020O,37.91,121,Gasoline,3.182,Fan & Song 2022\nHyundai ix35,3J1020O,78.78,121,Gasoline,5.9305,Fan & Song 2022\nHyundai ix35,3J1020O,18.86,121,Gasoline,1.7437,Fan & Song 2022\nHyundai ix35,3J1020O,31.88,121,Gasoline,2.0389,Fan & Song 2022\nHyundai ix35,3J1020O,163.68,121,Gasoline,12.083,Fan & Song 2022\nHyundai ix35,3J1020O,173.22,121,Gasoline,12.5281,Fan & Song 2022\nHyundai ix35,3J1020O,7.19,121,Gasoline,0.7734,Fan & Song 2022\nHyundai ix35,3J1020O,117.47,121,Gasoline,9.1085,Fan & Song 2022\nHyundai ix35,3J1020O,67.29,121,Gasoline,5.2171,Fan & Song 2022\nHyundai ix35,3J1020O,58.87,121,Gasoline,4.5091,Fan & Song 2022\nHyundai ix35,3J1020O,58.49,121,Gasoline,4.8472,Fan & Song 2022\nHyundai ix35,3J1020O,26.87,121,Gasoline,2.9313,Fan & Song 2022\nHyundai ix35,3J1020O,120.15,121,Gasoline,11.2729,Fan & Song 2022\nHyundai ix35,3J1020O,143.94,121,Gasoline,12.8986,Fan & Song 2022\nHyundai ix35,3J1020O,43.35,121,Gasoline,3.6228,Fan & Song 2022\nHyundai ix35,3J1020O,44.62,121,Gasoline,3.6474,Fan & Song 2022\nHyundai ix35,3J1020O,131.86,121,Gasoline,12.4374,Fan & Song 2022\nHyundai ix35,3J1020O,38.26,121,Gasoline,3.5667,Fan & Song 2022\nHyundai ix35,3J1020O,12.55,121,Gasoline,1.3915,Fan & Song 2022\nHyundai ix35,3J1020O,3.52,121,Gasoline,0.5615,Fan & Song 2022\nHyundai ix35,3J1020O,23.44,121,Gasoline,2.3314,Fan & Song 2022\nHyundai ix35,3J1020O,11.93,121,Gasoline,1.1237,Fan & Song 2022\nHyundai ix35,3J1020O,4.95,121,Gasoline,0.6868,Fan & Song 2022\nHyundai ix35,3J1020O,14.73,121,Gasoline,1.1157,Fan & Song 2022\nHyundai ix35,3J1020O,4.58,121,Gasoline,0.6331,Fan & Song 2022\nHyundai ix35,3J1020O,10.71,121,Gasoline,0.8496,Fan & Song 2022\nHyundai ix35,3J1020O,2.41,121,Gasoline,0.266,Fan & Song 2022\nHyundai ix35,3J1020O,179.19,121,Gasoline,15.3575,Fan & Song 2022\nHyundai ix35,3J1020O,165.33,121,Gasoline,14.3004,Fan & Song 2022\nHyundai ix35,3J1020O,20.4,121,Gasoline,2.2401,Fan & Song 2022\nHyundai ix35,3J1020O,16.69,121,Gasoline,1.7033,Fan & Song 2022\nHyundai ix35,3J1020O,118.85,121,Gasoline,8.6161,Fan & Song 2022\nHyundai ix35,3J1020O,69.68,121,Gasoline,5.8425,Fan & Song 2022\nHyundai ix35,3J1020O,2.08,121,Gasoline,0.2783,Fan & Song 2022\nHyundai ix35,3J1020O,33.26,121,Gasoline,2.4714,Fan & Song 2022\nHyundai ix35,3J1020O,22.77,121,Gasoline,1.5898,Fan & Song 2022\nHyundai ix35,3J1020O,108.14,121,Gasoline,8.9649,Fan & Song 2022\nHyundai ix35,3J1020O,124.44,121,Gasoline,10.6295,Fan & Song 2022\nHyundai ix35,3J1020O,181.51,121,Gasoline,14.8941,Fan & Song 2022\nHyundai ix35,3J1020O,118.48,121,Gasoline,8.6318,Fan & Song 2022\nHyundai ix35,3J1020O,18.04,121,Gasoline,1.9829,Fan & Song 2022\nHyundai ix35,3J1020O,16.29,121,Gasoline,1.3607,Fan & Song 2022\nHyundai ix35,3J1020O,30.06,121,Gasoline,2.5662,Fan & Song 2022\nHyundai ix35,3J1020O,1.62,121,Gasoline,0.2041,Fan & Song 2022\nHyundai ix35,3J1020O,116.47,121,Gasoline,10.3183,Fan & Song 2022\nHyundai ix35,3J1020O,17.99,121,Gasoline,1.7098,Fan & Song 2022\nHyundai ix35,3J1020O,14.84,121,Gasoline,1.3427,Fan & Song 2022\nHyundai ix35,3J1020O,129.1,121,Gasoline,9.9068,Fan & Song 2022\nHyundai ix35,3J1020O,68.23,121,Gasoline,5.3869,Fan & Song 2022\nHyundai ix35,3J1020O,10.7,121,Gasoline,1.0852,Fan & Song 2022\nHyundai ix35,3J1020O,237.38,121,Gasoline,19.6258,Fan & Song 2022\nHyundai ix35,3J1020O,80.0,121,Gasoline,6.2625,Fan & Song 2022\nHyundai ix35,3J1020O,17.22,121,Gasoline,1.4946,Fan & Song 2022\nHyundai ix35,3J1020O,111.13,121,Gasoline,8.4666,Fan & Song 2022\nHyundai ix35,3J1020O,241.82,121,Gasoline,19.7483,Fan & Song 2022\nHyundai ix35,3J1020O,45.6,121,Gasoline,3.8435,Fan & Song 2022\nHyundai ix35,3J1020O,190.47,121,Gasoline,14.5427,Fan & Song 2022\nHyundai ix35,3J1020O,143.57,121,Gasoline,10.6682,Fan & Song 2022\nHyundai ix35,3J1020O,2.55,121,Gasoline,0.4175,Fan & Song 2022\nHyundai ix35,3J1020O,491.39,121,Gasoline,38.0902,Fan & Song 2022\nHyundai ix35,3J1020O,151.15,121,Gasoline,13.1746,Fan & Song 2022\nHyundai ix35,3J1020O,121.29,121,Gasoline,9.7548,Fan & Song 2022\nHyundai ix35,3J1020O,182.19,121,Gasoline,16.7072,Fan & Song 2022\nHyundai ix35,3J1020O,121.41,121,Gasoline,9.2559,Fan & Song 2022\nHyundai ix35,3J1020O,164.65,121,Gasoline,14.1007,Fan & Song 2022\nHyundai ix35,3J1020O,224.59,121,Gasoline,19.4192,Fan & Song 2022\nHyundai ix35,3J1020O,59.08,121,Gasoline,4.9317,Fan & Song 2022\nHyundai ix35,3J1020O,102.47,121,Gasoline,7.8718,Fan & Song 2022\nHyundai ix35,3J1020O,169.95,121,Gasoline,13.8372,Fan & Song 2022\nHyundai ix35,3J1020O,8.09,121,Gasoline,0.7761,Fan & Song 2022\nHyundai ix35,3J1020O,102.69,121,Gasoline,8.6207,Fan & Song 2022\nHyundai ix35,3J1020O,132.1,121,Gasoline,10.0879,Fan & Song 2022\nHyundai ix35,3J1020O,141.81,121,Gasoline,10.2579,Fan & Song 2022\nHyundai ix35,3J1020O,56.54,121,Gasoline,4.871,Fan & Song 2022\nHyundai ix35,3J1020O,85.67,121,Gasoline,5.8636,Fan & Song 2022\nHyundai ix35,3J1020O,201.51,121,Gasoline,16.6553,Fan & Song 2022\nHyundai ix35,3J1020O,78.45,121,Gasoline,8.3855,Fan & Song 2022\nHyundai ix35,3J1020O,7.31,121,Gasoline,1.1394,Fan & Song 2022\nHyundai ix35,3J1020O,54.91,121,Gasoline,5.3218,Fan & Song 2022\nHyundai ix35,3J1020O,10.4,121,Gasoline,0.7821,Fan & Song 2022\nHyundai ix35,3J1020O,232.2,121,Gasoline,18.561,Fan & Song 2022\nHyundai ix35,3J1020O,285.15,121,Gasoline,21.454,Fan & Song 2022\nHyundai ix35,3J1020O,155.71,121,Gasoline,14.389,Fan & Song 2022\nHyundai ix35,3J1020O,145.74,121,Gasoline,12.4071,Fan & Song 2022\nHyundai ix35,3J1020O,75.76,121,Gasoline,6.9926,Fan & Song 2022\nHyundai ix35,3J1020O,33.92,121,Gasoline,2.9405,Fan & Song 2022\nHyundai ix35,3J1020O,36.07,121,Gasoline,2.9311,Fan & Song 2022\nHyundai ix35,3J1020O,30.8,121,Gasoline,2.3753,Fan & Song 2022\nHyundai ix35,3J1020O,2.98,121,Gasoline,0.4329,Fan & Song 2022\nHyundai ix35,3J1020O,30.32,121,Gasoline,2.3338,Fan & Song 2022\nHyundai ix35,3J1020O,18.82,121,Gasoline,1.4583,Fan & Song 2022\nHyundai ix35,3J1020O,121.74,121,Gasoline,8.7966,Fan & Song 2022\nHyundai ix35,3J1020O,35.44,121,Gasoline,3.0093,Fan & Song 2022\nHyundai ix35,3J1020O,5.45,121,Gasoline,0.5662,Fan & Song 2022\nHyundai ix35,3J1020O,11.36,121,Gasoline,0.9326,Fan & Song 2022\nHyundai ix35,3J1020O,15.55,121,Gasoline,1.2473,Fan & Song 2022\nHyundai ix35,3J1020O,188.46,121,Gasoline,13.822,Fan & Song 2022\nHyundai ix35,3J1020O,50.22,121,Gasoline,4.1892,Fan & Song 2022\nHyundai ix35,3J1020O,255.03,121,Gasoline,20.5121,Fan & Song 2022\nHyundai ix35,3J1020O,11.71,121,Gasoline,1.0047,Fan & Song 2022\nHyundai ix35,3J1020O,257.51,121,Gasoline,21.291,Fan & Song 2022\nHyundai ix35,3J1020O,30.87,121,Gasoline,3.2083,Fan & Song 2022\nHyundai ix35,3J1020O,3.7,121,Gasoline,0.3315,Fan & Song 2022\nHyundai ix35,3J1020O,2.51,121,Gasoline,0.3733,Fan & Song 2022\nHyundai ix35,3J1020O,128.96,121,Gasoline,12.4559,Fan & Song 2022\nHyundai ix35,3J1020O,132.89,121,Gasoline,10.3537,Fan & Song 2022\nHyundai ix35,3J1020O,32.45,121,Gasoline,4.0432,Fan & Song 2022\nHyundai ix35,3J1020O,4.82,121,Gasoline,0.7181,Fan & Song 2022\nHyundai ix35,3J1020O,5.54,121,Gasoline,0.8514,Fan & Song 2022\nHyundai ix35,3J1020O,11.2,121,Gasoline,1.6213,Fan & Song 2022\nHyundai ix35,3J1020O,42.7,121,Gasoline,3.625,Fan & Song 2022\nHyundai ix35,3J1020O,1.6,121,Gasoline,0.1568,Fan & Song 2022\nHyundai ix35,3J1020O,34.97,121,Gasoline,2.7206,Fan & Song 2022\nHyundai ix35,3J1020O,18.01,121,Gasoline,1.7756,Fan & Song 2022\nHyundai ix35,3J1020O,18.85,121,Gasoline,1.6335,Fan & Song 2022\nHyundai ix35,3J1020O,4.63,121,Gasoline,0.6959,Fan & Song 2022\nHyundai ix35,3J1020O,5.81,121,Gasoline,0.7596,Fan & Song 2022\nHyundai ix35,3J1020O,23.36,121,Gasoline,2.1208,Fan & Song 2022\nHyundai ix35,3J1020O,37.63,121,Gasoline,3.3883,Fan & Song 2022\nHyundai ix35,3J1020O,134.51,121,Gasoline,10.9143,Fan & Song 2022\nHyundai ix35,3J1020O,48.62,121,Gasoline,3.8541,Fan & Song 2022\nHyundai ix35,3J1020O,13.16,121,Gasoline,1.4056,Fan & Song 2022\nHyundai ix35,3J1020O,18.08,121,Gasoline,1.7952,Fan & Song 2022\nHyundai ix35,3J1020O,12.58,121,Gasoline,1.4233,Fan & Song 2022\nHyundai ix35,3J1020O,14.27,121,Gasoline,1.0502,Fan & Song 2022\nHyundai ix35,3J1020O,18.84,121,Gasoline,1.3738,Fan & Song 2022\nHyundai ix35,3J1020O,4.41,121,Gasoline,0.4864,Fan & Song 2022\nHyundai ix35,3J1020O,21.18,121,Gasoline,2.3697,Fan & Song 2022\nHyundai ix35,3J1020O,17.97,121,Gasoline,2.1488,Fan & Song 2022\nHyundai ix35,3J1020O,29.28,121,Gasoline,2.7534,Fan & Song 2022\nHyundai ix35,3J1020O,18.84,121,Gasoline,1.4769,Fan & Song 2022\nHyundai ix35,3J1020O,12.49,121,Gasoline,1.4723,Fan & Song 2022\nHyundai ix35,3J1020O,3.76,121,Gasoline,0.4053,Fan & Song 2022\nHyundai ix35,3J1020O,2.44,121,Gasoline,0.2939,Fan & Song 2022\nHyundai ix35,3J1020O,28.59,121,Gasoline,2.0751,Fan & Song 2022\nHyundai ix35,3J1020O,16.5,121,Gasoline,1.6124,Fan & Song 2022\nHyundai ix35,3J1020O,18.85,121,Gasoline,1.5873,Fan & Song 2022\nHyundai ix35,3J1020O,13.36,121,Gasoline,1.2359,Fan & Song 2022\nHyundai ix35,3J1020O,42.98,121,Gasoline,3.5448,Fan & Song 2022\nHyundai ix35,3J1020O,42.25,121,Gasoline,3.2812,Fan & Song 2022\nHyundai ix35,3J1020O,4.84,121,Gasoline,0.9411,Fan & Song 2022\nHyundai ix35,3J1020O,11.58,121,Gasoline,1.0156,Fan & Song 2022\nHyundai ix35,3J1020O,159.36,121,Gasoline,13.3218,Fan & Song 2022\nHyundai ix35,3J1020O,81.56,121,Gasoline,6.706,Fan & Song 2022\nHyundai ix35,3J1020O,265.22,121,Gasoline,21.9808,Fan & Song 2022\nHyundai ix35,3J1020O,23.71,121,Gasoline,2.5381,Fan & Song 2022\nHyundai ix35,3J1020O,5.28,121,Gasoline,0.6685,Fan & Song 2022\nHyundai ix35,3J1020O,4.59,121,Gasoline,0.6469,Fan & Song 2022\nHyundai ix35,3J1020O,15.14,121,Gasoline,1.4288,Fan & Song 2022\nHyundai ix35,3J1020O,75.92,121,Gasoline,7.5968,Fan & Song 2022\nHyundai ix35,3J1020O,38.48,121,Gasoline,3.3602,Fan & Song 2022\nHyundai ix35,3J1020O,106.97,121,Gasoline,7.5975,Fan & Song 2022\nHyundai ix35,3J1020O,14.83,121,Gasoline,1.6278,Fan & Song 2022\nHyundai ix35,3J1020O,5.27,121,Gasoline,0.6888,Fan & Song 2022\nHyundai ix35,3J1020O,133.16,121,Gasoline,12.3228,Fan & Song 2022\nHyundai ix35,3J1020O,51.86,121,Gasoline,4.8824,Fan & Song 2022\nHyundai ix35,3J1020O,19.61,121,Gasoline,1.7187,Fan & Song 2022\nHyundai ix35,3J1020O,144.57,121,Gasoline,11.672,Fan & Song 2022\nHyundai ix35,3J1020O,15.25,121,Gasoline,1.2337,Fan & Song 2022\nHyundai ix35,3J1020O,139.0,121,Gasoline,10.588,Fan & Song 2022\nHyundai ix35,3J1020O,13.68,121,Gasoline,1.3053,Fan & Song 2022\nHyundai ix35,3J1020O,12.57,121,Gasoline,1.0217,Fan & Song 2022\nHyundai ix35,3J1020O,44.79,121,Gasoline,4.0896,Fan & Song 2022\nHyundai ix35,3J1020O,16.58,121,Gasoline,1.8234,Fan & Song 2022\nHyundai ix35,3J1020O,4.84,121,Gasoline,0.5828,Fan & Song 2022\nHyundai ix35,3J1020O,11.44,121,Gasoline,0.9603,Fan & Song 2022\nHyundai ix35,3J1020O,7.23,121,Gasoline,0.9129,Fan & Song 2022\nHyundai ix35,3J1020O,1.9,121,Gasoline,0.3486,Fan & Song 2022\nHyundai ix35,3J1020O,2.7,121,Gasoline,0.5592,Fan & Song 2022\nHyundai ix35,3J1020O,5.84,121,Gasoline,0.8806,Fan & Song 2022\nHyundai ix35,3J1020O,4.42,121,Gasoline,0.488,Fan & Song 2022\nNissan Teana,2X10829,166.29,110,Gasoline,14.9333,Fan & Song 2022\nNissan Teana,2X1080Q,32.13,104,Gasoline,3.0415,Fan & Song 2022\nNissan Teana,2X10829,85.9,110,Gasoline,6.3947,Fan & Song 2022\nNissan Teana,2X1080Q,1.6,104,Gasoline,0.2118,Fan & Song 2022\nNissan Teana,2X10829,8.77,110,Gasoline,1.096,Fan & Song 2022\nNissan Teana,2X10829,10.14,110,Gasoline,1.131,Fan & Song 2022\nNissan Teana,2X10829,38.62,110,Gasoline,2.688,Fan & Song 2022\nNissan Teana,2X1080Q,7.92,104,Gasoline,0.7534,Fan & Song 2022\nNissan Teana,2X10829,114.99,110,Gasoline,10.9968,Fan & Song 2022\nNissan Teana,2X1080Q,14.18,104,Gasoline,1.374,Fan & Song 2022\nNissan Teana,2X10829,102.86,110,Gasoline,6.3384,Fan & Song 2022\nNissan Teana,2X10829,26.23,110,Gasoline,2.8089,Fan & Song 2022\nNissan Teana,2X10829,3.14,110,Gasoline,0.4034,Fan & Song 2022\nNissan Teana,2X10829,28.77,110,Gasoline,3.1768,Fan & Song 2022\nNissan Teana,2X10829,123.05,110,Gasoline,10.0071,Fan & Song 2022\nNissan Teana,2X10829,65.28,110,Gasoline,5.711,Fan & Song 2022\nNissan Teana,2X10829,5.55,110,Gasoline,0.4924,Fan & Song 2022\nNissan Teana,2X1080Q,6.25,104,Gasoline,0.7266,Fan & Song 2022\nNissan Teana,2X10829,154.66,110,Gasoline,10.4608,Fan & Song 2022\nNissan Teana,2X10829,17.89,110,Gasoline,1.597,Fan & Song 2022\nNissan Teana,2X10829,32.58,110,Gasoline,3.2636,Fan & Song 2022\nNissan Teana,2X10829,2.4,110,Gasoline,0.3347,Fan & Song 2022\nNissan Teana,2X10829,132.4,110,Gasoline,11.3636,Fan & Song 2022\nNissan Teana,2X1080Q,46.42,104,Gasoline,3.9477,Fan & Song 2022\nNissan Teana,2X10829,69.55,110,Gasoline,7.0511,Fan & Song 2022\nNissan Teana,2X1080Q,12.71,104,Gasoline,1.4462,Fan & Song 2022\nNissan Teana,2X1080Q,10.92,104,Gasoline,0.9886,Fan & Song 2022\nNissan Teana,2X1080Q,21.74,104,Gasoline,2.4432,Fan & Song 2022\nNissan Teana,2X10829,15.81,110,Gasoline,1.4129,Fan & Song 2022\nNissan Teana,2X10829,32.56,110,Gasoline,3.1103,Fan & Song 2022\nNissan Teana,2X10829,53.36,110,Gasoline,4.2886,Fan & Song 2022\nNissan Teana,2X10829,31.7,110,Gasoline,2.8122,Fan & Song 2022\nNissan Teana,2X10829,148.99,110,Gasoline,12.3223,Fan & Song 2022\nNissan Teana,2X1080Q,25.08,104,Gasoline,2.1868,Fan & Song 2022\nNissan Teana,2X10829,123.55,110,Gasoline,10.923,Fan & Song 2022\nNissan Teana,2X1080Q,74.99,104,Gasoline,7.0529,Fan & Song 2022\nNissan Teana,2X10829,55.18,110,Gasoline,6.1628,Fan & Song 2022\nNissan Teana,2X10829,102.74,110,Gasoline,11.0976,Fan & Song 2022\nNissan Teana,2X10829,225.14,110,Gasoline,19.9434,Fan & Song 2022\nNissan Teana,2X1080Q,141.39,104,Gasoline,12.7084,Fan & Song 2022\nNissan Teana,2X1080Q,7.63,104,Gasoline,0.7177,Fan & Song 2022\nNissan Teana,2X10829,107.49,110,Gasoline,9.3111,Fan & Song 2022\nNissan Teana,2X10829,12.46,110,Gasoline,1.1393,Fan & Song 2022\nNissan Teana,2X10829,3.92,110,Gasoline,0.4681,Fan & Song 2022\nNissan Teana,2X10829,37.09,110,Gasoline,2.7859,Fan & Song 2022\nNissan Teana,2X10829,90.98,110,Gasoline,8.3425,Fan & Song 2022\nNissan Teana,2X10829,52.29,110,Gasoline,4.7478,Fan & Song 2022\nNissan Teana,2X1080Q,73.05,104,Gasoline,9.0789,Fan & Song 2022\nNissan Teana,2X1080Q,10.04,104,Gasoline,1.293,Fan & Song 2022\nNissan Teana,2X10829,35.59,110,Gasoline,3.7098,Fan & Song 2022\nNissan Teana,2X1080Q,39.54,104,Gasoline,4.3517,Fan & Song 2022\nNissan Teana,2X1080Q,186.2,104,Gasoline,12.3769,Fan & Song 2022\nNissan Teana,2X10829,13.87,110,Gasoline,1.571,Fan & Song 2022\nNissan Teana,2X10829,24.56,110,Gasoline,1.8272,Fan & Song 2022\nNissan Teana,2X10829,117.71,110,Gasoline,9.9523,Fan & Song 2022\nNissan Teana,2X10829,145.49,110,Gasoline,8.7493,Fan & Song 2022\nNissan Teana,2X1080Q,17.03,104,Gasoline,1.3435,Fan & Song 2022\nNissan Teana,2X10829,124.2,110,Gasoline,9.0673,Fan & Song 2022\nNissan Teana,2X10829,151.5,110,Gasoline,8.5959,Fan & Song 2022\nNissan Teana,2X1080Q,33.47,104,Gasoline,3.1208,Fan & Song 2022\nNissan Teana,2X10829,35.71,110,Gasoline,4.4501,Fan & Song 2022\nNissan Teana,2X1080Q,12.14,104,Gasoline,1.0351,Fan & Song 2022\nNissan Teana,2X10829,240.77,110,Gasoline,15.1478,Fan & Song 2022\nNissan Teana,2X10829,18.57,110,Gasoline,1.7157,Fan & Song 2022\nNissan Teana,2X1080Q,31.88,104,Gasoline,3.1236,Fan & Song 2022\nNissan Teana,2X10829,155.51,110,Gasoline,9.6623,Fan & Song 2022\nNissan Teana,2X10829,3.32,110,Gasoline,0.3037,Fan & Song 2022\nNissan Teana,2X10829,16.65,110,Gasoline,1.2052,Fan & Song 2022\nNissan Teana,2X10829,11.28,110,Gasoline,0.7501,Fan & Song 2022\nNissan Teana,2X10829,24.96,110,Gasoline,2.3759,Fan & Song 2022\nNissan Teana,2X1080Q,112.52,104,Gasoline,8.3352,Fan & Song 2022\nNissan Teana,2X1080Q,297.0,104,Gasoline,19.3584,Fan & Song 2022\nNissan Teana,2X1080Q,6.27,104,Gasoline,0.6269,Fan & Song 2022\nNissan Teana,2X10829,21.63,110,Gasoline,2.3876,Fan & Song 2022\nNissan Teana,2X10829,136.82,110,Gasoline,11.0038,Fan & Song 2022\nNissan Teana,2X1080Q,246.95,104,Gasoline,17.2132,Fan & Song 2022\nNissan Teana,2X10829,129.75,110,Gasoline,10.1214,Fan & Song 2022\nNissan Teana,2X1080Q,27.61,104,Gasoline,2.1688,Fan & Song 2022\nNissan Teana,2X1080Q,114.28,104,Gasoline,8.5743,Fan & Song 2022\nNissan Teana,2X10829,57.53,110,Gasoline,4.5823,Fan & Song 2022\nNissan Teana,2X10829,96.2,110,Gasoline,7.1226,Fan & Song 2022\nNissan Teana,2X1080Q,57.3,104,Gasoline,3.267,Fan & Song 2022\nNissan Teana,2X10829,150.95,110,Gasoline,11.1283,Fan & Song 2022\nNissan Teana,2X10829,310.58,110,Gasoline,22.0068,Fan & Song 2022\nNissan Teana,2X1080Q,19.2,104,Gasoline,1.2566,Fan & Song 2022\nNissan Teana,2X10829,8.24,110,Gasoline,0.6086,Fan & Song 2022\nNissan Teana,2X1080Q,192.05,104,Gasoline,15.5864,Fan & Song 2022\nNissan Teana,2X1080Q,2.66,104,Gasoline,0.4809,Fan & Song 2022\nNissan Teana,2X1080Q,3.17,104,Gasoline,0.4269,Fan & Song 2022\nNissan Teana,2X10829,6.15,110,Gasoline,0.852,Fan & Song 2022\nNissan Teana,2X10829,87.3,110,Gasoline,5.4268,Fan & Song 2022\nNissan Teana,2X10829,7.29,110,Gasoline,0.5384,Fan & Song 2022\nNissan Teana,2X10829,10.47,110,Gasoline,0.938,Fan & Song 2022\nNissan Teana,2X10829,106.53,110,Gasoline,7.6999,Fan & Song 2022\nNissan Teana,2X1080Q,30.75,104,Gasoline,2.7213,Fan & Song 2022\nNissan Teana,2X10829,4.41,110,Gasoline,0.4072,Fan & Song 2022\nNissan Teana,2X10829,25.33,110,Gasoline,1.6954,Fan & Song 2022\nNissan Teana,2X1080Q,120.91,104,Gasoline,8.7987,Fan & Song 2022\nNissan Teana,2X1080Q,3.05,104,Gasoline,0.3582,Fan & Song 2022\nNissan Teana,2X10829,152.11,110,Gasoline,13.5452,Fan & Song 2022\nNissan Teana,2X10829,14.37,110,Gasoline,0.8741,Fan & Song 2022\nNissan Teana,2X10829,33.18,110,Gasoline,2.9211,Fan & Song 2022\nNissan Teana,2X10829,41.61,110,Gasoline,3.9825,Fan & Song 2022\nNissan Teana,2X1080Q,19.36,104,Gasoline,1.7688,Fan & Song 2022\nNissan Teana,2X10829,20.08,110,Gasoline,1.7681,Fan & Song 2022\nNissan Teana,2X1080Q,17.67,104,Gasoline,1.4264,Fan & Song 2022\nNissan Teana,2X10829,2.75,110,Gasoline,0.519,Fan & Song 2022\nNissan Teana,2X10829,110.83,110,Gasoline,9.8973,Fan & Song 2022\nNissan Teana,2X1080Q,9.53,104,Gasoline,1.0216,Fan & Song 2022\nNissan Teana,2X1080Q,21.74,104,Gasoline,1.796,Fan & Song 2022\nNissan Teana,2X10829,3.72,110,Gasoline,0.5689,Fan & Song 2022\nNissan Teana,2X10829,2.51,110,Gasoline,0.3487,Fan & Song 2022\nNissan Teana,2X10829,20.78,110,Gasoline,1.4878,Fan & Song 2022\nNissan Teana,2X1080Q,34.05,104,Gasoline,2.7173,Fan & Song 2022\nNissan Teana,2X1080Q,32.83,104,Gasoline,2.3324,Fan & Song 2022\nNissan Teana,2X1080Q,7.86,104,Gasoline,0.795,Fan & Song 2022\nNissan Teana,2X10829,12.25,110,Gasoline,1.2311,Fan & Song 2022\nNissan Teana,2X1080Q,18.82,104,Gasoline,1.2296,Fan & Song 2022\nNissan Teana,2X10829,39.57,110,Gasoline,3.3394,Fan & Song 2022\nNissan Teana,2X1080Q,13.7,104,Gasoline,1.0047,Fan & Song 2022\nNissan Teana,2X10829,10.27,110,Gasoline,1.3584,Fan & Song 2022\nNissan Teana,2X1080Q,86.26,104,Gasoline,8.6731,Fan & Song 2022\nNissan Teana,2X1080Q,47.66,104,Gasoline,5.7471,Fan & Song 2022\nNissan Teana,2X10829,49.17,110,Gasoline,5.9337,Fan & Song 2022\nNissan Teana,2X1080Q,10.38,104,Gasoline,1.0968,Fan & Song 2022\nNissan Teana,2X1080Q,221.46,104,Gasoline,15.2378,Fan & Song 2022\nNissan Teana,2X10829,8.75,110,Gasoline,1.0508,Fan & Song 2022\nNissan Teana,2X1080Q,39.07,104,Gasoline,2.8194,Fan & Song 2022\nNissan Teana,2X10829,5.33,110,Gasoline,0.6404,Fan & Song 2022\nNissan Teana,2X10829,13.41,110,Gasoline,1.2358,Fan & Song 2022\nNissan Teana,2X1080Q,41.39,104,Gasoline,4.2998,Fan & Song 2022\nNissan Teana,2X10829,19.02,110,Gasoline,1.3285,Fan & Song 2022\nNissan Teana,2X10829,24.15,110,Gasoline,1.9775,Fan & Song 2022\nNissan Teana,2X10829,26.15,110,Gasoline,2.303,Fan & Song 2022\nNissan Teana,2X1080Q,5.21,104,Gasoline,0.6094,Fan & Song 2022\nNissan Teana,2X10829,44.07,110,Gasoline,3.6284,Fan & Song 2022\nNissan Teana,2X1080Q,15.7,104,Gasoline,1.0195,Fan & Song 2022\nNissan Teana,2X10829,3.56,110,Gasoline,0.4984,Fan & Song 2022\nNissan Teana,2X1080Q,186.33,104,Gasoline,13.274,Fan & Song 2022\nNissan Teana,2X10829,13.06,110,Gasoline,1.7575,Fan & Song 2022\nNissan Teana,2X1080Q,29.31,104,Gasoline,2.4017,Fan & Song 2022\nNissan Teana,2X1080Q,90.18,104,Gasoline,7.8662,Fan & Song 2022\nNissan Teana,2X10829,14.76,110,Gasoline,1.3245,Fan & Song 2022\nNissan Teana,2X1080Q,16.68,104,Gasoline,2.0961,Fan & Song 2022\nNissan Teana,2X1080Q,16.76,104,Gasoline,1.5641,Fan & Song 2022\nNissan Teana,2X1080Q,20.27,104,Gasoline,1.8282,Fan & Song 2022\nNissan Teana,2X10829,17.14,110,Gasoline,1.4543,Fan & Song 2022\nNissan Teana,2X1080Q,48.36,104,Gasoline,5.1578,Fan & Song 2022\nNissan Teana,2X10829,102.74,110,Gasoline,6.5544,Fan & Song 2022\nNissan Teana,2X10829,1.97,110,Gasoline,0.1991,Fan & Song 2022\nNissan Teana,2X1080Q,40.63,104,Gasoline,3.7005,Fan & Song 2022\nNissan Teana,2X1080Q,14.25,104,Gasoline,1.5932,Fan & Song 2022\nNissan Teana,2X1080Q,4.95,104,Gasoline,0.9388,Fan & Song 2022\nNissan Teana,2X1080Q,259.16,104,Gasoline,18.646,Fan & Song 2022\nNissan Teana,2X1080Q,32.71,104,Gasoline,3.2312,Fan & Song 2022\nNissan Teana,2X10829,3.71,110,Gasoline,0.6022,Fan & Song 2022\nNissan Teana,2X1080Q,186.61,104,Gasoline,12.5666,Fan & Song 2022\nNissan Teana,2X1080Q,109.85,104,Gasoline,7.5035,Fan & Song 2022\nNissan Teana,2X1080Q,1.88,104,Gasoline,0.2086,Fan & Song 2022\nNissan Teana,2X10829,3.83,110,Gasoline,0.433,Fan & Song 2022\nNissan Teana,2X1080Q,2.37,104,Gasoline,0.2441,Fan & Song 2022\nNissan Teana,2X1080Q,2.67,104,Gasoline,0.3862,Fan & Song 2022\nNissan Teana,2X1080Q,14.65,104,Gasoline,1.8426,Fan & Song 2022\nNissan Teana,2X10829,177.52,110,Gasoline,14.1122,Fan & Song 2022\nNissan Teana,2X1080Q,7.66,104,Gasoline,1.0529,Fan & Song 2022\nNissan Teana,2X1080Q,1.79,104,Gasoline,0.2449,Fan & Song 2022\nNissan Teana,2X1080Q,204.17,104,Gasoline,15.3882,Fan & Song 2022\nNissan Teana,2X10829,28.81,110,Gasoline,2.8946,Fan & Song 2022\nNissan Teana,2X10829,1.62,110,Gasoline,0.1431,Fan & Song 2022\nNissan Teana,2X1080Q,10.1,104,Gasoline,0.7574,Fan & Song 2022\nNissan Teana,2X1080Q,138.08,104,Gasoline,9.7017,Fan & Song 2022\nNissan Teana,2X10829,193.83,110,Gasoline,10.8173,Fan & Song 2022\nNissan Teana,2X10829,144.28,110,Gasoline,8.3514,Fan & Song 2022\nNissan Teana,2X1080Q,5.73,104,Gasoline,0.6573,Fan & Song 2022\nNissan Teana,2X1080Q,12.07,104,Gasoline,1.2922,Fan & Song 2022\nNissan Teana,2X1080Q,11.59,104,Gasoline,1.163,Fan & Song 2022\nNissan Teana,2X10829,35.36,110,Gasoline,3.8481,Fan & Song 2022\nNissan Teana,2X10829,11.34,110,Gasoline,0.8197,Fan & Song 2022\nNissan Teana,2X10829,157.53,110,Gasoline,13.1028,Fan & Song 2022\nNissan Teana,2X10829,122.04,110,Gasoline,9.1805,Fan & Song 2022\nNissan Teana,2X10829,5.98,110,Gasoline,0.5557,Fan & Song 2022\nNissan Teana,2X10829,100.52,110,Gasoline,10.9928,Fan & Song 2022\nNissan Teana,2X1080Q,35.75,104,Gasoline,3.1489,Fan & Song 2022\nNissan Teana,2X1080Q,17.73,104,Gasoline,1.6636,Fan & Song 2022\nNissan Teana,2X10829,18.56,110,Gasoline,1.8535,Fan & Song 2022\nNissan Teana,2X10829,101.88,110,Gasoline,8.4718,Fan & Song 2022\nNissan Teana,2X10829,106.01,110,Gasoline,9.4993,Fan & Song 2022\nNissan Teana,2X1080Q,4.21,104,Gasoline,0.5356,Fan & Song 2022\nNissan Teana,2X10829,5.24,110,Gasoline,0.6411,Fan & Song 2022\nNissan Teana,2X10829,6.62,110,Gasoline,0.7075,Fan & Song 2022\nNissan Teana,2X1080Q,17.63,104,Gasoline,1.5174,Fan & Song 2022\nNissan Teana,2X1080Q,4.4,104,Gasoline,0.4158,Fan & Song 2022\nNissan Teana,2X1080Q,105.26,104,Gasoline,9.5942,Fan & Song 2022\nNissan Teana,2X10829,3.92,110,Gasoline,0.364,Fan & Song 2022\nNissan Teana,2X10829,45.89,110,Gasoline,4.442,Fan & Song 2022\nNissan Teana,2X10829,213.83,110,Gasoline,23.3468,Fan & Song 2022\nNissan Teana,2X1080Q,13.66,104,Gasoline,1.3407,Fan & Song 2022\nNissan Teana,2X10829,40.68,110,Gasoline,3.184,Fan & Song 2022\nNissan Teana,2X1080Q,7.23,104,Gasoline,0.7733,Fan & Song 2022\nNissan Teana,2X10829,143.75,110,Gasoline,14.4547,Fan & Song 2022\nNissan Teana,2X10829,43.58,110,Gasoline,3.1038,Fan & Song 2022\nNissan Teana,2X1080Q,146.03,104,Gasoline,9.7892,Fan & Song 2022\nNissan Teana,2X1080Q,2.31,104,Gasoline,0.2789,Fan & Song 2022\nNissan Teana,2X10829,3.31,110,Gasoline,0.3771,Fan & Song 2022\nNissan Teana,2X10829,191.94,110,Gasoline,12.6236,Fan & Song 2022\nNissan Teana,2X1080Q,72.32,104,Gasoline,6.2811,Fan & Song 2022\nNissan Teana,2X10829,67.61,110,Gasoline,5.4601,Fan & Song 2022\nNissan Teana,2X1080Q,21.56,104,Gasoline,1.6976,Fan & Song 2022\nNissan Teana,2X1080Q,127.29,104,Gasoline,7.4547,Fan & Song 2022\nNissan Teana,2X10829,94.92,110,Gasoline,9.0289,Fan & Song 2022\nNissan Teana,2X10829,5.07,110,Gasoline,0.4568,Fan & Song 2022\nNissan Teana,2X10829,252.02,110,Gasoline,23.3194,Fan & Song 2022\nNissan Teana,2X10829,4.31,110,Gasoline,0.3576,Fan & Song 2022\nNissan Teana,2X1080Q,4.56,104,Gasoline,0.6489,Fan & Song 2022\nNissan Teana,2X10829,3.88,110,Gasoline,0.4415,Fan & Song 2022\nNissan Teana,2X10829,128.05,110,Gasoline,8.3498,Fan & Song 2022\nNissan Teana,2X10829,123.7,110,Gasoline,8.2736,Fan & Song 2022\nNissan Teana,2X10829,56.84,110,Gasoline,5.4822,Fan & Song 2022\nNissan Teana,2X1080Q,127.74,104,Gasoline,7.9756,Fan & Song 2022\nNissan Teana,2X10829,12.62,110,Gasoline,1.24,Fan & Song 2022\nNissan Teana,2X10829,15.47,110,Gasoline,1.8967,Fan & Song 2022\nNissan Teana,2X1080Q,4.43,104,Gasoline,0.9374,Fan & Song 2022\nNissan Teana,2X10829,118.06,110,Gasoline,6.7525,Fan & Song 2022\nNissan Teana,2X1080Q,15.52,104,Gasoline,1.16,Fan & Song 2022\nNissan Teana,2X10829,64.95,110,Gasoline,4.3345,Fan & Song 2022\nNissan Teana,2X10829,34.3,110,Gasoline,3.3724,Fan & Song 2022\nNissan Teana,2X1080Q,185.33,104,Gasoline,12.1152,Fan & Song 2022\nNissan Teana,2X10829,64.99,110,Gasoline,5.8713,Fan & Song 2022\nNissan Teana,2X10829,63.27,110,Gasoline,9.7604,Fan & Song 2022\nNissan Teana,2X10829,17.51,110,Gasoline,2.1426,Fan & Song 2022\nNissan Teana,2X10829,158.03,110,Gasoline,10.186,Fan & Song 2022\nNissan Teana,2X10829,153.44,110,Gasoline,11.4864,Fan & Song 2022\nNissan Teana,2X10829,19.38,110,Gasoline,2.5023,Fan & Song 2022\nNissan Teana,2X10829,30.89,110,Gasoline,2.3173,Fan & Song 2022\nNissan Teana,2X1080Q,1.68,104,Gasoline,0.2216,Fan & Song 2022\nNissan Teana,2X10829,7.22,110,Gasoline,1.2097,Fan & Song 2022\nNissan Teana,2X10829,16.63,110,Gasoline,1.9522,Fan & Song 2022\nNissan Teana,2X1080Q,26.15,104,Gasoline,3.5037,Fan & Song 2022\nNissan Teana,2X1080Q,10.93,104,Gasoline,0.7638,Fan & Song 2022\nNissan Teana,2X1080Q,61.46,104,Gasoline,5.061,Fan & Song 2022\nNissan Teana,2X10829,20.01,110,Gasoline,2.2685,Fan & Song 2022\nNissan Teana,2X1080Q,34.95,104,Gasoline,2.5524,Fan & Song 2022\nNissan Teana,2X1080Q,10.28,104,Gasoline,1.245,Fan & Song 2022\nNissan Teana,2X10829,12.34,110,Gasoline,0.9155,Fan & Song 2022\nNissan Teana,2X10829,40.65,110,Gasoline,3.3253,Fan & Song 2022\nNissan Teana,2X10829,44.29,110,Gasoline,6.6199,Fan & Song 2022\nNissan Teana,2X1080Q,10.12,104,Gasoline,1.3769,Fan & Song 2022\nNissan Teana,2X1080Q,54.47,104,Gasoline,5.396,Fan & Song 2022\nNissan Teana,2X10829,155.23,110,Gasoline,12.0997,Fan & Song 2022\nNissan Teana,2X10829,13.03,110,Gasoline,0.9536,Fan & Song 2022\nNissan Teana,2X10829,12.98,110,Gasoline,2.1416,Fan & Song 2022\nNissan Teana,2X10829,175.1,110,Gasoline,14.468,Fan & Song 2022\nNissan Teana,2X1080Q,26.49,104,Gasoline,2.6171,Fan & Song 2022\nNissan Teana,2X1080Q,22.76,104,Gasoline,1.7091,Fan & Song 2022\nNissan Teana,2X10829,28.99,110,Gasoline,4.3001,Fan & Song 2022\nNissan Teana,2X10829,82.8,110,Gasoline,4.9172,Fan & Song 2022\nNissan Teana,2X10829,20.51,110,Gasoline,2.9584,Fan & Song 2022\nNissan Teana,2X10829,131.69,110,Gasoline,14.1397,Fan & Song 2022\nNissan Teana,2X1080Q,4.07,104,Gasoline,0.6744,Fan & Song 2022\nNissan Teana,2X10829,8.77,110,Gasoline,0.8926,Fan & Song 2022\nNissan Teana,2X10829,31.75,110,Gasoline,2.2737,Fan & Song 2022\nNissan Teana,2X10829,27.65,110,Gasoline,3.7665,Fan & Song 2022\nNissan Teana,2X1080Q,5.77,104,Gasoline,0.8647,Fan & Song 2022\nNissan Teana,2X10829,15.15,110,Gasoline,1.4712,Fan & Song 2022\nNissan Teana,2X1080Q,2.59,104,Gasoline,0.2632,Fan & Song 2022\nNissan Teana,2X10829,29.26,110,Gasoline,1.9491,Fan & Song 2022\nNissan Teana,2X1080Q,8.75,104,Gasoline,1.2199,Fan & Song 2022\nNissan Teana,2X10829,124.82,110,Gasoline,12.2966,Fan & Song 2022\nNissan Teana,2X10829,2.45,110,Gasoline,0.3461,Fan & Song 2022\nNissan Teana,2X1080Q,26.27,104,Gasoline,2.2675,Fan & Song 2022\nNissan Teana,2X1080Q,32.32,104,Gasoline,3.5055,Fan & Song 2022\nNissan Teana,2X1080Q,43.69,104,Gasoline,4.0948,Fan & Song 2022\nNissan Teana,2X10829,101.73,110,Gasoline,10.4624,Fan & Song 2022\nNissan Teana,2X1080Q,4.18,104,Gasoline,0.6086,Fan & Song 2022\nNissan Teana,2X1080Q,6.04,104,Gasoline,0.9562,Fan & Song 2022\nNissan Teana,2X1080Q,48.0,104,Gasoline,5.2945,Fan & Song 2022\nNissan Teana,2X1080Q,19.2,104,Gasoline,1.7305,Fan & Song 2022\nNissan Teana,2X10829,56.43,110,Gasoline,4.1893,Fan & Song 2022\nNissan Teana,2X10829,133.26,110,Gasoline,13.0842,Fan & Song 2022\nNissan Teana,2X10829,110.31,110,Gasoline,10.3963,Fan & Song 2022\nNissan Teana,2X10829,36.0,110,Gasoline,2.8642,Fan & Song 2022\nNissan Teana,2X10829,102.61,110,Gasoline,11.0323,Fan & Song 2022\nNissan Teana,2X10829,112.17,110,Gasoline,11.7821,Fan & Song 2022\nNissan Teana,2X10829,41.12,110,Gasoline,4.4519,Fan & Song 2022\nNissan Teana,2X1080Q,20.33,104,Gasoline,1.9623,Fan & Song 2022\nNissan Teana,2X1080Q,52.19,104,Gasoline,7.8897,Fan & Song 2022\nNissan Teana,2X10829,10.56,110,Gasoline,1.3744,Fan & Song 2022\nNissan Teana,2X1080Q,27.61,104,Gasoline,2.7943,Fan & Song 2022\nNissan Teana,2X1080Q,8.09,104,Gasoline,0.9142,Fan & Song 2022\nNissan Teana,2X1080Q,114.63,104,Gasoline,12.1346,Fan & Song 2022\nNissan Teana,2X10829,36.02,110,Gasoline,3.5905,Fan & Song 2022\nNissan Teana,2X10829,44.46,110,Gasoline,3.1149,Fan & Song 2022\nNissan Teana,2X10829,18.6,110,Gasoline,2.4007,Fan & Song 2022\nNissan Teana,2X1080Q,74.57,104,Gasoline,7.259,Fan & Song 2022\nNissan Teana,2X1080Q,32.98,104,Gasoline,2.3656,Fan & Song 2022\nNissan Teana,2X10829,11.15,110,Gasoline,0.8877,Fan & Song 2022\nNissan Teana,2X10829,114.08,110,Gasoline,7.5607,Fan & Song 2022\nNissan Teana,2X1080Q,2.65,104,Gasoline,0.5958,Fan & Song 2022\nNissan Teana,2X1080Q,61.17,104,Gasoline,3.9768,Fan & Song 2022\nNissan Teana,2X10829,82.74,110,Gasoline,8.1406,Fan & Song 2022\nNissan Teana,2X1080Q,10.67,104,Gasoline,1.6336,Fan & Song 2022\nNissan Teana,2X10829,179.7,110,Gasoline,12.5723,Fan & Song 2022\nNissan Teana,2X10829,15.78,110,Gasoline,1.5662,Fan & Song 2022\nNissan Teana,2X1080Q,34.57,104,Gasoline,4.7074,Fan & Song 2022\nNissan Teana,2X10829,684.17,110,Gasoline,44.3412,Fan & Song 2022\nNissan Teana,2X1080Q,3.42,104,Gasoline,0.4599,Fan & Song 2022\nNissan Teana,2X10829,10.36,110,Gasoline,1.5931,Fan & Song 2022\nNissan Teana,2X1080Q,10.53,104,Gasoline,0.7883,Fan & Song 2022\nNissan Teana,2X1080Q,35.29,104,Gasoline,3.3365,Fan & Song 2022\nNissan Teana,2X10829,36.79,110,Gasoline,2.3631,Fan & Song 2022\nNissan Teana,2X10829,17.68,110,Gasoline,3.2606,Fan & Song 2022\nNissan Teana,2X1080Q,155.31,104,Gasoline,10.5774,Fan & Song 2022\nNissan Teana,2X10829,112.89,110,Gasoline,7.7027,Fan & Song 2022\nNissan Teana,2X10829,113.78,110,Gasoline,11.7268,Fan & Song 2022\nNissan Teana,2X10829,103.72,110,Gasoline,8.2277,Fan & Song 2022\nNissan Teana,2X10829,26.64,110,Gasoline,2.5695,Fan & Song 2022\nNissan Teana,2X10829,26.2,110,Gasoline,2.6628,Fan & Song 2022\nNissan Teana,2X10829,172.26,110,Gasoline,19.7339,Fan & Song 2022\nNissan Teana,2X1080Q,15.03,104,Gasoline,1.4378,Fan & Song 2022\nNissan Teana,2X10829,39.47,110,Gasoline,2.6308,Fan & Song 2022\nNissan Teana,2X1080Q,258.11,104,Gasoline,17.3981,Fan & Song 2022\nNissan Teana,2X10829,36.84,110,Gasoline,3.9339,Fan & Song 2022\nNissan Teana,2X10829,25.64,110,Gasoline,2.2682,Fan & Song 2022\nNissan Teana,2X1080Q,37.14,104,Gasoline,2.7227,Fan & Song 2022\nNissan Teana,2X10829,41.87,110,Gasoline,4.3704,Fan & Song 2022\nNissan Teana,2X10829,111.44,110,Gasoline,7.8342,Fan & Song 2022\nNissan Teana,2X10829,63.26,110,Gasoline,8.518,Fan & Song 2022\nNissan Teana,2X1080Q,2.46,104,Gasoline,0.436,Fan & Song 2022\nNissan Teana,2X10829,49.05,110,Gasoline,6.1689,Fan & Song 2022\nNissan Teana,2X1080Q,24.92,104,Gasoline,3.8613,Fan & Song 2022\nNissan Teana,2X10829,108.18,110,Gasoline,8.0709,Fan & Song 2022\nNissan Teana,2X10829,5.01,110,Gasoline,0.6114,Fan & Song 2022\nNissan Teana,2X1080Q,5.75,104,Gasoline,0.84,Fan & Song 2022\nNissan Teana,2X10829,43.5,110,Gasoline,3.5639,Fan & Song 2022\nNissan Teana,2X10829,224.69,110,Gasoline,15.1228,Fan & Song 2022\nNissan Teana,2X10829,189.31,110,Gasoline,17.6471,Fan & Song 2022\nNissan Teana,2X10829,30.48,110,Gasoline,3.4908,Fan & Song 2022\nNissan Teana,2X10829,36.47,110,Gasoline,2.6988,Fan & Song 2022\nNissan Teana,2X10829,27.38,110,Gasoline,4.0156,Fan & Song 2022\nNissan Teana,2X1080Q,9.45,104,Gasoline,0.9827,Fan & Song 2022\nNissan Teana,2X1080Q,32.54,104,Gasoline,2.781,Fan & Song 2022\nNissan Teana,2X10829,36.58,110,Gasoline,2.7904,Fan & Song 2022\nNissan Teana,2X10829,41.01,110,Gasoline,4.4222,Fan & Song 2022\nNissan Teana,2X10829,23.81,110,Gasoline,2.3197,Fan & Song 2022\nNissan Teana,2X10829,36.35,110,Gasoline,4.6759,Fan & Song 2022\nNissan Teana,2X1080Q,228.56,104,Gasoline,15.9018,Fan & Song 2022\nNissan Teana,2X10829,182.21,110,Gasoline,16.6133,Fan & Song 2022\nNissan Teana,2X10829,169.12,110,Gasoline,18.8345,Fan & Song 2022\nNissan Teana,2X10829,19.25,110,Gasoline,1.8689,Fan & Song 2022\nNissan Teana,2X10829,62.28,110,Gasoline,10.6754,Fan & Song 2022\nNissan Teana,2X10829,53.1,110,Gasoline,6.0615,Fan & Song 2022\nNissan Teana,2X10829,125.69,110,Gasoline,11.2304,Fan & Song 2022\nNissan Teana,2X10829,108.11,110,Gasoline,7.6728,Fan & Song 2022\nNissan Teana,2X10829,39.67,110,Gasoline,2.8157,Fan & Song 2022\nNissan Teana,2X1080Q,7.87,104,Gasoline,1.3863,Fan & Song 2022\nNissan Teana,2X10829,86.3,110,Gasoline,11.5457,Fan & Song 2022\nNissan Teana,2X1080Q,4.62,104,Gasoline,0.5153,Fan & Song 2022\nNissan Teana,2X10829,113.68,110,Gasoline,11.871,Fan & Song 2022\nNissan Teana,2X1080Q,61.63,104,Gasoline,3.9448,Fan & Song 2022\nNissan Teana,2X1080Q,132.52,104,Gasoline,11.8354,Fan & Song 2022\nNissan Teana,2X1080Q,2.79,104,Gasoline,0.4856,Fan & Song 2022\nNissan Teana,2X10829,57.79,110,Gasoline,5.5192,Fan & Song 2022\nNissan Teana,2X10829,33.06,110,Gasoline,3.4191,Fan & Song 2022\nNissan Teana,2X10829,28.4,110,Gasoline,4.383,Fan & Song 2022\nNissan Teana,2X10829,53.76,110,Gasoline,6.3551,Fan & Song 2022\nNissan Teana,2X1080Q,22.18,104,Gasoline,1.9636,Fan & Song 2022\nNissan Teana,2X1080Q,31.03,104,Gasoline,3.1974,Fan & Song 2022\nNissan Teana,2X10829,52.74,110,Gasoline,3.7627,Fan & Song 2022\nNissan Teana,2X1080Q,112.35,104,Gasoline,10.906,Fan & Song 2022\nNissan Teana,2X1080Q,106.6,104,Gasoline,9.5931,Fan & Song 2022\nNissan Teana,2X10829,110.88,110,Gasoline,10.7378,Fan & Song 2022\nNissan Teana,2X10829,33.29,110,Gasoline,4.0668,Fan & Song 2022\nNissan Teana,2X10829,8.92,110,Gasoline,0.9245,Fan & Song 2022\nNissan Teana,2X1080Q,139.15,104,Gasoline,9.0177,Fan & Song 2022\nNissan Teana,2X10829,148.25,110,Gasoline,11.1764,Fan & Song 2022\nNissan Teana,2X10829,4.88,110,Gasoline,0.6032,Fan & Song 2022\nNissan Teana,2X1080Q,189.78,104,Gasoline,11.3128,Fan & Song 2022\nNissan Teana,2X10829,2.51,110,Gasoline,0.4828,Fan & Song 2022\nNissan Teana,2X10829,32.28,110,Gasoline,5.5093,Fan & Song 2022\nNissan Teana,2X10829,239.96,110,Gasoline,15.2088,Fan & Song 2022\nNissan Teana,2X1080Q,113.38,104,Gasoline,13.2862,Fan & Song 2022\nNissan Teana,2X1080Q,2.9,104,Gasoline,0.4004,Fan & Song 2022\nNissan Teana,2X10829,18.3,110,Gasoline,3.018,Fan & Song 2022\nNissan Teana,2X1080Q,105.97,104,Gasoline,11.7516,Fan & Song 2022\nNissan Teana,2X1080Q,21.54,104,Gasoline,1.9136,Fan & Song 2022\nNissan Teana,2X1080Q,35.52,104,Gasoline,6.0445,Fan & Song 2022\nNissan Teana,2X10829,4.94,110,Gasoline,0.6314,Fan & Song 2022\nNissan Teana,2X10829,24.97,110,Gasoline,2.5713,Fan & Song 2022\nNissan Teana,2X10829,147.3,110,Gasoline,13.7445,Fan & Song 2022\nNissan Teana,2X10829,56.58,110,Gasoline,4.5602,Fan & Song 2022\nNissan Teana,2X1080Q,2.48,104,Gasoline,0.413,Fan & Song 2022\nNissan Teana,2X10829,13.08,110,Gasoline,1.5132,Fan & Song 2022\nNissan Teana,2X10829,115.06,110,Gasoline,12.0877,Fan & Song 2022\nNissan Teana,2X10829,30.62,110,Gasoline,3.5181,Fan & Song 2022\nNissan Teana,2X1080Q,116.63,104,Gasoline,7.7302,Fan & Song 2022\nNissan Teana,2X10829,163.94,110,Gasoline,15.008,Fan & Song 2022\nNissan Teana,2X10829,26.68,110,Gasoline,1.9059,Fan & Song 2022\nNissan Teana,2X10829,299.87,110,Gasoline,19.6044,Fan & Song 2022\nNissan Teana,2X1080Q,2.03,104,Gasoline,0.3745,Fan & Song 2022\nNissan Teana,2X1080Q,113.26,104,Gasoline,10.0649,Fan & Song 2022\nNissan Teana,2X10829,50.49,110,Gasoline,7.4206,Fan & Song 2022\nNissan Teana,2X1080Q,1.77,104,Gasoline,0.19,Fan & Song 2022\nNissan Teana,2X10829,64.52,110,Gasoline,3.6289,Fan & Song 2022\nNissan Teana,2X10829,33.22,110,Gasoline,2.0136,Fan & Song 2022\nNissan Teana,2X1080Q,28.44,104,Gasoline,3.0959,Fan & Song 2022\nNissan Teana,2X1080Q,32.54,104,Gasoline,2.2818,Fan & Song 2022\nNissan Teana,2X1080Q,12.01,104,Gasoline,1.9723,Fan & Song 2022\nNissan Teana,2X10829,33.4,110,Gasoline,3.828,Fan & Song 2022\nNissan Teana,2X10829,5.14,110,Gasoline,0.8979,Fan & Song 2022\nNissan Teana,2X1080Q,2.23,104,Gasoline,0.2533,Fan & Song 2022\nNissan Teana,2X1080Q,3.36,104,Gasoline,0.4465,Fan & Song 2022\nNissan Teana,2X10829,104.26,110,Gasoline,6.2558,Fan & Song 2022\nNissan Teana,2X1080Q,37.03,104,Gasoline,3.6812,Fan & Song 2022\nNissan Teana,2X1080Q,10.02,104,Gasoline,1.4182,Fan & Song 2022\nNissan Teana,2X10829,136.2,110,Gasoline,12.3972,Fan & Song 2022\nNissan Teana,2X10829,37.7,110,Gasoline,4.4402,Fan & Song 2022\nNissan Teana,2X1080Q,8.44,104,Gasoline,1.3389,Fan & Song 2022\nNissan Teana,2X1080Q,15.51,104,Gasoline,1.4603,Fan & Song 2022\nNissan Teana,2X10829,4.29,110,Gasoline,0.3665,Fan & Song 2022\nNissan Teana,2X1080Q,17.31,104,Gasoline,1.6602,Fan & Song 2022\nNissan Teana,2X1080Q,15.12,104,Gasoline,2.1224,Fan & Song 2022\nNissan Teana,2X10829,160.93,110,Gasoline,11.1735,Fan & Song 2022\nNissan Teana,2X10829,3.38,110,Gasoline,0.3192,Fan & Song 2022\nNissan Teana,2X1080Q,10.48,104,Gasoline,0.6909,Fan & Song 2022\nNissan Teana,2X1080Q,7.7,104,Gasoline,1.3877,Fan & Song 2022\nNissan Teana,2X10829,20.24,110,Gasoline,2.0217,Fan & Song 2022\nNissan Teana,2X10829,17.1,110,Gasoline,2.3452,Fan & Song 2022\nNissan Teana,2X10829,3.66,110,Gasoline,0.4562,Fan & Song 2022\nNissan Teana,2X10829,9.53,110,Gasoline,0.8087,Fan & Song 2022\nNissan Teana,2X10829,27.02,110,Gasoline,2.9742,Fan & Song 2022\nNissan Teana,2X10829,9.82,110,Gasoline,0.8258,Fan & Song 2022\nNissan Teana,2X10829,15.44,110,Gasoline,1.3951,Fan & Song 2022\nNissan Teana,2X10829,192.23,110,Gasoline,16.0895,Fan & Song 2022\nNissan Teana,2X10829,9.6,110,Gasoline,0.7229,Fan & Song 2022\nNissan Teana,2X10829,25.05,110,Gasoline,2.385,Fan & Song 2022\nNissan Teana,2X10829,184.84,110,Gasoline,15.1696,Fan & Song 2022\nNissan Teana,2X10829,123.98,110,Gasoline,13.9413,Fan & Song 2022\nNissan Teana,2X1080Q,36.44,104,Gasoline,2.9802,Fan & Song 2022\nNissan Teana,2X1080Q,32.6,104,Gasoline,2.7529,Fan & Song 2022\nNissan Teana,2X1080Q,1.55,104,Gasoline,0.2113,Fan & Song 2022\nNissan Teana,2X1080Q,6.1,104,Gasoline,0.4898,Fan & Song 2022\nNissan Teana,2X10829,6.29,110,Gasoline,0.6284,Fan & Song 2022\nNissan Teana,2X10829,41.81,110,Gasoline,3.451,Fan & Song 2022\nNissan Teana,2X10829,8.51,110,Gasoline,0.8054,Fan & Song 2022\nNissan Teana,2X1080Q,100.17,104,Gasoline,7.9393,Fan & Song 2022\nNissan Teana,2X1080Q,6.28,104,Gasoline,0.868,Fan & Song 2022\nNissan Teana,2X1080Q,5.57,104,Gasoline,0.7951,Fan & Song 2022\nNissan Teana,2X10829,29.08,110,Gasoline,2.409,Fan & Song 2022\nNissan Teana,2X10829,4.89,110,Gasoline,0.5144,Fan & Song 2022\nNissan Teana,2X1080Q,192.72,104,Gasoline,13.94,Fan & Song 2022\nNissan Teana,2X1080Q,232.53,104,Gasoline,17.7389,Fan & Song 2022\nNissan Teana,2X10829,25.82,110,Gasoline,2.1992,Fan & Song 2022\nNissan Teana,2X10829,2.24,110,Gasoline,0.3023,Fan & Song 2022\nNissan Teana,2X10829,6.94,110,Gasoline,0.6346,Fan & Song 2022\nNissan Teana,2X1080Q,35.19,104,Gasoline,2.6239,Fan & Song 2022\nNissan Teana,2X10829,8.7,110,Gasoline,0.5699,Fan & Song 2022\nNissan Teana,2X10829,231.5,110,Gasoline,19.2424,Fan & Song 2022\nNissan Teana,2X10829,271.26,110,Gasoline,16.5205,Fan & Song 2022\nNissan Teana,2X1080Q,296.29,104,Gasoline,23.3245,Fan & Song 2022\nNissan Teana,2X1080Q,5.17,104,Gasoline,0.6932,Fan & Song 2022\nNissan Teana,2X1080Q,109.94,104,Gasoline,9.5769,Fan & Song 2022\nNissan Teana,2X10829,20.53,110,Gasoline,1.7003,Fan & Song 2022\nNissan Teana,2X10829,5.03,110,Gasoline,0.4715,Fan & Song 2022\nNissan Teana,2X10829,1.57,110,Gasoline,0.2279,Fan & Song 2022\nNissan Teana,2X1080Q,11.4,104,Gasoline,0.8909,Fan & Song 2022\nNissan Teana,2X1080Q,158.64,104,Gasoline,9.8191,Fan & Song 2022\nNissan Teana,2X1080Q,37.34,104,Gasoline,3.0691,Fan & Song 2022\nNissan Teana,2X1080Q,27.45,104,Gasoline,2.849,Fan & Song 2022\nNissan Teana,2X1080Q,135.47,104,Gasoline,9.2397,Fan & Song 2022\nNissan Teana,2X10829,8.26,110,Gasoline,0.8342,Fan & Song 2022\nNissan Teana,2X1080Q,22.85,104,Gasoline,2.6679,Fan & Song 2022\nNissan Teana,2X1080Q,2.13,104,Gasoline,0.2824,Fan & Song 2022\nNissan Teana,2X10829,30.9,110,Gasoline,1.948,Fan & Song 2022\nNissan Teana,2X1080Q,32.31,104,Gasoline,2.2814,Fan & Song 2022\nNissan Teana,2X10829,51.77,110,Gasoline,3.5436,Fan & Song 2022\nNissan Teana,2X10829,123.36,110,Gasoline,10.5223,Fan & Song 2022\nNissan Teana,2X1080Q,15.24,104,Gasoline,1.0204,Fan & Song 2022\nNissan Teana,2X1080Q,4.23,104,Gasoline,0.4944,Fan & Song 2022\nNissan Teana,2X10829,137.29,110,Gasoline,11.6188,Fan & Song 2022\nNissan Teana,2X1080Q,13.87,104,Gasoline,1.5218,Fan & Song 2022\nNissan Teana,2X10829,46.86,110,Gasoline,2.9015,Fan & Song 2022\nNissan Teana,2X1080Q,2.12,104,Gasoline,0.3165,Fan & Song 2022\nNissan Teana,2X10829,83.99,110,Gasoline,5.831,Fan & Song 2022\nNissan Teana,2X10829,63.29,110,Gasoline,5.0173,Fan & Song 2022\nNissan Teana,2X1080Q,51.49,104,Gasoline,4.6206,Fan & Song 2022\nNissan Teana,2X1080Q,13.02,104,Gasoline,0.9554,Fan & Song 2022\nNissan Teana,2X10829,13.7,110,Gasoline,1.0405,Fan & Song 2022\nNissan Teana,2X1080Q,63.82,104,Gasoline,4.9662,Fan & Song 2022\nNissan Teana,2X10829,26.93,110,Gasoline,3.2098,Fan & Song 2022\nNissan Teana,2X1080Q,129.7,104,Gasoline,9.2786,Fan & Song 2022\nNissan Teana,2X10829,1.52,110,Gasoline,0.2979,Fan & Song 2022\nNissan Teana,2X1080Q,329.0,104,Gasoline,22.9105,Fan & Song 2022\nNissan Teana,2X10829,57.04,110,Gasoline,3.8142,Fan & Song 2022\nNissan Teana,2X10829,114.71,110,Gasoline,8.8174,Fan & Song 2022\nNissan Teana,2X10829,108.56,110,Gasoline,7.6305,Fan & Song 2022\nNissan Teana,2X10829,4.15,110,Gasoline,0.3614,Fan & Song 2022\nNissan Teana,2X10829,100.77,110,Gasoline,10.7985,Fan & Song 2022\nNissan Teana,2X10829,190.22,110,Gasoline,14.6824,Fan & Song 2022\nNissan Teana,2X1080Q,189.01,104,Gasoline,11.8305,Fan & Song 2022\nNissan Teana,2X10829,2.43,110,Gasoline,0.264,Fan & Song 2022\nNissan Teana,2X10829,3.12,110,Gasoline,0.3797,Fan & Song 2022\nNissan Teana,2X1080Q,6.17,104,Gasoline,0.4717,Fan & Song 2022\nNissan Teana,2X1080Q,6.01,104,Gasoline,1.0535,Fan & Song 2022\nNissan Teana,2X10829,10.67,110,Gasoline,1.1223,Fan & Song 2022\nNissan Teana,2X10829,24.74,110,Gasoline,2.3057,Fan & Song 2022\nNissan Teana,2X10829,15.58,110,Gasoline,1.6584,Fan & Song 2022\nNissan Teana,2X1080Q,41.8,104,Gasoline,3.2583,Fan & Song 2022\nNissan Teana,2X1080Q,53.44,104,Gasoline,4.622,Fan & Song 2022\nNissan Teana,2X10829,57.26,110,Gasoline,4.2008,Fan & Song 2022\nNissan Teana,2X1080Q,137.69,104,Gasoline,9.263,Fan & Song 2022\nNissan Teana,2X10829,9.76,110,Gasoline,0.9352,Fan & Song 2022\nNissan Teana,2X10829,5.23,110,Gasoline,0.5523,Fan & Song 2022\nNissan Teana,2X10829,25.53,110,Gasoline,1.6242,Fan & Song 2022\nNissan Teana,2X10829,123.94,110,Gasoline,9.7791,Fan & Song 2022\nNissan Teana,2X1080Q,33.25,104,Gasoline,2.2223,Fan & Song 2022\nNissan Teana,2X1080Q,9.65,104,Gasoline,1.0525,Fan & Song 2022\nNissan Teana,2X10829,106.29,110,Gasoline,8.0278,Fan & Song 2022\nNissan Teana,2X10829,23.01,110,Gasoline,2.62,Fan & Song 2022\nNissan Teana,2X10829,39.78,110,Gasoline,3.6306,Fan & Song 2022\nNissan Teana,2X10829,28.64,110,Gasoline,1.9903,Fan & Song 2022\nNissan Teana,2X1080Q,14.62,104,Gasoline,1.6466,Fan & Song 2022\nNissan Teana,2X10829,5.89,110,Gasoline,0.542,Fan & Song 2022\nNissan Teana,2X10829,17.6,110,Gasoline,2.0578,Fan & Song 2022\nNissan Teana,2X1080Q,35.23,104,Gasoline,2.1788,Fan & Song 2022\nNissan Teana,2X1080Q,41.15,104,Gasoline,3.2656,Fan & Song 2022\nNissan Teana,2X1080Q,1.56,104,Gasoline,0.2573,Fan & Song 2022\nNissan Teana,2X10829,60.6,110,Gasoline,4.9026,Fan & Song 2022\nNissan Teana,2X10829,6.74,110,Gasoline,0.6232,Fan & Song 2022\nNissan Teana,2X1080Q,18.45,104,Gasoline,1.6443,Fan & Song 2022\nNissan Teana,2X10829,5.5,110,Gasoline,0.743,Fan & Song 2022\nNissan Teana,2X10829,5.84,110,Gasoline,0.6437,Fan & Song 2022\nNissan Teana,2X1080Q,12.28,104,Gasoline,1.2928,Fan & Song 2022\nNissan Teana,2X10829,111.4,110,Gasoline,10.7612,Fan & Song 2022\nNissan Teana,2X10829,2.1,110,Gasoline,0.2234,Fan & Song 2022\nNissan Teana,2X10829,5.65,110,Gasoline,0.5758,Fan & Song 2022\nNissan Teana,2X1080Q,10.9,104,Gasoline,1.2856,Fan & Song 2022\nNissan Teana,2X1080Q,1.54,104,Gasoline,0.2455,Fan & Song 2022\nNissan Teana,2X1080Q,6.36,104,Gasoline,0.978,Fan & Song 2022\nNissan Teana,2X1080Q,3.75,104,Gasoline,0.622,Fan & Song 2022\nNissan Teana,2X1080Q,9.18,104,Gasoline,0.7436,Fan & Song 2022\nNissan Teana,2X10829,42.78,110,Gasoline,3.4949,Fan & Song 2022\nNissan Teana,2X1080Q,6.02,104,Gasoline,1.1655,Fan & Song 2022\nNissan Teana,2X1080Q,10.96,104,Gasoline,0.8596,Fan & Song 2022\nNissan Teana,2X10829,100.8,110,Gasoline,7.8911,Fan & Song 2022\nNissan Teana,2X1080Q,2.38,104,Gasoline,0.3137,Fan & Song 2022\nNissan Teana,2X10829,109.52,110,Gasoline,7.0415,Fan & Song 2022\nNissan Teana,2X1080Q,17.61,104,Gasoline,1.8091,Fan & Song 2022\nNissan Teana,2X10829,6.23,110,Gasoline,0.556,Fan & Song 2022\nNissan Teana,2X1080Q,30.09,104,Gasoline,2.7952,Fan & Song 2022\nNissan Teana,2X10829,1.93,110,Gasoline,0.3237,Fan & Song 2022\nNissan Teana,2X10829,154.74,110,Gasoline,15.9634,Fan & Song 2022\nNissan Teana,2X1080Q,17.02,104,Gasoline,1.3565,Fan & Song 2022\nNissan Teana,2X10829,221.28,110,Gasoline,15.2072,Fan & Song 2022\nNissan Teana,2X10829,2.3,110,Gasoline,0.4492,Fan & Song 2022\nNissan Teana,2X1080Q,106.04,104,Gasoline,8.3727,Fan & Song 2022\nNissan Teana,2X1080Q,23.7,104,Gasoline,3.0222,Fan & Song 2022\nNissan Teana,2X10829,96.17,110,Gasoline,6.0218,Fan & Song 2022\nNissan Teana,2X10829,6.88,110,Gasoline,0.8763,Fan & Song 2022\nNissan Teana,2X1080Q,4.82,104,Gasoline,0.6541,Fan & Song 2022\nNissan Teana,2X1080Q,31.45,104,Gasoline,2.2827,Fan & Song 2022\nNissan Teana,2X1080Q,84.41,104,Gasoline,7.2107,Fan & Song 2022\nNissan Teana,2X10829,10.67,110,Gasoline,1.7457,Fan & Song 2022\nNissan Teana,2X10829,5.52,110,Gasoline,0.724,Fan & Song 2022\nNissan Teana,2X10829,6.78,110,Gasoline,0.8134,Fan & Song 2022\nNissan Teana,2X10829,115.19,110,Gasoline,7.5482,Fan & Song 2022\nNissan Teana,2X10829,37.86,110,Gasoline,3.6277,Fan & Song 2022\nNissan Teana,2X10829,107.06,110,Gasoline,9.5205,Fan & Song 2022\nNissan Teana,2X1080Q,4.68,104,Gasoline,0.6915,Fan & Song 2022\nNissan Teana,2X10829,1.8,110,Gasoline,0.2477,Fan & Song 2022\nNissan Teana,2X10829,2.11,110,Gasoline,0.2936,Fan & Song 2022\nNissan Teana,2X10829,6.8,110,Gasoline,0.5396,Fan & Song 2022\nNissan Teana,2X10829,13.32,110,Gasoline,1.3842,Fan & Song 2022\nNissan Teana,2X10829,61.51,110,Gasoline,5.2932,Fan & Song 2022\nNissan Teana,2X10829,1.83,110,Gasoline,0.3178,Fan & Song 2022\nNissan Teana,2X10829,80.44,110,Gasoline,7.9775,Fan & Song 2022\nNissan Teana,2X10829,177.83,110,Gasoline,11.3057,Fan & Song 2022\nNissan Teana,2X10829,103.48,110,Gasoline,6.9447,Fan & Song 2022\nNissan Teana,2X10829,41.56,110,Gasoline,3.7375,Fan & Song 2022\nNissan Teana,2X1080Q,101.82,104,Gasoline,7.7488,Fan & Song 2022\nNissan Teana,2X10829,55.9,110,Gasoline,5.398,Fan & Song 2022\nNissan Teana,2X10829,8.25,110,Gasoline,0.9144,Fan & Song 2022\nNissan Teana,2X10829,103.21,110,Gasoline,9.862,Fan & Song 2022\nNissan Teana,2X1080Q,8.36,104,Gasoline,0.8457,Fan & Song 2022\nNissan Teana,2X10829,49.7,110,Gasoline,5.4888,Fan & Song 2022\nNissan Teana,2X1080Q,91.48,104,Gasoline,6.2727,Fan & Song 2022\nNissan Teana,2X10829,19.91,110,Gasoline,2.0015,Fan & Song 2022\nNissan Teana,2X10829,152.3,110,Gasoline,14.4774,Fan & Song 2022\nNissan Teana,2X1080Q,3.68,104,Gasoline,0.569,Fan & Song 2022\nNissan Teana,2X1080Q,1.65,104,Gasoline,0.1844,Fan & Song 2022\nNissan Teana,2X10829,10.4,110,Gasoline,0.7474,Fan & Song 2022\nNissan Teana,2X10829,239.67,110,Gasoline,17.5372,Fan & Song 2022\nNissan Teana,2X1080Q,43.34,104,Gasoline,4.447,Fan & Song 2022\nNissan Teana,2X1080Q,3.1,104,Gasoline,0.3034,Fan & Song 2022\nNissan Teana,2X1080Q,1.82,104,Gasoline,0.267,Fan & Song 2022\nNissan Teana,2X1080Q,6.02,104,Gasoline,0.6544,Fan & Song 2022\nNissan Teana,2X10829,294.39,110,Gasoline,20.1759,Fan & Song 2022\nNissan Teana,2X10829,152.74,110,Gasoline,15.4014,Fan & Song 2022\nNissan Teana,2X10829,44.28,110,Gasoline,6.3042,Fan & Song 2022\nNissan Teana,2X1080Q,8.27,104,Gasoline,0.7798,Fan & Song 2022\nNissan Teana,2X1080Q,41.81,104,Gasoline,3.1037,Fan & Song 2022\nNissan Tiida,2X10608,4.79,140,Gasoline,0.5922,Fan & Song 2022\nNissan Tiida,2X10608,120.54,140,Gasoline,8.1449,Fan & Song 2022\nNissan Tiida,2X10608,34.22,140,Gasoline,3.2203,Fan & Song 2022\nNissan Tiida,2X10608,19.02,140,Gasoline,1.2144,Fan & Song 2022\nNissan Tiida,2X10608,6.6,140,Gasoline,0.8814,Fan & Song 2022\nNissan Tiida,2X10608,7.88,140,Gasoline,0.7857,Fan & Song 2022\nNissan Tiida,2X10608,33.96,140,Gasoline,2.4418,Fan & Song 2022\nNissan Tiida,2X10608,14.53,140,Gasoline,1.1783,Fan & Song 2022\nNissan Tiida,2X10608,156.35,140,Gasoline,10.6661,Fan & Song 2022\nNissan Tiida,2X10608,42.95,140,Gasoline,3.901,Fan & Song 2022\nNissan Tiida,2X10608,8.64,140,Gasoline,0.8609,Fan & Song 2022\nNissan Tiida,2X10608,9.37,140,Gasoline,0.7262,Fan & Song 2022\nNissan Tiida,2X10608,2.96,140,Gasoline,0.2862,Fan & Song 2022\nNissan Tiida,2X10608,63.6,140,Gasoline,5.1409,Fan & Song 2022\nNissan Tiida,2X10608,82.45,140,Gasoline,5.6821,Fan & Song 2022\nNissan Tiida,2X10608,26.28,140,Gasoline,1.9875,Fan & Song 2022\nNissan Tiida,2X10608,41.47,140,Gasoline,2.9412,Fan & Song 2022\nNissan Tiida,2X10608,7.54,140,Gasoline,0.8027,Fan & Song 2022\nNissan Tiida,2X10608,46.81,140,Gasoline,3.4927,Fan & Song 2022\nNissan Tiida,2X10608,19.6,140,Gasoline,1.2946,Fan & Song 2022\nNissan Tiida,2X10608,7.1,140,Gasoline,0.6852,Fan & Song 2022\nNissan Tiida,2X10608,3.49,140,Gasoline,0.3395,Fan & Song 2022\nNissan Tiida,2X10608,65.24,140,Gasoline,4.8832,Fan & Song 2022\nNissan Tiida,2X10608,198.64,140,Gasoline,12.6187,Fan & Song 2022\nNissan Tiida,2X10608,24.27,140,Gasoline,1.5582,Fan & Song 2022\nNissan Tiida,2X10608,15.74,140,Gasoline,1.0363,Fan & Song 2022\nNissan Tiida,2X10608,14.0,140,Gasoline,1.3241,Fan & Song 2022\nNissan Tiida,2X10608,108.71,140,Gasoline,7.5853,Fan & Song 2022\nNissan Tiida,2X10608,12.12,140,Gasoline,1.156,Fan & Song 2022\nNissan Tiida,2X10608,475.87,140,Gasoline,32.2584,Fan & Song 2022\nNissan Tiida,2X10608,2.18,140,Gasoline,0.201,Fan & Song 2022\nNissan Tiida,2X10608,152.24,140,Gasoline,9.9341,Fan & Song 2022\nNissan Tiida,2X10608,4.46,140,Gasoline,0.3993,Fan & Song 2022\nNissan Tiida,2X10608,8.29,140,Gasoline,0.4534,Fan & Song 2022\nNissan Tiida,2X10608,436.2,140,Gasoline,29.627,Fan & Song 2022\nNissan Tiida,2X10608,17.19,140,Gasoline,1.2549,Fan & Song 2022\nNissan Tiida,2X10608,3.94,140,Gasoline,0.2754,Fan & Song 2022\nNissan Tiida,2X10608,14.09,140,Gasoline,1.161,Fan & Song 2022\nNissan Tiida,2X10608,7.77,140,Gasoline,0.7415,Fan & Song 2022\nNissan Tiida,2X10608,10.59,140,Gasoline,0.6965,Fan & Song 2022\nNissan Tiida,2X10608,3.14,140,Gasoline,0.3095,Fan & Song 2022\nNissan Tiida,2X10608,2.17,140,Gasoline,0.1546,Fan & Song 2022\nNissan Tiida,2X10608,167.93,140,Gasoline,10.7118,Fan & Song 2022\nNissan Tiida,2X10608,24.57,140,Gasoline,1.5858,Fan & Song 2022\nNissan Tiida,2X10608,97.66,140,Gasoline,5.7692,Fan & Song 2022\nNissan Tiida,2X10608,109.13,140,Gasoline,5.8732,Fan & Song 2022\nNissan Tiida,2X10608,37.06,140,Gasoline,2.1611,Fan & Song 2022\nNissan Tiida,2X10608,147.65,140,Gasoline,10.3588,Fan & Song 2022\nNissan Tiida,2X10608,147.07,140,Gasoline,10.0156,Fan & Song 2022\nNissan Tiida,2X10608,128.99,140,Gasoline,8.5889,Fan & Song 2022\nNissan Tiida,2X10608,54.92,140,Gasoline,3.5599,Fan & Song 2022\nNissan Tiida,2X10608,17.34,140,Gasoline,1.2797,Fan & Song 2022\nNissan Tiida,2X10608,2.76,140,Gasoline,0.4064,Fan & Song 2022\nNissan Tiida,2X10608,3.9,140,Gasoline,0.2537,Fan & Song 2022\nNissan Tiida,2X10608,273.36,140,Gasoline,19.2918,Fan & Song 2022\nNissan Tiida,2X10608,16.97,140,Gasoline,1.2774,Fan & Song 2022\nNissan Tiida,2X10608,8.04,140,Gasoline,0.6482,Fan & Song 2022\nNissan Tiida,2X10608,131.18,140,Gasoline,9.9561,Fan & Song 2022\nNissan Tiida,2X10608,3.28,140,Gasoline,0.2567,Fan & Song 2022\nNissan Tiida,2X10608,14.32,140,Gasoline,1.1073,Fan & Song 2022\nNissan Tiida,2X10608,4.03,140,Gasoline,0.4631,Fan & Song 2022\nNissan Tiida,2X10608,104.68,140,Gasoline,6.4998,Fan & Song 2022\nNissan Tiida,2X10608,106.84,140,Gasoline,7.2493,Fan & Song 2022\nNissan Tiida,2X10608,3.39,140,Gasoline,0.466,Fan & Song 2022\nNissan Tiida,2X10608,34.06,140,Gasoline,2.3588,Fan & Song 2022\nNissan Tiida,2X10608,30.33,140,Gasoline,2.2953,Fan & Song 2022\nNissan Tiida,2X10608,42.75,140,Gasoline,2.1662,Fan & Song 2022\nNissan Tiida,2X10608,33.35,140,Gasoline,1.5088,Fan & Song 2022\nNissan Tiida,2X10608,67.82,140,Gasoline,3.4155,Fan & Song 2022\nNissan Tiida,2X10608,10.52,140,Gasoline,0.9178,Fan & Song 2022\nNissan Tiida,2X10608,120.96,140,Gasoline,8.3189,Fan & Song 2022\nNissan Tiida,2X10608,14.24,140,Gasoline,1.3965,Fan & Song 2022\nNissan Tiida,2X10608,11.06,140,Gasoline,1.0425,Fan & Song 2022\nNissan Tiida,2X10608,122.76,140,Gasoline,8.5897,Fan & Song 2022\nNissan Tiida,2X10608,62.61,140,Gasoline,3.7332,Fan & Song 2022\nNissan Tiida,2X10608,103.84,140,Gasoline,6.513,Fan & Song 2022\nNissan Tiida,2X10608,6.51,140,Gasoline,0.4699,Fan & Song 2022\nNissan Tiida,2X10608,13.94,140,Gasoline,0.9397,Fan & Song 2022\nNissan Tiida,2X10608,24.4,140,Gasoline,1.9054,Fan & Song 2022\nNissan Tiida,2X10608,13.1,140,Gasoline,0.8566,Fan & Song 2022\nNissan Tiida,2X10608,143.15,140,Gasoline,9.0319,Fan & Song 2022\nNissan Tiida,2X10608,3.19,140,Gasoline,0.3566,Fan & Song 2022\nNissan Tiida,2X10608,2.17,140,Gasoline,0.2609,Fan & Song 2022\nNissan Tiida,2X10608,7.57,140,Gasoline,0.4885,Fan & Song 2022\nNissan Tiida,2X10608,121.71,140,Gasoline,7.7896,Fan & Song 2022\nNissan Tiida,2X10608,5.22,140,Gasoline,0.4074,Fan & Song 2022\nNissan Tiida,2X10608,112.09,140,Gasoline,7.4927,Fan & Song 2022\nNissan Tiida,2X10608,9.21,140,Gasoline,0.6681,Fan & Song 2022\nNissan Tiida,2X10608,20.76,140,Gasoline,1.4296,Fan & Song 2022\nNissan Tiida,2X10608,1.54,140,Gasoline,0.2005,Fan & Song 2022\nNissan Tiida,2X10608,16.54,140,Gasoline,1.0269,Fan & Song 2022\nNissan Tiida,2X10608,4.75,140,Gasoline,0.4824,Fan & Song 2022\nNissan Tiida,2X10608,180.09,140,Gasoline,12.572,Fan & Song 2022\nNissan Tiida,2X10608,16.25,140,Gasoline,0.9823,Fan & Song 2022\nNissan Tiida,2X10608,148.96,140,Gasoline,8.8565,Fan & Song 2022\nNissan Tiida,2X10608,21.47,140,Gasoline,1.6496,Fan & Song 2022\nNissan Tiida,2X10608,44.16,140,Gasoline,2.7852,Fan & Song 2022\nNissan Tiida,2X10608,290.46,140,Gasoline,19.3477,Fan & Song 2022\nNissan Tiida,2X10608,24.94,140,Gasoline,1.6682,Fan & Song 2022\nNissan Tiida,2X10608,165.73,140,Gasoline,11.4698,Fan & Song 2022\nNissan Tiida,2X10608,128.25,140,Gasoline,8.2114,Fan & Song 2022\nNissan Tiida,2X10608,66.31,140,Gasoline,4.1512,Fan & Song 2022\nNissan Tiida,2X10608,2.99,140,Gasoline,0.3456,Fan & Song 2022\nNissan Tiida,2X10608,34.27,140,Gasoline,2.6777,Fan & Song 2022\nNissan Tiida,2X10608,47.22,140,Gasoline,2.6656,Fan & Song 2022\nNissan Tiida,2X10608,127.46,140,Gasoline,8.8123,Fan & Song 2022\nNissan Tiida,2X10608,2.16,140,Gasoline,0.3235,Fan & Song 2022\nNissan Tiida,2X10608,122.09,140,Gasoline,8.5268,Fan & Song 2022\nNissan Tiida,2X10608,18.53,140,Gasoline,1.2505,Fan & Song 2022\nNissan Tiida,2X10608,13.14,140,Gasoline,0.9251,Fan & Song 2022\nNissan Tiida,2X10608,24.76,140,Gasoline,1.7077,Fan & Song 2022\nNissan Tiida,2X10608,105.45,140,Gasoline,7.9955,Fan & Song 2022\nNissan Tiida,2X10608,32.97,140,Gasoline,2.4386,Fan & Song 2022\nNissan Tiida,2X10608,107.68,140,Gasoline,8.5672,Fan & Song 2022\nNissan Tiida,2X10608,48.5,140,Gasoline,3.0135,Fan & Song 2022\nNissan Tiida,2X10608,116.14,140,Gasoline,8.3276,Fan & Song 2022\nNissan Tiida,2X10608,105.58,140,Gasoline,7.4435,Fan & Song 2022\nNissan Tiida,2X10608,12.6,140,Gasoline,0.9272,Fan & Song 2022\nNissan Tiida,2X10608,24.7,140,Gasoline,1.6245,Fan & Song 2022\nNissan Tiida,2X10608,33.08,140,Gasoline,2.9958,Fan & Song 2022\nNissan Tiida,2X10608,3.73,140,Gasoline,0.2843,Fan & Song 2022\nNissan Tiida,2X10608,12.81,140,Gasoline,0.8286,Fan & Song 2022\nNissan Tiida,2X10608,2.4,140,Gasoline,0.3481,Fan & Song 2022\nNissan Tiida,2X10608,34.24,140,Gasoline,3.0555,Fan & Song 2022\nNissan Tiida,2X10608,13.06,140,Gasoline,0.8912,Fan & Song 2022\nNissan Tiida,2X10608,11.0,140,Gasoline,0.7354,Fan & Song 2022\nNissan Tiida,2X10608,34.05,140,Gasoline,2.3103,Fan & Song 2022\nNissan Tiida,2X10608,133.16,140,Gasoline,8.9931,Fan & Song 2022\nNissan Tiida,2X10608,85.41,140,Gasoline,5.3662,Fan & Song 2022\nNissan Tiida,2X10608,15.92,140,Gasoline,0.8991,Fan & Song 2022\nNissan Tiida,2X10608,57.89,140,Gasoline,3.4585,Fan & Song 2022\nNissan Tiida,2X10608,4.59,140,Gasoline,0.4376,Fan & Song 2022\nNissan Tiida,2X10608,50.88,140,Gasoline,3.4427,Fan & Song 2022\nNissan Tiida,2X10608,152.55,140,Gasoline,9.1935,Fan & Song 2022\nNissan Tiida,2X10608,109.9,140,Gasoline,5.7795,Fan & Song 2022\nNissan Tiida,2X10608,106.77,140,Gasoline,5.7926,Fan & Song 2022\nNissan Tiida,2X10608,48.24,140,Gasoline,3.2403,Fan & Song 2022\nNissan Tiida,2X10608,4.78,140,Gasoline,0.4984,Fan & Song 2022\nNissan Tiida,2X10608,34.15,140,Gasoline,2.5603,Fan & Song 2022\nNissan Tiida,2X10608,13.84,140,Gasoline,1.1142,Fan & Song 2022\nNissan Tiida,2X10608,22.56,140,Gasoline,1.425,Fan & Song 2022\nNissan Tiida,2X10608,30.68,140,Gasoline,2.8919,Fan & Song 2022\nNissan Tiida,2X10608,11.74,140,Gasoline,0.7344,Fan & Song 2022\nNissan Tiida,2X10608,1.7,140,Gasoline,0.1842,Fan & Song 2022\nNissan Tiida,2X10608,47.19,140,Gasoline,3.1989,Fan & Song 2022\nNissan Tiida,2X10608,9.25,140,Gasoline,0.6958,Fan & Song 2022\nNissan Tiida,2X10608,37.2,140,Gasoline,3.1314,Fan & Song 2022\nNissan Tiida,2X10608,10.83,140,Gasoline,0.7813,Fan & Song 2022\nNissan Tiida,2X10608,12.66,140,Gasoline,1.0877,Fan & Song 2022\nNissan Tiida,2X10608,11.28,140,Gasoline,0.9393,Fan & Song 2022\nNissan Tiida,2X10608,9.52,140,Gasoline,1.0022,Fan & Song 2022\nNissan Tiida,2X10608,31.49,140,Gasoline,2.0894,Fan & Song 2022\nNissan Tiida,2X10608,15.37,140,Gasoline,1.2865,Fan & Song 2022\nNissan Tiida,2X10608,72.52,140,Gasoline,3.8876,Fan & Song 2022\nNissan Tiida,2X10608,66.98,140,Gasoline,3.5855,Fan & Song 2022\nNissan Tiida,2X10608,31.78,140,Gasoline,1.9792,Fan & Song 2022\nNissan Tiida,2X10608,61.15,140,Gasoline,5.5357,Fan & Song 2022\nNissan Tiida,2X10608,5.63,140,Gasoline,0.5159,Fan & Song 2022\nNissan Tiida,2X10608,1.9,140,Gasoline,0.182,Fan & Song 2022\nNissan Tiida,2X10608,6.74,140,Gasoline,0.7528,Fan & Song 2022\nNissan Tiida,2X10608,11.17,140,Gasoline,0.8918,Fan & Song 2022\nNissan Tiida,2X10608,2.39,140,Gasoline,0.2527,Fan & Song 2022\nNissan Tiida,2X10608,104.45,140,Gasoline,7.1788,Fan & Song 2022\nNissan Tiida,2X10608,27.71,140,Gasoline,1.6384,Fan & Song 2022\nNissan Tiida,2X10608,24.31,140,Gasoline,1.9089,Fan & Song 2022\nNissan Tiida,2X10608,121.96,140,Gasoline,7.1375,Fan & Song 2022\nNissan Tiida,2X10608,13.2,140,Gasoline,1.3595,Fan & Song 2022\nNissan Tiida,2X10608,151.1,140,Gasoline,8.9495,Fan & Song 2022\nNissan Tiida,2X10608,109.88,140,Gasoline,5.9131,Fan & Song 2022\nNissan Tiida,2X10608,1.81,140,Gasoline,0.1916,Fan & Song 2022\nNissan Tiida,2X10608,185.01,140,Gasoline,10.1911,Fan & Song 2022\nNissan Tiida,2X10608,1.84,140,Gasoline,0.2096,Fan & Song 2022\nNissan Tiida,2X10608,117.08,140,Gasoline,8.4413,Fan & Song 2022\nNissan Tiida,2X10608,36.88,140,Gasoline,2.1989,Fan & Song 2022\nNissan Tiida,2X10608,31.84,140,Gasoline,2.0168,Fan & Song 2022\nNissan Tiida,2X10608,7.27,140,Gasoline,0.7293,Fan & Song 2022\nNissan Tiida,2X10608,1.56,140,Gasoline,0.2108,Fan & Song 2022\nNissan Tiida,2X10608,110.86,140,Gasoline,7.9841,Fan & Song 2022\nNissan Tiida,2X10608,120.12,140,Gasoline,7.8609,Fan & Song 2022\nNissan Tiida,2X10608,14.49,140,Gasoline,1.076,Fan & Song 2022\nNissan Tiida,2X10608,102.22,140,Gasoline,6.397,Fan & Song 2022\nNissan Tiida,2X10608,134.27,140,Gasoline,6.9157,Fan & Song 2022\nNissan Tiida,2X10608,105.51,140,Gasoline,6.08,Fan & Song 2022\nNissan Tiida,2X10608,14.11,140,Gasoline,0.8737,Fan & Song 2022\nNissan Tiida,2X10608,3.54,140,Gasoline,0.3086,Fan & Song 2022\nNissan Tiida,2X10608,10.34,140,Gasoline,0.7472,Fan & Song 2022\nNissan Tiida,2X10608,144.99,140,Gasoline,9.3501,Fan & Song 2022\nNissan Tiida,2X10608,1.53,140,Gasoline,0.1375,Fan & Song 2022\nNissan Tiida,2X10608,153.96,140,Gasoline,9.3588,Fan & Song 2022\nNissan Tiida,2X10608,11.18,140,Gasoline,0.7212,Fan & Song 2022\nNissan Tiida,2X10608,330.74,140,Gasoline,20.0879,Fan & Song 2022\nNissan Tiida,2X10608,33.64,140,Gasoline,2.0489,Fan & Song 2022\nNissan Tiida,2X10608,101.52,140,Gasoline,6.0068,Fan & Song 2022\nNissan Tiida,2X10608,53.85,140,Gasoline,3.031,Fan & Song 2022\nNissan Tiida,2X10608,1.8,140,Gasoline,0.1938,Fan & Song 2022\nNissan Tiida,2X10608,11.2,140,Gasoline,0.9011,Fan & Song 2022\nNissan Tiida,2X10608,170.37,140,Gasoline,9.5763,Fan & Song 2022\nNissan Tiida,2X10608,1.86,140,Gasoline,0.2481,Fan & Song 2022\nNissan Tiida,2X10608,121.64,140,Gasoline,9.029,Fan & Song 2022\nNissan Tiida,2X10608,4.86,140,Gasoline,0.4541,Fan & Song 2022\nNissan Tiida,2X10608,6.24,140,Gasoline,0.5506,Fan & Song 2022\nNissan Tiida,2X10608,129.67,140,Gasoline,7.7423,Fan & Song 2022\nNissan Tiida,2X10608,44.02,140,Gasoline,2.3549,Fan & Song 2022\nNissan Tiida,2X10608,10.66,140,Gasoline,0.766,Fan & Song 2022\nNissan Tiida,2X10608,2.5,140,Gasoline,0.2178,Fan & Song 2022\nNissan Tiida,2X10608,8.97,140,Gasoline,0.7961,Fan & Song 2022\nNissan Tiida,2X10608,2.05,140,Gasoline,0.3162,Fan & Song 2022\nNissan Tiida,2X10608,40.35,140,Gasoline,2.3312,Fan & Song 2022\nNissan Tiida,2X10608,16.76,140,Gasoline,1.2288,Fan & Song 2022\nNissan Tiida,2X10608,167.86,140,Gasoline,9.2028,Fan & Song 2022\nNissan Tiida,2X10608,2.34,140,Gasoline,0.2299,Fan & Song 2022\nNissan Tiida,2X10608,3.13,140,Gasoline,0.2723,Fan & Song 2022\nNissan Tiida,2X10608,6.46,140,Gasoline,0.6126,Fan & Song 2022\nNissan Tiida,2X10608,102.9,140,Gasoline,5.7174,Fan & Song 2022\nNissan Tiida,2X10608,6.6,140,Gasoline,0.6958,Fan & Song 2022\nNissan Tiida,2X10608,13.21,140,Gasoline,1.1139,Fan & Song 2022\nNissan Tiida,2X10608,27.68,140,Gasoline,2.4085,Fan & Song 2022\nNissan Tiida,2X10608,14.09,140,Gasoline,1.6579,Fan & Song 2022\nNissan Tiida,2X10608,33.78,140,Gasoline,3.8684,Fan & Song 2022\nNissan Tiida,2X10608,15.39,140,Gasoline,1.552,Fan & Song 2022\nNissan Tiida,2X10608,12.31,140,Gasoline,0.8678,Fan & Song 2022\nNissan Tiida,2X10608,174.73,140,Gasoline,9.9516,Fan & Song 2022\nNissan Tiida,2X10608,28.95,140,Gasoline,1.8183,Fan & Song 2022\nNissan Tiida,2X10608,1.83,140,Gasoline,0.2405,Fan & Song 2022\nNissan Tiida,2X10608,20.68,140,Gasoline,1.4064,Fan & Song 2022\nNissan Tiida,2X10608,109.48,140,Gasoline,7.5539,Fan & Song 2022\nNissan Tiida,2X10608,14.23,140,Gasoline,1.292,Fan & Song 2022\nNissan Tiida,2X10608,34.13,140,Gasoline,2.7448,Fan & Song 2022\nNissan Tiida,2X10608,102.77,140,Gasoline,8.6494,Fan & Song 2022\nNissan Tiida,2X10608,3.94,140,Gasoline,0.3008,Fan & Song 2022\nNissan Tiida,2X10608,3.27,140,Gasoline,0.2635,Fan & Song 2022\nNissan Tiida,2X10608,11.72,140,Gasoline,0.849,Fan & Song 2022\nNissan Tiida,2X10608,158.19,140,Gasoline,9.3205,Fan & Song 2022\nNissan Tiida,2X10608,7.99,140,Gasoline,0.5822,Fan & Song 2022\nNissan Tiida,2X10608,109.74,140,Gasoline,6.3542,Fan & Song 2022\nNissan Tiida,2X10608,123.43,140,Gasoline,7.5056,Fan & Song 2022\nNissan Tiida,2X10608,7.51,140,Gasoline,0.6291,Fan & Song 2022\nNissan Tiida,2X10608,5.33,140,Gasoline,0.4067,Fan & Song 2022\nNissan Tiida,2X10608,192.18,140,Gasoline,10.9794,Fan & Song 2022\nNissan Tiida,2X10608,154.46,140,Gasoline,9.4252,Fan & Song 2022\nNissan Tiida,2X10608,14.24,140,Gasoline,1.4243,Fan & Song 2022\nNissan Tiida,2X10608,122.89,140,Gasoline,7.2948,Fan & Song 2022\nNissan Tiida,2X10608,65.29,140,Gasoline,4.8265,Fan & Song 2022\nNissan Tiida,2X10608,121.55,140,Gasoline,6.7789,Fan & Song 2022\nNissan Tiida,2X10608,11.78,140,Gasoline,0.8503,Fan & Song 2022\nNissan Tiida,2X10608,118.56,140,Gasoline,7.3395,Fan & Song 2022\nNissan Tiida,2X10608,81.06,140,Gasoline,4.7082,Fan & Song 2022\nNissan Tiida,2X10608,246.77,140,Gasoline,15.0899,Fan & Song 2022\nNissan Tiida,2X10608,119.35,140,Gasoline,7.0773,Fan & Song 2022\nNissan Tiida,2X10608,2.73,140,Gasoline,0.2527,Fan & Song 2022\nNissan Tiida,2X10608,20.6,140,Gasoline,1.551,Fan & Song 2022\nNissan Tiida,2X10608,120.75,140,Gasoline,9.6317,Fan & Song 2022\nNissan Tiida,2X10608,140.5,140,Gasoline,8.3465,Fan & Song 2022\nNissan Tiida,2X10608,179.47,140,Gasoline,10.8604,Fan & Song 2022\nNissan Tiida,2X10608,160.27,140,Gasoline,9.556,Fan & Song 2022\nNissan Tiida,2X10608,4.65,140,Gasoline,0.3849,Fan & Song 2022\nNissan Tiida,2X10608,19.59,140,Gasoline,1.2589,Fan & Song 2022\nNissan Tiida,2X10608,68.22,140,Gasoline,3.9449,Fan & Song 2022\nNissan Tiida,2X10608,10.5,140,Gasoline,0.6428,Fan & Song 2022\nNissan Tiida,2X10608,103.48,140,Gasoline,6.7579,Fan & Song 2022\nNissan Tiida,2X10608,3.47,140,Gasoline,0.4313,Fan & Song 2022\nNissan Tiida,2X10608,51.22,140,Gasoline,5.0799,Fan & Song 2022\nNissan Tiida,2X10608,77.28,140,Gasoline,4.7466,Fan & Song 2022\nNissan Tiida,2X10608,7.38,140,Gasoline,0.5489,Fan & Song 2022\nNissan Tiida,2X10608,33.74,140,Gasoline,2.4923,Fan & Song 2022\nNissan Tiida,2X10608,25.15,140,Gasoline,1.9281,Fan & Song 2022\nNissan Tiida,2X10608,7.14,140,Gasoline,0.5905,Fan & Song 2022\nNissan Tiida,2X10608,10.87,140,Gasoline,0.9704,Fan & Song 2022\nNissan Tiida,2X10608,2.86,140,Gasoline,0.3404,Fan & Song 2022\nNissan Tiida,2X10608,12.11,140,Gasoline,1.417,Fan & Song 2022\nNissan Tiida,2X10608,10.99,140,Gasoline,1.0976,Fan & Song 2022\nNissan Tiida,2X10608,7.75,140,Gasoline,0.6231,Fan & Song 2022\nNissan Tiida,2X10608,56.73,140,Gasoline,3.834,Fan & Song 2022\nNissan Tiida,2X10608,4.6,140,Gasoline,0.3484,Fan & Song 2022\nNissan Tiida,2X10608,116.14,140,Gasoline,8.6941,Fan & Song 2022\nNissan Tiida,2X10608,107.34,140,Gasoline,7.8463,Fan & Song 2022\nNissan Tiida,2X10608,107.11,140,Gasoline,8.482,Fan & Song 2022\nNissan Tiida,2X10608,3.75,140,Gasoline,0.3043,Fan & Song 2022\nNissan Tiida,2X10608,40.96,140,Gasoline,3.441,Fan & Song 2022\nNissan Tiida,2X10608,120.69,140,Gasoline,7.608,Fan & Song 2022\nNissan Tiida,2X10608,3.65,140,Gasoline,0.3184,Fan & Song 2022\nNissan Tiida,2X10608,104.11,140,Gasoline,5.8661,Fan & Song 2022\nNissan Tiida,2X10608,108.79,140,Gasoline,6.1554,Fan & Song 2022\nNissan Tiida,2X10608,101.17,140,Gasoline,9.9627,Fan & Song 2022\nNissan Tiida,2X10608,55.35,140,Gasoline,2.8479,Fan & Song 2022\nNissan Tiida,2X10608,6.64,140,Gasoline,0.6013,Fan & Song 2022\nNissan Tiida,2X10608,105.63,140,Gasoline,7.4308,Fan & Song 2022\nNissan Tiida,2X10608,4.14,140,Gasoline,0.5421,Fan & Song 2022\nNissan Tiida,2X10608,47.49,140,Gasoline,2.8915,Fan & Song 2022\nNissan Tiida,2X10608,22.2,140,Gasoline,1.6728,Fan & Song 2022\nNissan Tiida,2X10608,74.34,140,Gasoline,5.7177,Fan & Song 2022\nNissan Tiida,2X10608,150.23,140,Gasoline,10.2121,Fan & Song 2022\nNissan Tiida,2X10608,304.67,140,Gasoline,19.7921,Fan & Song 2022\nNissan Tiida,2X10608,183.36,140,Gasoline,10.5895,Fan & Song 2022\nNissan Tiida,2X10608,36.26,140,Gasoline,4.8855,Fan & Song 2022\nNissan Tiida,2X10608,202.2,140,Gasoline,11.4791,Fan & Song 2022\nNissan Tiida,2X10608,117.69,140,Gasoline,9.9399,Fan & Song 2022\nNissan Tiida,2X10608,12.0,140,Gasoline,1.0518,Fan & Song 2022\nNissan Tiida,2X10608,32.56,140,Gasoline,2.2098,Fan & Song 2022\nNissan Tiida,2X10608,31.93,140,Gasoline,1.9281,Fan & Song 2022\nNissan Tiida,2X10608,103.24,140,Gasoline,7.185,Fan & Song 2022\nNissan Tiida,2X10608,30.93,140,Gasoline,2.1549,Fan & Song 2022\nNissan Tiida,2X10608,25.1,140,Gasoline,1.7379,Fan & Song 2022\nNissan Tiida,2X10608,10.59,140,Gasoline,0.7025,Fan & Song 2022\nNissan Tiida,2X10608,48.59,140,Gasoline,2.5605,Fan & Song 2022\nNissan Tiida,2X10608,34.34,140,Gasoline,3.4864,Fan & Song 2022\nNissan Tiida,2X10608,30.24,140,Gasoline,1.9204,Fan & Song 2022\nNissan Tiida,2X10608,48.4,140,Gasoline,3.3955,Fan & Song 2022\nNissan Tiida,2X10608,34.49,140,Gasoline,3.3778,Fan & Song 2022\nNissan Tiida,2X10608,19.7,140,Gasoline,1.5892,Fan & Song 2022\nNissan Tiida,2X10608,35.59,140,Gasoline,2.4338,Fan & Song 2022\nNissan Tiida,2X10608,125.35,140,Gasoline,9.0555,Fan & Song 2022\nNissan Tiida,2X10608,36.08,140,Gasoline,3.1682,Fan & Song 2022\nNissan Tiida,2X10608,129.53,140,Gasoline,7.2042,Fan & Song 2022\nNissan Tiida,2X10608,2.53,140,Gasoline,0.3076,Fan & Song 2022\nNissan Tiida,2X10608,130.79,140,Gasoline,7.9817,Fan & Song 2022\nNissan Tiida,2X10608,56.24,140,Gasoline,3.65,Fan & Song 2022\nNissan Tiida,2X10608,30.48,140,Gasoline,2.0611,Fan & Song 2022\nNissan Tiida,2X10608,2.47,140,Gasoline,0.2047,Fan & Song 2022\nNissan Tiida,2X10608,12.2,140,Gasoline,0.8771,Fan & Song 2022\nNissan Tiida,2X10608,19.05,140,Gasoline,1.5714,Fan & Song 2022\nNissan Tiida,2X10608,24.1,140,Gasoline,1.8404,Fan & Song 2022\nNissan Tiida,2X10608,4.22,140,Gasoline,0.4527,Fan & Song 2022\nNissan Tiida,2X10608,7.21,140,Gasoline,0.9174,Fan & Song 2022\nNissan Tiida,2X10608,10.59,140,Gasoline,0.9726,Fan & Song 2022\nNissan Tiida,2X10608,103.21,140,Gasoline,6.6077,Fan & Song 2022\nNissan Tiida,2X10608,6.89,140,Gasoline,0.6567,Fan & Song 2022\nNissan Tiida,2X10608,1.8,140,Gasoline,0.2992,Fan & Song 2022\nNissan Tiida,2X10608,40.05,140,Gasoline,4.2139,Fan & Song 2022\nNissan Tiida,2X10608,27.38,140,Gasoline,1.9751,Fan & Song 2022\nNissan Tiida,2X10608,13.22,140,Gasoline,1.3379,Fan & Song 2022\nNissan Tiida,2X10608,129.11,140,Gasoline,8.9933,Fan & Song 2022\nNissan Tiida,2X10608,2.04,140,Gasoline,0.2215,Fan & Song 2022\nNissan Tiida,2X10608,127.94,140,Gasoline,11.6902,Fan & Song 2022\nNissan Tiida,2X10608,17.08,140,Gasoline,1.6753,Fan & Song 2022\nNissan Tiida,2X10608,4.22,140,Gasoline,0.4349,Fan & Song 2022\nNissan Tiida,2X10608,120.26,140,Gasoline,11.7191,Fan & Song 2022\nNissan Tiida,2X10608,31.59,140,Gasoline,2.1896,Fan & Song 2022\nNissan Tiida,2X10608,14.23,140,Gasoline,1.33,Fan & Song 2022\nNissan Tiida,2X10608,100.55,140,Gasoline,9.934,Fan & Song 2022\nNissan Tiida,2X10608,3.59,140,Gasoline,0.4147,Fan & Song 2022\nNissan Tiida,2X10608,34.31,140,Gasoline,2.7327,Fan & Song 2022\nNissan Tiida,2X10608,24.71,140,Gasoline,2.3507,Fan & Song 2022\nNissan Tiida,2X10608,77.64,140,Gasoline,5.1235,Fan & Song 2022\nNissan Tiida,2X10608,4.49,140,Gasoline,0.4879,Fan & Song 2022\nNissan Tiida,2X10608,105.45,140,Gasoline,7.0658,Fan & Song 2022\nNissan Tiida,2X10608,213.06,140,Gasoline,13.4185,Fan & Song 2022\nNissan Tiida,2X10608,27.48,140,Gasoline,2.0506,Fan & Song 2022\nNissan Tiida,2X10608,10.97,140,Gasoline,0.8965,Fan & Song 2022\nNissan Tiida,2X10608,1.76,140,Gasoline,0.2344,Fan & Song 2022\nNissan Tiida,2X10608,2.73,140,Gasoline,0.2526,Fan & Song 2022\nNissan Tiida,2X10608,27.67,140,Gasoline,1.8105,Fan & Song 2022\nNissan Tiida,2X10608,4.23,140,Gasoline,0.5265,Fan & Song 2022\nNissan Tiida,2X10608,19.99,140,Gasoline,1.5436,Fan & Song 2022\nNissan Tiida,2X10608,2.09,140,Gasoline,0.2146,Fan & Song 2022\nNissan Tiida,2X10608,171.83,140,Gasoline,11.7163,Fan & Song 2022\nNissan Tiida,2X10608,37.79,140,Gasoline,3.4044,Fan & Song 2022\nNissan Tiida,2X10608,13.24,140,Gasoline,1.1236,Fan & Song 2022\nNissan Tiida,2X10608,12.55,140,Gasoline,0.9108,Fan & Song 2022\nNissan Tiida,2X10608,11.06,140,Gasoline,0.8053,Fan & Song 2022\nNissan Tiida,2X10608,7.16,140,Gasoline,0.9614,Fan & Song 2022\nNissan Tiida,2X10608,24.06,140,Gasoline,2.1229,Fan & Song 2022\nNissan Tiida,2X10608,9.8,140,Gasoline,0.6215,Fan & Song 2022\nNissan Tiida,2X10608,1.8,140,Gasoline,0.1398,Fan & Song 2022\nNissan Tiida,2X10608,26.3,140,Gasoline,2.2068,Fan & Song 2022\nNissan Tiida,2X10608,116.13,140,Gasoline,8.7795,Fan & Song 2022\nNissan Tiida,2X10608,19.11,140,Gasoline,1.2683,Fan & Song 2022\nNissan Tiida,2X10608,15.43,140,Gasoline,1.1577,Fan & Song 2022\nNissan Tiida,2X10608,3.34,140,Gasoline,0.4589,Fan & Song 2022\nNissan Tiida,2X10608,5.19,140,Gasoline,0.4043,Fan & Song 2022\nNissan Tiida,2X10608,7.06,140,Gasoline,0.684,Fan & Song 2022\nNissan Tiida,2X10608,56.34,140,Gasoline,3.4124,Fan & Song 2022\nNissan Tiida,2X10608,2.1,140,Gasoline,0.2277,Fan & Song 2022\nNissan Tiida,2X10608,41.34,140,Gasoline,3.2089,Fan & Song 2022\nNissan Tiida,2X10608,30.98,140,Gasoline,1.9242,Fan & Song 2022\nNissan Tiida,2X10608,9.0,140,Gasoline,0.5908,Fan & Song 2022\nNissan Tiida,2X10608,69.91,140,Gasoline,4.9517,Fan & Song 2022\nNissan Tiida,2X10608,18.62,140,Gasoline,1.6403,Fan & Song 2022\nNissan Tiida,2X10608,36.06,140,Gasoline,2.5416,Fan & Song 2022\nNissan Tiida,2X10608,39.67,140,Gasoline,2.7211,Fan & Song 2022\nNissan Tiida,2X10608,50.23,140,Gasoline,3.1616,Fan & Song 2022\nNissan Tiida,2X10608,9.33,140,Gasoline,0.8502,Fan & Song 2022\nNissan Tiida,2X10608,16.47,140,Gasoline,1.4984,Fan & Song 2022\nNissan Tiida,2X10608,7.31,140,Gasoline,0.5274,Fan & Song 2022\nNissan Tiida,2X10608,5.77,140,Gasoline,0.3716,Fan & Song 2022\nNissan Tiida,2X10608,102.37,140,Gasoline,9.2069,Fan & Song 2022\nNissan Tiida,2X10608,9.43,140,Gasoline,1.1074,Fan & Song 2022\nNissan Tiida,2X10608,42.79,140,Gasoline,2.1337,Fan & Song 2022\nNissan Tiida,2X10608,14.33,140,Gasoline,1.5055,Fan & Song 2022\nNissan Tiida,2X10608,109.82,140,Gasoline,8.913,Fan & Song 2022\nNissan Tiida,2X10608,55.09,140,Gasoline,3.2192,Fan & Song 2022\nNissan Tiida,2X10608,19.35,140,Gasoline,1.462,Fan & Song 2022\nNissan Tiida,2X10608,54.76,140,Gasoline,3.6869,Fan & Song 2022\nNissan Tiida,2X10608,18.4,140,Gasoline,1.4314,Fan & Song 2022\nNissan Tiida,2X10608,31.26,140,Gasoline,2.3279,Fan & Song 2022\nNissan Tiida,2X10608,2.14,140,Gasoline,0.3229,Fan & Song 2022\nNissan Tiida,2X10608,26.15,140,Gasoline,2.3335,Fan & Song 2022\nNissan Tiida,2X10608,21.35,140,Gasoline,1.5538,Fan & Song 2022\nNissan Tiida,2X10608,103.28,140,Gasoline,6.3527,Fan & Song 2022\nNissan Tiida,2X10608,7.89,140,Gasoline,0.9283,Fan & Song 2022\nNissan Tiida,2X10608,71.24,140,Gasoline,5.0744,Fan & Song 2022\nNissan Tiida,2X10608,37.24,140,Gasoline,2.3394,Fan & Song 2022\nNissan Tiida,2X10608,136.77,140,Gasoline,12.1609,Fan & Song 2022\nNissan Tiida,2X10608,132.88,140,Gasoline,9.4535,Fan & Song 2022\nNissan Tiida,2X10608,53.32,140,Gasoline,4.3642,Fan & Song 2022\nNissan Tiida,2X10608,109.76,140,Gasoline,7.8741,Fan & Song 2022\nNissan Tiida,2X10608,115.26,140,Gasoline,9.9743,Fan & Song 2022\nNissan Tiida,2X10608,171.58,140,Gasoline,12.0302,Fan & Song 2022\nNissan Tiida,2X10608,53.49,140,Gasoline,3.6221,Fan & Song 2022\nNissan Tiida,2X10608,63.26,140,Gasoline,5.3439,Fan & Song 2022\nNissan Tiida,2X10608,4.62,140,Gasoline,0.4815,Fan & Song 2022\nNissan Tiida,2X10608,58.8,140,Gasoline,3.1333,Fan & Song 2022\nNissan Tiida,2X10608,101.25,140,Gasoline,6.1031,Fan & Song 2022\nNissan Tiida,2X10608,7.36,140,Gasoline,0.6252,Fan & Song 2022\nNissan Tiida,2X10608,42.39,140,Gasoline,3.4233,Fan & Song 2022\nNissan Tiida,2X10608,58.02,140,Gasoline,5.19,Fan & Song 2022\nNissan Tiida,2X10608,12.78,140,Gasoline,0.9096,Fan & Song 2022\nNissan Tiida,2X10608,10.0,140,Gasoline,0.919,Fan & Song 2022\nNissan Tiida,2X10608,27.22,140,Gasoline,2.1921,Fan & Song 2022\nNissan Tiida,2X10608,7.03,140,Gasoline,0.6405,Fan & Song 2022\nNissan Tiida,2X10608,11.35,140,Gasoline,0.8766,Fan & Song 2022\nNissan Tiida,2X10608,4.02,140,Gasoline,0.3936,Fan & Song 2022\nNissan Tiida,2X10608,7.19,140,Gasoline,0.4837,Fan & Song 2022\nNissan Tiida,2X10608,23.79,140,Gasoline,1.411,Fan & Song 2022\nNissan Tiida,2X10608,65.02,140,Gasoline,5.8715,Fan & Song 2022\nNissan Tiida,2X10608,121.28,140,Gasoline,8.6287,Fan & Song 2022\nNissan Tiida,2X10608,4.82,140,Gasoline,0.5667,Fan & Song 2022\nNissan Tiida,2X10608,25.72,140,Gasoline,1.3705,Fan & Song 2022\nNissan Tiida,2X10608,8.54,140,Gasoline,0.5628,Fan & Song 2022\nNissan Tiida,2X10608,2.67,140,Gasoline,0.1991,Fan & Song 2022\nNissan Tiida,2X10608,31.41,140,Gasoline,1.8326,Fan & Song 2022\nNissan Tiida,2X10608,9.72,140,Gasoline,0.7079,Fan & Song 2022\nNissan Tiida,2X10608,76.22,140,Gasoline,4.7284,Fan & Song 2022\nNissan Tiida,2X10608,117.64,140,Gasoline,9.2401,Fan & Song 2022\nNissan Tiida,2X10608,98.49,140,Gasoline,8.1253,Fan & Song 2022\nNissan Tiida,2X10608,47.23,140,Gasoline,2.6383,Fan & Song 2022\nNissan Tiida,2X10608,46.13,140,Gasoline,3.09,Fan & Song 2022\nNissan Tiida,2X10608,27.42,140,Gasoline,1.9371,Fan & Song 2022\nNissan Tiida,2X10608,66.08,140,Gasoline,5.2703,Fan & Song 2022\nNissan Tiida,2X10608,20.97,140,Gasoline,1.7421,Fan & Song 2022\nNissan Tiida,2X10608,54.36,140,Gasoline,3.2003,Fan & Song 2022\nNissan Tiida,2X10608,45.66,140,Gasoline,2.7342,Fan & Song 2022\nNissan Tiida,2X10608,41.16,140,Gasoline,2.3225,Fan & Song 2022\nNissan Tiida,2X10608,97.27,140,Gasoline,7.803,Fan & Song 2022\nNissan Tiida,2X10608,109.31,140,Gasoline,6.1751,Fan & Song 2022\nNissan Tiida,2X10608,10.1,140,Gasoline,0.7134,Fan & Song 2022\nNissan Tiida,2X10608,148.69,140,Gasoline,10.2537,Fan & Song 2022\nNissan Tiida,2X10608,10.43,140,Gasoline,0.766,Fan & Song 2022\nNissan Tiida,2X10608,172.08,140,Gasoline,13.6812,Fan & Song 2022\nNissan Tiida,2X10608,34.48,140,Gasoline,1.893,Fan & Song 2022\nNissan Tiida,2X10608,38.35,140,Gasoline,2.8975,Fan & Song 2022\nNissan Tiida,2X10608,2.98,140,Gasoline,0.2677,Fan & Song 2022\nNissan Tiida,2X10608,178.47,140,Gasoline,9.3997,Fan & Song 2022\nNissan Tiida,2X10608,21.09,140,Gasoline,1.1823,Fan & Song 2022\nNissan Tiida,2X10608,110.49,140,Gasoline,8.5114,Fan & Song 2022\nNissan Tiida,2X10608,37.54,140,Gasoline,1.9673,Fan & Song 2022\nNissan Tiida,2X10608,30.14,140,Gasoline,1.9883,Fan & Song 2022\nNissan Tiida,2X10608,42.4,140,Gasoline,2.555,Fan & Song 2022\nNissan Tiida,2X10608,18.81,140,Gasoline,1.7856,Fan & Song 2022\nNissan Tiida,2X10608,33.8,140,Gasoline,1.8671,Fan & Song 2022\nNissan Tiida,2X10608,170.68,140,Gasoline,9.6885,Fan & Song 2022\nNissan Tiida,2X10608,71.48,140,Gasoline,4.0633,Fan & Song 2022\nNissan Tiida,2X10608,4.22,140,Gasoline,0.4026,Fan & Song 2022\nNissan Tiida,2X10608,21.49,140,Gasoline,1.2459,Fan & Song 2022\nNissan Tiida,2X10608,104.76,140,Gasoline,7.2551,Fan & Song 2022\nNissan Tiida,2X10608,37.26,140,Gasoline,3.7065,Fan & Song 2022\nNissan Tiida,2X10608,34.68,140,Gasoline,2.3471,Fan & Song 2022\nNissan Tiida,2X10608,36.07,140,Gasoline,1.4492,Fan & Song 2022\nNissan Tiida,2X10608,122.91,140,Gasoline,6.7281,Fan & Song 2022\nNissan Tiida,2X10608,47.51,140,Gasoline,3.8782,Fan & Song 2022\nNissan Tiida,2X10608,10.84,140,Gasoline,0.644,Fan & Song 2022\nNissan Tiida,2X10608,37.83,140,Gasoline,2.2891,Fan & Song 2022\nNissan Tiida,2X10608,39.92,140,Gasoline,2.9333,Fan & Song 2022\nNissan Tiida,2X10608,3.16,140,Gasoline,0.2497,Fan & Song 2022\nNissan Tiida,2X10608,111.64,140,Gasoline,8.7021,Fan & Song 2022\nNissan Tiida,2X10608,4.28,140,Gasoline,0.4646,Fan & Song 2022\nNissan Tiida,2X10608,10.01,140,Gasoline,0.9213,Fan & Song 2022\nNissan Tiida,2X10608,34.42,140,Gasoline,1.6975,Fan & Song 2022\nNissan Tiida,2X10608,11.32,140,Gasoline,0.7696,Fan & Song 2022\nNissan Tiida,2X10608,113.12,140,Gasoline,9.0655,Fan & Song 2022\nNissan Tiida,2X10608,4.64,140,Gasoline,0.5731,Fan & Song 2022\nNissan Tiida,2X10608,47.17,140,Gasoline,4.5886,Fan & Song 2022\nNissan Tiida,2X10608,28.28,140,Gasoline,1.5874,Fan & Song 2022\nNissan Tiida,2X10608,63.29,140,Gasoline,3.6864,Fan & Song 2022\nNissan Tiida,2X10608,40.61,140,Gasoline,2.4139,Fan & Song 2022\nNissan Tiida,2X10608,100.82,140,Gasoline,8.1221,Fan & Song 2022\nNissan Tiida,2X10608,38.52,140,Gasoline,2.5254,Fan & Song 2022\nNissan Tiida,2X10608,42.64,140,Gasoline,2.6811,Fan & Song 2022\nNissan Tiida,2X10608,14.46,140,Gasoline,1.1616,Fan & Song 2022\nNissan Tiida,2X10608,113.14,140,Gasoline,8.7948,Fan & Song 2022\nNissan Tiida,2X10608,21.34,140,Gasoline,1.4359,Fan & Song 2022\nNissan Tiida,2X10608,2.74,140,Gasoline,0.2082,Fan & Song 2022\nNissan Tiida,2X10608,101.89,140,Gasoline,7.7763,Fan & Song 2022\nNissan Tiida,2X10608,4.5,140,Gasoline,0.6627,Fan & Song 2022\nNissan Tiida,2X10608,12.14,140,Gasoline,0.9184,Fan & Song 2022\nNissan Tiida,2X10608,4.63,140,Gasoline,0.5485,Fan & Song 2022\nNissan Tiida,2X10608,10.72,140,Gasoline,0.9792,Fan & Song 2022\nNissan Tiida,2X10608,43.44,140,Gasoline,2.4302,Fan & Song 2022\nNissan Tiida,2X10608,5.86,140,Gasoline,0.5675,Fan & Song 2022\nNissan Tiida,2X10608,4.28,140,Gasoline,0.4534,Fan & Song 2022\nNissan Tiida,2X10608,103.28,140,Gasoline,8.7737,Fan & Song 2022\nNissan Tiida,2X10608,1.72,140,Gasoline,0.1762,Fan & Song 2022\nNissan Tiida,2X10608,3.87,140,Gasoline,0.3028,Fan & Song 2022\nNissan Tiida,2X10608,42.96,140,Gasoline,2.6489,Fan & Song 2022\nNissan Tiida,2X10608,59.61,140,Gasoline,3.6732,Fan & Song 2022\nNissan Tiida,2X10608,6.66,140,Gasoline,0.467,Fan & Song 2022\nNissan Tiida,2X10608,6.01,140,Gasoline,0.5462,Fan & Song 2022\nNissan Tiida,2X10608,10.46,140,Gasoline,0.8444,Fan & Song 2022\nNissan Tiida,2X10608,3.29,140,Gasoline,0.312,Fan & Song 2022\nNissan Tiida,2X10608,24.14,140,Gasoline,1.6699,Fan & Song 2022\nNissan Tiida,2X10608,35.24,140,Gasoline,2.2078,Fan & Song 2022\nNissan Tiida,2X10608,10.28,140,Gasoline,0.8167,Fan & Song 2022\nNissan Tiida,2X10608,20.24,140,Gasoline,1.4956,Fan & Song 2022\nNissan Tiida,2X10608,43.06,140,Gasoline,3.3923,Fan & Song 2022\nNissan Tiida,2X10608,101.77,140,Gasoline,6.2296,Fan & Song 2022\nNissan Tiida,2X10608,10.61,140,Gasoline,0.7541,Fan & Song 2022\nNissan Tiida,2X10608,63.78,140,Gasoline,4.4071,Fan & Song 2022\nNissan Tiida,2X10608,85.93,140,Gasoline,6.6714,Fan & Song 2022\nNissan Tiida,2X10608,166.82,140,Gasoline,10.9229,Fan & Song 2022\nNissan Tiida,2X10608,47.8,140,Gasoline,3.396,Fan & Song 2022\nNissan Tiida,2X10608,35.94,140,Gasoline,2.1283,Fan & Song 2022\nNissan Tiida,2X10608,35.9,140,Gasoline,2.3369,Fan & Song 2022\nNissan Tiida,2X10608,16.3,140,Gasoline,1.6812,Fan & Song 2022\nNissan Tiida,2X10608,149.51,140,Gasoline,10.1261,Fan & Song 2022\nNissan Tiida,2X10608,52.59,140,Gasoline,4.1861,Fan & Song 2022\nNissan Tiida,2X10608,4.19,140,Gasoline,0.5262,Fan & Song 2022\nNissan Tiida,2X10608,16.06,140,Gasoline,1.4397,Fan & Song 2022\nNissan Tiida,2X10608,10.66,140,Gasoline,0.8619,Fan & Song 2022\nNissan Tiida,2X10608,6.53,140,Gasoline,0.5375,Fan & Song 2022\nNissan Tiida,2X10608,6.57,140,Gasoline,0.7256,Fan & Song 2022\nNissan Tiida,2X10608,11.33,140,Gasoline,1.0521,Fan & Song 2022\nNissan Tiida,2X10608,14.06,140,Gasoline,1.5252,Fan & Song 2022\nNissan Tiida,2X10608,4.64,140,Gasoline,0.4297,Fan & Song 2022\nNissan Tiida,2X10608,4.61,140,Gasoline,0.4686,Fan & Song 2022\nNissan Tiida,2X10608,10.32,140,Gasoline,0.8089,Fan & Song 2022\nNissan Tiida,2X10608,3.53,140,Gasoline,0.5119,Fan & Song 2022\nNissan Tiida,2X10608,100.05,140,Gasoline,6.3144,Fan & Song 2022\nNissan Tiida,2X10608,187.35,140,Gasoline,11.9372,Fan & Song 2022\nNissan Tiida,2X10608,33.32,140,Gasoline,1.8405,Fan & Song 2022\nNissan Tiida,2X10608,35.24,140,Gasoline,2.8101,Fan & Song 2022\nNissan Tiida,2X10608,120.83,140,Gasoline,8.0068,Fan & Song 2022\nNissan Tiida,2X10608,159.81,140,Gasoline,11.389,Fan & Song 2022\nNissan Tiida,2X10608,15.36,140,Gasoline,1.2184,Fan & Song 2022\nNissan Tiida,2X10608,15.38,140,Gasoline,0.8323,Fan & Song 2022\nNissan Tiida,2X10608,13.33,140,Gasoline,1.0429,Fan & Song 2022\nNissan Tiida,2X10608,4.62,140,Gasoline,0.4556,Fan & Song 2022\nNissan Tiida,2X10608,21.46,140,Gasoline,1.8687,Fan & Song 2022\nNissan Tiida,2X10608,5.66,140,Gasoline,0.8188,Fan & Song 2022\nNissan Tiida,2X10608,40.49,140,Gasoline,2.7737,Fan & Song 2022\nNissan Tiida,2X10608,37.56,140,Gasoline,2.2929,Fan & Song 2022\nNissan Tiida,2X10608,4.56,140,Gasoline,0.5868,Fan & Song 2022\nNissan Tiida,2X10608,4.71,140,Gasoline,0.6816,Fan & Song 2022\nNissan Tiida,2X10608,16.1,140,Gasoline,1.2135,Fan & Song 2022\nNissan Tiida,2X10608,9.3,140,Gasoline,0.8403,Fan & Song 2022\nNissan Tiida,2X10608,38.49,140,Gasoline,2.2167,Fan & Song 2022\nNissan Tiida,2X10608,106.36,140,Gasoline,7.135,Fan & Song 2022\nNissan Tiida,2X10608,141.63,140,Gasoline,11.824,Fan & Song 2022\nNissan Tiida,2X10608,98.0,140,Gasoline,7.7187,Fan & Song 2022\nNissan Tiida,2X10608,34.9,140,Gasoline,3.0945,Fan & Song 2022\nNissan Tiida,2X10608,45.45,140,Gasoline,2.3745,Fan & Song 2022\nNissan Tiida,2X10608,10.59,140,Gasoline,0.9582,Fan & Song 2022\nNissan Tiida,2X10608,2.63,140,Gasoline,0.2789,Fan & Song 2022\nNissan Tiida,2X10608,129.7,140,Gasoline,10.4039,Fan & Song 2022\nNissan Tiida,2X10608,17.8,140,Gasoline,1.3533,Fan & Song 2022\nNissan Tiida,2X10608,124.36,140,Gasoline,9.8506,Fan & Song 2022\nNissan Tiida,2X10608,16.5,140,Gasoline,1.1468,Fan & Song 2022\nNissan Tiida,2X10608,34.29,140,Gasoline,2.5406,Fan & Song 2022\nNissan Tiida,2X10608,9.07,140,Gasoline,0.6718,Fan & Song 2022\nNissan Tiida,2X10608,132.78,140,Gasoline,9.365,Fan & Song 2022\nNissan Tiida,2X10608,14.32,140,Gasoline,0.9027,Fan & Song 2022\nNissan Tiida,2X10608,13.12,140,Gasoline,0.9581,Fan & Song 2022\nNissan Tiida,2X10608,10.1,140,Gasoline,0.8537,Fan & Song 2022\nNissan Tiida,2X10608,153.57,140,Gasoline,10.6553,Fan & Song 2022\nNissan Tiida,2X10608,202.56,140,Gasoline,14.1861,Fan & Song 2022\nNissan Tiida,2X10608,107.76,140,Gasoline,7.3226,Fan & Song 2022\nNissan Tiida,2X10608,4.28,140,Gasoline,0.426,Fan & Song 2022\nNissan Tiida,2X10608,160.86,140,Gasoline,13.2893,Fan & Song 2022\nNissan Tiida,2X10608,28.11,140,Gasoline,1.7096,Fan & Song 2022\nNissan Tiida,2X10608,19.92,140,Gasoline,2.4013,Fan & Song 2022\nNissan Tiida,2X10608,9.4,140,Gasoline,0.7488,Fan & Song 2022\nNissan Tiida,2X10608,147.58,140,Gasoline,11.8423,Fan & Song 2022\nNissan Tiida,2X10608,161.88,140,Gasoline,11.02,Fan & Song 2022\nNissan Tiida,2X10608,79.99,140,Gasoline,6.5045,Fan & Song 2022\nNissan Tiida,2X10608,119.1,140,Gasoline,8.6426,Fan & Song 2022\nNissan Tiida,2X10608,114.06,140,Gasoline,7.4664,Fan & Song 2022\nNissan Tiida,2X10608,20.39,140,Gasoline,1.4877,Fan & Song 2022\nNissan Tiida,2X10608,34.51,140,Gasoline,3.0047,Fan & Song 2022\nNissan Tiida,2X10608,58.85,140,Gasoline,3.9555,Fan & Song 2022\nNissan Tiida,2X10608,115.08,140,Gasoline,8.093,Fan & Song 2022\nNissan Tiida,2X10608,8.44,140,Gasoline,0.8757,Fan & Song 2022\nNissan Tiida,2X10608,9.9,140,Gasoline,1.0387,Fan & Song 2022\nNissan Tiida,2X10608,137.83,140,Gasoline,8.4758,Fan & Song 2022\nNissan Tiida,2X10608,38.76,140,Gasoline,2.0668,Fan & Song 2022\nNissan Tiida,2X10608,15.12,140,Gasoline,1.241,Fan & Song 2022\nNissan Tiida,2X10608,44.82,140,Gasoline,2.406,Fan & Song 2022\nNissan Tiida,2X10608,3.69,140,Gasoline,0.3578,Fan & Song 2022\nNissan Tiida,2X10608,113.3,140,Gasoline,7.9147,Fan & Song 2022\nNissan Tiida,2X10608,6.09,140,Gasoline,0.5011,Fan & Song 2022\nNissan Tiida,2X10608,3.1,140,Gasoline,0.3878,Fan & Song 2022\nNissan Tiida,2X10608,189.46,140,Gasoline,14.0215,Fan & Song 2022\nVolkswagen Magotan,0X3050M,188.12,118,Gasoline,15.9218,Fan & Song 2022\nVolkswagen Magotan,0X3050X,45.01,118,Gasoline,2.8705,Fan & Song 2022\nVolkswagen Magotan,0X3050X,66.3,118,Gasoline,5.3039,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.67,118,Gasoline,0.7953,Fan & Song 2022\nVolkswagen Magotan,0X3050M,1.9,118,Gasoline,0.33,Fan & Song 2022\nVolkswagen Magotan,0X3050M,34.16,118,Gasoline,2.5843,Fan & Song 2022\nVolkswagen Magotan,0X3050M,140.34,118,Gasoline,11.2399,Fan & Song 2022\nVolkswagen Magotan,0X3050X,65.44,118,Gasoline,5.1925,Fan & Song 2022\nVolkswagen Magotan,0X3050M,19.28,118,Gasoline,1.5226,Fan & Song 2022\nVolkswagen Magotan,0X3050X,10.26,118,Gasoline,0.969,Fan & Song 2022\nVolkswagen Magotan,0X3050X,107.38,118,Gasoline,8.6327,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.13,118,Gasoline,0.3917,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.83,118,Gasoline,0.6495,Fan & Song 2022\nVolkswagen Magotan,0X3050X,10.66,118,Gasoline,1.2885,Fan & Song 2022\nVolkswagen Magotan,0X3050M,78.12,118,Gasoline,7.1967,Fan & Song 2022\nVolkswagen Magotan,0X3050M,42.86,118,Gasoline,4.3047,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.54,118,Gasoline,0.4505,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.41,118,Gasoline,0.6947,Fan & Song 2022\nVolkswagen Magotan,0X3050X,8.74,118,Gasoline,0.974,Fan & Song 2022\nVolkswagen Magotan,0X3050X,39.52,118,Gasoline,3.2976,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.24,118,Gasoline,1.4972,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.82,118,Gasoline,0.3173,Fan & Song 2022\nVolkswagen Magotan,0X3050X,31.14,118,Gasoline,4.1615,Fan & Song 2022\nVolkswagen Magotan,0X3050M,157.84,118,Gasoline,13.6553,Fan & Song 2022\nVolkswagen Magotan,0X3050M,42.37,118,Gasoline,4.7366,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.43,118,Gasoline,0.3776,Fan & Song 2022\nVolkswagen Magotan,0X3050X,76.24,118,Gasoline,5.8501,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.82,118,Gasoline,0.8309,Fan & Song 2022\nVolkswagen Magotan,0X3050X,66.17,118,Gasoline,5.1114,Fan & Song 2022\nVolkswagen Magotan,0X3050X,10.72,118,Gasoline,1.1129,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.65,118,Gasoline,0.7803,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.47,118,Gasoline,1.3135,Fan & Song 2022\nVolkswagen Magotan,0X3050X,7.84,118,Gasoline,0.8995,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.57,118,Gasoline,2.8776,Fan & Song 2022\nVolkswagen Magotan,0X3050M,32.26,118,Gasoline,2.3543,Fan & Song 2022\nVolkswagen Magotan,0X3050X,38.48,118,Gasoline,3.511,Fan & Song 2022\nVolkswagen Magotan,0X3050M,19.15,118,Gasoline,2.664,Fan & Song 2022\nVolkswagen Magotan,0X3050X,66.35,118,Gasoline,5.0148,Fan & Song 2022\nVolkswagen Magotan,0X3050M,115.62,118,Gasoline,8.3181,Fan & Song 2022\nVolkswagen Magotan,0X3050X,100.41,118,Gasoline,6.8885,Fan & Song 2022\nVolkswagen Magotan,0X3050M,114.29,118,Gasoline,8.1606,Fan & Song 2022\nVolkswagen Magotan,0X3050X,49.21,118,Gasoline,3.5336,Fan & Song 2022\nVolkswagen Magotan,0X3050X,103.3,118,Gasoline,7.0515,Fan & Song 2022\nVolkswagen Magotan,0X3050X,15.12,118,Gasoline,1.5887,Fan & Song 2022\nVolkswagen Magotan,0X3050X,151.31,118,Gasoline,11.1565,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.01,118,Gasoline,3.5412,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.66,118,Gasoline,1.5545,Fan & Song 2022\nVolkswagen Magotan,0X3050X,183.38,118,Gasoline,16.4662,Fan & Song 2022\nVolkswagen Magotan,0X3050X,78.93,118,Gasoline,5.0881,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.41,118,Gasoline,1.3121,Fan & Song 2022\nVolkswagen Magotan,0X3050X,370.35,118,Gasoline,29.5737,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.5,118,Gasoline,0.3342,Fan & Song 2022\nVolkswagen Magotan,0X3050M,25.1,118,Gasoline,2.1037,Fan & Song 2022\nVolkswagen Magotan,0X3050X,145.38,118,Gasoline,11.0587,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.39,118,Gasoline,0.4576,Fan & Song 2022\nVolkswagen Magotan,0X3050X,12.82,118,Gasoline,1.3285,Fan & Song 2022\nVolkswagen Magotan,0X3050X,142.04,118,Gasoline,10.5084,Fan & Song 2022\nVolkswagen Magotan,0X3050M,9.13,118,Gasoline,1.2028,Fan & Song 2022\nVolkswagen Magotan,0X3050M,23.08,118,Gasoline,1.8624,Fan & Song 2022\nVolkswagen Magotan,0X3050X,36.65,118,Gasoline,2.7885,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.06,118,Gasoline,0.3064,Fan & Song 2022\nVolkswagen Magotan,0X3050M,4.13,118,Gasoline,0.4838,Fan & Song 2022\nVolkswagen Magotan,0X3050X,114.2,118,Gasoline,9.3128,Fan & Song 2022\nVolkswagen Magotan,0X3050X,138.91,118,Gasoline,13.1504,Fan & Song 2022\nVolkswagen Magotan,0X3050X,142.37,118,Gasoline,11.6573,Fan & Song 2022\nVolkswagen Magotan,0X3050X,182.65,118,Gasoline,14.0711,Fan & Song 2022\nVolkswagen Magotan,0X3050X,135.93,118,Gasoline,10.4006,Fan & Song 2022\nVolkswagen Magotan,0X3050X,227.69,118,Gasoline,17.9207,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.12,118,Gasoline,1.185,Fan & Song 2022\nVolkswagen Magotan,0X3050X,121.1,118,Gasoline,9.3661,Fan & Song 2022\nVolkswagen Magotan,0X3050X,148.68,118,Gasoline,11.2461,Fan & Song 2022\nVolkswagen Magotan,0X3050X,104.75,118,Gasoline,8.5792,Fan & Song 2022\nVolkswagen Magotan,0X3050X,129.37,118,Gasoline,9.5347,Fan & Song 2022\nVolkswagen Magotan,0X3050X,181.38,118,Gasoline,12.6805,Fan & Song 2022\nVolkswagen Magotan,0X3050X,242.44,118,Gasoline,21.0012,Fan & Song 2022\nVolkswagen Magotan,0X3050X,16.8,118,Gasoline,1.4258,Fan & Song 2022\nVolkswagen Magotan,0X3050M,22.62,118,Gasoline,1.7979,Fan & Song 2022\nVolkswagen Magotan,0X3050X,1.84,118,Gasoline,0.3078,Fan & Song 2022\nVolkswagen Magotan,0X3050X,60.61,118,Gasoline,3.9318,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.25,118,Gasoline,1.9393,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.89,118,Gasoline,0.5137,Fan & Song 2022\nVolkswagen Magotan,0X3050X,13.81,118,Gasoline,1.3604,Fan & Song 2022\nVolkswagen Magotan,0X3050X,7.9,118,Gasoline,0.835,Fan & Song 2022\nVolkswagen Magotan,0X3050X,180.38,118,Gasoline,17.9587,Fan & Song 2022\nVolkswagen Magotan,0X3050X,40.41,118,Gasoline,3.2518,Fan & Song 2022\nVolkswagen Magotan,0X3050X,241.08,118,Gasoline,22.8804,Fan & Song 2022\nVolkswagen Magotan,0X3050X,137.89,118,Gasoline,9.952,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.86,118,Gasoline,0.7175,Fan & Song 2022\nVolkswagen Magotan,0X3050M,101.06,118,Gasoline,6.5657,Fan & Song 2022\nVolkswagen Magotan,0X3050M,104.81,118,Gasoline,6.8989,Fan & Song 2022\nVolkswagen Magotan,0X3050M,89.35,118,Gasoline,6.454,Fan & Song 2022\nVolkswagen Magotan,0X3050X,61.7,118,Gasoline,4.0392,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.9,118,Gasoline,3.072,Fan & Song 2022\nVolkswagen Magotan,0X3050X,12.57,118,Gasoline,1.1399,Fan & Song 2022\nVolkswagen Magotan,0X3050X,15.8,118,Gasoline,1.2786,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.87,118,Gasoline,0.7256,Fan & Song 2022\nVolkswagen Magotan,0X3050X,12.45,118,Gasoline,1.3547,Fan & Song 2022\nVolkswagen Magotan,0X3050X,104.17,118,Gasoline,5.8973,Fan & Song 2022\nVolkswagen Magotan,0X3050M,40.71,118,Gasoline,3.032,Fan & Song 2022\nVolkswagen Magotan,0X3050M,207.06,118,Gasoline,16.1903,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.18,118,Gasoline,0.6648,Fan & Song 2022\nVolkswagen Magotan,0X3050M,239.54,118,Gasoline,18.3215,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.96,118,Gasoline,0.5031,Fan & Song 2022\nVolkswagen Magotan,0X3050X,89.76,118,Gasoline,7.5883,Fan & Song 2022\nVolkswagen Magotan,0X3050X,117.54,118,Gasoline,10.5547,Fan & Song 2022\nVolkswagen Magotan,0X3050M,17.75,118,Gasoline,1.1866,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.19,118,Gasoline,0.4389,Fan & Song 2022\nVolkswagen Magotan,0X3050X,106.72,118,Gasoline,7.6472,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.26,118,Gasoline,0.9285,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.24,118,Gasoline,1.7269,Fan & Song 2022\nVolkswagen Magotan,0X3050X,37.8,118,Gasoline,2.8645,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.71,118,Gasoline,1.598,Fan & Song 2022\nVolkswagen Magotan,0X3050M,63.93,118,Gasoline,4.7857,Fan & Song 2022\nVolkswagen Magotan,0X3050M,105.49,118,Gasoline,7.6609,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.35,118,Gasoline,1.2781,Fan & Song 2022\nVolkswagen Magotan,0X3050M,69.96,118,Gasoline,5.2352,Fan & Song 2022\nVolkswagen Magotan,0X3050X,15.11,118,Gasoline,1.24,Fan & Song 2022\nVolkswagen Magotan,0X3050M,18.02,118,Gasoline,1.9426,Fan & Song 2022\nVolkswagen Magotan,0X3050X,100.54,118,Gasoline,8.0215,Fan & Song 2022\nVolkswagen Magotan,0X3050X,103.42,118,Gasoline,6.9501,Fan & Song 2022\nVolkswagen Magotan,0X3050X,103.57,118,Gasoline,7.8951,Fan & Song 2022\nVolkswagen Magotan,0X3050M,5.82,118,Gasoline,0.6071,Fan & Song 2022\nVolkswagen Magotan,0X3050M,25.53,118,Gasoline,2.3582,Fan & Song 2022\nVolkswagen Magotan,0X3050X,27.92,118,Gasoline,2.4909,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.31,118,Gasoline,0.39,Fan & Song 2022\nVolkswagen Magotan,0X3050X,190.21,118,Gasoline,13.8131,Fan & Song 2022\nVolkswagen Magotan,0X3050M,17.28,118,Gasoline,1.8338,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.72,118,Gasoline,0.3666,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.43,118,Gasoline,1.3379,Fan & Song 2022\nVolkswagen Magotan,0X3050M,9.9,118,Gasoline,1.4054,Fan & Song 2022\nVolkswagen Magotan,0X3050X,1.75,118,Gasoline,0.3141,Fan & Song 2022\nVolkswagen Magotan,0X3050X,1.94,118,Gasoline,0.2847,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.14,118,Gasoline,0.57,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.33,118,Gasoline,1.2493,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.48,118,Gasoline,0.5297,Fan & Song 2022\nVolkswagen Magotan,0X3050M,29.26,118,Gasoline,2.4754,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.53,118,Gasoline,1.1893,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.2,118,Gasoline,0.2184,Fan & Song 2022\nVolkswagen Magotan,0X3050M,8.48,118,Gasoline,0.8346,Fan & Song 2022\nVolkswagen Magotan,0X3050X,121.93,118,Gasoline,7.7285,Fan & Song 2022\nVolkswagen Magotan,0X3050X,119.94,118,Gasoline,7.8107,Fan & Song 2022\nVolkswagen Magotan,0X3050X,103.49,118,Gasoline,8.3008,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.68,118,Gasoline,0.5105,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.1,118,Gasoline,1.0903,Fan & Song 2022\nVolkswagen Magotan,0X3050M,5.98,118,Gasoline,0.6706,Fan & Song 2022\nVolkswagen Magotan,0X3050X,66.78,118,Gasoline,5.5576,Fan & Song 2022\nVolkswagen Magotan,0X3050X,101.17,118,Gasoline,5.796,Fan & Song 2022\nVolkswagen Magotan,0X3050X,20.22,118,Gasoline,2.7866,Fan & Song 2022\nVolkswagen Magotan,0X3050X,32.68,118,Gasoline,2.9427,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.44,118,Gasoline,1.3111,Fan & Song 2022\nVolkswagen Magotan,0X3050X,102.04,118,Gasoline,7.2979,Fan & Song 2022\nVolkswagen Magotan,0X3050M,27.27,118,Gasoline,2.5356,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.0,118,Gasoline,0.3675,Fan & Song 2022\nVolkswagen Magotan,0X3050M,10.91,118,Gasoline,1.4369,Fan & Song 2022\nVolkswagen Magotan,0X3050M,152.14,118,Gasoline,10.9475,Fan & Song 2022\nVolkswagen Magotan,0X3050X,9.52,118,Gasoline,0.9372,Fan & Song 2022\nVolkswagen Magotan,0X3050X,97.17,118,Gasoline,7.8722,Fan & Song 2022\nVolkswagen Magotan,0X3050X,81.13,118,Gasoline,5.571,Fan & Song 2022\nVolkswagen Magotan,0X3050M,153.12,118,Gasoline,11.8545,Fan & Song 2022\nVolkswagen Magotan,0X3050X,30.78,118,Gasoline,2.3202,Fan & Song 2022\nVolkswagen Magotan,0X3050X,79.11,118,Gasoline,5.1087,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.61,118,Gasoline,1.46,Fan & Song 2022\nVolkswagen Magotan,0X3050M,177.89,118,Gasoline,13.5424,Fan & Song 2022\nVolkswagen Magotan,0X3050X,7.46,118,Gasoline,1.0408,Fan & Song 2022\nVolkswagen Magotan,0X3050X,13.02,118,Gasoline,1.4944,Fan & Song 2022\nVolkswagen Magotan,0X3050X,12.55,118,Gasoline,1.0442,Fan & Song 2022\nVolkswagen Magotan,0X3050M,60.26,118,Gasoline,4.948,Fan & Song 2022\nVolkswagen Magotan,0X3050M,109.31,118,Gasoline,8.4614,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.93,118,Gasoline,1.3062,Fan & Song 2022\nVolkswagen Magotan,0X3050M,113.34,118,Gasoline,7.3947,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.94,118,Gasoline,0.4197,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.13,118,Gasoline,1.8665,Fan & Song 2022\nVolkswagen Magotan,0X3050X,36.57,118,Gasoline,2.6371,Fan & Song 2022\nVolkswagen Magotan,0X3050M,31.81,118,Gasoline,4.0688,Fan & Song 2022\nVolkswagen Magotan,0X3050M,24.7,118,Gasoline,2.3464,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.42,118,Gasoline,0.3379,Fan & Song 2022\nVolkswagen Magotan,0X3050M,59.24,118,Gasoline,4.41,Fan & Song 2022\nVolkswagen Magotan,0X3050X,31.56,118,Gasoline,2.2539,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.72,118,Gasoline,0.7399,Fan & Song 2022\nVolkswagen Magotan,0X3050X,143.97,118,Gasoline,10.2673,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.31,118,Gasoline,0.6552,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.29,118,Gasoline,1.3369,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.33,118,Gasoline,2.0697,Fan & Song 2022\nVolkswagen Magotan,0X3050X,33.58,118,Gasoline,2.762,Fan & Song 2022\nVolkswagen Magotan,0X3050M,162.8,118,Gasoline,11.7408,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.92,118,Gasoline,0.3148,Fan & Song 2022\nVolkswagen Magotan,0X3050M,124.32,118,Gasoline,10.3356,Fan & Song 2022\nVolkswagen Magotan,0X3050M,114.98,118,Gasoline,9.873,Fan & Song 2022\nVolkswagen Magotan,0X3050X,88.22,118,Gasoline,6.2653,Fan & Song 2022\nVolkswagen Magotan,0X3050X,22.76,118,Gasoline,1.7032,Fan & Song 2022\nVolkswagen Magotan,0X3050M,68.84,118,Gasoline,4.6009,Fan & Song 2022\nVolkswagen Magotan,0X3050M,25.77,118,Gasoline,2.2397,Fan & Song 2022\nVolkswagen Magotan,0X3050X,123.79,118,Gasoline,8.1046,Fan & Song 2022\nVolkswagen Magotan,0X3050M,55.19,118,Gasoline,4.5849,Fan & Song 2022\nVolkswagen Magotan,0X3050M,60.08,118,Gasoline,4.8839,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.24,118,Gasoline,0.4192,Fan & Song 2022\nVolkswagen Magotan,0X3050M,47.85,118,Gasoline,4.9392,Fan & Song 2022\nVolkswagen Magotan,0X3050X,24.12,118,Gasoline,2.2771,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.84,118,Gasoline,0.754,Fan & Song 2022\nVolkswagen Magotan,0X3050X,197.6,118,Gasoline,13.7958,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.2,118,Gasoline,0.2229,Fan & Song 2022\nVolkswagen Magotan,0X3050X,33.06,118,Gasoline,3.254,Fan & Song 2022\nVolkswagen Magotan,0X3050M,11.68,118,Gasoline,1.3615,Fan & Song 2022\nVolkswagen Magotan,0X3050X,23.0,118,Gasoline,2.5145,Fan & Song 2022\nVolkswagen Magotan,0X3050M,123.12,118,Gasoline,10.5942,Fan & Song 2022\nVolkswagen Magotan,0X3050M,21.21,118,Gasoline,1.5969,Fan & Song 2022\nVolkswagen Magotan,0X3050M,1.74,118,Gasoline,0.2534,Fan & Song 2022\nVolkswagen Magotan,0X3050X,120.36,118,Gasoline,9.775,Fan & Song 2022\nVolkswagen Magotan,0X3050X,118.62,118,Gasoline,9.4905,Fan & Song 2022\nVolkswagen Magotan,0X3050X,89.76,118,Gasoline,6.3297,Fan & Song 2022\nVolkswagen Magotan,0X3050X,60.07,118,Gasoline,5.3088,Fan & Song 2022\nVolkswagen Magotan,0X3050X,157.67,118,Gasoline,10.568,Fan & Song 2022\nVolkswagen Magotan,0X3050X,81.67,118,Gasoline,5.4392,Fan & Song 2022\nVolkswagen Magotan,0X3050X,76.64,118,Gasoline,5.545,Fan & Song 2022\nVolkswagen Magotan,0X3050X,226.12,118,Gasoline,18.6235,Fan & Song 2022\nVolkswagen Magotan,0X3050X,92.41,118,Gasoline,6.0929,Fan & Song 2022\nVolkswagen Magotan,0X3050M,93.52,118,Gasoline,6.9522,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.16,118,Gasoline,0.2648,Fan & Song 2022\nVolkswagen Magotan,0X3050X,157.16,118,Gasoline,10.6957,Fan & Song 2022\nVolkswagen Magotan,0X3050X,28.36,118,Gasoline,2.2454,Fan & Song 2022\nVolkswagen Magotan,0X3050X,39.75,118,Gasoline,3.0594,Fan & Song 2022\nVolkswagen Magotan,0X3050X,106.56,118,Gasoline,9.5008,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.74,118,Gasoline,0.7425,Fan & Song 2022\nVolkswagen Magotan,0X3050X,56.04,118,Gasoline,5.5437,Fan & Song 2022\nVolkswagen Magotan,0X3050M,141.06,118,Gasoline,9.4278,Fan & Song 2022\nVolkswagen Magotan,0X3050X,121.3,118,Gasoline,8.8237,Fan & Song 2022\nVolkswagen Magotan,0X3050M,49.88,118,Gasoline,2.9444,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.2,118,Gasoline,0.247,Fan & Song 2022\nVolkswagen Magotan,0X3050X,18.53,118,Gasoline,1.7763,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.26,118,Gasoline,1.5503,Fan & Song 2022\nVolkswagen Magotan,0X3050M,104.47,118,Gasoline,9.4581,Fan & Song 2022\nVolkswagen Magotan,0X3050M,37.42,118,Gasoline,4.8288,Fan & Song 2022\nVolkswagen Magotan,0X3050X,101.01,118,Gasoline,7.3048,Fan & Song 2022\nVolkswagen Magotan,0X3050M,116.59,118,Gasoline,8.5499,Fan & Song 2022\nVolkswagen Magotan,0X3050X,36.19,118,Gasoline,2.6312,Fan & Song 2022\nVolkswagen Magotan,0X3050M,9.4,118,Gasoline,1.293,Fan & Song 2022\nVolkswagen Magotan,0X3050M,33.81,118,Gasoline,4.0109,Fan & Song 2022\nVolkswagen Magotan,0X3050M,71.33,118,Gasoline,6.0818,Fan & Song 2022\nVolkswagen Magotan,0X3050M,23.46,118,Gasoline,2.0659,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.59,118,Gasoline,0.2699,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.0,118,Gasoline,0.676,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.95,118,Gasoline,0.5807,Fan & Song 2022\nVolkswagen Magotan,0X3050X,12.25,118,Gasoline,1.0623,Fan & Song 2022\nVolkswagen Magotan,0X3050X,11.97,118,Gasoline,0.9492,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.7,118,Gasoline,1.6371,Fan & Song 2022\nVolkswagen Magotan,0X3050X,38.83,118,Gasoline,2.7729,Fan & Song 2022\nVolkswagen Magotan,0X3050X,10.7,118,Gasoline,1.3662,Fan & Song 2022\nVolkswagen Magotan,0X3050M,38.35,118,Gasoline,4.2364,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.82,118,Gasoline,3.478,Fan & Song 2022\nVolkswagen Magotan,0X3050M,5.11,118,Gasoline,0.8647,Fan & Song 2022\nVolkswagen Magotan,0X3050X,48.87,118,Gasoline,4.0183,Fan & Song 2022\nVolkswagen Magotan,0X3050M,54.63,118,Gasoline,4.252,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.16,118,Gasoline,0.5507,Fan & Song 2022\nVolkswagen Magotan,0X3050M,6.06,118,Gasoline,0.5289,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.91,118,Gasoline,0.3556,Fan & Song 2022\nVolkswagen Magotan,0X3050X,42.56,118,Gasoline,3.2841,Fan & Song 2022\nVolkswagen Magotan,0X3050M,125.98,118,Gasoline,9.3095,Fan & Song 2022\nVolkswagen Magotan,0X3050M,258.74,118,Gasoline,19.5664,Fan & Song 2022\nVolkswagen Magotan,0X3050X,153.05,118,Gasoline,12.0291,Fan & Song 2022\nVolkswagen Magotan,0X3050M,8.37,118,Gasoline,0.9824,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.8,118,Gasoline,0.3562,Fan & Song 2022\nVolkswagen Magotan,0X3050M,32.61,118,Gasoline,3.6736,Fan & Song 2022\nVolkswagen Magotan,0X3050M,10.6,118,Gasoline,0.9274,Fan & Song 2022\nVolkswagen Magotan,0X3050M,28.2,118,Gasoline,2.6548,Fan & Song 2022\nVolkswagen Magotan,0X3050M,24.09,118,Gasoline,2.5503,Fan & Song 2022\nVolkswagen Magotan,0X3050M,16.24,118,Gasoline,2.3762,Fan & Song 2022\nVolkswagen Magotan,0X3050X,31.81,118,Gasoline,2.9072,Fan & Song 2022\nVolkswagen Magotan,0X3050M,55.72,118,Gasoline,4.7956,Fan & Song 2022\nVolkswagen Magotan,0X3050X,15.11,118,Gasoline,1.1035,Fan & Song 2022\nVolkswagen Magotan,0X3050X,18.03,118,Gasoline,1.4064,Fan & Song 2022\nVolkswagen Magotan,0X3050X,9.74,118,Gasoline,0.8285,Fan & Song 2022\nVolkswagen Magotan,0X3050X,41.03,118,Gasoline,3.5504,Fan & Song 2022\nVolkswagen Magotan,0X3050X,10.1,118,Gasoline,1.1173,Fan & Song 2022\nVolkswagen Magotan,0X3050X,7.2,118,Gasoline,0.8191,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.89,118,Gasoline,0.2653,Fan & Song 2022\nVolkswagen Magotan,0X3050X,25.99,118,Gasoline,2.4852,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.74,118,Gasoline,0.6349,Fan & Song 2022\nVolkswagen Magotan,0X3050X,17.87,118,Gasoline,1.9122,Fan & Song 2022\nVolkswagen Magotan,0X3050X,29.34,118,Gasoline,2.9858,Fan & Song 2022\nVolkswagen Magotan,0X3050M,36.33,118,Gasoline,3.6293,Fan & Song 2022\nVolkswagen Magotan,0X3050M,261.32,118,Gasoline,19.7331,Fan & Song 2022\nVolkswagen Magotan,0X3050X,191.95,118,Gasoline,11.8912,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.48,118,Gasoline,2.68,Fan & Song 2022\nVolkswagen Magotan,0X3050M,360.35,118,Gasoline,27.546,Fan & Song 2022\nVolkswagen Magotan,0X3050X,165.12,118,Gasoline,11.6037,Fan & Song 2022\nVolkswagen Magotan,0X3050M,177.16,118,Gasoline,14.7186,Fan & Song 2022\nVolkswagen Magotan,0X3050X,34.96,118,Gasoline,3.5279,Fan & Song 2022\nVolkswagen Magotan,0X3050X,13.9,118,Gasoline,1.4022,Fan & Song 2022\nVolkswagen Magotan,0X3050M,29.31,118,Gasoline,3.3911,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.29,118,Gasoline,1.3091,Fan & Song 2022\nVolkswagen Magotan,0X3050X,13.82,118,Gasoline,1.4418,Fan & Song 2022\nVolkswagen Magotan,0X3050X,10.0,118,Gasoline,1.0281,Fan & Song 2022\nVolkswagen Magotan,0X3050M,1.5,118,Gasoline,0.2213,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.88,118,Gasoline,1.7224,Fan & Song 2022\nVolkswagen Magotan,0X3050M,4.47,118,Gasoline,0.3732,Fan & Song 2022\nVolkswagen Magotan,0X3050M,133.73,118,Gasoline,11.1993,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.82,118,Gasoline,2.2649,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.71,118,Gasoline,0.722,Fan & Song 2022\nVolkswagen Magotan,0X3050X,89.49,118,Gasoline,9.4148,Fan & Song 2022\nVolkswagen Magotan,0X3050X,32.37,118,Gasoline,3.5845,Fan & Song 2022\nVolkswagen Magotan,0X3050M,106.25,118,Gasoline,8.3703,Fan & Song 2022\nVolkswagen Magotan,0X3050X,17.74,118,Gasoline,2.0955,Fan & Song 2022\nVolkswagen Magotan,0X3050M,53.44,118,Gasoline,4.5015,Fan & Song 2022\nVolkswagen Magotan,0X3050X,38.23,118,Gasoline,3.0478,Fan & Song 2022\nVolkswagen Magotan,0X3050X,39.49,118,Gasoline,3.5426,Fan & Song 2022\nVolkswagen Magotan,0X3050X,13.92,118,Gasoline,1.1542,Fan & Song 2022\nVolkswagen Magotan,0X3050M,58.97,118,Gasoline,7.0264,Fan & Song 2022\nVolkswagen Magotan,0X3050M,21.89,118,Gasoline,1.6622,Fan & Song 2022\nVolkswagen Magotan,0X3050M,84.05,118,Gasoline,7.4909,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.28,118,Gasoline,3.3567,Fan & Song 2022\nVolkswagen Magotan,0X3050M,36.1,118,Gasoline,4.8757,Fan & Song 2022\nVolkswagen Magotan,0X3050M,59.11,118,Gasoline,7.2288,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.25,118,Gasoline,0.2709,Fan & Song 2022\nVolkswagen Magotan,0X3050X,49.51,118,Gasoline,7.1717,Fan & Song 2022\nVolkswagen Magotan,0X3050M,36.45,118,Gasoline,2.8326,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.57,118,Gasoline,0.3078,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.42,118,Gasoline,2.3748,Fan & Song 2022\nVolkswagen Magotan,0X3050M,138.35,118,Gasoline,13.8723,Fan & Song 2022\nVolkswagen Magotan,0X3050X,137.32,118,Gasoline,9.2188,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.75,118,Gasoline,0.6383,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.08,118,Gasoline,1.0683,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.98,118,Gasoline,1.3221,Fan & Song 2022\nVolkswagen Magotan,0X3050X,137.26,118,Gasoline,9.8381,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.61,118,Gasoline,1.2824,Fan & Song 2022\nVolkswagen Magotan,0X3050M,50.77,118,Gasoline,3.7888,Fan & Song 2022\nVolkswagen Magotan,0X3050X,8.66,118,Gasoline,0.7723,Fan & Song 2022\nVolkswagen Magotan,0X3050X,48.15,118,Gasoline,3.8988,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.64,118,Gasoline,1.53,Fan & Song 2022\nVolkswagen Magotan,0X3050M,6.22,118,Gasoline,0.693,Fan & Song 2022\nVolkswagen Magotan,0X3050X,17.89,118,Gasoline,1.9936,Fan & Song 2022\nVolkswagen Magotan,0X3050X,103.64,118,Gasoline,6.9079,Fan & Song 2022\nVolkswagen Magotan,0X3050M,41.33,118,Gasoline,6.0109,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.86,118,Gasoline,1.4224,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.79,118,Gasoline,0.9689,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.28,118,Gasoline,0.3089,Fan & Song 2022\nVolkswagen Magotan,0X3050X,105.1,118,Gasoline,6.9162,Fan & Song 2022\nVolkswagen Magotan,0X3050M,35.53,118,Gasoline,3.5854,Fan & Song 2022\nVolkswagen Magotan,0X3050X,34.35,118,Gasoline,2.4568,Fan & Song 2022\nVolkswagen Magotan,0X3050M,6.3,118,Gasoline,0.6921,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.7,118,Gasoline,0.5238,Fan & Song 2022\nVolkswagen Magotan,0X3050M,28.43,118,Gasoline,3.7952,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.23,118,Gasoline,0.7645,Fan & Song 2022\nVolkswagen Magotan,0X3050M,58.98,118,Gasoline,8.1572,Fan & Song 2022\nVolkswagen Magotan,0X3050X,347.24,118,Gasoline,26.1172,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.97,118,Gasoline,1.3587,Fan & Song 2022\nVolkswagen Magotan,0X3050M,60.29,118,Gasoline,4.8992,Fan & Song 2022\nVolkswagen Magotan,0X3050M,134.76,118,Gasoline,12.5057,Fan & Song 2022\nVolkswagen Magotan,0X3050M,45.32,118,Gasoline,3.8848,Fan & Song 2022\nVolkswagen Magotan,0X3050M,55.82,118,Gasoline,4.7061,Fan & Song 2022\nVolkswagen Magotan,0X3050X,190.84,118,Gasoline,13.6161,Fan & Song 2022\nVolkswagen Magotan,0X3050M,21.65,118,Gasoline,3.019,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.78,118,Gasoline,0.4749,Fan & Song 2022\nVolkswagen Magotan,0X3050M,194.59,118,Gasoline,18.1185,Fan & Song 2022\nVolkswagen Magotan,0X3050M,11.68,118,Gasoline,1.2216,Fan & Song 2022\nVolkswagen Magotan,0X3050M,32.75,118,Gasoline,2.1282,Fan & Song 2022\nVolkswagen Magotan,0X3050X,185.21,118,Gasoline,12.8335,Fan & Song 2022\nVolkswagen Magotan,0X3050M,147.38,118,Gasoline,11.9143,Fan & Song 2022\nVolkswagen Magotan,0X3050M,129.4,118,Gasoline,10.279,Fan & Song 2022\nVolkswagen Magotan,0X3050M,163.98,118,Gasoline,13.1125,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.18,118,Gasoline,0.3023,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.18,118,Gasoline,0.3573,Fan & Song 2022\nVolkswagen Magotan,0X3050M,108.4,118,Gasoline,9.4048,Fan & Song 2022\nVolkswagen Magotan,0X3050M,132.43,118,Gasoline,14.1866,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.9,118,Gasoline,0.6507,Fan & Song 2022\nVolkswagen Magotan,0X3050X,136.09,118,Gasoline,9.9195,Fan & Song 2022\nVolkswagen Magotan,0X3050M,18.48,118,Gasoline,2.8182,Fan & Song 2022\nVolkswagen Magotan,0X3050M,35.39,118,Gasoline,4.4258,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.6,118,Gasoline,1.5846,Fan & Song 2022\nVolkswagen Magotan,0X3050M,204.84,118,Gasoline,16.1392,Fan & Song 2022\nVolkswagen Magotan,0X3050X,92.75,118,Gasoline,7.9247,Fan & Song 2022\nVolkswagen Magotan,0X3050M,32.84,118,Gasoline,3.1975,Fan & Song 2022\nVolkswagen Magotan,0X3050M,38.06,118,Gasoline,4.1334,Fan & Song 2022\nVolkswagen Magotan,0X3050X,15.22,118,Gasoline,1.2094,Fan & Song 2022\nVolkswagen Magotan,0X3050M,103.28,118,Gasoline,8.1152,Fan & Song 2022\nVolkswagen Magotan,0X3050M,61.68,118,Gasoline,6.5475,Fan & Song 2022\nVolkswagen Magotan,0X3050M,53.44,118,Gasoline,4.4073,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.77,118,Gasoline,0.4319,Fan & Song 2022\nVolkswagen Magotan,0X3050M,105.22,118,Gasoline,9.3554,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.27,118,Gasoline,1.497,Fan & Song 2022\nVolkswagen Magotan,0X3050X,105.44,118,Gasoline,6.9574,Fan & Song 2022\nVolkswagen Magotan,0X3050M,1.67,118,Gasoline,0.2694,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.47,118,Gasoline,0.569,Fan & Song 2022\nVolkswagen Magotan,0X3050M,44.12,118,Gasoline,3.8738,Fan & Song 2022\nVolkswagen Magotan,0X3050X,8.21,118,Gasoline,0.5502,Fan & Song 2022\nVolkswagen Magotan,0X3050M,189.94,118,Gasoline,16.5156,Fan & Song 2022\nVolkswagen Magotan,0X3050X,234.43,118,Gasoline,20.1598,Fan & Song 2022\nVolkswagen Magotan,0X3050X,128.89,118,Gasoline,7.1705,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.57,118,Gasoline,0.545,Fan & Song 2022\nVolkswagen Magotan,0X3050X,105.0,118,Gasoline,6.6088,Fan & Song 2022\nVolkswagen Magotan,0X3050X,150.9,118,Gasoline,8.9398,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.3,118,Gasoline,0.7576,Fan & Song 2022\nVolkswagen Magotan,0X3050X,239.67,118,Gasoline,16.3596,Fan & Song 2022\nVolkswagen Magotan,0X3050X,142.0,118,Gasoline,10.3291,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.45,118,Gasoline,0.3553,Fan & Song 2022\nVolkswagen Magotan,0X3050X,136.41,118,Gasoline,11.4779,Fan & Song 2022\nVolkswagen Magotan,0X3050M,10.69,118,Gasoline,1.2001,Fan & Song 2022\nVolkswagen Magotan,0X3050M,110.81,118,Gasoline,8.1848,Fan & Song 2022\nVolkswagen Magotan,0X3050M,81.31,118,Gasoline,6.7277,Fan & Song 2022\nVolkswagen Magotan,0X3050M,19.79,118,Gasoline,2.4664,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.41,118,Gasoline,0.3999,Fan & Song 2022\nVolkswagen Magotan,0X3050X,13.89,118,Gasoline,1.3406,Fan & Song 2022\nVolkswagen Magotan,0X3050M,1.63,118,Gasoline,0.2954,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.02,118,Gasoline,0.2612,Fan & Song 2022\nVolkswagen Magotan,0X3050M,102.14,118,Gasoline,8.3351,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.59,118,Gasoline,1.5351,Fan & Song 2022\nVolkswagen Magotan,0X3050M,34.83,118,Gasoline,3.9518,Fan & Song 2022\nVolkswagen Magotan,0X3050X,17.18,118,Gasoline,1.6469,Fan & Song 2022\nVolkswagen Magotan,0X3050X,34.76,118,Gasoline,3.0308,Fan & Song 2022\nVolkswagen Magotan,0X3050X,113.17,118,Gasoline,6.7526,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.42,118,Gasoline,0.3375,Fan & Song 2022\nVolkswagen Magotan,0X3050M,132.93,118,Gasoline,12.2665,Fan & Song 2022\nVolkswagen Magotan,0X3050M,164.17,118,Gasoline,14.0208,Fan & Song 2022\nVolkswagen Magotan,0X3050X,43.95,118,Gasoline,2.8723,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.51,118,Gasoline,0.3767,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.56,118,Gasoline,0.5241,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.64,118,Gasoline,0.342,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.48,118,Gasoline,1.2468,Fan & Song 2022\nVolkswagen Magotan,0X3050X,8.03,118,Gasoline,0.8524,Fan & Song 2022\nVolkswagen Magotan,0X3050X,13.86,118,Gasoline,1.3302,Fan & Song 2022\nVolkswagen Magotan,0X3050X,9.4,118,Gasoline,0.7974,Fan & Song 2022\nVolkswagen Magotan,0X3050M,163.95,118,Gasoline,11.562,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.16,118,Gasoline,0.66,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.29,118,Gasoline,0.5841,Fan & Song 2022\nVolkswagen Magotan,0X3050M,29.6,118,Gasoline,2.9199,Fan & Song 2022\nVolkswagen Magotan,0X3050M,117.53,118,Gasoline,9.2039,Fan & Song 2022\nVolkswagen Magotan,0X3050X,7.0,118,Gasoline,0.7212,Fan & Song 2022\nVolkswagen Magotan,0X3050M,175.75,118,Gasoline,11.8422,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.64,118,Gasoline,0.689,Fan & Song 2022\nVolkswagen Magotan,0X3050M,1.94,118,Gasoline,0.3107,Fan & Song 2022\nVolkswagen Magotan,0X3050M,124.68,118,Gasoline,8.4759,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.64,118,Gasoline,0.6787,Fan & Song 2022\nVolkswagen Magotan,0X3050X,121.88,118,Gasoline,9.7089,Fan & Song 2022\nVolkswagen Magotan,0X3050M,103.78,118,Gasoline,7.4453,Fan & Song 2022\nVolkswagen Magotan,0X3050M,8.53,118,Gasoline,0.7968,Fan & Song 2022\nVolkswagen Magotan,0X3050X,40.8,118,Gasoline,3.2306,Fan & Song 2022\nVolkswagen Magotan,0X3050X,32.12,118,Gasoline,2.7547,Fan & Song 2022\nVolkswagen Magotan,0X3050X,18.67,118,Gasoline,1.3043,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.04,118,Gasoline,0.3919,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.08,118,Gasoline,0.451,Fan & Song 2022\nVolkswagen Magotan,0X3050M,22.26,118,Gasoline,1.9676,Fan & Song 2022\nVolkswagen Magotan,0X3050X,44.68,118,Gasoline,4.0699,Fan & Song 2022\nVolkswagen Magotan,0X3050M,43.36,118,Gasoline,3.6723,Fan & Song 2022\nVolkswagen Magotan,0X3050M,11.56,118,Gasoline,0.9967,Fan & Song 2022\nVolkswagen Magotan,0X3050X,18.64,118,Gasoline,1.4356,Fan & Song 2022\nVolkswagen Magotan,0X3050M,12.7,118,Gasoline,1.1848,Fan & Song 2022\nVolkswagen Magotan,0X3050X,13.12,118,Gasoline,1.2463,Fan & Song 2022\nVolkswagen Magotan,0X3050M,11.9,118,Gasoline,1.2164,Fan & Song 2022\nVolkswagen Magotan,0X3050M,52.62,118,Gasoline,3.6949,Fan & Song 2022\nVolkswagen Magotan,0X3050M,52.08,118,Gasoline,3.3057,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.89,118,Gasoline,0.4043,Fan & Song 2022\nVolkswagen Magotan,0X3050X,115.34,118,Gasoline,7.8856,Fan & Song 2022\nVolkswagen Magotan,0X3050X,115.24,118,Gasoline,7.2319,Fan & Song 2022\nVolkswagen Magotan,0X3050M,190.46,118,Gasoline,14.0522,Fan & Song 2022\nVolkswagen Magotan,0X3050M,261.7,118,Gasoline,17.8483,Fan & Song 2022\nVolkswagen Magotan,0X3050X,173.68,118,Gasoline,12.15,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.4,118,Gasoline,0.6881,Fan & Song 2022\nVolkswagen Magotan,0X3050M,42.38,118,Gasoline,3.3438,Fan & Song 2022\nVolkswagen Magotan,0X3050X,145.17,118,Gasoline,10.3022,Fan & Song 2022\nVolkswagen Magotan,0X3050M,360.8,118,Gasoline,29.5901,Fan & Song 2022\nVolkswagen Magotan,0X3050X,112.06,118,Gasoline,7.872,Fan & Song 2022\nVolkswagen Magotan,0X3050M,124.8,118,Gasoline,9.0711,Fan & Song 2022\nVolkswagen Magotan,0X3050M,105.02,118,Gasoline,7.1204,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.1,118,Gasoline,2.0766,Fan & Song 2022\nVolkswagen Magotan,0X3050M,140.43,118,Gasoline,9.875,Fan & Song 2022\nVolkswagen Magotan,0X3050M,3.81,118,Gasoline,0.3265,Fan & Song 2022\nVolkswagen Magotan,0X3050M,35.33,118,Gasoline,3.8089,Fan & Song 2022\nVolkswagen Magotan,0X3050X,7.26,118,Gasoline,0.7148,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.41,118,Gasoline,0.3009,Fan & Song 2022\nVolkswagen Magotan,0X3050M,17.27,118,Gasoline,1.7782,Fan & Song 2022\nVolkswagen Magotan,0X3050X,46.33,118,Gasoline,3.4526,Fan & Song 2022\nVolkswagen Magotan,0X3050X,8.19,118,Gasoline,0.6873,Fan & Song 2022\nVolkswagen Magotan,0X3050M,6.55,118,Gasoline,0.8379,Fan & Song 2022\nVolkswagen Magotan,0X3050M,47.19,118,Gasoline,3.5243,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.87,118,Gasoline,0.6464,Fan & Song 2022\nVolkswagen Magotan,0X3050M,31.66,118,Gasoline,4.0555,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.85,118,Gasoline,0.7042,Fan & Song 2022\nVolkswagen Magotan,0X3050M,51.59,118,Gasoline,5.4614,Fan & Song 2022\nVolkswagen Magotan,0X3050M,4.3,118,Gasoline,0.8228,Fan & Song 2022\nVolkswagen Magotan,0X3050M,55.2,118,Gasoline,4.6815,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.67,118,Gasoline,0.6481,Fan & Song 2022\nVolkswagen Magotan,0X3050M,146.55,118,Gasoline,11.5362,Fan & Song 2022\nVolkswagen Magotan,0X3050M,59.39,118,Gasoline,4.714,Fan & Song 2022\nVolkswagen Magotan,0X3050M,63.26,118,Gasoline,4.6685,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.48,118,Gasoline,0.6548,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.58,118,Gasoline,1.5676,Fan & Song 2022\nVolkswagen Magotan,0X3050X,28.42,118,Gasoline,3.1214,Fan & Song 2022\nVolkswagen Magotan,0X3050M,74.05,118,Gasoline,5.216,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.04,118,Gasoline,1.1762,Fan & Song 2022\nVolkswagen Magotan,0X3050M,30.93,118,Gasoline,3.0801,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.25,118,Gasoline,1.0878,Fan & Song 2022\nVolkswagen Magotan,0X3050X,15.7,118,Gasoline,1.8586,Fan & Song 2022\nVolkswagen Magotan,0X3050M,109.28,118,Gasoline,8.464,Fan & Song 2022\nVolkswagen Magotan,0X3050X,16.09,118,Gasoline,1.3348,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.22,118,Gasoline,0.493,Fan & Song 2022\nVolkswagen Magotan,0X3050M,11.75,118,Gasoline,1.3376,Fan & Song 2022\nVolkswagen Magotan,0X3050M,7.22,118,Gasoline,1.0226,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.25,118,Gasoline,1.4576,Fan & Song 2022\nVolkswagen Magotan,0X3050M,144.54,118,Gasoline,10.1016,Fan & Song 2022\nVolkswagen Magotan,0X3050M,17.6,118,Gasoline,1.5453,Fan & Song 2022\nVolkswagen Magotan,0X3050X,15.78,118,Gasoline,1.2674,Fan & Song 2022\nVolkswagen Magotan,0X3050M,63.27,118,Gasoline,4.2733,Fan & Song 2022\nVolkswagen Magotan,0X3050M,8.59,118,Gasoline,0.888,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.86,118,Gasoline,0.5169,Fan & Song 2022\nVolkswagen Magotan,0X3050M,44.24,118,Gasoline,5.2027,Fan & Song 2022\nVolkswagen Magotan,0X3050M,109.48,118,Gasoline,8.7564,Fan & Song 2022\nVolkswagen Magotan,0X3050M,22.85,118,Gasoline,1.5476,Fan & Song 2022\nVolkswagen Magotan,0X3050X,26.05,118,Gasoline,2.9619,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.98,118,Gasoline,0.4095,Fan & Song 2022\nVolkswagen Magotan,0X3050X,24.91,118,Gasoline,1.9332,Fan & Song 2022\nVolkswagen Magotan,0X3050M,111.24,118,Gasoline,9.2859,Fan & Song 2022\nVolkswagen Magotan,0X3050X,9.8,118,Gasoline,0.8939,Fan & Song 2022\nVolkswagen Magotan,0X3050M,120.94,118,Gasoline,10.2537,Fan & Song 2022\nVolkswagen Magotan,0X3050M,11.72,118,Gasoline,1.3966,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.45,118,Gasoline,1.392,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.59,118,Gasoline,0.7983,Fan & Song 2022\nVolkswagen Magotan,0X3050X,24.02,118,Gasoline,1.6345,Fan & Song 2022\nVolkswagen Magotan,0X3050M,62.88,118,Gasoline,4.9576,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.29,118,Gasoline,0.5673,Fan & Song 2022\nVolkswagen Magotan,0X3050M,59.93,118,Gasoline,3.7398,Fan & Song 2022\nVolkswagen Magotan,0X3050X,7.12,118,Gasoline,0.9039,Fan & Song 2022\nVolkswagen Magotan,0X3050X,3.92,118,Gasoline,0.4973,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.69,118,Gasoline,0.3265,Fan & Song 2022\nVolkswagen Magotan,0X3050M,25.98,118,Gasoline,2.1309,Fan & Song 2022\nVolkswagen Magotan,0X3050X,17.37,118,Gasoline,1.2393,Fan & Song 2022\nVolkswagen Magotan,0X3050M,64.85,118,Gasoline,6.3494,Fan & Song 2022\nVolkswagen Magotan,0X3050M,13.25,118,Gasoline,1.2721,Fan & Song 2022\nVolkswagen Magotan,0X3050M,150.0,118,Gasoline,12.3641,Fan & Song 2022\nVolkswagen Magotan,0X3050M,164.04,118,Gasoline,14.9306,Fan & Song 2022\nVolkswagen Magotan,0X3050M,18.49,118,Gasoline,2.5754,Fan & Song 2022\nVolkswagen Magotan,0X3050M,22.67,118,Gasoline,2.0999,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.5,118,Gasoline,1.4301,Fan & Song 2022\nVolkswagen Magotan,0X3050M,59.39,118,Gasoline,5.0414,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.79,118,Gasoline,1.267,Fan & Song 2022\nVolkswagen Magotan,0X3050X,1.59,118,Gasoline,0.2278,Fan & Song 2022\nVolkswagen Magotan,0X3050M,18.85,118,Gasoline,2.4251,Fan & Song 2022\nVolkswagen Magotan,0X3050X,14.24,118,Gasoline,1.4878,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.33,118,Gasoline,0.5925,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.56,118,Gasoline,0.8226,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.77,118,Gasoline,0.4251,Fan & Song 2022\nVolkswagen Magotan,0X3050M,37.74,118,Gasoline,2.8482,Fan & Song 2022\nVolkswagen Magotan,0X3050M,8.47,118,Gasoline,1.1416,Fan & Song 2022\nVolkswagen Magotan,0X3050M,115.07,118,Gasoline,12.4735,Fan & Song 2022\nVolkswagen Magotan,0X3050M,74.0,118,Gasoline,5.1975,Fan & Song 2022\nVolkswagen Magotan,0X3050M,76.34,118,Gasoline,6.4937,Fan & Song 2022\nVolkswagen Magotan,0X3050X,38.24,118,Gasoline,2.598,Fan & Song 2022\nVolkswagen Magotan,0X3050M,114.64,118,Gasoline,10.7074,Fan & Song 2022\nVolkswagen Magotan,0X3050M,180.95,118,Gasoline,17.6402,Fan & Song 2022\nVolkswagen Magotan,0X3050M,34.4,118,Gasoline,2.911,Fan & Song 2022\nVolkswagen Magotan,0X3050X,65.22,118,Gasoline,3.7045,Fan & Song 2022\nVolkswagen Magotan,0X3050X,10.5,118,Gasoline,1.0489,Fan & Song 2022\nVolkswagen Magotan,0X3050M,87.4,118,Gasoline,7.4564,Fan & Song 2022\nVolkswagen Magotan,0X3050M,15.85,118,Gasoline,1.6267,Fan & Song 2022\nVolkswagen Magotan,0X3050M,121.24,118,Gasoline,12.2854,Fan & Song 2022\nVolkswagen Magotan,0X3050M,33.94,118,Gasoline,3.5854,Fan & Song 2022\nVolkswagen Magotan,0X3050X,4.84,118,Gasoline,0.6932,Fan & Song 2022\nVolkswagen Magotan,0X3050M,24.87,118,Gasoline,2.4802,Fan & Song 2022\nVolkswagen Magotan,0X3050X,91.8,118,Gasoline,6.8156,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.31,118,Gasoline,0.3372,Fan & Song 2022\nVolkswagen Magotan,0X3050X,15.81,118,Gasoline,1.3834,Fan & Song 2022\nVolkswagen Magotan,0X3050X,26.14,118,Gasoline,2.939,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.61,118,Gasoline,2.2972,Fan & Song 2022\nVolkswagen Magotan,0X3050M,20.88,118,Gasoline,2.5418,Fan & Song 2022\nVolkswagen Magotan,0X3050M,53.72,118,Gasoline,3.7823,Fan & Song 2022\nVolkswagen Magotan,0X3050M,41.51,118,Gasoline,3.3817,Fan & Song 2022\nVolkswagen Magotan,0X3050M,130.02,118,Gasoline,10.6861,Fan & Song 2022\nVolkswagen Magotan,0X3050X,16.76,118,Gasoline,1.2893,Fan & Song 2022\nVolkswagen Magotan,0X3050M,103.84,118,Gasoline,8.7559,Fan & Song 2022\nVolkswagen Magotan,0X3050M,11.0,118,Gasoline,1.1694,Fan & Song 2022\nVolkswagen Magotan,0X3050M,11.6,118,Gasoline,0.9591,Fan & Song 2022\nVolkswagen Magotan,0X3050M,102.78,118,Gasoline,9.604,Fan & Song 2022\nVolkswagen Magotan,0X3050X,10.36,118,Gasoline,1.1169,Fan & Song 2022\nVolkswagen Magotan,0X3050X,9.7,118,Gasoline,0.9364,Fan & Song 2022\nVolkswagen Magotan,0X3050M,64.25,118,Gasoline,4.4857,Fan & Song 2022\nVolkswagen Magotan,0X3050M,110.96,118,Gasoline,11.677,Fan & Song 2022\nVolkswagen Magotan,0X3050X,18.94,118,Gasoline,1.7009,Fan & Song 2022\nVolkswagen Magotan,0X3050M,73.4,118,Gasoline,6.0954,Fan & Song 2022\nVolkswagen Magotan,0X3050X,18.53,118,Gasoline,1.335,Fan & Song 2022\nVolkswagen Magotan,0X3050X,16.3,118,Gasoline,1.5867,Fan & Song 2022\nVolkswagen Magotan,0X3050M,116.99,118,Gasoline,11.0964,Fan & Song 2022\nVolkswagen Magotan,0X3050M,4.05,118,Gasoline,0.5804,Fan & Song 2022\nVolkswagen Magotan,0X3050M,69.48,118,Gasoline,5.5949,Fan & Song 2022\nVolkswagen Magotan,0X3050M,49.14,118,Gasoline,4.619,Fan & Song 2022\nVolkswagen Magotan,0X3050M,150.67,118,Gasoline,12.0852,Fan & Song 2022\nVolkswagen Magotan,0X3050M,2.91,118,Gasoline,0.4119,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.63,118,Gasoline,1.4838,Fan & Song 2022\nVolkswagen Magotan,0X3050M,123.91,118,Gasoline,11.3385,Fan & Song 2022\nVolkswagen Magotan,0X3050X,37.11,118,Gasoline,3.1228,Fan & Song 2022\nVolkswagen Magotan,0X3050M,137.39,118,Gasoline,12.8837,Fan & Song 2022\nVolkswagen Magotan,0X3050X,2.13,118,Gasoline,0.251,Fan & Song 2022\nVolkswagen Magotan,0X3050M,14.6,118,Gasoline,1.8152,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.73,118,Gasoline,0.7995,Fan & Song 2022\nVolkswagen Magotan,0X3050X,9.77,118,Gasoline,0.8681,Fan & Song 2022\nVolkswagen Magotan,0X3050X,6.54,118,Gasoline,0.7988,Fan & Song 2022\nVolkswagen Magotan,0X3050M,106.47,118,Gasoline,8.5679,Fan & Song 2022\nVolkswagen Magotan,0X3050M,45.02,118,Gasoline,3.7581,Fan & Song 2022\nVolkswagen Magotan,0X3050M,49.1,118,Gasoline,4.7464,Fan & Song 2022\nVolkswagen Magotan,0X3050M,116.6,118,Gasoline,11.6503,Fan & Song 2022\nVolkswagen Magotan,0X3050M,10.11,118,Gasoline,0.7063,Fan & Song 2022\nVolkswagen Magotan,0X3050M,33.5,118,Gasoline,2.5784,Fan & Song 2022\nVolkswagen Magotan,0X3050X,5.79,118,Gasoline,0.6321,Fan & Song 2022\nVolkswagen Passat,0X2072T,117.86,118,Gasoline,8.9624,Fan & Song 2022\nVolkswagen Passat,0X2072T,34.41,118,Gasoline,2.4417,Fan & Song 2022\nVolkswagen Passat,0X2072T,3.22,118,Gasoline,0.4127,Fan & Song 2022\nVolkswagen Passat,0X2072R,44.38,118,Gasoline,5.4209,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.06,118,Gasoline,0.5472,Fan & Song 2022\nVolkswagen Passat,0X2072T,3.89,118,Gasoline,0.4001,Fan & Song 2022\nVolkswagen Passat,0X2072R,46.16,118,Gasoline,4.436,Fan & Song 2022\nVolkswagen Passat,0X2072S,209.12,118,Gasoline,14.8715,Fan & Song 2022\nVolkswagen Passat,0X2072R,134.57,118,Gasoline,10.6775,Fan & Song 2022\nVolkswagen Passat,0X2072T,37.66,118,Gasoline,3.439,Fan & Song 2022\nVolkswagen Passat,0X2072T,19.36,118,Gasoline,1.5899,Fan & Song 2022\nVolkswagen Passat,0X2072S,22.65,118,Gasoline,2.6283,Fan & Song 2022\nVolkswagen Passat,0X2072R,11.12,118,Gasoline,1.1923,Fan & Song 2022\nVolkswagen Passat,0X2072T,43.16,118,Gasoline,2.8738,Fan & Song 2022\nVolkswagen Passat,0X2072T,79.86,118,Gasoline,5.9211,Fan & Song 2022\nVolkswagen Passat,0X2072S,85.73,118,Gasoline,6.1538,Fan & Song 2022\nVolkswagen Passat,0X2072T,20.95,118,Gasoline,2.3463,Fan & Song 2022\nVolkswagen Passat,0X2072T,2.44,118,Gasoline,0.2139,Fan & Song 2022\nVolkswagen Passat,0X2072S,30.26,118,Gasoline,2.2797,Fan & Song 2022\nVolkswagen Passat,0X2072S,106.88,118,Gasoline,9.0285,Fan & Song 2022\nVolkswagen Passat,0X2072T,5.34,118,Gasoline,0.6716,Fan & Song 2022\nVolkswagen Passat,0X2072S,126.96,118,Gasoline,9.8811,Fan & Song 2022\nVolkswagen Passat,0X2072T,13.4,118,Gasoline,1.3766,Fan & Song 2022\nVolkswagen Passat,0X2072S,37.31,118,Gasoline,2.7131,Fan & Song 2022\nVolkswagen Passat,0X2072S,104.62,118,Gasoline,10.9274,Fan & Song 2022\nVolkswagen Passat,0X2072R,17.46,118,Gasoline,1.6938,Fan & Song 2022\nVolkswagen Passat,0X2072T,47.5,118,Gasoline,4.4997,Fan & Song 2022\nVolkswagen Passat,0X2072T,10.18,118,Gasoline,0.9045,Fan & Song 2022\nVolkswagen Passat,0X2072S,36.65,118,Gasoline,2.6149,Fan & Song 2022\nVolkswagen Passat,0X2072R,126.15,118,Gasoline,9.6986,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.6,118,Gasoline,0.3189,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.62,118,Gasoline,0.3567,Fan & Song 2022\nVolkswagen Passat,0X2072R,31.63,118,Gasoline,3.0002,Fan & Song 2022\nVolkswagen Passat,0X2072T,16.85,118,Gasoline,1.5432,Fan & Song 2022\nVolkswagen Passat,0X2072S,5.98,118,Gasoline,0.6855,Fan & Song 2022\nVolkswagen Passat,0X2072S,1.93,118,Gasoline,0.1733,Fan & Song 2022\nVolkswagen Passat,0X2072S,147.31,118,Gasoline,11.7163,Fan & Song 2022\nVolkswagen Passat,0X2072T,32.14,118,Gasoline,2.7235,Fan & Song 2022\nVolkswagen Passat,0X2072S,30.93,118,Gasoline,3.5376,Fan & Song 2022\nVolkswagen Passat,0X2072T,199.95,118,Gasoline,12.8973,Fan & Song 2022\nVolkswagen Passat,0X2072T,11.09,118,Gasoline,1.0311,Fan & Song 2022\nVolkswagen Passat,0X2072R,16.39,118,Gasoline,1.2929,Fan & Song 2022\nVolkswagen Passat,0X2072S,29.03,118,Gasoline,2.468,Fan & Song 2022\nVolkswagen Passat,0X2072S,138.96,118,Gasoline,12.0541,Fan & Song 2022\nVolkswagen Passat,0X2072R,412.54,118,Gasoline,35.4871,Fan & Song 2022\nVolkswagen Passat,0X2072S,110.99,118,Gasoline,9.7038,Fan & Song 2022\nVolkswagen Passat,0X2072T,11.82,118,Gasoline,1.1047,Fan & Song 2022\nVolkswagen Passat,0X2072T,25.39,118,Gasoline,2.0438,Fan & Song 2022\nVolkswagen Passat,0X2072R,1.96,118,Gasoline,0.2587,Fan & Song 2022\nVolkswagen Passat,0X2072T,2.39,118,Gasoline,0.3192,Fan & Song 2022\nVolkswagen Passat,0X2072T,105.3,118,Gasoline,10.2148,Fan & Song 2022\nVolkswagen Passat,0X2072R,60.34,118,Gasoline,5.4359,Fan & Song 2022\nVolkswagen Passat,0X2072T,1.84,118,Gasoline,0.2905,Fan & Song 2022\nVolkswagen Passat,0X2072T,193.36,118,Gasoline,15.7093,Fan & Song 2022\nVolkswagen Passat,0X2072T,2.25,118,Gasoline,0.3111,Fan & Song 2022\nVolkswagen Passat,0X2072S,37.34,118,Gasoline,3.3422,Fan & Song 2022\nVolkswagen Passat,0X2072S,9.0,118,Gasoline,0.7888,Fan & Song 2022\nVolkswagen Passat,0X2072S,152.6,118,Gasoline,11.2772,Fan & Song 2022\nVolkswagen Passat,0X2072S,689.99,118,Gasoline,54.3992,Fan & Song 2022\nVolkswagen Passat,0X2072S,214.7,118,Gasoline,16.9352,Fan & Song 2022\nVolkswagen Passat,0X2072T,335.71,118,Gasoline,23.7141,Fan & Song 2022\nVolkswagen Passat,0X2072S,107.59,118,Gasoline,6.6269,Fan & Song 2022\nVolkswagen Passat,0X2072T,100.55,118,Gasoline,8.4341,Fan & Song 2022\nVolkswagen Passat,0X2072S,6.72,118,Gasoline,0.5769,Fan & Song 2022\nVolkswagen Passat,0X2072S,140.98,118,Gasoline,11.5561,Fan & Song 2022\nVolkswagen Passat,0X2072R,208.88,118,Gasoline,13.6914,Fan & Song 2022\nVolkswagen Passat,0X2072S,115.12,118,Gasoline,7.9669,Fan & Song 2022\nVolkswagen Passat,0X2072T,22.28,118,Gasoline,1.9588,Fan & Song 2022\nVolkswagen Passat,0X2072T,207.16,118,Gasoline,14.5586,Fan & Song 2022\nVolkswagen Passat,0X2072T,21.54,118,Gasoline,1.7043,Fan & Song 2022\nVolkswagen Passat,0X2072R,42.48,118,Gasoline,4.2533,Fan & Song 2022\nVolkswagen Passat,0X2072S,1.56,118,Gasoline,0.1494,Fan & Song 2022\nVolkswagen Passat,0X2072T,169.57,118,Gasoline,14.4071,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.34,118,Gasoline,0.3573,Fan & Song 2022\nVolkswagen Passat,0X2072R,110.83,118,Gasoline,7.8306,Fan & Song 2022\nVolkswagen Passat,0X2072S,44.85,118,Gasoline,3.417,Fan & Song 2022\nVolkswagen Passat,0X2072S,64.74,118,Gasoline,4.4716,Fan & Song 2022\nVolkswagen Passat,0X2072T,44.5,118,Gasoline,2.8394,Fan & Song 2022\nVolkswagen Passat,0X2072T,12.34,118,Gasoline,0.9755,Fan & Song 2022\nVolkswagen Passat,0X2072T,49.46,118,Gasoline,3.7551,Fan & Song 2022\nVolkswagen Passat,0X2072T,3.9,118,Gasoline,0.3782,Fan & Song 2022\nVolkswagen Passat,0X2072R,18.87,118,Gasoline,1.7924,Fan & Song 2022\nVolkswagen Passat,0X2072T,12.72,118,Gasoline,1.3277,Fan & Song 2022\nVolkswagen Passat,0X2072T,42.68,118,Gasoline,3.1501,Fan & Song 2022\nVolkswagen Passat,0X2072R,25.58,118,Gasoline,3.0432,Fan & Song 2022\nVolkswagen Passat,0X2072R,196.97,118,Gasoline,12.0557,Fan & Song 2022\nVolkswagen Passat,0X2072S,13.17,118,Gasoline,1.1992,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.62,118,Gasoline,0.4127,Fan & Song 2022\nVolkswagen Passat,0X2072T,40.93,118,Gasoline,3.0324,Fan & Song 2022\nVolkswagen Passat,0X2072S,5.03,118,Gasoline,0.4218,Fan & Song 2022\nVolkswagen Passat,0X2072T,2.33,118,Gasoline,0.4036,Fan & Song 2022\nVolkswagen Passat,0X2072S,12.41,118,Gasoline,1.1296,Fan & Song 2022\nVolkswagen Passat,0X2072S,7.23,118,Gasoline,0.693,Fan & Song 2022\nVolkswagen Passat,0X2072T,21.29,118,Gasoline,1.7722,Fan & Song 2022\nVolkswagen Passat,0X2072S,19.32,118,Gasoline,1.7825,Fan & Song 2022\nVolkswagen Passat,0X2072R,6.99,118,Gasoline,0.8086,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.84,118,Gasoline,0.2931,Fan & Song 2022\nVolkswagen Passat,0X2072T,259.12,118,Gasoline,17.4159,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.09,118,Gasoline,0.2705,Fan & Song 2022\nVolkswagen Passat,0X2072S,37.17,118,Gasoline,2.6909,Fan & Song 2022\nVolkswagen Passat,0X2072T,8.32,118,Gasoline,0.8773,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.84,118,Gasoline,0.4568,Fan & Song 2022\nVolkswagen Passat,0X2072R,182.9,118,Gasoline,13.1056,Fan & Song 2022\nVolkswagen Passat,0X2072S,138.76,118,Gasoline,12.973,Fan & Song 2022\nVolkswagen Passat,0X2072R,24.96,118,Gasoline,3.4232,Fan & Song 2022\nVolkswagen Passat,0X2072R,37.27,118,Gasoline,3.2365,Fan & Song 2022\nVolkswagen Passat,0X2072S,80.75,118,Gasoline,5.3678,Fan & Song 2022\nVolkswagen Passat,0X2072R,35.79,118,Gasoline,3.8562,Fan & Song 2022\nVolkswagen Passat,0X2072S,6.15,118,Gasoline,0.7689,Fan & Song 2022\nVolkswagen Passat,0X2072S,30.76,118,Gasoline,2.0032,Fan & Song 2022\nVolkswagen Passat,0X2072S,39.04,118,Gasoline,4.6926,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.13,118,Gasoline,0.5268,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.86,118,Gasoline,0.2267,Fan & Song 2022\nVolkswagen Passat,0X2072T,43.58,118,Gasoline,3.6078,Fan & Song 2022\nVolkswagen Passat,0X2072R,6.43,118,Gasoline,0.5915,Fan & Song 2022\nVolkswagen Passat,0X2072S,26.08,118,Gasoline,2.2851,Fan & Song 2022\nVolkswagen Passat,0X2072T,16.27,118,Gasoline,1.2322,Fan & Song 2022\nVolkswagen Passat,0X2072T,214.84,118,Gasoline,14.9066,Fan & Song 2022\nVolkswagen Passat,0X2072S,45.64,118,Gasoline,3.5263,Fan & Song 2022\nVolkswagen Passat,0X2072R,81.23,118,Gasoline,6.6154,Fan & Song 2022\nVolkswagen Passat,0X2072T,4.1,118,Gasoline,0.5191,Fan & Song 2022\nVolkswagen Passat,0X2072S,36.51,118,Gasoline,3.6357,Fan & Song 2022\nVolkswagen Passat,0X2072S,19.0,118,Gasoline,1.5663,Fan & Song 2022\nVolkswagen Passat,0X2072S,71.39,118,Gasoline,6.8774,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.91,118,Gasoline,0.4039,Fan & Song 2022\nVolkswagen Passat,0X2072S,1.64,118,Gasoline,0.2178,Fan & Song 2022\nVolkswagen Passat,0X2072T,33.55,118,Gasoline,3.1028,Fan & Song 2022\nVolkswagen Passat,0X2072T,13.54,118,Gasoline,1.2665,Fan & Song 2022\nVolkswagen Passat,0X2072T,52.57,118,Gasoline,4.1762,Fan & Song 2022\nVolkswagen Passat,0X2072R,12.06,118,Gasoline,1.2738,Fan & Song 2022\nVolkswagen Passat,0X2072R,16.25,118,Gasoline,1.889,Fan & Song 2022\nVolkswagen Passat,0X2072T,10.5,118,Gasoline,1.0714,Fan & Song 2022\nVolkswagen Passat,0X2072T,39.22,118,Gasoline,5.1346,Fan & Song 2022\nVolkswagen Passat,0X2072S,32.6,118,Gasoline,2.3633,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.9,118,Gasoline,0.6741,Fan & Song 2022\nVolkswagen Passat,0X2072S,16.25,118,Gasoline,1.136,Fan & Song 2022\nVolkswagen Passat,0X2072R,16.3,118,Gasoline,1.0881,Fan & Song 2022\nVolkswagen Passat,0X2072R,28.94,118,Gasoline,2.3503,Fan & Song 2022\nVolkswagen Passat,0X2072S,6.07,118,Gasoline,0.7811,Fan & Song 2022\nVolkswagen Passat,0X2072R,23.38,118,Gasoline,1.7501,Fan & Song 2022\nVolkswagen Passat,0X2072R,23.0,118,Gasoline,2.5688,Fan & Song 2022\nVolkswagen Passat,0X2072S,32.3,118,Gasoline,2.6104,Fan & Song 2022\nVolkswagen Passat,0X2072R,4.31,118,Gasoline,0.5688,Fan & Song 2022\nVolkswagen Passat,0X2072S,16.84,118,Gasoline,1.2272,Fan & Song 2022\nVolkswagen Passat,0X2072T,27.24,118,Gasoline,1.9678,Fan & Song 2022\nVolkswagen Passat,0X2072T,23.85,118,Gasoline,1.8241,Fan & Song 2022\nVolkswagen Passat,0X2072T,129.96,118,Gasoline,10.3185,Fan & Song 2022\nVolkswagen Passat,0X2072S,129.9,118,Gasoline,11.4208,Fan & Song 2022\nVolkswagen Passat,0X2072T,14.01,118,Gasoline,1.8078,Fan & Song 2022\nVolkswagen Passat,0X2072T,7.35,118,Gasoline,0.8184,Fan & Song 2022\nVolkswagen Passat,0X2072R,15.3,118,Gasoline,2.2257,Fan & Song 2022\nVolkswagen Passat,0X2072T,133.43,118,Gasoline,9.9063,Fan & Song 2022\nVolkswagen Passat,0X2072T,154.53,118,Gasoline,9.5601,Fan & Song 2022\nVolkswagen Passat,0X2072R,27.29,118,Gasoline,2.0156,Fan & Song 2022\nVolkswagen Passat,0X2072R,22.66,118,Gasoline,2.0701,Fan & Song 2022\nVolkswagen Passat,0X2072T,20.98,118,Gasoline,1.883,Fan & Song 2022\nVolkswagen Passat,0X2072T,55.66,118,Gasoline,3.7516,Fan & Song 2022\nVolkswagen Passat,0X2072R,189.32,118,Gasoline,14.1355,Fan & Song 2022\nVolkswagen Passat,0X2072T,23.4,118,Gasoline,1.5182,Fan & Song 2022\nVolkswagen Passat,0X2072T,32.3,118,Gasoline,2.0633,Fan & Song 2022\nVolkswagen Passat,0X2072T,109.86,118,Gasoline,7.5381,Fan & Song 2022\nVolkswagen Passat,0X2072S,55.89,118,Gasoline,5.8977,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.55,118,Gasoline,0.4448,Fan & Song 2022\nVolkswagen Passat,0X2072T,3.97,118,Gasoline,0.3611,Fan & Song 2022\nVolkswagen Passat,0X2072S,39.19,118,Gasoline,3.5035,Fan & Song 2022\nVolkswagen Passat,0X2072S,6.46,118,Gasoline,0.6201,Fan & Song 2022\nVolkswagen Passat,0X2072S,114.76,118,Gasoline,9.6976,Fan & Song 2022\nVolkswagen Passat,0X2072S,4.26,118,Gasoline,0.4838,Fan & Song 2022\nVolkswagen Passat,0X2072S,65.77,118,Gasoline,6.3996,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.14,118,Gasoline,0.1875,Fan & Song 2022\nVolkswagen Passat,0X2072S,10.07,118,Gasoline,0.9204,Fan & Song 2022\nVolkswagen Passat,0X2072S,13.67,118,Gasoline,1.701,Fan & Song 2022\nVolkswagen Passat,0X2072S,52.53,118,Gasoline,5.7766,Fan & Song 2022\nVolkswagen Passat,0X2072R,79.98,118,Gasoline,4.9231,Fan & Song 2022\nVolkswagen Passat,0X2072S,17.89,118,Gasoline,1.7123,Fan & Song 2022\nVolkswagen Passat,0X2072S,36.02,118,Gasoline,3.4298,Fan & Song 2022\nVolkswagen Passat,0X2072T,21.23,118,Gasoline,2.0539,Fan & Song 2022\nVolkswagen Passat,0X2072S,121.79,118,Gasoline,12.1123,Fan & Song 2022\nVolkswagen Passat,0X2072R,123.83,118,Gasoline,9.2186,Fan & Song 2022\nVolkswagen Passat,0X2072R,6.32,118,Gasoline,0.9183,Fan & Song 2022\nVolkswagen Passat,0X2072R,103.99,118,Gasoline,7.8252,Fan & Song 2022\nVolkswagen Passat,0X2072S,152.19,118,Gasoline,9.8839,Fan & Song 2022\nVolkswagen Passat,0X2072R,113.26,118,Gasoline,9.1502,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.67,118,Gasoline,0.8873,Fan & Song 2022\nVolkswagen Passat,0X2072T,120.75,118,Gasoline,7.6442,Fan & Song 2022\nVolkswagen Passat,0X2072S,45.67,118,Gasoline,3.3458,Fan & Song 2022\nVolkswagen Passat,0X2072T,152.6,118,Gasoline,10.1678,Fan & Song 2022\nVolkswagen Passat,0X2072S,34.08,118,Gasoline,2.4664,Fan & Song 2022\nVolkswagen Passat,0X2072R,137.08,118,Gasoline,8.3428,Fan & Song 2022\nVolkswagen Passat,0X2072T,13.54,118,Gasoline,1.4257,Fan & Song 2022\nVolkswagen Passat,0X2072T,105.13,118,Gasoline,7.2027,Fan & Song 2022\nVolkswagen Passat,0X2072R,100.89,118,Gasoline,7.9764,Fan & Song 2022\nVolkswagen Passat,0X2072T,8.23,118,Gasoline,0.7745,Fan & Song 2022\nVolkswagen Passat,0X2072S,45.65,118,Gasoline,3.1414,Fan & Song 2022\nVolkswagen Passat,0X2072R,149.52,118,Gasoline,12.3709,Fan & Song 2022\nVolkswagen Passat,0X2072T,10.87,118,Gasoline,1.289,Fan & Song 2022\nVolkswagen Passat,0X2072T,2.41,118,Gasoline,0.2876,Fan & Song 2022\nVolkswagen Passat,0X2072S,14.01,118,Gasoline,1.3775,Fan & Song 2022\nVolkswagen Passat,0X2072S,35.92,118,Gasoline,2.6957,Fan & Song 2022\nVolkswagen Passat,0X2072T,33.35,118,Gasoline,2.0519,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.41,118,Gasoline,0.3361,Fan & Song 2022\nVolkswagen Passat,0X2072T,271.1,118,Gasoline,20.6155,Fan & Song 2022\nVolkswagen Passat,0X2072S,4.9,118,Gasoline,0.62,Fan & Song 2022\nVolkswagen Passat,0X2072S,36.94,118,Gasoline,2.3476,Fan & Song 2022\nVolkswagen Passat,0X2072R,6.8,118,Gasoline,0.8969,Fan & Song 2022\nVolkswagen Passat,0X2072T,6.87,118,Gasoline,0.7149,Fan & Song 2022\nVolkswagen Passat,0X2072R,158.02,118,Gasoline,13.0705,Fan & Song 2022\nVolkswagen Passat,0X2072S,55.74,118,Gasoline,3.7464,Fan & Song 2022\nVolkswagen Passat,0X2072T,8.89,118,Gasoline,0.7764,Fan & Song 2022\nVolkswagen Passat,0X2072S,102.6,118,Gasoline,10.2051,Fan & Song 2022\nVolkswagen Passat,0X2072T,6.34,118,Gasoline,0.6457,Fan & Song 2022\nVolkswagen Passat,0X2072R,1.97,118,Gasoline,0.2988,Fan & Song 2022\nVolkswagen Passat,0X2072R,63.41,118,Gasoline,4.7118,Fan & Song 2022\nVolkswagen Passat,0X2072T,23.19,118,Gasoline,2.3137,Fan & Song 2022\nVolkswagen Passat,0X2072R,15.03,118,Gasoline,1.4798,Fan & Song 2022\nVolkswagen Passat,0X2072R,9.69,118,Gasoline,1.3299,Fan & Song 2022\nVolkswagen Passat,0X2072S,137.0,118,Gasoline,12.6086,Fan & Song 2022\nVolkswagen Passat,0X2072T,17.54,118,Gasoline,1.6006,Fan & Song 2022\nVolkswagen Passat,0X2072T,10.78,118,Gasoline,0.8957,Fan & Song 2022\nVolkswagen Passat,0X2072T,292.93,118,Gasoline,20.9489,Fan & Song 2022\nVolkswagen Passat,0X2072T,153.13,118,Gasoline,14.2016,Fan & Song 2022\nVolkswagen Passat,0X2072T,16.32,118,Gasoline,3.124,Fan & Song 2022\nVolkswagen Passat,0X2072R,258.29,118,Gasoline,17.3402,Fan & Song 2022\nVolkswagen Passat,0X2072S,14.52,118,Gasoline,1.2073,Fan & Song 2022\nVolkswagen Passat,0X2072S,16.22,118,Gasoline,1.0469,Fan & Song 2022\nVolkswagen Passat,0X2072T,16.1,118,Gasoline,1.332,Fan & Song 2022\nVolkswagen Passat,0X2072S,46.52,118,Gasoline,5.4687,Fan & Song 2022\nVolkswagen Passat,0X2072T,120.06,118,Gasoline,8.5563,Fan & Song 2022\nVolkswagen Passat,0X2072R,19.72,118,Gasoline,1.5605,Fan & Song 2022\nVolkswagen Passat,0X2072S,24.06,118,Gasoline,2.5773,Fan & Song 2022\nVolkswagen Passat,0X2072R,8.8,118,Gasoline,0.8878,Fan & Song 2022\nVolkswagen Passat,0X2072T,12.1,118,Gasoline,1.3674,Fan & Song 2022\nVolkswagen Passat,0X2072T,25.53,118,Gasoline,1.712,Fan & Song 2022\nVolkswagen Passat,0X2072S,108.67,118,Gasoline,11.2886,Fan & Song 2022\nVolkswagen Passat,0X2072S,4.06,118,Gasoline,0.5372,Fan & Song 2022\nVolkswagen Passat,0X2072S,82.38,118,Gasoline,6.9603,Fan & Song 2022\nVolkswagen Passat,0X2072S,34.29,118,Gasoline,2.3279,Fan & Song 2022\nVolkswagen Passat,0X2072R,12.06,118,Gasoline,1.4064,Fan & Song 2022\nVolkswagen Passat,0X2072T,20.29,118,Gasoline,1.4843,Fan & Song 2022\nVolkswagen Passat,0X2072T,18.53,118,Gasoline,1.7952,Fan & Song 2022\nVolkswagen Passat,0X2072S,119.16,118,Gasoline,11.1266,Fan & Song 2022\nVolkswagen Passat,0X2072T,5.11,118,Gasoline,0.6075,Fan & Song 2022\nVolkswagen Passat,0X2072R,30.23,118,Gasoline,2.9608,Fan & Song 2022\nVolkswagen Passat,0X2072T,164.4,118,Gasoline,11.3456,Fan & Song 2022\nVolkswagen Passat,0X2072T,84.44,118,Gasoline,8.4707,Fan & Song 2022\nVolkswagen Passat,0X2072T,15.09,118,Gasoline,1.4681,Fan & Song 2022\nVolkswagen Passat,0X2072T,1.66,118,Gasoline,0.2266,Fan & Song 2022\nVolkswagen Passat,0X2072S,11.96,118,Gasoline,2.1423,Fan & Song 2022\nVolkswagen Passat,0X2072S,257.14,118,Gasoline,23.6811,Fan & Song 2022\nVolkswagen Passat,0X2072S,273.36,118,Gasoline,17.6986,Fan & Song 2022\nVolkswagen Passat,0X2072T,80.25,118,Gasoline,5.9939,Fan & Song 2022\nVolkswagen Passat,0X2072S,75.56,118,Gasoline,9.1931,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.89,118,Gasoline,0.4917,Fan & Song 2022\nVolkswagen Passat,0X2072T,104.06,118,Gasoline,12.3057,Fan & Song 2022\nVolkswagen Passat,0X2072T,103.98,118,Gasoline,11.8248,Fan & Song 2022\nVolkswagen Passat,0X2072R,10.46,118,Gasoline,0.8036,Fan & Song 2022\nVolkswagen Passat,0X2072S,166.92,118,Gasoline,11.6535,Fan & Song 2022\nVolkswagen Passat,0X2072S,125.15,118,Gasoline,9.0521,Fan & Song 2022\nVolkswagen Passat,0X2072T,37.8,118,Gasoline,3.3037,Fan & Song 2022\nVolkswagen Passat,0X2072S,62.8,118,Gasoline,5.6534,Fan & Song 2022\nVolkswagen Passat,0X2072T,4.88,118,Gasoline,0.4808,Fan & Song 2022\nVolkswagen Passat,0X2072S,60.58,118,Gasoline,5.0114,Fan & Song 2022\nVolkswagen Passat,0X2072R,4.98,118,Gasoline,0.4856,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.79,118,Gasoline,0.2723,Fan & Song 2022\nVolkswagen Passat,0X2072S,1.7,118,Gasoline,0.2673,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.79,118,Gasoline,0.2906,Fan & Song 2022\nVolkswagen Passat,0X2072T,40.8,118,Gasoline,5.4397,Fan & Song 2022\nVolkswagen Passat,0X2072R,123.41,118,Gasoline,9.9286,Fan & Song 2022\nVolkswagen Passat,0X2072T,21.12,118,Gasoline,2.8673,Fan & Song 2022\nVolkswagen Passat,0X2072T,27.79,118,Gasoline,2.5959,Fan & Song 2022\nVolkswagen Passat,0X2072T,31.73,118,Gasoline,3.8128,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.57,118,Gasoline,0.7143,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.99,118,Gasoline,0.4414,Fan & Song 2022\nVolkswagen Passat,0X2072S,1.76,118,Gasoline,0.172,Fan & Song 2022\nVolkswagen Passat,0X2072S,10.66,118,Gasoline,0.8773,Fan & Song 2022\nVolkswagen Passat,0X2072S,37.47,118,Gasoline,3.3474,Fan & Song 2022\nVolkswagen Passat,0X2072T,19.44,118,Gasoline,1.6743,Fan & Song 2022\nVolkswagen Passat,0X2072R,39.75,118,Gasoline,4.5446,Fan & Song 2022\nVolkswagen Passat,0X2072T,3.47,118,Gasoline,0.4379,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.18,118,Gasoline,0.6272,Fan & Song 2022\nVolkswagen Passat,0X2072R,4.5,118,Gasoline,0.3386,Fan & Song 2022\nVolkswagen Passat,0X2072S,238.61,118,Gasoline,15.1552,Fan & Song 2022\nVolkswagen Passat,0X2072T,13.54,118,Gasoline,1.0961,Fan & Song 2022\nVolkswagen Passat,0X2072R,7.78,118,Gasoline,0.709,Fan & Song 2022\nVolkswagen Passat,0X2072T,8.46,118,Gasoline,1.1784,Fan & Song 2022\nVolkswagen Passat,0X2072R,4.47,118,Gasoline,0.5861,Fan & Song 2022\nVolkswagen Passat,0X2072T,48.88,118,Gasoline,3.7779,Fan & Song 2022\nVolkswagen Passat,0X2072S,102.39,118,Gasoline,7.6732,Fan & Song 2022\nVolkswagen Passat,0X2072R,25.28,118,Gasoline,2.4862,Fan & Song 2022\nVolkswagen Passat,0X2072T,4.0,118,Gasoline,0.5344,Fan & Song 2022\nVolkswagen Passat,0X2072S,167.79,118,Gasoline,16.2989,Fan & Song 2022\nVolkswagen Passat,0X2072S,51.08,118,Gasoline,3.8605,Fan & Song 2022\nVolkswagen Passat,0X2072S,12.95,118,Gasoline,2.3022,Fan & Song 2022\nVolkswagen Passat,0X2072T,18.48,118,Gasoline,2.099,Fan & Song 2022\nVolkswagen Passat,0X2072S,119.85,118,Gasoline,10.3736,Fan & Song 2022\nVolkswagen Passat,0X2072R,27.01,118,Gasoline,5.914,Fan & Song 2022\nVolkswagen Passat,0X2072R,209.24,118,Gasoline,14.5146,Fan & Song 2022\nVolkswagen Passat,0X2072T,37.63,118,Gasoline,4.3479,Fan & Song 2022\nVolkswagen Passat,0X2072S,17.05,118,Gasoline,2.1521,Fan & Song 2022\nVolkswagen Passat,0X2072T,108.09,118,Gasoline,7.5497,Fan & Song 2022\nVolkswagen Passat,0X2072T,38.17,118,Gasoline,4.1364,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.77,118,Gasoline,0.5358,Fan & Song 2022\nVolkswagen Passat,0X2072R,63.01,118,Gasoline,4.7326,Fan & Song 2022\nVolkswagen Passat,0X2072S,136.21,118,Gasoline,8.1377,Fan & Song 2022\nVolkswagen Passat,0X2072S,153.73,118,Gasoline,9.8338,Fan & Song 2022\nVolkswagen Passat,0X2072S,95.38,118,Gasoline,11.2222,Fan & Song 2022\nVolkswagen Passat,0X2072T,34.56,118,Gasoline,2.7673,Fan & Song 2022\nVolkswagen Passat,0X2072S,104.03,118,Gasoline,8.147,Fan & Song 2022\nVolkswagen Passat,0X2072S,8.01,118,Gasoline,0.8733,Fan & Song 2022\nVolkswagen Passat,0X2072S,11.37,118,Gasoline,0.6376,Fan & Song 2022\nVolkswagen Passat,0X2072T,16.51,118,Gasoline,1.5442,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.12,118,Gasoline,0.5972,Fan & Song 2022\nVolkswagen Passat,0X2072T,18.05,118,Gasoline,1.3538,Fan & Song 2022\nVolkswagen Passat,0X2072T,16.96,118,Gasoline,1.8003,Fan & Song 2022\nVolkswagen Passat,0X2072S,364.66,118,Gasoline,25.0547,Fan & Song 2022\nVolkswagen Passat,0X2072T,108.84,118,Gasoline,7.7649,Fan & Song 2022\nVolkswagen Passat,0X2072S,237.04,118,Gasoline,18.2991,Fan & Song 2022\nVolkswagen Passat,0X2072S,70.07,118,Gasoline,4.0896,Fan & Song 2022\nVolkswagen Passat,0X2072T,1.74,118,Gasoline,0.2307,Fan & Song 2022\nVolkswagen Passat,0X2072R,39.74,118,Gasoline,3.273,Fan & Song 2022\nVolkswagen Passat,0X2072R,30.85,118,Gasoline,3.1764,Fan & Song 2022\nVolkswagen Passat,0X2072S,102.66,118,Gasoline,7.5983,Fan & Song 2022\nVolkswagen Passat,0X2072S,61.21,118,Gasoline,5.9935,Fan & Song 2022\nVolkswagen Passat,0X2072R,197.74,118,Gasoline,13.3316,Fan & Song 2022\nVolkswagen Passat,0X2072T,13.02,118,Gasoline,1.2149,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.66,118,Gasoline,0.4528,Fan & Song 2022\nVolkswagen Passat,0X2072T,25.71,118,Gasoline,2.4764,Fan & Song 2022\nVolkswagen Passat,0X2072T,133.17,118,Gasoline,13.5086,Fan & Song 2022\nVolkswagen Passat,0X2072S,13.37,118,Gasoline,2.3227,Fan & Song 2022\nVolkswagen Passat,0X2072T,43.62,118,Gasoline,4.1807,Fan & Song 2022\nVolkswagen Passat,0X2072S,22.98,118,Gasoline,2.0675,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.37,118,Gasoline,0.4645,Fan & Song 2022\nVolkswagen Passat,0X2072R,29.62,118,Gasoline,2.6973,Fan & Song 2022\nVolkswagen Passat,0X2072R,8.1,118,Gasoline,0.9236,Fan & Song 2022\nVolkswagen Passat,0X2072T,44.56,118,Gasoline,4.3483,Fan & Song 2022\nVolkswagen Passat,0X2072T,10.12,118,Gasoline,1.1,Fan & Song 2022\nVolkswagen Passat,0X2072S,103.58,118,Gasoline,9.6398,Fan & Song 2022\nVolkswagen Passat,0X2072T,8.63,118,Gasoline,1.6098,Fan & Song 2022\nVolkswagen Passat,0X2072R,124.96,118,Gasoline,7.6279,Fan & Song 2022\nVolkswagen Passat,0X2072S,45.96,118,Gasoline,3.7409,Fan & Song 2022\nVolkswagen Passat,0X2072R,145.44,118,Gasoline,9.093,Fan & Song 2022\nVolkswagen Passat,0X2072R,4.72,118,Gasoline,0.6269,Fan & Song 2022\nVolkswagen Passat,0X2072R,134.88,118,Gasoline,13.8572,Fan & Song 2022\nVolkswagen Passat,0X2072T,10.4,118,Gasoline,1.3477,Fan & Song 2022\nVolkswagen Passat,0X2072R,46.07,118,Gasoline,3.0879,Fan & Song 2022\nVolkswagen Passat,0X2072T,104.33,118,Gasoline,12.494,Fan & Song 2022\nVolkswagen Passat,0X2072S,72.4,118,Gasoline,8.4018,Fan & Song 2022\nVolkswagen Passat,0X2072T,34.95,118,Gasoline,3.6648,Fan & Song 2022\nVolkswagen Passat,0X2072S,22.16,118,Gasoline,3.25,Fan & Song 2022\nVolkswagen Passat,0X2072S,21.65,118,Gasoline,1.9768,Fan & Song 2022\nVolkswagen Passat,0X2072R,1.59,118,Gasoline,0.2289,Fan & Song 2022\nVolkswagen Passat,0X2072R,327.42,118,Gasoline,21.2782,Fan & Song 2022\nVolkswagen Passat,0X2072S,27.43,118,Gasoline,2.3088,Fan & Song 2022\nVolkswagen Passat,0X2072R,16.16,118,Gasoline,1.4748,Fan & Song 2022\nVolkswagen Passat,0X2072R,177.95,118,Gasoline,13.3453,Fan & Song 2022\nVolkswagen Passat,0X2072S,70.51,118,Gasoline,6.662,Fan & Song 2022\nVolkswagen Passat,0X2072S,32.76,118,Gasoline,3.0387,Fan & Song 2022\nVolkswagen Passat,0X2072R,87.41,118,Gasoline,10.9554,Fan & Song 2022\nVolkswagen Passat,0X2072S,11.97,118,Gasoline,1.6555,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.62,118,Gasoline,0.7033,Fan & Song 2022\nVolkswagen Passat,0X2072R,46.96,118,Gasoline,5.815,Fan & Song 2022\nVolkswagen Passat,0X2072S,122.05,118,Gasoline,12.879,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.84,118,Gasoline,0.5027,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.33,118,Gasoline,0.2431,Fan & Song 2022\nVolkswagen Passat,0X2072T,107.35,118,Gasoline,11.5603,Fan & Song 2022\nVolkswagen Passat,0X2072R,113.25,118,Gasoline,7.1154,Fan & Song 2022\nVolkswagen Passat,0X2072R,129.54,118,Gasoline,8.5281,Fan & Song 2022\nVolkswagen Passat,0X2072R,9.38,118,Gasoline,1.0104,Fan & Song 2022\nVolkswagen Passat,0X2072T,12.03,118,Gasoline,0.9737,Fan & Song 2022\nVolkswagen Passat,0X2072S,4.7,118,Gasoline,0.5503,Fan & Song 2022\nVolkswagen Passat,0X2072T,3.97,118,Gasoline,0.3224,Fan & Song 2022\nVolkswagen Passat,0X2072S,120.5,118,Gasoline,13.4684,Fan & Song 2022\nVolkswagen Passat,0X2072T,221.11,118,Gasoline,16.5594,Fan & Song 2022\nVolkswagen Passat,0X2072S,178.15,118,Gasoline,11.5255,Fan & Song 2022\nVolkswagen Passat,0X2072R,13.17,118,Gasoline,1.3024,Fan & Song 2022\nVolkswagen Passat,0X2072S,240.8,118,Gasoline,20.6385,Fan & Song 2022\nVolkswagen Passat,0X2072S,126.72,118,Gasoline,12.1778,Fan & Song 2022\nVolkswagen Passat,0X2072S,60.0,118,Gasoline,5.0991,Fan & Song 2022\nVolkswagen Passat,0X2072S,50.0,118,Gasoline,3.8615,Fan & Song 2022\nVolkswagen Passat,0X2072S,306.21,118,Gasoline,19.7999,Fan & Song 2022\nVolkswagen Passat,0X2072T,127.69,118,Gasoline,12.0868,Fan & Song 2022\nVolkswagen Passat,0X2072R,117.16,118,Gasoline,10.4308,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.22,118,Gasoline,0.239,Fan & Song 2022\nVolkswagen Passat,0X2072T,52.89,118,Gasoline,5.5609,Fan & Song 2022\nVolkswagen Passat,0X2072S,27.72,118,Gasoline,1.7142,Fan & Song 2022\nVolkswagen Passat,0X2072S,23.1,118,Gasoline,1.7541,Fan & Song 2022\nVolkswagen Passat,0X2072R,171.27,118,Gasoline,13.9281,Fan & Song 2022\nVolkswagen Passat,0X2072R,238.73,118,Gasoline,16.4847,Fan & Song 2022\nVolkswagen Passat,0X2072R,22.81,118,Gasoline,1.9475,Fan & Song 2022\nVolkswagen Passat,0X2072R,9.7,118,Gasoline,1.3529,Fan & Song 2022\nVolkswagen Passat,0X2072T,101.81,118,Gasoline,8.2239,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.22,118,Gasoline,0.6945,Fan & Song 2022\nVolkswagen Passat,0X2072T,103.46,118,Gasoline,10.9085,Fan & Song 2022\nVolkswagen Passat,0X2072R,70.64,118,Gasoline,6.1372,Fan & Song 2022\nVolkswagen Passat,0X2072T,8.53,118,Gasoline,1.0793,Fan & Song 2022\nVolkswagen Passat,0X2072S,15.8,118,Gasoline,1.9899,Fan & Song 2022\nVolkswagen Passat,0X2072S,103.29,118,Gasoline,11.8013,Fan & Song 2022\nVolkswagen Passat,0X2072R,20.96,118,Gasoline,1.7842,Fan & Song 2022\nVolkswagen Passat,0X2072T,30.66,118,Gasoline,2.3141,Fan & Song 2022\nVolkswagen Passat,0X2072R,90.08,118,Gasoline,7.6207,Fan & Song 2022\nVolkswagen Passat,0X2072T,61.64,118,Gasoline,7.0661,Fan & Song 2022\nVolkswagen Passat,0X2072T,22.06,118,Gasoline,1.5966,Fan & Song 2022\nVolkswagen Passat,0X2072R,103.59,118,Gasoline,10.0803,Fan & Song 2022\nVolkswagen Passat,0X2072R,1.57,118,Gasoline,0.2032,Fan & Song 2022\nVolkswagen Passat,0X2072S,37.8,118,Gasoline,3.6473,Fan & Song 2022\nVolkswagen Passat,0X2072T,76.74,118,Gasoline,5.518,Fan & Song 2022\nVolkswagen Passat,0X2072S,40.58,118,Gasoline,4.6565,Fan & Song 2022\nVolkswagen Passat,0X2072R,11.06,118,Gasoline,0.8886,Fan & Song 2022\nVolkswagen Passat,0X2072S,8.32,118,Gasoline,0.575,Fan & Song 2022\nVolkswagen Passat,0X2072T,23.12,118,Gasoline,2.6857,Fan & Song 2022\nVolkswagen Passat,0X2072T,4.83,118,Gasoline,0.6393,Fan & Song 2022\nVolkswagen Passat,0X2072R,24.17,118,Gasoline,1.7202,Fan & Song 2022\nVolkswagen Passat,0X2072R,16.8,118,Gasoline,1.5216,Fan & Song 2022\nVolkswagen Passat,0X2072R,103.83,118,Gasoline,12.2127,Fan & Song 2022\nVolkswagen Passat,0X2072S,23.17,118,Gasoline,1.879,Fan & Song 2022\nVolkswagen Passat,0X2072T,91.9,118,Gasoline,8.0854,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.06,118,Gasoline,0.6388,Fan & Song 2022\nVolkswagen Passat,0X2072S,51.53,118,Gasoline,6.0058,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.06,118,Gasoline,0.3665,Fan & Song 2022\nVolkswagen Passat,0X2072T,3.9,118,Gasoline,0.5265,Fan & Song 2022\nVolkswagen Passat,0X2072S,33.75,118,Gasoline,4.1647,Fan & Song 2022\nVolkswagen Passat,0X2072R,150.95,118,Gasoline,11.8726,Fan & Song 2022\nVolkswagen Passat,0X2072T,79.26,118,Gasoline,6.8209,Fan & Song 2022\nVolkswagen Passat,0X2072S,78.35,118,Gasoline,5.0538,Fan & Song 2022\nVolkswagen Passat,0X2072T,102.77,118,Gasoline,7.6547,Fan & Song 2022\nVolkswagen Passat,0X2072S,101.38,118,Gasoline,6.1882,Fan & Song 2022\nVolkswagen Passat,0X2072S,39.86,118,Gasoline,3.5863,Fan & Song 2022\nVolkswagen Passat,0X2072R,44.1,118,Gasoline,3.6031,Fan & Song 2022\nVolkswagen Passat,0X2072T,31.02,118,Gasoline,3.1968,Fan & Song 2022\nVolkswagen Passat,0X2072S,87.84,118,Gasoline,7.4176,Fan & Song 2022\nVolkswagen Passat,0X2072T,4.0,118,Gasoline,0.5627,Fan & Song 2022\nVolkswagen Passat,0X2072T,25.1,118,Gasoline,2.5064,Fan & Song 2022\nVolkswagen Passat,0X2072S,85.25,118,Gasoline,9.667,Fan & Song 2022\nVolkswagen Passat,0X2072T,32.1,118,Gasoline,2.8196,Fan & Song 2022\nVolkswagen Passat,0X2072S,31.69,118,Gasoline,3.4746,Fan & Song 2022\nVolkswagen Passat,0X2072R,24.96,118,Gasoline,2.285,Fan & Song 2022\nVolkswagen Passat,0X2072R,117.72,118,Gasoline,8.9545,Fan & Song 2022\nVolkswagen Passat,0X2072S,85.35,118,Gasoline,6.1502,Fan & Song 2022\nVolkswagen Passat,0X2072S,109.88,118,Gasoline,7.4798,Fan & Song 2022\nVolkswagen Passat,0X2072S,31.88,118,Gasoline,3.6235,Fan & Song 2022\nVolkswagen Passat,0X2072S,68.43,118,Gasoline,6.1394,Fan & Song 2022\nVolkswagen Passat,0X2072R,42.66,118,Gasoline,3.3318,Fan & Song 2022\nVolkswagen Passat,0X2072R,192.12,118,Gasoline,13.8259,Fan & Song 2022\nVolkswagen Passat,0X2072S,21.81,118,Gasoline,1.9359,Fan & Song 2022\nVolkswagen Passat,0X2072S,41.1,118,Gasoline,4.7486,Fan & Song 2022\nVolkswagen Passat,0X2072S,18.14,118,Gasoline,1.825,Fan & Song 2022\nVolkswagen Passat,0X2072T,2.36,118,Gasoline,0.3304,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.39,118,Gasoline,0.223,Fan & Song 2022\nVolkswagen Passat,0X2072R,144.63,118,Gasoline,10.2558,Fan & Song 2022\nVolkswagen Passat,0X2072S,111.24,118,Gasoline,7.1283,Fan & Song 2022\nVolkswagen Passat,0X2072S,271.21,118,Gasoline,19.1249,Fan & Song 2022\nVolkswagen Passat,0X2072R,6.26,118,Gasoline,0.7311,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.01,118,Gasoline,0.5667,Fan & Song 2022\nVolkswagen Passat,0X2072R,210.11,118,Gasoline,14.1765,Fan & Song 2022\nVolkswagen Passat,0X2072T,9.16,118,Gasoline,0.7972,Fan & Song 2022\nVolkswagen Passat,0X2072T,7.26,118,Gasoline,0.7847,Fan & Song 2022\nVolkswagen Passat,0X2072R,175.41,118,Gasoline,12.4947,Fan & Song 2022\nVolkswagen Passat,0X2072T,17.7,118,Gasoline,1.2334,Fan & Song 2022\nVolkswagen Passat,0X2072R,19.21,118,Gasoline,1.581,Fan & Song 2022\nVolkswagen Passat,0X2072S,151.18,118,Gasoline,10.9588,Fan & Song 2022\nVolkswagen Passat,0X2072R,32.38,118,Gasoline,2.5342,Fan & Song 2022\nVolkswagen Passat,0X2072R,224.36,118,Gasoline,14.7112,Fan & Song 2022\nVolkswagen Passat,0X2072R,224.61,118,Gasoline,14.7485,Fan & Song 2022\nVolkswagen Passat,0X2072T,42.2,118,Gasoline,2.9711,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.52,118,Gasoline,0.6544,Fan & Song 2022\nVolkswagen Passat,0X2072S,31.47,118,Gasoline,1.8727,Fan & Song 2022\nVolkswagen Passat,0X2072T,17.23,118,Gasoline,1.3248,Fan & Song 2022\nVolkswagen Passat,0X2072S,1.79,118,Gasoline,0.225,Fan & Song 2022\nVolkswagen Passat,0X2072S,77.07,118,Gasoline,5.7299,Fan & Song 2022\nVolkswagen Passat,0X2072R,34.73,118,Gasoline,2.5898,Fan & Song 2022\nVolkswagen Passat,0X2072R,50.76,118,Gasoline,5.3701,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.15,118,Gasoline,0.4882,Fan & Song 2022\nVolkswagen Passat,0X2072R,57.62,118,Gasoline,5.6231,Fan & Song 2022\nVolkswagen Passat,0X2072R,120.66,118,Gasoline,10.455,Fan & Song 2022\nVolkswagen Passat,0X2072S,30.94,118,Gasoline,2.0725,Fan & Song 2022\nVolkswagen Passat,0X2072T,54.65,118,Gasoline,4.6813,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.78,118,Gasoline,0.6491,Fan & Song 2022\nVolkswagen Passat,0X2072T,36.95,118,Gasoline,2.4516,Fan & Song 2022\nVolkswagen Passat,0X2072S,1.77,118,Gasoline,0.2181,Fan & Song 2022\nVolkswagen Passat,0X2072T,4.68,118,Gasoline,0.4212,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.53,118,Gasoline,0.292,Fan & Song 2022\nVolkswagen Passat,0X2072T,15.58,118,Gasoline,1.2978,Fan & Song 2022\nVolkswagen Passat,0X2072S,20.93,118,Gasoline,2.4605,Fan & Song 2022\nVolkswagen Passat,0X2072T,27.58,118,Gasoline,2.9986,Fan & Song 2022\nVolkswagen Passat,0X2072S,33.68,118,Gasoline,3.5491,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.83,118,Gasoline,0.5065,Fan & Song 2022\nVolkswagen Passat,0X2072S,23.6,118,Gasoline,2.3583,Fan & Song 2022\nVolkswagen Passat,0X2072T,106.25,118,Gasoline,7.776,Fan & Song 2022\nVolkswagen Passat,0X2072R,124.66,118,Gasoline,11.089,Fan & Song 2022\nVolkswagen Passat,0X2072S,121.77,118,Gasoline,8.6275,Fan & Song 2022\nVolkswagen Passat,0X2072T,35.9,118,Gasoline,3.21,Fan & Song 2022\nVolkswagen Passat,0X2072R,8.04,118,Gasoline,0.7625,Fan & Song 2022\nVolkswagen Passat,0X2072S,121.67,118,Gasoline,8.9023,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.48,118,Gasoline,0.3023,Fan & Song 2022\nVolkswagen Passat,0X2072S,17.56,118,Gasoline,2.0055,Fan & Song 2022\nVolkswagen Passat,0X2072R,21.62,118,Gasoline,2.4773,Fan & Song 2022\nVolkswagen Passat,0X2072T,13.69,118,Gasoline,1.2196,Fan & Song 2022\nVolkswagen Passat,0X2072T,42.43,118,Gasoline,3.1286,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.66,118,Gasoline,0.3018,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.52,118,Gasoline,0.3209,Fan & Song 2022\nVolkswagen Passat,0X2072S,15.66,118,Gasoline,1.4696,Fan & Song 2022\nVolkswagen Passat,0X2072R,16.31,118,Gasoline,1.4071,Fan & Song 2022\nVolkswagen Passat,0X2072R,101.99,118,Gasoline,9.9864,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.11,118,Gasoline,0.2255,Fan & Song 2022\nVolkswagen Passat,0X2072S,46.24,118,Gasoline,4.3712,Fan & Song 2022\nVolkswagen Passat,0X2072R,29.16,118,Gasoline,3.0528,Fan & Song 2022\nVolkswagen Passat,0X2072T,35.27,118,Gasoline,2.8967,Fan & Song 2022\nVolkswagen Passat,0X2072R,25.18,118,Gasoline,3.757,Fan & Song 2022\nVolkswagen Passat,0X2072S,5.54,118,Gasoline,0.6832,Fan & Song 2022\nVolkswagen Passat,0X2072S,18.13,118,Gasoline,1.9549,Fan & Song 2022\nVolkswagen Passat,0X2072S,11.6,118,Gasoline,1.1532,Fan & Song 2022\nVolkswagen Passat,0X2072T,40.27,118,Gasoline,3.5429,Fan & Song 2022\nVolkswagen Passat,0X2072S,48.6,118,Gasoline,2.9685,Fan & Song 2022\nVolkswagen Passat,0X2072R,124.89,118,Gasoline,9.8022,Fan & Song 2022\nVolkswagen Passat,0X2072T,102.26,118,Gasoline,9.0359,Fan & Song 2022\nVolkswagen Passat,0X2072R,151.13,118,Gasoline,12.6978,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.13,118,Gasoline,0.2689,Fan & Song 2022\nVolkswagen Passat,0X2072R,48.41,118,Gasoline,4.3469,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.2,118,Gasoline,0.2955,Fan & Song 2022\nVolkswagen Passat,0X2072T,71.6,118,Gasoline,5.8994,Fan & Song 2022\nVolkswagen Passat,0X2072T,6.08,118,Gasoline,0.7597,Fan & Song 2022\nVolkswagen Passat,0X2072S,239.2,118,Gasoline,17.1487,Fan & Song 2022\nVolkswagen Passat,0X2072S,39.3,118,Gasoline,3.013,Fan & Song 2022\nVolkswagen Passat,0X2072S,24.01,118,Gasoline,1.6751,Fan & Song 2022\nVolkswagen Passat,0X2072T,115.33,118,Gasoline,9.259,Fan & Song 2022\nVolkswagen Passat,0X2072T,106.47,118,Gasoline,9.4473,Fan & Song 2022\nVolkswagen Passat,0X2072T,6.01,118,Gasoline,0.7236,Fan & Song 2022\nVolkswagen Passat,0X2072T,25.49,118,Gasoline,2.1181,Fan & Song 2022\nVolkswagen Passat,0X2072S,70.76,118,Gasoline,6.6219,Fan & Song 2022\nVolkswagen Passat,0X2072R,81.76,118,Gasoline,8.6124,Fan & Song 2022\nVolkswagen Passat,0X2072S,127.1,118,Gasoline,10.4388,Fan & Song 2022\nVolkswagen Passat,0X2072S,14.8,118,Gasoline,1.5607,Fan & Song 2022\nVolkswagen Passat,0X2072S,124.18,118,Gasoline,12.061,Fan & Song 2022\nVolkswagen Passat,0X2072R,21.95,118,Gasoline,1.7457,Fan & Song 2022\nVolkswagen Passat,0X2072S,10.26,118,Gasoline,1.0961,Fan & Song 2022\nVolkswagen Passat,0X2072T,24.65,118,Gasoline,2.0215,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.25,118,Gasoline,0.3598,Fan & Song 2022\nVolkswagen Passat,0X2072T,10.62,118,Gasoline,1.1327,Fan & Song 2022\nVolkswagen Passat,0X2072S,17.24,118,Gasoline,1.1702,Fan & Song 2022\nVolkswagen Passat,0X2072T,100.78,118,Gasoline,10.0388,Fan & Song 2022\nVolkswagen Passat,0X2072R,23.76,118,Gasoline,1.7294,Fan & Song 2022\nVolkswagen Passat,0X2072R,109.36,118,Gasoline,7.4319,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.72,118,Gasoline,0.3745,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.45,118,Gasoline,0.286,Fan & Song 2022\nVolkswagen Passat,0X2072S,18.16,118,Gasoline,2.2879,Fan & Song 2022\nVolkswagen Passat,0X2072S,10.34,118,Gasoline,0.7393,Fan & Song 2022\nVolkswagen Passat,0X2072S,105.48,118,Gasoline,9.1797,Fan & Song 2022\nVolkswagen Passat,0X2072R,147.36,118,Gasoline,11.9156,Fan & Song 2022\nVolkswagen Passat,0X2072S,12.43,118,Gasoline,1.3353,Fan & Song 2022\nVolkswagen Passat,0X2072T,5.12,118,Gasoline,0.5692,Fan & Song 2022\nVolkswagen Passat,0X2072R,4.77,118,Gasoline,0.6158,Fan & Song 2022\nVolkswagen Passat,0X2072R,104.47,118,Gasoline,8.4174,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.24,118,Gasoline,0.3962,Fan & Song 2022\nVolkswagen Passat,0X2072R,3.1,118,Gasoline,0.2856,Fan & Song 2022\nVolkswagen Passat,0X2072T,21.94,118,Gasoline,2.0229,Fan & Song 2022\nVolkswagen Passat,0X2072R,17.9,118,Gasoline,2.0853,Fan & Song 2022\nVolkswagen Passat,0X2072R,1.53,118,Gasoline,0.1435,Fan & Song 2022\nVolkswagen Passat,0X2072T,59.52,118,Gasoline,4.1186,Fan & Song 2022\nVolkswagen Passat,0X2072S,212.08,118,Gasoline,14.3049,Fan & Song 2022\nVolkswagen Passat,0X2072R,42.52,118,Gasoline,3.1342,Fan & Song 2022\nVolkswagen Passat,0X2072R,4.1,118,Gasoline,0.4013,Fan & Song 2022\nVolkswagen Passat,0X2072S,86.4,118,Gasoline,6.2197,Fan & Song 2022\nVolkswagen Passat,0X2072R,104.26,118,Gasoline,8.901,Fan & Song 2022\nVolkswagen Passat,0X2072S,3.78,118,Gasoline,0.6431,Fan & Song 2022\nVolkswagen Passat,0X2072T,20.89,118,Gasoline,1.8083,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.3,118,Gasoline,0.2217,Fan & Song 2022\nVolkswagen Passat,0X2072T,17.82,118,Gasoline,1.4668,Fan & Song 2022\nVolkswagen Passat,0X2072T,15.07,118,Gasoline,1.516,Fan & Song 2022\nVolkswagen Passat,0X2072T,9.98,118,Gasoline,1.6219,Fan & Song 2022\nVolkswagen Passat,0X2072S,17.78,118,Gasoline,1.1397,Fan & Song 2022\nVolkswagen Passat,0X2072T,3.45,118,Gasoline,0.396,Fan & Song 2022\nVolkswagen Passat,0X2072R,4.95,118,Gasoline,0.6034,Fan & Song 2022\nVolkswagen Passat,0X2072S,23.39,118,Gasoline,1.7062,Fan & Song 2022\nVolkswagen Passat,0X2072T,37.78,118,Gasoline,3.4416,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.34,118,Gasoline,0.1816,Fan & Song 2022\nVolkswagen Passat,0X2072T,28.1,118,Gasoline,2.3589,Fan & Song 2022\nVolkswagen Passat,0X2072T,8.89,118,Gasoline,0.6757,Fan & Song 2022\nVolkswagen Passat,0X2072R,5.91,118,Gasoline,0.6186,Fan & Song 2022\nVolkswagen Passat,0X2072S,5.67,118,Gasoline,0.7052,Fan & Song 2022\nVolkswagen Passat,0X2072S,106.98,118,Gasoline,11.3806,Fan & Song 2022\nVolkswagen Passat,0X2072T,8.0,118,Gasoline,0.8803,Fan & Song 2022\nVolkswagen Passat,0X2072S,8.05,118,Gasoline,1.1607,Fan & Song 2022\nVolkswagen Passat,0X2072S,149.28,118,Gasoline,14.4109,Fan & Song 2022\nVolkswagen Passat,0X2072T,40.48,118,Gasoline,7.0457,Fan & Song 2022\nVolkswagen Passat,0X2072R,2.3,118,Gasoline,0.2595,Fan & Song 2022\nVolkswagen Passat,0X2072R,78.98,118,Gasoline,4.972,Fan & Song 2022\nVolkswagen Passat,0X2072S,147.06,118,Gasoline,11.8055,Fan & Song 2022\nVolkswagen Passat,0X2072T,120.05,118,Gasoline,12.6301,Fan & Song 2022\nVolkswagen Passat,0X2072R,148.46,118,Gasoline,10.2424,Fan & Song 2022\nVolkswagen Passat,0X2072T,216.55,118,Gasoline,16.9845,Fan & Song 2022\nVolkswagen Passat,0X2072S,32.19,118,Gasoline,2.7864,Fan & Song 2022\nVolkswagen Passat,0X2072T,4.93,118,Gasoline,0.5914,Fan & Song 2022\nVolkswagen Passat,0X2072R,116.05,118,Gasoline,9.6063,Fan & Song 2022\nVolkswagen Passat,0X2072S,16.18,118,Gasoline,1.6079,Fan & Song 2022\nVolkswagen Passat,0X2072T,2.62,118,Gasoline,0.2729,Fan & Song 2022\nVolkswagen Passat,0X2072S,30.62,118,Gasoline,2.4453,Fan & Song 2022\nVolkswagen Passat,0X2072R,67.31,118,Gasoline,5.9978,Fan & Song 2022\nVolkswagen Passat,0X2072R,38.38,118,Gasoline,2.9474,Fan & Song 2022\nVolkswagen Passat,0X2072S,2.52,118,Gasoline,0.4429,Fan & Song 2022\nVolkswagen Passat,0X2072R,87.04,118,Gasoline,7.5522,Fan & Song 2022\nRenault Kangoo Van,,30.0,55,Diesel,4.23,EEA 2024 (OBFCM)\nVolkswagen Caddy,,30.5,55,Diesel,3.8,EEA 2024 (OBFCM)\nRenault Express,,31.6,55,Diesel,2.93,EEA 2024 (OBFCM)\nRenault Express,,31.9,55,Diesel,4.14,EEA 2024 (OBFCM)\nVolkswagen Caddy,,32.3,55,Diesel,3.8,EEA 2024 (OBFCM)\nVolkswagen Caddy,,33.6,55,Diesel,3.9,EEA 2024 (OBFCM)\nVolkswagen Caddy,,33.9,55,Diesel,3.7,EEA 2024 (OBFCM)\nRenault Kangoo Van,,34.6,55,Diesel,3.81,EEA 2024 (OBFCM)\nVolkswagen Caddy,,34.8,55,Diesel,6.3,EEA 2024 (OBFCM)\nRenault Express,,35.6,55,Diesel,3.11,EEA 2024 (OBFCM)\nRenault Kangoo Van,,36.7,55,Diesel,3.22,EEA 2024 (OBFCM)\nRenault Kangoo Van,,37.5,55,Diesel,3.41,EEA 2024 (OBFCM)\nVolkswagen Caddy,,38.0,55,Diesel,4.1,EEA 2024 (OBFCM)\nRenault Kangoo Van,,38.6,55,Diesel,3.3,EEA 2024 (OBFCM)\nRenault Kangoo Van,,39.0,55,Diesel,3.21,EEA 2024 (OBFCM)\nVolkswagen Caddy,,39.8,55,Diesel,5.0,EEA 2024 (OBFCM)\nVolkswagen Caddy,,40.3,55,Diesel,5.2,EEA 2024 (OBFCM)\nRenault Kangoo Van,,48.5,55,Diesel,4.47,EEA 2024 (OBFCM)\nRenault Express,,50.8,55,Diesel,5.32,EEA 2024 (OBFCM)\nRenault Kangoo,,51.1,55,Diesel,5.23,EEA 2024 (OBFCM)\nVolkswagen Caddy,,51.4,55,Diesel,3.8,EEA 2024 (OBFCM)\nRenault Express,,52.3,55,Diesel,5.76,EEA 2024 (OBFCM)\nRenault Kangoo,,53.6,55,Diesel,6.0,EEA 2024 (OBFCM)\nVolkswagen Caddy,,55.5,55,Diesel,3.9,EEA 2024 (OBFCM)\nRenault Kangoo Van,,56.0,55,Diesel,4.45,EEA 2024 (OBFCM)\nRenault Express,,57.0,55,Diesel,5.0,EEA 2024 (OBFCM)\nRenault Express,,57.4,55,Diesel,5.32,EEA 2024 (OBFCM)\nRenault Kangoo Van,,57.5,55,Diesel,4.86,EEA 2024 (OBFCM)\nRenault Express,,60.3,55,Diesel,6.27,EEA 2024 (OBFCM)\nVolkswagen Caddy,,61.9,55,Diesel,4.5,EEA 2024 (OBFCM)\nRenault Kangoo Van,,64.3,55,Diesel,5.28,EEA 2024 (OBFCM)\nRenault Kangoo Van,,65.3,55,Diesel,5.93,EEA 2024 (OBFCM)\nRenault Kangoo Van,,65.9,55,Diesel,4.83,EEA 2024 (OBFCM)\nRenault Express,,66.8,55,Diesel,4.42,EEA 2024 (OBFCM)\nRenault Express,,67.3,55,Diesel,5.21,EEA 2024 (OBFCM)\nVolkswagen Caddy,,69.7,55,Diesel,4.8,EEA 2024 (OBFCM)\nRenault Kangoo Van,,70.1,55,Diesel,4.46,EEA 2024 (OBFCM)\nRenault Kangoo Van,,71.0,55,Diesel,6.19,EEA 2024 (OBFCM)\nVolkswagen Caddy,,73.4,55,Diesel,6.3,EEA 2024 (OBFCM)\nRenault Kangoo Van,,73.5,55,Diesel,6.33,EEA 2024 (OBFCM)\nVolkswagen Caddy,,73.7,55,Diesel,5.5,EEA 2024 (OBFCM)\nRenault Kangoo Van,,76.0,55,Diesel,4.91,EEA 2024 (OBFCM)\nRenault Kangoo Van,,80.1,55,Diesel,5.54,EEA 2024 (OBFCM)\nRenault Kangoo Van,,80.4,55,Diesel,6.73,EEA 2024 (OBFCM)\nRenault Express,,80.4,55,Diesel,5.29,EEA 2024 (OBFCM)\nRenault Express,,80.4,55,Diesel,6.54,EEA 2024 (OBFCM)\nRenault Express,,82.2,55,Diesel,7.28,EEA 2024 (OBFCM)\nRenault Express,,86.5,55,Diesel,7.85,EEA 2024 (OBFCM)\nVolkswagen Caddy,,90.3,55,Diesel,6.5,EEA 2024 (OBFCM)\nRenault Express,,91.5,55,Diesel,5.26,EEA 2024 (OBFCM)\nRenault Express,,98.2,55,Diesel,8.59,EEA 2024 (OBFCM)\nRenault Express,,107.2,55,Diesel,6.46,EEA 2024 (OBFCM)\nRenault Kangoo Van,,114.0,55,Diesel,7.72,EEA 2024 (OBFCM)\nRenault Kangoo Van,,116.6,55,Diesel,9.09,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.1,55,Diesel,9.97,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.1,55,Diesel,9.88,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.1,55,Diesel,10.86,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.1,55,Diesel,10.85,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,55,Diesel,10.38,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,55,Diesel,10.19,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,55,Diesel,9.07,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,55,Diesel,10.13,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.3,55,Diesel,14.1,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.3,55,Diesel,11.89,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.8,55,Diesel,9.1,EEA 2024 (OBFCM)\nRenault Kangoo Van,,169.8,55,Diesel,11.39,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.0,55,Diesel,17.0,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.0,55,Diesel,13.28,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.1,55,Diesel,12.93,EEA 2024 (OBFCM)\nRenault Express,,200.2,55,Diesel,16.86,EEA 2024 (OBFCM)\nRenault Express,,200.2,55,Diesel,14.5,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.4,55,Diesel,13.51,EEA 2024 (OBFCM)\nRenault Express,,200.7,55,Diesel,13.7,EEA 2024 (OBFCM)\nRenault Kangoo Van,,201.3,55,Diesel,13.49,EEA 2024 (OBFCM)\nRenault Express,,201.6,55,Diesel,13.3,EEA 2024 (OBFCM)\nRenault Kangoo Van,,201.6,55,Diesel,10.28,EEA 2024 (OBFCM)\nRenault Kangoo Van,,202.7,55,Diesel,11.84,EEA 2024 (OBFCM)\nRenault Express,,203.0,55,Diesel,20.62,EEA 2024 (OBFCM)\nRenault Kangoo Van,,203.4,55,Diesel,16.85,EEA 2024 (OBFCM)\nRenault Express,,203.7,55,Diesel,11.04,EEA 2024 (OBFCM)\nRenault Kangoo Van,,204.8,55,Diesel,12.31,EEA 2024 (OBFCM)\nRenault Kangoo Van,,205.1,55,Diesel,14.92,EEA 2024 (OBFCM)\nRenault Express,,205.4,55,Diesel,17.06,EEA 2024 (OBFCM)\nRenault Kangoo Van,,205.6,55,Diesel,14.4,EEA 2024 (OBFCM)\nRenault Kangoo Van,,205.7,55,Diesel,12.42,EEA 2024 (OBFCM)\nRenault Kangoo Van,,206.0,55,Diesel,14.25,EEA 2024 (OBFCM)\nRenault Kangoo Van,,206.2,55,Diesel,13.51,EEA 2024 (OBFCM)\nRenault Express,,206.3,55,Diesel,14.66,EEA 2024 (OBFCM)\nRenault Kangoo Van,,207.0,55,Diesel,14.9,EEA 2024 (OBFCM)\nRenault Express,,207.5,55,Diesel,16.3,EEA 2024 (OBFCM)\nRenault Kangoo Van,,207.7,55,Diesel,12.2,EEA 2024 (OBFCM)\nRenault Kangoo Van,,207.7,55,Diesel,15.13,EEA 2024 (OBFCM)\nRenault Kangoo Van,,207.8,55,Diesel,11.48,EEA 2024 (OBFCM)\nRenault Kangoo Van,,210.9,55,Diesel,12.79,EEA 2024 (OBFCM)\nRenault Kangoo Van,,212.4,55,Diesel,13.22,EEA 2024 (OBFCM)\nRenault Kangoo Van,,214.9,55,Diesel,14.97,EEA 2024 (OBFCM)\nRenault Kangoo Van,,219.6,55,Diesel,16.16,EEA 2024 (OBFCM)\nRenault Kangoo Van,,221.5,55,Diesel,16.29,EEA 2024 (OBFCM)\nRenault Express,,230.0,55,Diesel,12.21,EEA 2024 (OBFCM)\nRenault Kangoo Van,,230.4,55,Diesel,17.5,EEA 2024 (OBFCM)\nRenault Kangoo Van,,231.5,55,Diesel,16.53,EEA 2024 (OBFCM)\nRenault Kangoo Van,,238.7,55,Diesel,16.38,EEA 2024 (OBFCM)\nRenault Kangoo Van,,249.1,55,Diesel,16.99,EEA 2024 (OBFCM)\nRenault Kangoo Van,,251.6,55,Diesel,16.26,EEA 2024 (OBFCM)\nRenault Express,,253.1,55,Diesel,17.5,EEA 2024 (OBFCM)\nRenault Express,,257.5,55,Diesel,13.41,EEA 2024 (OBFCM)\nRenault Express,,262.9,55,Diesel,19.2,EEA 2024 (OBFCM)\nRenault Kangoo Van,,284.3,55,Diesel,17.15,EEA 2024 (OBFCM)\nRenault Kangoo Van,,286.6,55,Diesel,16.63,EEA 2024 (OBFCM)\nRenault Kangoo,,298.1,55,Diesel,28.59,EEA 2024 (OBFCM)\nRenault Kangoo Van,,302.2,55,Diesel,26.49,EEA 2024 (OBFCM)\nRenault Kangoo Van,,302.3,55,Diesel,27.47,EEA 2024 (OBFCM)\nRenault Express,,308.3,55,Diesel,18.8,EEA 2024 (OBFCM)\nFord Transit Courier,,311.0,55,Diesel,22.24,EEA 2024 (OBFCM)\nRenault Express,,324.1,55,Diesel,24.64,EEA 2024 (OBFCM)\nRenault Express,,333.2,55,Diesel,24.91,EEA 2024 (OBFCM)\nFord Transit Courier,,343.1,55,Diesel,20.97,EEA 2024 (OBFCM)\nRenault Kangoo Van,,355.9,55,Diesel,22.74,EEA 2024 (OBFCM)\nRenault Express,,356.5,55,Diesel,22.62,EEA 2024 (OBFCM)\nRenault Express,,366.6,55,Diesel,26.63,EEA 2024 (OBFCM)\nRenault Express,,373.4,55,Diesel,26.29,EEA 2024 (OBFCM)\nRenault Kangoo Van,,394.2,55,Diesel,27.69,EEA 2024 (OBFCM)\nRenault Express,,398.4,55,Diesel,28.7,EEA 2024 (OBFCM)\nFord Transit Connect,,404.0,55,Diesel,32.77,EEA 2024 (OBFCM)\nRenault Kangoo Van,,415.4,55,Diesel,27.72,EEA 2024 (OBFCM)\nFord Transit Courier,,424.3,55,Diesel,42.93,EEA 2024 (OBFCM)\nRenault Kangoo Van,,452.0,55,Diesel,29.13,EEA 2024 (OBFCM)\nRenault Kangoo Van,,470.0,55,Diesel,36.18,EEA 2024 (OBFCM)\nRenault Express,,477.5,55,Diesel,41.08,EEA 2024 (OBFCM)\nFord Transit Connect,,486.1,55,Diesel,30.85,EEA 2024 (OBFCM)\nRenault Kangoo Van,,489.1,55,Diesel,34.55,EEA 2024 (OBFCM)\nRenault Express,,501.1,55,Diesel,31.14,EEA 2024 (OBFCM)\nRenault Express,,534.3,55,Diesel,38.52,EEA 2024 (OBFCM)\nRenault Express,,539.6,55,Diesel,36.26,EEA 2024 (OBFCM)\nRenault Express,,540.0,55,Diesel,31.2,EEA 2024 (OBFCM)\nRenault Kangoo Van,,544.5,55,Diesel,34.02,EEA 2024 (OBFCM)\nRenault Express,,549.5,55,Diesel,38.72,EEA 2024 (OBFCM)\nRenault Kangoo Van,,555.6,55,Diesel,37.1,EEA 2024 (OBFCM)\nRenault Express,,577.9,55,Diesel,37.06,EEA 2024 (OBFCM)\nRenault Express,,590.5,55,Diesel,55.14,EEA 2024 (OBFCM)\nFord Transit Courier,,596.9,55,Diesel,45.71,EEA 2024 (OBFCM)\nRenault Express,,603.5,55,Diesel,33.67,EEA 2024 (OBFCM)\nRenault Kangoo Van,,608.1,55,Diesel,45.47,EEA 2024 (OBFCM)\nFord Transit Courier,,630.2,55,Diesel,38.51,EEA 2024 (OBFCM)\nRenault Kangoo Van,,637.3,55,Diesel,39.29,EEA 2024 (OBFCM)\nRenault Kangoo Van,,654.5,55,Diesel,42.66,EEA 2024 (OBFCM)\nRenault Express,,720.1,55,Diesel,48.67,EEA 2024 (OBFCM)\nRenault Express,,751.9,55,Diesel,55.83,EEA 2024 (OBFCM)\nRenault Kangoo Van,,767.0,55,Diesel,61.07,EEA 2024 (OBFCM)\nFord Transit Courier,,776.0,55,Diesel,45.12,EEA 2024 (OBFCM)\nRenault Express,,788.0,55,Diesel,43.4,EEA 2024 (OBFCM)\nRenault Kangoo Van,,788.7,55,Diesel,47.01,EEA 2024 (OBFCM)\nRenault Express,,793.4,55,Diesel,50.7,EEA 2024 (OBFCM)\nRenault Express,,804.2,55,Diesel,50.57,EEA 2024 (OBFCM)\nRenault Express,,832.4,55,Diesel,50.05,EEA 2024 (OBFCM)\nRenault Kangoo Van,,836.7,55,Diesel,53.66,EEA 2024 (OBFCM)\nFord Transit Courier,,838.0,55,Diesel,50.29,EEA 2024 (OBFCM)\nRenault Express,,855.6,55,Diesel,42.35,EEA 2024 (OBFCM)\nRenault Kangoo Van,,863.1,55,Diesel,47.89,EEA 2024 (OBFCM)\nRenault Express,,878.2,55,Diesel,61.21,EEA 2024 (OBFCM)\nRenault Kangoo Van,,890.1,55,Diesel,65.69,EEA 2024 (OBFCM)\nRenault Express,,890.7,55,Diesel,58.04,EEA 2024 (OBFCM)\nRenault Express,,930.3,55,Diesel,60.69,EEA 2024 (OBFCM)\nRenault Express,,944.5,55,Diesel,72.67,EEA 2024 (OBFCM)\nRenault Express,,945.6,55,Diesel,53.4,EEA 2024 (OBFCM)\nRenault Express,,945.9,55,Diesel,67.04,EEA 2024 (OBFCM)\nRenault Express,,949.6,55,Diesel,53.08,EEA 2024 (OBFCM)\nRenault Express,,978.3,55,Diesel,64.26,EEA 2024 (OBFCM)\nRenault Express,,992.4,55,Diesel,75.05,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,31.6,59,Diesel,2.4,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,34.6,59,Diesel,3.4,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,42.7,59,Diesel,7.0,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,43.3,59,Diesel,4.2,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,43.8,59,Diesel,5.3,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,47.0,59,Diesel,5.4,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,47.9,59,Diesel,5.5,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,68.8,59,Diesel,4.5,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,81.0,59,Diesel,6.4,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,153.8,59,Diesel,9.8,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,501.0,59,Diesel,28.1,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,30.7,70,Diesel,4.5,EEA 2024 (OBFCM)\nRenault Express,,32.9,70,Diesel,3.52,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,34.0,70,Diesel,5.6,EEA 2024 (OBFCM)\nRenault Kangoo Van,,34.4,70,Diesel,3.9,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,34.9,70,Diesel,2.8,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,41.9,70,Diesel,3.1,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,42.2,70,Diesel,4.0,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,51.8,70,Diesel,4.5,EEA 2024 (OBFCM)\nRenault Express,,55.1,70,Diesel,3.46,EEA 2024 (OBFCM)\nRenault Express,,64.0,70,Diesel,4.62,EEA 2024 (OBFCM)\nRenault Kangoo Van,,69.4,70,Diesel,5.36,EEA 2024 (OBFCM)\nRenault Kangoo Van,,70.3,70,Diesel,5.38,EEA 2024 (OBFCM)\nRenault Express,,71.9,70,Diesel,5.49,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,72.2,70,Diesel,5.9,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,74.6,70,Diesel,7.6,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,77.9,70,Diesel,4.6,EEA 2024 (OBFCM)\nMercedes-Benz Citan,,79.0,70,Diesel,7.0,EEA 2024 (OBFCM)\nRenault Kangoo Van,,85.9,70,Diesel,6.97,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,87.6,70,Diesel,5.2,EEA 2024 (OBFCM)\nFiat Fiat Fiorino,,138.4,70,Diesel,7.9,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.1,70,Diesel,15.75,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.1,70,Diesel,11.14,EEA 2024 (OBFCM)\n104 Kangoo Van,,150.1,70,Diesel,11.81,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,70,Diesel,11.41,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,70,Diesel,9.45,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,70,Diesel,11.98,EEA 2024 (OBFCM)\nRenault Express,,150.2,70,Diesel,9.26,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,70,Diesel,11.19,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,70,Diesel,10.43,EEA 2024 (OBFCM)\nRenault Express,,150.2,70,Diesel,7.8,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.2,70,Diesel,11.67,EEA 2024 (OBFCM)\nRenault Kangoo Van,,150.3,70,Diesel,10.55,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.0,70,Diesel,15.08,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.0,70,Diesel,16.88,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.0,70,Diesel,17.12,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.0,70,Diesel,19.69,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.0,70,Diesel,17.38,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.1,70,Diesel,12.4,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.2,70,Diesel,13.99,EEA 2024 (OBFCM)\nRenault Express,,200.6,70,Diesel,13.8,EEA 2024 (OBFCM)\nRenault Kangoo Van,,201.0,70,Diesel,13.15,EEA 2024 (OBFCM)\nRenault Kangoo Van,,201.1,70,Diesel,12.35,EEA 2024 (OBFCM)\nRenault Express,,201.3,70,Diesel,13.0,EEA 2024 (OBFCM)\nRenault Kangoo Van,,201.3,70,Diesel,19.09,EEA 2024 (OBFCM)\nRenault Kangoo Van,,201.3,70,Diesel,15.76,EEA 2024 (OBFCM)\nRenault Express,,201.5,70,Diesel,13.4,EEA 2024 (OBFCM)\nRenault Kangoo Van,,201.7,70,Diesel,14.39,EEA 2024 (OBFCM)\nRenault Kangoo Van,,203.6,70,Diesel,12.53,EEA 2024 (OBFCM)\nRenault Kangoo Van,,204.4,70,Diesel,11.11,EEA 2024 (OBFCM)\nRenault Kangoo Van,,209.3,70,Diesel,11.66,EEA 2024 (OBFCM)\nRenault Kangoo Van,,209.5,70,Diesel,14.08,EEA 2024 (OBFCM)\nRenault Kangoo Van,,211.2,70,Diesel,12.71,EEA 2024 (OBFCM)\nRenault Kangoo Van,,211.8,70,Diesel,12.24,EEA 2024 (OBFCM)\nRenault Kangoo Van,,212.0,70,Diesel,16.21,EEA 2024 (OBFCM)\nRenault Kangoo Van,,213.2,70,Diesel,14.46,EEA 2024 (OBFCM)\nRenault Kangoo Van,,213.3,70,Diesel,14.29,EEA 2024 (OBFCM)\nRenault Kangoo Van,,215.0,70,Diesel,16.12,EEA 2024 (OBFCM)\nRenault Kangoo Van,,215.6,70,Diesel,14.9,EEA 2024 (OBFCM)\nRenault Kangoo Van,,216.9,70,Diesel,15.47,EEA 2024 (OBFCM)\nRenault Kangoo Van,,217.4,70,Diesel,12.24,EEA 2024 (OBFCM)\nRenault Kangoo Van,,222.0,70,Diesel,15.59,EEA 2024 (OBFCM)\nRenault Kangoo Van,,229.0,70,Diesel,15.56,EEA 2024 (OBFCM)\nRenault Kangoo Van,,230.6,70,Diesel,15.48,EEA 2024 (OBFCM)\nRenault Kangoo Van,,230.6,70,Diesel,13.32,EEA 2024 (OBFCM)\nRenault Express,,233.3,70,Diesel,19.61,EEA 2024 (OBFCM)\nRenault Kangoo Van,,240.4,70,Diesel,23.29,EEA 2024 (OBFCM)\nRenault Express,,247.9,70,Diesel,16.9,EEA 2024 (OBFCM)\nRenault Kangoo Van,,248.8,70,Diesel,14.82,EEA 2024 (OBFCM)\nRenault Kangoo Van,,275.1,70,Diesel,17.17,EEA 2024 (OBFCM)\nRenault Kangoo Van,,296.2,70,Diesel,23.52,EEA 2024 (OBFCM)\nRenault Kangoo Van,,297.4,70,Diesel,35.11,EEA 2024 (OBFCM)\nRenault Kangoo Van,,303.9,70,Diesel,17.28,EEA 2024 (OBFCM)\nRenault Kangoo Van,,343.8,70,Diesel,23.72,EEA 2024 (OBFCM)\nRenault Express,,354.0,70,Diesel,32.07,EEA 2024 (OBFCM)\nRenault Kangoo Van,,367.8,70,Diesel,29.45,EEA 2024 (OBFCM)\nRenault Express,,398.7,70,Diesel,24.78,EEA 2024 (OBFCM)\nRenault Kangoo Van,,400.6,70,Diesel,34.24,EEA 2024 (OBFCM)\nRenault Kangoo Van,,421.8,70,Diesel,31.98,EEA 2024 (OBFCM)\nRenault Express,,430.7,70,Diesel,25.24,EEA 2024 (OBFCM)\nRenault Express,,455.3,70,Diesel,29.82,EEA 2024 (OBFCM)\nRenault Express,,457.8,70,Diesel,29.54,EEA 2024 (OBFCM)\nRenault Kangoo Van,,486.2,70,Diesel,35.07,EEA 2024 (OBFCM)\nRenault Kangoo Van,,490.3,70,Diesel,31.95,EEA 2024 (OBFCM)\nRenault Express,,492.1,70,Diesel,33.64,EEA 2024 (OBFCM)\nRenault Express,,501.5,70,Diesel,31.07,EEA 2024 (OBFCM)\nRenault Kangoo Van,,506.1,70,Diesel,37.42,EEA 2024 (OBFCM)\nRenault Express,,509.5,70,Diesel,28.42,EEA 2024 (OBFCM)\nRenault Kangoo Van,,531.1,70,Diesel,29.66,EEA 2024 (OBFCM)\nRenault Express,,538.0,70,Diesel,36.26,EEA 2024 (OBFCM)\nRenault Kangoo Van,,544.5,70,Diesel,35.48,EEA 2024 (OBFCM)\nRenault Kangoo Van,,575.9,70,Diesel,43.45,EEA 2024 (OBFCM)\nRenault Kangoo Van,,577.3,70,Diesel,38.41,EEA 2024 (OBFCM)\nRenault Kangoo Van,,613.0,70,Diesel,36.53,EEA 2024 (OBFCM)\nRenault Kangoo Van,,636.8,70,Diesel,43.33,EEA 2024 (OBFCM)\nRenault Kangoo Van,,642.2,70,Diesel,48.53,EEA 2024 (OBFCM)\nRenault Express,,643.9,70,Diesel,48.38,EEA 2024 (OBFCM)\nRenault Kangoo Van,,663.9,70,Diesel,56.74,EEA 2024 (OBFCM)\nRenault Kangoo Van,,663.9,70,Diesel,44.25,EEA 2024 (OBFCM)\nRenault Kangoo Van,,707.0,70,Diesel,50.97,EEA 2024 (OBFCM)\nRenault Kangoo Van,,721.3,70,Diesel,43.18,EEA 2024 (OBFCM)\nRenault Kangoo Van,,787.4,70,Diesel,61.53,EEA 2024 (OBFCM)\nRenault Kangoo Van,,806.4,70,Diesel,52.98,EEA 2024 (OBFCM)\nRenault Kangoo Van,,868.7,70,Diesel,59.69,EEA 2024 (OBFCM)\nRenault Kangoo Van,,910.6,70,Diesel,55.58,EEA 2024 (OBFCM)\nRenault Kangoo Van,,935.6,70,Diesel,79.57,EEA 2024 (OBFCM)\nRenault Kangoo Van,,984.5,70,Diesel,63.82,EEA 2024 (OBFCM)\nFord Transit Connect,,90.5,73,Diesel,8.65,EEA 2024 (OBFCM)\nFord Transit Courier Sortimo,,102.3,73,Diesel,9.53,EEA 2024 (OBFCM)\nFord Transit Connect,,376.0,73,Diesel,24.92,EEA 2024 (OBFCM)\nFord Transit Courier,,614.1,73,Diesel,44.41,EEA 2024 (OBFCM)\nFord Transit Connect,,624.1,73,Diesel,33.6,EEA 2024 (OBFCM)\nFord Transit Connect,,704.2,73,Diesel,63.18,EEA 2024 (OBFCM)\nFord Transit Courier,,968.4,73,Diesel,61.89,EEA 2024 (OBFCM)\nRenault Clio,,150.2,74,Diesel,7.88,EEA 2024 (OBFCM)\nRenault Clio,,150.4,74,Diesel,11.45,EEA 2024 (OBFCM)\nOpel Vivaro,,30.2,75,Diesel,3.2,EEA 2024 (OBFCM)\nCitroen Berlingo,,31.8,75,Diesel,3.0,EEA 2024 (OBFCM)\nVolkswagen Caddy,,31.9,75,Diesel,3.6,EEA 2024 (OBFCM)\nCitroen Berlingo,,32.0,75,Diesel,2.7,EEA 2024 (OBFCM)\nCitroen Jumpy,,33.3,75,Diesel,3.4,EEA 2024 (OBFCM)\nPeugeot Partner,,33.8,75,Diesel,3.6,EEA 2024 (OBFCM)\nPeugeot Partner,,34.0,75,Diesel,2.5,EEA 2024 (OBFCM)\nPeugeot Partner,,34.7,75,Diesel,2.6,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,35.8,75,Diesel,6.2,EEA 2024 (OBFCM)\nCitroen Berlingo,,35.9,75,Diesel,3.1,EEA 2024 (OBFCM)\nPeugeot Partner Crew Cab,,36.6,75,Diesel,4.3,EEA 2024 (OBFCM)\nPeugeot Partner,,38.0,75,Diesel,3.4,EEA 2024 (OBFCM)\nPeugeot Partner,,38.1,75,Diesel,3.6,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,38.3,75,Diesel,4.0,EEA 2024 (OBFCM)\nCitroen Berlingo,,39.8,75,Diesel,5.0,EEA 2024 (OBFCM)\nCitroen Berlingo,,40.0,75,Diesel,3.9,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,40.2,75,Diesel,3.9,EEA 2024 (OBFCM)\nPeugeot Partner,,42.5,75,Diesel,3.0,EEA 2024 (OBFCM)\nPeugeot Partner,,45.4,75,Diesel,4.1,EEA 2024 (OBFCM)\nVolkswagen Caddy,,46.7,75,Diesel,4.7,EEA 2024 (OBFCM)\nToyota Proace City,,48.4,75,Diesel,4.81,EEA 2024 (OBFCM)\nOpel Combo,,53.6,75,Diesel,4.2,EEA 2024 (OBFCM)\nPeugeot Partner,,54.0,75,Diesel,3.5,EEA 2024 (OBFCM)\nOpel Combo,,58.2,75,Diesel,6.4,EEA 2024 (OBFCM)\nPeugeot Partner,,60.8,75,Diesel,4.5,EEA 2024 (OBFCM)\nPeugeot Nuevo Partner Standard 600Kg Bluehdi 100 S&S 6 Vel,,61.8,75,Diesel,4.6,EEA 2024 (OBFCM)\nToyota Proace City,,62.1,75,Diesel,8.08,EEA 2024 (OBFCM)\nCitroen Berlingo,,63.4,75,Diesel,4.8,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,70.3,75,Diesel,4.7,EEA 2024 (OBFCM)\nOpel Combo,,78.3,75,Diesel,4.5,EEA 2024 (OBFCM)\nPeugeot Partner,,78.3,75,Diesel,5.9,EEA 2024 (OBFCM)\nCitroen Berlingo,,80.4,75,Diesel,4.7,EEA 2024 (OBFCM)\nCitroen Berlingo,,80.7,75,Diesel,6.0,EEA 2024 (OBFCM)\nPeugeot Rifter Active Pack Business Standard Bluehdi 100 S,,81.7,75,Diesel,10.3,EEA 2024 (OBFCM)\nOpel Nuevo Combo N1 100 Cv 1.5 Td S/S Mt6 6.4,,82.9,75,Diesel,8.6,EEA 2024 (OBFCM)\nCitroen Berlingo,,89.0,75,Diesel,5.1,EEA 2024 (OBFCM)\nVolkswagen Caddy,,90.1,75,Diesel,5.9,EEA 2024 (OBFCM)\nCitroen Berlingo,,100.1,75,Diesel,7.0,EEA 2024 (OBFCM)\nCitroen Berlingo,,102.9,75,Diesel,7.7,EEA 2024 (OBFCM)\nOpel Combo,,111.4,75,Diesel,7.0,EEA 2024 (OBFCM)\nOpel Combo,,118.3,75,Diesel,11.9,EEA 2024 (OBFCM)\nCitroen Berlingo,,240.7,75,Diesel,18.3,EEA 2024 (OBFCM)\nFiat Fiat E-Doblo,,402.0,75,Diesel,27.0,EEA 2024 (OBFCM)\nPeugeot Partner,,451.5,75,Diesel,31.8,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,497.7,75,Diesel,31.5,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,539.3,75,Diesel,42.0,EEA 2024 (OBFCM)\nPeugeot Partner,,552.9,75,Diesel,32.1,EEA 2024 (OBFCM)\nCitroen Berlingo / 15 / Hdi L1,,588.3,75,Diesel,33.62,EEA 2024 (OBFCM)\nPeugeot Partner,,629.7,75,Diesel,38.6,EEA 2024 (OBFCM)\nToyota Proace City,,675.0,75,Diesel,62.05,EEA 2024 (OBFCM)\nVolkswagen Caddy,,711.3,75,Diesel,43.7,EEA 2024 (OBFCM)\nOpel Combo,,776.1,75,Diesel,59.5,EEA 2024 (OBFCM)\nOpel Combo,,812.9,75,Diesel,51.7,EEA 2024 (OBFCM)\nToyota Proace City,,863.9,75,Diesel,69.93,EEA 2024 (OBFCM)\nFord Transit Custom,,65.4,77,Diesel,17.81,EEA 2024 (OBFCM)\nFord Transit Custom,,75.0,77,Diesel,12.7,EEA 2024 (OBFCM)\nFiat Fiat Doblo,,81.0,77,Diesel,6.5,EEA 2024 (OBFCM)\nFord Transit Custom,,321.1,77,Diesel,34.23,EEA 2024 (OBFCM)\nFord Transit Custom,,331.2,77,Diesel,26.59,EEA 2024 (OBFCM)\nFord Transit Custom,,335.4,77,Diesel,34.89,EEA 2024 (OBFCM)\nFord Transit Custom,,481.7,77,Diesel,41.17,EEA 2024 (OBFCM)\nFord Transit,,487.0,77,Diesel,37.91,EEA 2024 (OBFCM)\nFord Transit,,629.4,77,Diesel,56.54,EEA 2024 (OBFCM)\nFord Transit Custom,,664.1,77,Diesel,48.39,EEA 2024 (OBFCM)\nFord Transit Custom,,766.2,77,Diesel,74.41,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,33.5,81,Diesel,4.0,EEA 2024 (OBFCM)\nRenault Trafic,,51.9,81,Diesel,6.02,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,56.6,81,Diesel,7.1,EEA 2024 (OBFCM)\nRenault Trafic,,58.3,81,Diesel,5.37,EEA 2024 (OBFCM)\nRenault Trafic,,71.4,81,Diesel,7.3,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,88.1,81,Diesel,7.1,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,104.0,81,Diesel,10.0,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,106.2,81,Diesel,8.8,EEA 2024 (OBFCM)\nVolkswagen Transporter,,106.7,81,Diesel,8.8,EEA 2024 (OBFCM)\nVolkswagen Transporter,,116.5,81,Diesel,9.6,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,133.3,81,Diesel,13.1,EEA 2024 (OBFCM)\nNissan Primastar,,150.0,81,Diesel,18.0,EEA 2024 (OBFCM)\nRenault Trafic,,150.3,81,Diesel,16.41,EEA 2024 (OBFCM)\nRenault Trafic,,150.3,81,Diesel,13.01,EEA 2024 (OBFCM)\nRenault Trafic,,150.4,81,Diesel,13.06,EEA 2024 (OBFCM)\nRenault Trafic,,150.5,81,Diesel,15.38,EEA 2024 (OBFCM)\nRenault Trafic,,150.9,81,Diesel,19.68,EEA 2024 (OBFCM)\nVolkswagen Transporter,,181.4,81,Diesel,16.0,EEA 2024 (OBFCM)\nRenault Trafic,,201.1,81,Diesel,21.61,EEA 2024 (OBFCM)\nRenault Trafic,,225.2,81,Diesel,22.35,EEA 2024 (OBFCM)\nRenault Trafic,,235.1,81,Diesel,20.96,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,267.5,81,Diesel,27.2,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,323.0,81,Diesel,30.0,EEA 2024 (OBFCM)\nRenault Trafic,,829.2,81,Diesel,81.15,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,872.0,81,Diesel,65.0,EEA 2024 (OBFCM)\nDacia Duster,,32.2,85,Diesel,2.59,EEA 2024 (OBFCM)\nDacia Duster,,35.0,85,Diesel,3.21,EEA 2024 (OBFCM)\nDacia Duster,,35.7,85,Diesel,3.01,EEA 2024 (OBFCM)\nRenault Kangoo Van,,37.5,85,Diesel,3.11,EEA 2024 (OBFCM)\nDacia Duster,,51.3,85,Diesel,3.7,EEA 2024 (OBFCM)\nRenault Kangoo Van,,155.8,85,Diesel,12.03,EEA 2024 (OBFCM)\nRenault Kangoo Van,,194.5,85,Diesel,15.02,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.0,85,Diesel,14.73,EEA 2024 (OBFCM)\nRenault Kangoo Van,,200.5,85,Diesel,16.31,EEA 2024 (OBFCM)\nRenault Kangoo Van,,209.0,85,Diesel,13.35,EEA 2024 (OBFCM)\nRenault Kangoo Van,,227.2,85,Diesel,15.27,EEA 2024 (OBFCM)\nRenault Kangoo Van,,237.7,85,Diesel,18.61,EEA 2024 (OBFCM)\nRenault Kangoo Van,,355.4,85,Diesel,20.6,EEA 2024 (OBFCM)\nRenault Kangoo Van,,870.0,85,Diesel,63.15,EEA 2024 (OBFCM)\nRenault Kangoo Van,,896.6,85,Diesel,53.92,EEA 2024 (OBFCM)\nRenault Kangoo,,967.8,85,Diesel,61.1,EEA 2024 (OBFCM)\nRenault Kangoo Van,,970.7,85,Diesel,58.52,EEA 2024 (OBFCM)\nRenault Kangoo Van,,988.6,85,Diesel,71.9,EEA 2024 (OBFCM)\nPeugeot Expert,,32.2,88,Diesel,4.0,EEA 2024 (OBFCM)\nOpel Vivaro,,33.4,88,Diesel,3.9,EEA 2024 (OBFCM)\nCitroen Jumpy Furgon Talla M Bluehdi 120 S&S 6V (Solo Stoc,,35.6,88,Diesel,4.6,EEA 2024 (OBFCM)\nToyota Proace,,37.4,88,Diesel,4.44,EEA 2024 (OBFCM)\nToyota Proace Max,,44.0,88,Diesel,4.94,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,62.0,88,Diesel,6.1,EEA 2024 (OBFCM)\nCitroen Jumpy,,82.3,88,Diesel,9.5,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,89.0,88,Diesel,7.6,EEA 2024 (OBFCM)\nCitroen Jumper,,186.2,88,Diesel,16.6,EEA 2024 (OBFCM)\nOpel Vivaro,,227.5,88,Diesel,16.5,EEA 2024 (OBFCM)\nPeugeot Expert,,663.1,88,Diesel,45.3,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,43.8,90,Diesel,4.4,EEA 2024 (OBFCM)\nVolkswagen Caddy,,46.1,90,Diesel,4.5,EEA 2024 (OBFCM)\nVolkswagen Caddy,,46.6,90,Diesel,4.6,EEA 2024 (OBFCM)\nVolkswagen Caddy,,75.1,90,Diesel,9.0,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,91.2,90,Diesel,8.1,EEA 2024 (OBFCM)\nVolkswagen Caddy,,146.2,90,Diesel,13.2,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,356.5,90,Diesel,25.6,EEA 2024 (OBFCM)\nVolkswagen Vw Caddy,,464.1,90,Diesel,34.0,EEA 2024 (OBFCM)\nFord Transit,,39.8,95,Diesel,6.54,EEA 2024 (OBFCM)\nFord Transit,,256.8,95,Diesel,31.86,EEA 2024 (OBFCM)\nFord Transit,,478.7,95,Diesel,42.62,EEA 2024 (OBFCM)\nFord Transit,,664.2,95,Diesel,70.49,EEA 2024 (OBFCM)\nFord Transit Custom,,819.3,95,Diesel,61.64,EEA 2024 (OBFCM)\nFord Transit Custom,,954.3,95,Diesel,86.91,EEA 2024 (OBFCM)\nRenault Trafic,,30.8,96,Diesel,3.11,EEA 2024 (OBFCM)\nOpel Grandland Van,,31.1,96,Diesel,3.8,EEA 2024 (OBFCM)\nRenault Trafic,,31.8,96,Diesel,5.82,EEA 2024 (OBFCM)\nRenault Trafic,,32.0,96,Diesel,4.48,EEA 2024 (OBFCM)\nRenault Trafic,,35.0,96,Diesel,3.34,EEA 2024 (OBFCM)\nRenault Trafic,,38.3,96,Diesel,4.6,EEA 2024 (OBFCM)\nFord Transit Custom,,41.1,96,Diesel,5.17,EEA 2024 (OBFCM)\nRenault Trafic,,41.9,96,Diesel,5.28,EEA 2024 (OBFCM)\nRenault Trafic,,46.2,96,Diesel,7.07,EEA 2024 (OBFCM)\nToyota Proace City,,47.5,96,Diesel,4.92,EEA 2024 (OBFCM)\nPeugeot Partner,,51.2,96,Diesel,3.7,EEA 2024 (OBFCM)\nRenault Trafic,,51.4,96,Diesel,5.3,EEA 2024 (OBFCM)\nRenault Trafic,,53.7,96,Diesel,5.83,EEA 2024 (OBFCM)\nFord Transit,,59.3,96,Diesel,10.04,EEA 2024 (OBFCM)\nRenault Trafic,,59.7,96,Diesel,6.02,EEA 2024 (OBFCM)\nCitroen Berlingo,,63.7,96,Diesel,5.7,EEA 2024 (OBFCM)\nPeugeot Partner,,66.1,96,Diesel,6.8,EEA 2024 (OBFCM)\nRenault Trafic,,69.2,96,Diesel,7.2,EEA 2024 (OBFCM)\nRenault Trafic,,77.0,96,Diesel,7.41,EEA 2024 (OBFCM)\nRenault Trafic,,79.1,96,Diesel,8.24,EEA 2024 (OBFCM)\nRenault Trafic,,81.2,96,Diesel,8.03,EEA 2024 (OBFCM)\nFord Transit,,84.6,96,Diesel,17.86,EEA 2024 (OBFCM)\nRenault Trafic,,100.5,96,Diesel,9.91,EEA 2024 (OBFCM)\nRenault Trafic,,150.2,96,Diesel,14.64,EEA 2024 (OBFCM)\nRenault Trafic,,150.2,96,Diesel,13.76,EEA 2024 (OBFCM)\nRenault Trafic,,150.2,96,Diesel,14.39,EEA 2024 (OBFCM)\nRenault Trafic,,150.2,96,Diesel,18.43,EEA 2024 (OBFCM)\nRenault Trafic,,150.3,96,Diesel,18.07,EEA 2024 (OBFCM)\nRenault Trafic,,150.4,96,Diesel,18.55,EEA 2024 (OBFCM)\nRenault Trafic,,150.4,96,Diesel,14.46,EEA 2024 (OBFCM)\nRenault Trafic,,150.5,96,Diesel,17.83,EEA 2024 (OBFCM)\nRenault Trafic,,150.5,96,Diesel,14.84,EEA 2024 (OBFCM)\nRenault Trafic,,150.6,96,Diesel,14.11,EEA 2024 (OBFCM)\nRenault Trafic,,150.7,96,Diesel,11.59,EEA 2024 (OBFCM)\nRenault Master,,150.8,96,Diesel,17.8,EEA 2024 (OBFCM)\nRenault Master,,150.8,96,Diesel,14.04,EEA 2024 (OBFCM)\nRenault Master,,150.9,96,Diesel,18.29,EEA 2024 (OBFCM)\nRenault Master,,150.9,96,Diesel,16.4,EEA 2024 (OBFCM)\nNissan Nissan Interstar,,151.0,96,Diesel,16.0,EEA 2024 (OBFCM)\nNissan Nissan Interstar,,151.0,96,Diesel,15.0,EEA 2024 (OBFCM)\nNissan Nissan Interstar,,151.0,96,Diesel,21.0,EEA 2024 (OBFCM)\nRenault Master,,151.5,96,Diesel,13.95,EEA 2024 (OBFCM)\nRenault Master,,170.1,96,Diesel,19.2,EEA 2024 (OBFCM)\nRenault Trafic,,171.1,96,Diesel,16.08,EEA 2024 (OBFCM)\nRenault Trafic,,180.2,96,Diesel,16.43,EEA 2024 (OBFCM)\nRenault Trafic,,207.0,96,Diesel,18.16,EEA 2024 (OBFCM)\nRenault Trafic,,211.0,96,Diesel,17.59,EEA 2024 (OBFCM)\nFord Transit Trend,,230.0,96,Diesel,24.08,EEA 2024 (OBFCM)\nToyota Proace City,,269.7,96,Diesel,19.42,EEA 2024 (OBFCM)\nFord Transit,,280.0,96,Diesel,38.28,EEA 2024 (OBFCM)\nRenault Trafic,,292.4,96,Diesel,28.61,EEA 2024 (OBFCM)\nNissan Primastar,,322.0,96,Diesel,32.0,EEA 2024 (OBFCM)\nRenault Trafic,,328.4,96,Diesel,36.54,EEA 2024 (OBFCM)\nRenault Trafic,,337.8,96,Diesel,32.3,EEA 2024 (OBFCM)\nFord Transit,,351.7,96,Diesel,39.62,EEA 2024 (OBFCM)\nFord Transit,,372.0,96,Diesel,48.81,EEA 2024 (OBFCM)\nRenault Trafic,,380.1,96,Diesel,36.6,EEA 2024 (OBFCM)\nRenault Trafic,,417.9,96,Diesel,31.74,EEA 2024 (OBFCM)\nFord Transit,,483.0,96,Diesel,47.08,EEA 2024 (OBFCM)\nRenault Trafic,,549.4,96,Diesel,58.45,EEA 2024 (OBFCM)\nRenault Trafic,,549.7,96,Diesel,44.97,EEA 2024 (OBFCM)\nFord Transit,,572.2,96,Diesel,57.12,EEA 2024 (OBFCM)\nRenault Trafic,,579.5,96,Diesel,62.72,EEA 2024 (OBFCM)\nRenault Trafic,,587.8,96,Diesel,61.74,EEA 2024 (OBFCM)\nJeep Compass Van,,612.0,96,Diesel,45.8,EEA 2024 (OBFCM)\nRenault Trafic,,627.7,96,Diesel,52.69,EEA 2024 (OBFCM)\nRenault Trafic,,650.5,96,Diesel,81.86,EEA 2024 (OBFCM)\nRenault Trafic,,665.0,96,Diesel,60.1,EEA 2024 (OBFCM)\nRenault Trafic,,673.3,96,Diesel,57.58,EEA 2024 (OBFCM)\nRenault Trafic,,687.4,96,Diesel,52.46,EEA 2024 (OBFCM)\nNissan Primastar,,698.0,96,Diesel,69.0,EEA 2024 (OBFCM)\nRenault Trafic,,720.6,96,Diesel,82.31,EEA 2024 (OBFCM)\nRenault Trafic,,738.1,96,Diesel,54.6,EEA 2024 (OBFCM)\nRenault Trafic,,743.8,96,Diesel,72.02,EEA 2024 (OBFCM)\nRenault Trafic,,911.3,96,Diesel,69.63,EEA 2024 (OBFCM)\nRenault Trafic,,999.5,96,Diesel,83.51,EEA 2024 (OBFCM)\nRenault Master,,32.0,100,Diesel,7.43,EEA 2024 (OBFCM)\nNissan Interstar,,33.0,100,Diesel,5.0,EEA 2024 (OBFCM)\nRenault Master,,34.0,100,Diesel,5.27,EEA 2024 (OBFCM)\nRenault Master,,35.0,100,Diesel,6.42,EEA 2024 (OBFCM)\nRenault Master,,36.0,100,Diesel,5.01,EEA 2024 (OBFCM)\nRenault Master,,36.0,100,Diesel,5.91,EEA 2024 (OBFCM)\nRenault Master,,38.0,100,Diesel,5.56,EEA 2024 (OBFCM)\nRenault Master,,40.0,100,Diesel,5.72,EEA 2024 (OBFCM)\nRenault Master,,51.0,100,Diesel,6.15,EEA 2024 (OBFCM)\nRenault Master,,55.0,100,Diesel,7.51,EEA 2024 (OBFCM)\nRenault Master,,58.0,100,Diesel,5.91,EEA 2024 (OBFCM)\nRenault Master,,59.0,100,Diesel,7.67,EEA 2024 (OBFCM)\nRenault Master,,70.0,100,Diesel,10.12,EEA 2024 (OBFCM)\nRenault Master,,125.0,100,Diesel,19.37,EEA 2024 (OBFCM)\nMercedes-Benz Vito,,132.7,100,Diesel,11.2,EEA 2024 (OBFCM)\nRenault Master,,144.0,100,Diesel,15.84,EEA 2024 (OBFCM)\nRenault Master,,184.0,100,Diesel,26.8,EEA 2024 (OBFCM)\nFord Transit Custom,,329.0,100,Diesel,27.66,EEA 2024 (OBFCM)\nFord Transit Custom,,400.8,100,Diesel,32.88,EEA 2024 (OBFCM)\nFord Transit Custom,,456.2,100,Diesel,47.75,EEA 2024 (OBFCM)\nRenault Master,,512.0,100,Diesel,53.79,EEA 2024 (OBFCM)\nIveco 35S14,,531.0,100,Diesel,42.0,EEA 2024 (OBFCM)\nFord Transit Custom,,531.5,100,Diesel,39.18,EEA 2024 (OBFCM)\nIveco 35S14,,546.0,100,Diesel,54.0,EEA 2024 (OBFCM)\nRenault Master,,659.0,100,Diesel,66.12,EEA 2024 (OBFCM)\nFord Transit Custom,,662.0,100,Diesel,54.22,EEA 2024 (OBFCM)\nRenault Master,,718.0,100,Diesel,70.06,EEA 2024 (OBFCM)\nIveco 35S14,,733.0,100,Diesel,74.0,EEA 2024 (OBFCM)\nRenault Master,,758.0,100,Diesel,88.16,EEA 2024 (OBFCM)\nRenault Master,,765.0,100,Diesel,167.88,EEA 2024 (OBFCM)\nNissan Interstar,,783.0,100,Diesel,92.0,EEA 2024 (OBFCM)\nRenault Master,,813.0,100,Diesel,83.19,EEA 2024 (OBFCM)\nFord Transit Custom Van,,843.0,100,Diesel,64.07,EEA 2024 (OBFCM)\nRenault Master,,907.0,100,Diesel,101.95,EEA 2024 (OBFCM)\nRenault Master,,912.0,100,Diesel,97.31,EEA 2024 (OBFCM)\nIveco 35S14,,921.0,100,Diesel,97.0,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,30.3,103,Diesel,4.2,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,30.9,103,Diesel,3.7,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,31.0,103,Diesel,3.8,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,31.0,103,Diesel,3.5,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,31.9,103,Diesel,4.9,EEA 2024 (OBFCM)\nMan Tge,,32.0,103,Diesel,4.3,EEA 2024 (OBFCM)\nVolkswagen Crafter,,33.6,103,Diesel,6.1,EEA 2024 (OBFCM)\nVolkswagen Crafter,,34.7,103,Diesel,4.9,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,36.0,103,Diesel,6.4,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,38.0,103,Diesel,5.2,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,39.0,103,Diesel,3.5,EEA 2024 (OBFCM)\nVolkswagen Crafter,,39.1,103,Diesel,5.2,EEA 2024 (OBFCM)\nOpel Movano,,40.0,103,Diesel,4.3,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,46.0,103,Diesel,5.6,EEA 2024 (OBFCM)\nMan Tge,,47.0,103,Diesel,6.9,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,49.0,103,Diesel,5.6,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,50.0,103,Diesel,5.0,EEA 2024 (OBFCM)\nVolkswagen Crafter,,51.7,103,Diesel,5.6,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,53.0,103,Diesel,5.9,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,56.0,103,Diesel,6.2,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,60.0,103,Diesel,6.6,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,61.0,103,Diesel,6.5,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,64.0,103,Diesel,6.1,EEA 2024 (OBFCM)\nVolkswagen Crafter,,67.1,103,Diesel,6.7,EEA 2024 (OBFCM)\nVolkswagen Crafter,,73.5,103,Diesel,10.2,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,78.3,103,Diesel,10.6,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,83.0,103,Diesel,7.7,EEA 2024 (OBFCM)\nMan Tge / 3.140 / F,,83.4,103,Diesel,8.6,EEA 2024 (OBFCM)\nPeugeot Boxer Furgon 330 L1 H1 Bhdi 140 S&S 6 Vel. Man,,91.5,103,Diesel,8.7,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,95.0,103,Diesel,8.8,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,103.0,103,Diesel,15.2,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,106.0,103,Diesel,12.8,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,120.0,103,Diesel,10.7,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,123.0,103,Diesel,12.6,EEA 2024 (OBFCM)\nVolkswagen Crafter,,123.6,103,Diesel,14.0,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,126.0,103,Diesel,11.5,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,131.0,103,Diesel,11.3,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,134.0,103,Diesel,15.0,EEA 2024 (OBFCM)\nMan Tge,,136.8,103,Diesel,13.9,EEA 2024 (OBFCM)\nOpel Movano,,154.8,103,Diesel,16.35,EEA 2024 (OBFCM)\nOpel Movano Furgon Base Light L2 H2 33T Bluehdi 140 S&,,162.2,103,Diesel,20.8,EEA 2024 (OBFCM)\nFiat Nan,,173.0,103,Diesel,18.5,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,188.0,103,Diesel,17.8,EEA 2024 (OBFCM)\nOpel Movano,,208.0,103,Diesel,20.6,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,232.0,103,Diesel,18.2,EEA 2024 (OBFCM)\nPeugeot Boxer,,247.0,103,Diesel,21.6,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,254.0,103,Diesel,21.8,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,257.0,103,Diesel,24.9,EEA 2024 (OBFCM)\nVolkswagen Crafter,,281.3,103,Diesel,25.0,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,314.0,103,Diesel,36.9,EEA 2024 (OBFCM)\nVolkswagen Crafter,,376.7,103,Diesel,38.6,EEA 2024 (OBFCM)\nVolkswagen Crafter,,485.6,103,Diesel,44.4,EEA 2024 (OBFCM)\nVolkswagen Crafter,,488.2,103,Diesel,57.64,EEA 2024 (OBFCM)\nToyota Proace Max,,733.0,103,Diesel,65.86,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,865.0,103,Diesel,83.7,EEA 2024 (OBFCM)\nCitroen Jumpy,,35.7,106,Diesel,3.7,EEA 2024 (OBFCM)\nPeugeot Expert,,40.9,106,Diesel,4.1,EEA 2024 (OBFCM)\nDangel Vivaro 4X4,,43.2,106,Diesel,4.3,EEA 2024 (OBFCM)\nCitroen Jumpy,,48.1,106,Diesel,5.1,EEA 2024 (OBFCM)\nPeugeot Expert,,63.1,106,Diesel,7.6,EEA 2024 (OBFCM)\nOpel Vivaro,,68.3,106,Diesel,7.6,EEA 2024 (OBFCM)\nToyota Proace,,78.3,106,Diesel,9.92,EEA 2024 (OBFCM)\nPeugeot Expert,,80.9,106,Diesel,7.7,EEA 2024 (OBFCM)\nDangel Expert 4X4,,102.8,106,Diesel,9.34,EEA 2024 (OBFCM)\nOpel Vivaro,,772.7,106,Diesel,70.4,EEA 2024 (OBFCM)\nPeugeot Expert,,856.7,106,Diesel,57.6,EEA 2024 (OBFCM)\nRenault Trafic,,30.2,110,Diesel,5.74,EEA 2024 (OBFCM)\nRenault Master,,31.0,110,Diesel,7.51,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,31.8,110,Diesel,8.24,EEA 2024 (OBFCM)\nRenault Trafic,,33.0,110,Diesel,5.03,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,33.4,110,Diesel,7.0,EEA 2024 (OBFCM)\nVolkswagen Transporter,,35.2,110,Diesel,6.8,EEA 2024 (OBFCM)\nRenault Trafic,,37.5,110,Diesel,4.78,EEA 2024 (OBFCM)\nNissan Interstar,,42.0,110,Diesel,6.0,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,44.3,110,Diesel,5.1,EEA 2024 (OBFCM)\nRenault Master,,45.0,110,Diesel,8.25,EEA 2024 (OBFCM)\nRenault Master,,46.0,110,Diesel,11.98,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,48.0,110,Diesel,7.0,EEA 2024 (OBFCM)\nRenault Master,,49.0,110,Diesel,6.1,EEA 2024 (OBFCM)\nRenault Master,,49.0,110,Diesel,6.98,EEA 2024 (OBFCM)\nRenault Master,,60.0,110,Diesel,6.76,EEA 2024 (OBFCM)\nRenault Master,,61.0,110,Diesel,8.49,EEA 2024 (OBFCM)\nRenault Trafic,,61.6,110,Diesel,6.41,EEA 2024 (OBFCM)\nVolkswagen Transporter,,74.5,110,Diesel,9.8,EEA 2024 (OBFCM)\nRenault Master,,86.0,110,Diesel,12.11,EEA 2024 (OBFCM)\nRenault Master,,99.0,110,Diesel,14.18,EEA 2024 (OBFCM)\nRenault Master,,107.0,110,Diesel,12.3,EEA 2024 (OBFCM)\nVolkswagen Transporter,,110.7,110,Diesel,11.6,EEA 2024 (OBFCM)\nVolkswagen Transporter,,120.1,110,Diesel,12.6,EEA 2024 (OBFCM)\nRenault Master,,122.0,110,Diesel,15.37,EEA 2024 (OBFCM)\nVolkswagen Business Van,,127.9,110,Diesel,12.3,EEA 2024 (OBFCM)\nVolkswagen Transporter,,130.6,110,Diesel,11.4,EEA 2024 (OBFCM)\nRenault Master,,138.0,110,Diesel,15.16,EEA 2024 (OBFCM)\nRenault Master,,140.0,110,Diesel,20.75,EEA 2024 (OBFCM)\nRenault Trafic,,150.1,110,Diesel,26.32,EEA 2024 (OBFCM)\nRenault Trafic,,150.1,110,Diesel,16.53,EEA 2024 (OBFCM)\nRenault Trafic,,150.3,110,Diesel,17.76,EEA 2024 (OBFCM)\nRenault Trafic,,150.3,110,Diesel,16.08,EEA 2024 (OBFCM)\nRenault Trafic,,150.3,110,Diesel,15.46,EEA 2024 (OBFCM)\nRenault Trafic,,150.4,110,Diesel,15.02,EEA 2024 (OBFCM)\nRenault Master,,150.5,110,Diesel,14.64,EEA 2024 (OBFCM)\nRenault Trafic,,150.5,110,Diesel,14.16,EEA 2024 (OBFCM)\nRenault Trafic,,150.6,110,Diesel,13.21,EEA 2024 (OBFCM)\nRenault Master,,150.7,110,Diesel,14.0,EEA 2024 (OBFCM)\nRenault Master,,150.7,110,Diesel,14.7,EEA 2024 (OBFCM)\nRenault Master,,150.7,110,Diesel,16.5,EEA 2024 (OBFCM)\nNissan Nissan Interstar,,151.0,110,Diesel,16.0,EEA 2024 (OBFCM)\nRenault Master,,151.0,110,Diesel,16.25,EEA 2024 (OBFCM)\nNissan Nissan Interstar,,151.0,110,Diesel,13.0,EEA 2024 (OBFCM)\nRenault Trafic,,151.2,110,Diesel,14.21,EEA 2024 (OBFCM)\nRenault Master,,151.3,110,Diesel,17.44,EEA 2024 (OBFCM)\nRenault Trafic,,155.7,110,Diesel,14.58,EEA 2024 (OBFCM)\nRenault Trafic,,172.4,110,Diesel,14.83,EEA 2024 (OBFCM)\nRenault Master,,179.0,110,Diesel,18.64,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,189.9,110,Diesel,25.9,EEA 2024 (OBFCM)\nRenault Master,,190.0,110,Diesel,19.93,EEA 2024 (OBFCM)\nRenault Trafic,,201.7,110,Diesel,28.22,EEA 2024 (OBFCM)\nRenault Trafic,,202.3,110,Diesel,22.93,EEA 2024 (OBFCM)\nRenault Trafic,,236.9,110,Diesel,21.33,EEA 2024 (OBFCM)\nRenault Trafic,,254.8,110,Diesel,31.07,EEA 2024 (OBFCM)\nRenault Trafic,,269.8,110,Diesel,20.88,EEA 2024 (OBFCM)\nRenault Trafic,,286.2,110,Diesel,29.39,EEA 2024 (OBFCM)\nVolkswagen Transporter,,293.9,110,Diesel,28.3,EEA 2024 (OBFCM)\nRenault Trafic,,322.6,110,Diesel,25.53,EEA 2024 (OBFCM)\nRenault Trafic,,329.1,110,Diesel,28.59,EEA 2024 (OBFCM)\nRenault Master,,344.9,110,Diesel,34.9,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,356.3,110,Diesel,27.6,EEA 2024 (OBFCM)\nRenault Master,,361.1,110,Diesel,34.23,EEA 2024 (OBFCM)\nRenault Master,,366.0,110,Diesel,44.7,EEA 2024 (OBFCM)\nRenault Trafic,,368.1,110,Diesel,32.17,EEA 2024 (OBFCM)\nRenault Trafic,,369.3,110,Diesel,34.89,EEA 2024 (OBFCM)\nRenault Master,,371.0,110,Diesel,58.26,EEA 2024 (OBFCM)\nRenault Trafic,,373.8,110,Diesel,33.15,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,399.1,110,Diesel,35.1,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,403.2,110,Diesel,31.8,EEA 2024 (OBFCM)\nRenault Trafic,,407.9,110,Diesel,33.17,EEA 2024 (OBFCM)\nRenault Master,,438.8,110,Diesel,43.63,EEA 2024 (OBFCM)\nRenault Master,,463.0,110,Diesel,42.98,EEA 2024 (OBFCM)\nRenault Trafic,,478.2,110,Diesel,69.93,EEA 2024 (OBFCM)\nRenault Trafic,,484.2,110,Diesel,44.94,EEA 2024 (OBFCM)\nFord Custom Trend,,496.0,110,Diesel,41.63,EEA 2024 (OBFCM)\nRenault Master,,525.0,110,Diesel,64.69,EEA 2024 (OBFCM)\nRenault Trafic,,529.3,110,Diesel,48.46,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,556.6,110,Diesel,48.3,EEA 2024 (OBFCM)\nRenault Trafic,,585.3,110,Diesel,67.34,EEA 2024 (OBFCM)\nRenault Trafic,,595.9,110,Diesel,84.04,EEA 2024 (OBFCM)\nRenault Trafic,,596.7,110,Diesel,108.11,EEA 2024 (OBFCM)\nRenault Trafic,,624.8,110,Diesel,80.14,EEA 2024 (OBFCM)\nRenault Trafic,,628.6,110,Diesel,52.59,EEA 2024 (OBFCM)\nRenault Master,,642.4,110,Diesel,66.22,EEA 2024 (OBFCM)\nRenault Trafic,,656.1,110,Diesel,71.44,EEA 2024 (OBFCM)\nRenault Trafic,,662.3,110,Diesel,55.49,EEA 2024 (OBFCM)\nMercedes-Benz Sprinter,,662.7,110,Diesel,59.0,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,666.5,110,Diesel,66.98,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,722.1,110,Diesel,77.3,EEA 2024 (OBFCM)\nRenault Master,,743.4,110,Diesel,65.58,EEA 2024 (OBFCM)\nRenault Trafic,,753.8,110,Diesel,75.82,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,781.9,110,Diesel,72.1,EEA 2024 (OBFCM)\nVolkswagen Transporter,,807.5,110,Diesel,61.1,EEA 2024 (OBFCM)\nNissan Primastar,,823.0,110,Diesel,74.0,EEA 2024 (OBFCM)\nRenault Trafic,,827.3,110,Diesel,71.85,EEA 2024 (OBFCM)\nRenault Trafic,,835.7,110,Diesel,70.83,EEA 2024 (OBFCM)\nRenault Master,,861.8,110,Diesel,80.88,EEA 2024 (OBFCM)\nFord Transit Custom,,932.4,110,Diesel,86.08,EEA 2024 (OBFCM)\nRenault Master,,945.0,110,Diesel,92.94,EEA 2024 (OBFCM)\nRenault Trafic,,960.4,110,Diesel,114.5,EEA 2024 (OBFCM)\nRenault Trafic,,990.2,110,Diesel,94.84,EEA 2024 (OBFCM)\nFord Ford Transit Onnicar,,58.0,114,Diesel,7.19,EEA 2024 (OBFCM)\nFord Transit,,530.6,114,Diesel,50.76,EEA 2024 (OBFCM)\nIveco 35S16,,426.8,115,Diesel,28.91,EEA 2024 (OBFCM)\nIveco 35S16,,550.0,115,Diesel,46.0,EEA 2024 (OBFCM)\nIveco 35S16,,904.0,115,Diesel,103.0,EEA 2024 (OBFCM)\nIveco 35S16,,974.0,115,Diesel,87.0,EEA 2024 (OBFCM)\nIveco 35S16,,633.0,118,Diesel,73.0,EEA 2024 (OBFCM)\nMan Tge,,46.9,120,Diesel,8.3,EEA 2024 (OBFCM)\nMan Tge,,115.3,120,Diesel,15.6,EEA 2024 (OBFCM)\nVolkswagen Crafter,,115.7,120,Diesel,14.6,EEA 2024 (OBFCM)\nMan Tge,,142.3,120,Diesel,24.4,EEA 2024 (OBFCM)\nMan Tge,,580.2,120,Diesel,68.3,EEA 2024 (OBFCM)\nIsuzu D-Max,,622.0,120,Diesel,56.0,EEA 2024 (OBFCM)\nFord Transit,,44.2,121,Diesel,11.62,EEA 2024 (OBFCM)\nFord Transit,,180.8,121,Diesel,33.38,EEA 2024 (OBFCM)\nCitroen Jumper,,218.4,121,Diesel,32.46,EEA 2024 (OBFCM)\nFord Transit,,225.0,121,Diesel,27.17,EEA 2024 (OBFCM)\nFord Transit,,271.9,121,Diesel,33.02,EEA 2024 (OBFCM)\nRenault Trafic,,35.6,125,Diesel,6.18,EEA 2024 (OBFCM)\nMercedes-Benz Sprinter,,45.1,125,Diesel,6.36,EEA 2024 (OBFCM)\nFord Ranger,,50.0,125,Diesel,7.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,56.0,125,Diesel,8.2,EEA 2024 (OBFCM)\nRenault Master,,56.3,125,Diesel,6.16,EEA 2024 (OBFCM)\nFord Transit,,77.7,125,Diesel,8.98,EEA 2024 (OBFCM)\nRenault Master,,83.2,125,Diesel,12.87,EEA 2024 (OBFCM)\nFord Ranger,,89.3,125,Diesel,11.94,EEA 2024 (OBFCM)\nFord Transit,,93.0,125,Diesel,14.28,EEA 2024 (OBFCM)\nVolkswagen Amarok,,128.8,125,Diesel,20.9,EEA 2024 (OBFCM)\nFord Ranger,,142.8,125,Diesel,14.63,EEA 2024 (OBFCM)\nFord Ranger,,150.2,125,Diesel,18.51,EEA 2024 (OBFCM)\nRenault Trafic,,150.3,125,Diesel,16.79,EEA 2024 (OBFCM)\nRenault Trafic,,150.3,125,Diesel,16.81,EEA 2024 (OBFCM)\nRenault Trafic,,150.4,125,Diesel,17.73,EEA 2024 (OBFCM)\nRenault Trafic,,150.4,125,Diesel,15.6,EEA 2024 (OBFCM)\nRenault Trafic,,150.5,125,Diesel,12.69,EEA 2024 (OBFCM)\nRenault Trafic,,150.5,125,Diesel,16.36,EEA 2024 (OBFCM)\nRenault Master,,150.6,125,Diesel,15.2,EEA 2024 (OBFCM)\nRenault Master,,150.6,125,Diesel,18.38,EEA 2024 (OBFCM)\nRenault Master,,150.7,125,Diesel,14.99,EEA 2024 (OBFCM)\nRenault Master,,150.8,125,Diesel,15.8,EEA 2024 (OBFCM)\nRenault Master,,150.8,125,Diesel,18.32,EEA 2024 (OBFCM)\nRenault Master,,150.9,125,Diesel,18.63,EEA 2024 (OBFCM)\nRenault Trafic,,151.2,125,Diesel,19.48,EEA 2024 (OBFCM)\nRenault Master,,151.6,125,Diesel,20.52,EEA 2024 (OBFCM)\nVolkswagen Amarok,,186.0,125,Diesel,25.6,EEA 2024 (OBFCM)\nFord Transit,,189.0,125,Diesel,24.86,EEA 2024 (OBFCM)\nRenault Trafic,,193.6,125,Diesel,17.86,EEA 2024 (OBFCM)\nFord Transit Custom,,202.2,125,Diesel,18.34,EEA 2024 (OBFCM)\nFord Transit Custom,,220.1,125,Diesel,26.38,EEA 2024 (OBFCM)\nRenault Trafic,,229.6,125,Diesel,22.83,EEA 2024 (OBFCM)\nFord Transit Custom,,230.7,125,Diesel,34.5,EEA 2024 (OBFCM)\nFord Ranger,,240.0,125,Diesel,23.16,EEA 2024 (OBFCM)\nRenault Trafic,,244.3,125,Diesel,25.73,EEA 2024 (OBFCM)\nFord Transit Custom,,252.1,125,Diesel,21.62,EEA 2024 (OBFCM)\nRenault Trafic,,257.2,125,Diesel,32.62,EEA 2024 (OBFCM)\nFord Transit Custom,,257.6,125,Diesel,36.41,EEA 2024 (OBFCM)\nFord Transit,,299.1,125,Diesel,33.24,EEA 2024 (OBFCM)\nVolkswagen Amarok,,305.0,125,Diesel,31.1,EEA 2024 (OBFCM)\nFord Ranger,,312.4,125,Diesel,30.35,EEA 2024 (OBFCM)\nMercedes-Benz Sprinter,,335.5,125,Diesel,31.74,EEA 2024 (OBFCM)\nFord Transit Custom,,344.4,125,Diesel,31.77,EEA 2024 (OBFCM)\nFord Transit Custom,,362.4,125,Diesel,32.04,EEA 2024 (OBFCM)\nFord Transit Custom,,368.0,125,Diesel,39.3,EEA 2024 (OBFCM)\nFord Transit Custom,,402.7,125,Diesel,42.33,EEA 2024 (OBFCM)\nRenault Trafic,,497.2,125,Diesel,57.35,EEA 2024 (OBFCM)\nMercedes-Benz Sprinter,,526.7,125,Diesel,50.72,EEA 2024 (OBFCM)\nFord Transit Custom,,596.2,125,Diesel,66.26,EEA 2024 (OBFCM)\nFord Transit Custom,,612.0,125,Diesel,54.16,EEA 2024 (OBFCM)\nFord Transit,,616.2,125,Diesel,55.5,EEA 2024 (OBFCM)\nRenault Trafic,,621.1,125,Diesel,72.36,EEA 2024 (OBFCM)\nFord Transit,,623.0,125,Diesel,76.97,EEA 2024 (OBFCM)\nRenault Trafic,,632.9,125,Diesel,60.93,EEA 2024 (OBFCM)\nFord Transit,,633.8,125,Diesel,86.28,EEA 2024 (OBFCM)\nFord Transit Custom Trend,,640.4,125,Diesel,57.98,EEA 2024 (OBFCM)\nMercedes-Benz Sprinter,,646.3,125,Diesel,82.3,EEA 2024 (OBFCM)\nFord Transit,,676.4,125,Diesel,76.37,EEA 2024 (OBFCM)\nFord Ranger,,691.5,125,Diesel,73.55,EEA 2024 (OBFCM)\nFord Transit,,696.0,125,Diesel,84.78,EEA 2024 (OBFCM)\nFord Ranger,,718.0,125,Diesel,74.29,EEA 2024 (OBFCM)\nRenault Master,,720.7,125,Diesel,80.49,EEA 2024 (OBFCM)\nVolkswagen Amarok,,722.8,125,Diesel,86.5,EEA 2024 (OBFCM)\nFord Tourneo Custom Van,,741.3,125,Diesel,78.0,EEA 2024 (OBFCM)\nFord Transit Custom,,748.8,125,Diesel,72.82,EEA 2024 (OBFCM)\nRenault Trafic,,761.4,125,Diesel,72.86,EEA 2024 (OBFCM)\nFord Transit Custom,,779.4,125,Diesel,75.16,EEA 2024 (OBFCM)\nFord Transit Custom / 20 / 125 H1 320L A6,,789.8,125,Diesel,69.29,EEA 2024 (OBFCM)\nNissan Primastar,,797.0,125,Diesel,93.0,EEA 2024 (OBFCM)\nFord Ranger,,797.4,125,Diesel,66.78,EEA 2024 (OBFCM)\nFord Custom Limited,,867.7,125,Diesel,69.38,EEA 2024 (OBFCM)\nFord Transit Custom,,877.0,125,Diesel,83.95,EEA 2024 (OBFCM)\nRenault Trafic,,895.6,125,Diesel,74.43,EEA 2024 (OBFCM)\nNissan Primastar,,911.0,125,Diesel,90.0,EEA 2024 (OBFCM)\nRenault Trafic,,936.8,125,Diesel,78.74,EEA 2024 (OBFCM)\nFord Transit,,976.1,125,Diesel,99.89,EEA 2024 (OBFCM)\nFord Ranger,,980.4,125,Diesel,83.82,EEA 2024 (OBFCM)\nIveco 35S18,,506.0,129,Diesel,54.0,EEA 2024 (OBFCM)\nIveco 35S18,,593.0,129,Diesel,68.0,EEA 2024 (OBFCM)\nIveco 35S18,,632.0,129,Diesel,88.0,EEA 2024 (OBFCM)\nIveco 35C18,,652.0,129,Diesel,89.0,EEA 2024 (OBFCM)\nIveco 35S18,,675.0,129,Diesel,81.0,EEA 2024 (OBFCM)\nIveco 35S18,,678.0,129,Diesel,74.0,EEA 2024 (OBFCM)\nMan Tge,,31.5,130,Diesel,4.8,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,32.3,130,Diesel,8.4,EEA 2024 (OBFCM)\nVolkswagen Crafter,,32.9,130,Diesel,6.5,EEA 2024 (OBFCM)\nPeugeot Expert,,35.2,130,Diesel,5.0,EEA 2024 (OBFCM)\nPeugeot Expert,,36.7,130,Diesel,4.6,EEA 2024 (OBFCM)\nPeugeot Expert,,37.4,130,Diesel,4.2,EEA 2024 (OBFCM)\nVolkswagen Crafter,,37.6,130,Diesel,6.6,EEA 2024 (OBFCM)\nMan Tge,,37.9,130,Diesel,6.2,EEA 2024 (OBFCM)\nVolkswagen Crafter,,39.8,130,Diesel,5.5,EEA 2024 (OBFCM)\nMan Tge,,40.3,130,Diesel,5.0,EEA 2024 (OBFCM)\nVolkswagen Crafter,,42.4,130,Diesel,6.0,EEA 2024 (OBFCM)\nVolkswagen Crafter,,42.6,130,Diesel,4.5,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,44.8,130,Diesel,7.5,EEA 2024 (OBFCM)\nPeugeot Expert,,46.5,130,Diesel,6.3,EEA 2024 (OBFCM)\nMan Tge,,49.3,130,Diesel,6.3,EEA 2024 (OBFCM)\nVolkswagen Crafter,,50.2,130,Diesel,7.7,EEA 2024 (OBFCM)\nCitroen Jumpy,,51.3,130,Diesel,6.5,EEA 2024 (OBFCM)\nFiat Fiat Scudo,,55.0,130,Diesel,11.0,EEA 2024 (OBFCM)\nMan Tge,,60.1,130,Diesel,8.2,EEA 2024 (OBFCM)\nMan Tge,,60.8,130,Diesel,8.1,EEA 2024 (OBFCM)\nMan Tge,,62.4,130,Diesel,7.6,EEA 2024 (OBFCM)\nPeugeot Expert,,68.6,130,Diesel,7.7,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,71.5,130,Diesel,13.0,EEA 2024 (OBFCM)\nVolkswagen Crafter,,73.4,130,Diesel,8.9,EEA 2024 (OBFCM)\nMan Tge,,75.6,130,Diesel,11.2,EEA 2024 (OBFCM)\nMan Tge,,77.7,130,Diesel,9.8,EEA 2024 (OBFCM)\nMan Tge,,79.5,130,Diesel,8.3,EEA 2024 (OBFCM)\nPeugeot Expert,,81.2,130,Diesel,10.9,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,81.9,130,Diesel,9.3,EEA 2024 (OBFCM)\nMan Tge,,87.0,130,Diesel,10.8,EEA 2024 (OBFCM)\nMan Tge,,90.4,130,Diesel,13.0,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,90.5,130,Diesel,13.0,EEA 2024 (OBFCM)\nVolkswagen Crafter,,91.7,130,Diesel,16.9,EEA 2024 (OBFCM)\nVolkswagen Crafter,,97.0,130,Diesel,8.4,EEA 2024 (OBFCM)\nVolkswagen Crafter,,98.7,130,Diesel,12.2,EEA 2024 (OBFCM)\nMan Tge,,99.9,130,Diesel,14.7,EEA 2024 (OBFCM)\nVolkswagen Crafter,,100.2,130,Diesel,13.1,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,102.1,130,Diesel,13.3,EEA 2024 (OBFCM)\nVolkswagen Crafter,,107.2,130,Diesel,12.6,EEA 2024 (OBFCM)\nVolkswagen Crafter,,129.3,130,Diesel,15.5,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,148.4,130,Diesel,17.0,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,155.2,130,Diesel,17.1,EEA 2024 (OBFCM)\nMan Tge,,161.6,130,Diesel,17.6,EEA 2024 (OBFCM)\nOpel Vivaro,,186.5,130,Diesel,15.4,EEA 2024 (OBFCM)\nMan Tge,,190.1,130,Diesel,24.9,EEA 2024 (OBFCM)\nMan Tge,,192.9,130,Diesel,24.0,EEA 2024 (OBFCM)\nMan Tge,,204.5,130,Diesel,21.7,EEA 2024 (OBFCM)\nVolkswagen. Vw Crafter,,228.4,130,Diesel,30.4,EEA 2024 (OBFCM)\nMan Tge,,233.7,130,Diesel,24.7,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,248.4,130,Diesel,28.9,EEA 2024 (OBFCM)\nVolkswagen Crafter,,249.6,130,Diesel,33.4,EEA 2024 (OBFCM)\nMan Tge,,296.3,130,Diesel,28.9,EEA 2024 (OBFCM)\nMan Tge,,336.8,130,Diesel,40.8,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,364.1,130,Diesel,40.7,EEA 2024 (OBFCM)\nVolkswagen Crafter,,451.1,130,Diesel,51.9,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,473.0,130,Diesel,40.7,EEA 2024 (OBFCM)\nPeugeot Expert,,542.7,130,Diesel,51.4,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,574.0,130,Diesel,102.3,EEA 2024 (OBFCM)\nPeugeot Expert,,615.9,130,Diesel,47.1,EEA 2024 (OBFCM)\nMan Tge,,725.9,130,Diesel,68.5,EEA 2024 (OBFCM)\nVolkswagen Crafter,,754.4,130,Diesel,78.7,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,832.4,130,Diesel,94.6,EEA 2024 (OBFCM)\nVolkswagen Vw Crafter,,954.4,130,Diesel,85.9,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,32.0,132,Diesel,4.5,EEA 2024 (OBFCM)\nRenault Master,,34.0,132,Diesel,5.93,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,36.0,132,Diesel,3.9,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,36.0,132,Diesel,4.0,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,42.0,132,Diesel,5.4,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,44.0,132,Diesel,5.6,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,45.0,132,Diesel,8.4,EEA 2024 (OBFCM)\nRenault Master,,48.0,132,Diesel,6.73,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,49.0,132,Diesel,6.3,EEA 2024 (OBFCM)\nPeugeot Boxer,,64.0,132,Diesel,7.4,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,70.0,132,Diesel,11.6,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,84.0,132,Diesel,9.9,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,92.0,132,Diesel,9.3,EEA 2024 (OBFCM)\nRenault Master,,104.0,132,Diesel,14.45,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,105.0,132,Diesel,10.5,EEA 2024 (OBFCM)\nPeugeot Boxer,,115.0,132,Diesel,13.0,EEA 2024 (OBFCM)\nRenault Master,,142.0,132,Diesel,16.26,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,154.0,132,Diesel,19.6,EEA 2024 (OBFCM)\nRenault Master,,159.0,132,Diesel,21.82,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,161.0,132,Diesel,16.7,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,190.0,132,Diesel,18.5,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,229.0,132,Diesel,18.6,EEA 2024 (OBFCM)\nRenault Master,,241.0,132,Diesel,35.08,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,306.0,132,Diesel,30.7,EEA 2024 (OBFCM)\nRenault Master,,344.0,132,Diesel,39.84,EEA 2024 (OBFCM)\nRenault/Carpol Master,,524.0,132,Diesel,56.5,EEA 2024 (OBFCM)\nRenault Master,,878.0,132,Diesel,99.69,EEA 2024 (OBFCM)\nFiat Fiat Ducato,,879.0,132,Diesel,75.5,EEA 2024 (OBFCM)\nCitroen Jumper,,34.0,136,Diesel,5.7,EEA 2024 (OBFCM)\nLand Rover Defender,,102.0,143,Diesel,16.0,EEA 2024 (OBFCM)\nMercedes-Benz E 220 D,,31.5,145,Diesel,5.46,EEA 2024 (OBFCM)\nVolvo Xc60 Van,,39.1,145,Diesel,5.58,EEA 2024 (OBFCM)\nVolvo Xc60 Van,,97.6,145,Diesel,10.12,EEA 2024 (OBFCM)\nVolvo Xc60 Van,,170.4,145,Diesel,23.85,EEA 2024 (OBFCM)\nVolvo Xc60 Van,,229.3,145,Diesel,22.22,EEA 2024 (OBFCM)\nVolvo Xc60 Van,,264.1,145,Diesel,24.23,EEA 2024 (OBFCM)\nMercedes-Benz Glc 220 D 4Matic,,443.0,145,Diesel,40.8,EEA 2024 (OBFCM)\nVolvo Xc60 Van,,502.1,145,Diesel,43.02,EEA 2024 (OBFCM)\nVolvo Xc60 Van,,747.1,145,Diesel,68.94,EEA 2024 (OBFCM)\nVolkswagen Vw Passat,,32.0,147,Diesel,6.58,EEA 2024 (OBFCM)\nVolkswagen Vw Passat,,34.6,147,Diesel,8.65,EEA 2024 (OBFCM)\nVolkswagen Vw Passat,,35.2,147,Diesel,5.96,EEA 2024 (OBFCM)\nVolkswagen Vw Passat,,36.7,147,Diesel,6.6,EEA 2024 (OBFCM)\nVolkswagen Vw Passat,,38.1,147,Diesel,6.37,EEA 2024 (OBFCM)\nVolkswagen Vw Passat,,45.5,147,Diesel,6.76,EEA 2024 (OBFCM)\nVolkswagen Vw Passat,,52.3,147,Diesel,9.15,EEA 2024 (OBFCM)\nSsangyong Rexton,,38.0,149,Diesel,9.0,EEA 2024 (OBFCM)\nSsangyong Musso Sports,,111.5,149,Diesel,21.92,EEA 2024 (OBFCM)\nVolkswagen Transporter,,30.5,150,Diesel,7.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,31.1,150,Diesel,10.8,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,32.0,150,Diesel,7.85,EEA 2024 (OBFCM)\nVolkswagen Transporter,,32.2,150,Diesel,5.3,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,32.6,150,Diesel,5.44,EEA 2024 (OBFCM)\nVolkswagen Amarok,,33.1,150,Diesel,11.0,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,36.2,150,Diesel,4.46,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,37.3,150,Diesel,5.2,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,41.0,150,Diesel,13.0,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,41.8,150,Diesel,14.98,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,41.8,150,Diesel,4.2,EEA 2024 (OBFCM)\nVolkswagen Transporter,,42.1,150,Diesel,9.7,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,43.3,150,Diesel,15.61,EEA 2024 (OBFCM)\nVolkswagen Amarok,,44.0,150,Diesel,8.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,44.4,150,Diesel,9.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,47.1,150,Diesel,11.6,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,48.6,150,Diesel,7.6,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,49.0,150,Diesel,12.71,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,54.5,150,Diesel,12.81,EEA 2024 (OBFCM)\nVolkswagen Transporter,,61.3,150,Diesel,10.9,EEA 2024 (OBFCM)\nToyota Toyota Land Cruiser 150 Series,,62.4,150,Diesel,10.07,EEA 2024 (OBFCM)\nVolkswagen Amarok,,63.0,150,Diesel,14.6,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,64.1,150,Diesel,6.88,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,68.6,150,Diesel,10.02,EEA 2024 (OBFCM)\nFord Ranger,,75.5,150,Diesel,14.64,EEA 2024 (OBFCM)\nVolkswagen Business Van,,76.1,150,Diesel,10.5,EEA 2024 (OBFCM)\nVolkswagen Transporter,,82.7,150,Diesel,9.1,EEA 2024 (OBFCM)\nVolkswagen Transporter,,83.4,150,Diesel,11.3,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,86.0,150,Diesel,12.98,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,86.8,150,Diesel,15.5,EEA 2024 (OBFCM)\nFord Ranger,,91.5,150,Diesel,17.31,EEA 2024 (OBFCM)\nVolkswagen Transporter,,96.2,150,Diesel,17.0,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,101.1,150,Diesel,17.68,EEA 2024 (OBFCM)\nToyota Hilux,,102.0,150,Diesel,20.0,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,109.8,150,Diesel,10.3,EEA 2024 (OBFCM)\nVolkswagen Transporter,,121.6,150,Diesel,14.9,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,151.4,150,Diesel,23.83,EEA 2024 (OBFCM)\nVolkswagen Transporter,,151.6,150,Diesel,18.3,EEA 2024 (OBFCM)\nToyota Toyota Land Cruiser (150 Series),,156.8,150,Diesel,17.07,EEA 2024 (OBFCM)\nVolkswagen Transporter,,158.7,150,Diesel,18.1,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,164.5,150,Diesel,22.08,EEA 2024 (OBFCM)\nVolkswagen Transporter,,165.1,150,Diesel,16.9,EEA 2024 (OBFCM)\nVolkswagen Transporter,,165.3,150,Diesel,21.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,166.1,150,Diesel,22.7,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,173.4,150,Diesel,24.5,EEA 2024 (OBFCM)\nToyota Toyota Land Cruiser 150 Series,,187.9,150,Diesel,25.46,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,191.4,150,Diesel,24.1,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,201.8,150,Diesel,31.05,EEA 2024 (OBFCM)\nFord Ranger,,209.5,150,Diesel,11.82,EEA 2024 (OBFCM)\nFord Ranger,,219.4,150,Diesel,29.95,EEA 2024 (OBFCM)\nVolkswagen Transporter,,234.1,150,Diesel,26.5,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,235.5,150,Diesel,40.4,EEA 2024 (OBFCM)\nFord Ranger,,248.0,150,Diesel,34.99,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,250.0,150,Diesel,27.9,EEA 2024 (OBFCM)\nFord Ranger,,253.2,150,Diesel,32.61,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,258.1,150,Diesel,35.04,EEA 2024 (OBFCM)\nFord Ranger,,269.6,150,Diesel,33.15,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,276.5,150,Diesel,25.85,EEA 2024 (OBFCM)\nVolkswagen Transporter,,287.6,150,Diesel,30.7,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,289.0,150,Diesel,37.79,EEA 2024 (OBFCM)\nFord Ranger,,325.4,150,Diesel,37.17,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,333.8,150,Diesel,35.41,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,354.9,150,Diesel,48.77,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,356.5,150,Diesel,37.28,EEA 2024 (OBFCM)\nAudi Q5,,395.2,150,Diesel,36.18,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,416.1,150,Diesel,77.64,EEA 2024 (OBFCM)\nVolkswagen Transporter,,433.2,150,Diesel,47.3,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,460.3,150,Diesel,58.38,EEA 2024 (OBFCM)\nFord Ranger,,475.7,150,Diesel,55.14,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,560.3,150,Diesel,96.11,EEA 2024 (OBFCM)\nVolkswagen Transporter,,618.9,150,Diesel,54.5,EEA 2024 (OBFCM)\nVolkswagen Vw Transporter,,646.6,150,Diesel,69.3,EEA 2024 (OBFCM)\nFord Ranger,,679.5,150,Diesel,79.9,EEA 2024 (OBFCM)\nFord Ranger,,725.0,150,Diesel,80.41,EEA 2024 (OBFCM)\nAudi A6 Avant,,730.7,150,Diesel,67.43,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,927.7,150,Diesel,99.71,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,937.9,150,Diesel,113.22,EEA 2024 (OBFCM)\nToyota Toyota Hilux,,949.6,150,Diesel,111.0,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,34.0,151,Diesel,13.2,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,36.2,151,Diesel,9.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,39.0,151,Diesel,14.6,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,39.1,151,Diesel,8.0,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,39.3,151,Diesel,6.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,47.0,151,Diesel,10.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,48.7,151,Diesel,12.2,EEA 2024 (OBFCM)\nVolkswagen Amarok,,59.5,151,Diesel,15.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,60.0,151,Diesel,13.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,62.2,151,Diesel,10.5,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,62.8,151,Diesel,10.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,65.2,151,Diesel,10.7,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,68.1,151,Diesel,18.6,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,82.4,151,Diesel,15.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,103.0,151,Diesel,19.2,EEA 2024 (OBFCM)\nVolkswagen Amarok,,107.3,151,Diesel,21.0,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,109.0,151,Diesel,15.4,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,145.4,151,Diesel,22.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,146.2,151,Diesel,18.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,147.2,151,Diesel,17.6,EEA 2024 (OBFCM)\nVolkswagen Amarok,,153.3,151,Diesel,20.4,EEA 2024 (OBFCM)\nFord Ranger,,200.4,151,Diesel,29.04,EEA 2024 (OBFCM)\nFord Ranger,,214.0,151,Diesel,32.33,EEA 2024 (OBFCM)\nFord Ranger,,248.4,151,Diesel,34.35,EEA 2024 (OBFCM)\nFord Ranger,,254.7,151,Diesel,39.26,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,266.9,151,Diesel,30.7,EEA 2024 (OBFCM)\nFord Ranger,,299.4,151,Diesel,29.18,EEA 2024 (OBFCM)\nFord Ranger,,366.0,151,Diesel,42.59,EEA 2024 (OBFCM)\nFord Ranger,,384.2,151,Diesel,49.61,EEA 2024 (OBFCM)\nFord Ranger Wildtrak,,388.9,151,Diesel,48.43,EEA 2024 (OBFCM)\nFord Ranger Wildtrak,,394.2,151,Diesel,50.86,EEA 2024 (OBFCM)\nFord Ranger,,422.5,151,Diesel,36.48,EEA 2024 (OBFCM)\nFord Ranger,,443.2,151,Diesel,43.95,EEA 2024 (OBFCM)\nFord Ranger,,495.0,151,Diesel,65.39,EEA 2024 (OBFCM)\nFord Ranger,,519.0,151,Diesel,63.98,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,519.5,151,Diesel,63.5,EEA 2024 (OBFCM)\nVolkswagen Amarok,,541.0,151,Diesel,83.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,545.4,151,Diesel,81.4,EEA 2024 (OBFCM)\nFord Ranger,,548.9,151,Diesel,52.55,EEA 2024 (OBFCM)\nVolkswagen. Vw Amarok,,651.7,151,Diesel,85.6,EEA 2024 (OBFCM)\nFord Ranger,,658.8,151,Diesel,82.45,EEA 2024 (OBFCM)\nFord Ranger,,675.4,151,Diesel,77.44,EEA 2024 (OBFCM)\nFord Ranger,,684.3,151,Diesel,93.06,EEA 2024 (OBFCM)\nFord Ranger,,739.1,151,Diesel,88.09,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,743.1,151,Diesel,80.8,EEA 2024 (OBFCM)\nFord Ranger,,744.0,151,Diesel,83.31,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,843.1,151,Diesel,91.9,EEA 2024 (OBFCM)\nIveco 35C21,,565.0,152,Diesel,72.0,EEA 2024 (OBFCM)\nIveco 35S21,,974.0,152,Diesel,132.0,EEA 2024 (OBFCM)\nFord Ranger,,285.4,154,Diesel,37.03,EEA 2024 (OBFCM)\nFord Ranger,,377.6,154,Diesel,54.21,EEA 2024 (OBFCM)\nFord Ranger,,441.3,154,Diesel,73.12,EEA 2024 (OBFCM)\nFord Ranger,,212.0,155,Diesel,24.9,EEA 2024 (OBFCM)\nFord Ranger,,226.8,155,Diesel,32.48,EEA 2024 (OBFCM)\nFord Ranger,,229.0,155,Diesel,32.63,EEA 2024 (OBFCM)\nFord Ranger,,841.7,155,Diesel,124.5,EEA 2024 (OBFCM)\nFord Ranger,,122.0,156,Diesel,25.65,EEA 2024 (OBFCM)\nFord Ranger,,187.9,156,Diesel,29.34,EEA 2024 (OBFCM)\nFord Ranger,,493.5,156,Diesel,59.03,EEA 2024 (OBFCM)\nFord Ranger,,785.0,156,Diesel,109.06,EEA 2024 (OBFCM)\nFord Ranger,,34.1,157,Diesel,8.85,EEA 2024 (OBFCM)\nFord Ranger,,37.0,157,Diesel,6.17,EEA 2024 (OBFCM)\nFord Ranger,,57.0,157,Diesel,17.89,EEA 2024 (OBFCM)\nFord Ranger Raptor,,139.0,157,Diesel,17.0,EEA 2024 (OBFCM)\nFord Ranger,,164.4,157,Diesel,19.87,EEA 2024 (OBFCM)\nFord Ranger,,206.3,157,Diesel,21.56,EEA 2024 (OBFCM)\nFord Ranger / 20 / Tdci 1567 / Dc 4X4 A10,,217.0,157,Diesel,35.13,EEA 2024 (OBFCM)\nFord Ranger,,226.1,157,Diesel,66.69,EEA 2024 (OBFCM)\nFord Ranger,,241.0,157,Diesel,55.19,EEA 2024 (OBFCM)\nFord Ranger,,257.9,157,Diesel,29.1,EEA 2024 (OBFCM)\nFord Ranger,,291.3,157,Diesel,39.9,EEA 2024 (OBFCM)\nFord Ranger,,420.0,157,Diesel,65.4,EEA 2024 (OBFCM)\nFord Ranger,,471.2,157,Diesel,68.12,EEA 2024 (OBFCM)\nFord Ranger,,556.4,157,Diesel,53.23,EEA 2024 (OBFCM)\nFord Ranger,,626.1,157,Diesel,66.12,EEA 2024 (OBFCM)\nFord Ranger Raptor,,670.0,157,Diesel,89.0,EEA 2024 (OBFCM)\nFord Ranger,,699.4,157,Diesel,82.41,EEA 2024 (OBFCM)\nFord Ranger,,737.7,157,Diesel,64.09,EEA 2024 (OBFCM)\nFord Ranger,,804.1,157,Diesel,103.88,EEA 2024 (OBFCM)\nFord Nan,,837.3,157,Diesel,94.54,EEA 2024 (OBFCM)\nFord Ranger,,874.2,157,Diesel,86.59,EEA 2024 (OBFCM)\nFord Ranger,,886.4,157,Diesel,101.47,EEA 2024 (OBFCM)\nFord Ranger,,913.0,157,Diesel,101.86,EEA 2024 (OBFCM)\nVolkswagen Amarok,,31.0,177,Diesel,12.2,EEA 2024 (OBFCM)\nVolkswagen Amarok,,31.0,177,Diesel,11.7,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,31.1,177,Diesel,10.3,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,31.1,177,Diesel,8.2,EEA 2024 (OBFCM)\nVolkswagen Amarok,,31.5,177,Diesel,10.9,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,32.1,177,Diesel,7.5,EEA 2024 (OBFCM)\nVolkswagen Amarok,,32.3,177,Diesel,9.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,34.2,177,Diesel,9.2,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,35.4,177,Diesel,8.6,EEA 2024 (OBFCM)\nVolkswagen Amarok,,35.6,177,Diesel,10.3,EEA 2024 (OBFCM)\nVolkswagen Amarok,,36.4,177,Diesel,13.2,EEA 2024 (OBFCM)\nVolkswagen Amarok,,37.0,177,Diesel,9.4,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,37.0,177,Diesel,13.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,37.1,177,Diesel,13.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,37.2,177,Diesel,8.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,39.0,177,Diesel,9.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,40.0,177,Diesel,13.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,40.0,177,Diesel,14.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,40.3,177,Diesel,13.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,40.3,177,Diesel,8.1,EEA 2024 (OBFCM)\nVolkswagen Amarok,,40.5,177,Diesel,13.8,EEA 2024 (OBFCM)\nVolkswagen Amarok 3.0 / Tdi 4M Aut.,,44.0,177,Diesel,15.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,46.0,177,Diesel,13.6,EEA 2024 (OBFCM)\nVolkswagen Amarok,,47.0,177,Diesel,8.1,EEA 2024 (OBFCM)\nVolkswagen Amarok,,48.0,177,Diesel,12.2,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,48.0,177,Diesel,12.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,52.0,177,Diesel,13.2,EEA 2024 (OBFCM)\nVolkswagen Amarok,,52.0,177,Diesel,14.7,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,52.1,177,Diesel,17.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,52.1,177,Diesel,15.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,52.2,177,Diesel,14.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,52.4,177,Diesel,16.1,EEA 2024 (OBFCM)\nVolkswagen Amarok,,53.2,177,Diesel,14.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,56.0,177,Diesel,14.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,57.0,177,Diesel,14.6,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,57.0,177,Diesel,12.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,57.2,177,Diesel,11.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,57.3,177,Diesel,15.2,EEA 2024 (OBFCM)\nVolkswagen Amarok,,58.5,177,Diesel,11.4,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,58.6,177,Diesel,10.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,59.0,177,Diesel,14.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,62.1,177,Diesel,14.2,EEA 2024 (OBFCM)\nVolkswagen Amarok,,63.1,177,Diesel,15.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,64.0,177,Diesel,11.2,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,64.4,177,Diesel,12.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,66.6,177,Diesel,15.0,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,73.2,177,Diesel,18.9,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,73.3,177,Diesel,13.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,73.7,177,Diesel,11.5,EEA 2024 (OBFCM)\nVolkswagen Amarok,,74.9,177,Diesel,10.8,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,76.2,177,Diesel,19.7,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,81.9,177,Diesel,27.1,EEA 2024 (OBFCM)\nVolkswagen Amarok,,84.0,177,Diesel,16.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,100.3,177,Diesel,16.1,EEA 2024 (OBFCM)\nVw Amarok,,101.0,177,Diesel,17.1,EEA 2024 (OBFCM)\nVolkswagen Amarok,,101.8,177,Diesel,19.1,EEA 2024 (OBFCM)\nVolkswagen Amarok,,101.8,177,Diesel,14.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,103.8,177,Diesel,23.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,105.1,177,Diesel,16.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,106.1,177,Diesel,15.3,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,106.1,177,Diesel,31.1,EEA 2024 (OBFCM)\nVolkswagen Amarok,,106.6,177,Diesel,18.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,107.7,177,Diesel,34.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,109.4,177,Diesel,16.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,110.4,177,Diesel,22.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,114.4,177,Diesel,22.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,122.5,177,Diesel,18.6,EEA 2024 (OBFCM)\nVolkswagen Amarok,,123.1,177,Diesel,24.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,133.3,177,Diesel,23.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,138.4,177,Diesel,17.0,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,139.0,177,Diesel,24.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,139.1,177,Diesel,17.3,EEA 2024 (OBFCM)\nFord Ranger,,139.6,177,Diesel,21.22,EEA 2024 (OBFCM)\nVolkswagen Amarok,,143.2,177,Diesel,23.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,145.3,177,Diesel,27.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,150.3,177,Diesel,24.3,EEA 2024 (OBFCM)\nFord Ranger Wildtrak,,157.0,177,Diesel,24.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,157.6,177,Diesel,28.3,EEA 2024 (OBFCM)\nVolkswagen Amarok,,164.1,177,Diesel,26.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,186.0,177,Diesel,30.2,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,195.4,177,Diesel,36.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,195.6,177,Diesel,30.5,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,198.1,177,Diesel,25.6,EEA 2024 (OBFCM)\nVolkswagen Amarok,,204.5,177,Diesel,40.6,EEA 2024 (OBFCM)\nFord Ranger,,205.3,177,Diesel,40.39,EEA 2024 (OBFCM)\nFord Ranger,,208.5,177,Diesel,30.23,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,209.0,177,Diesel,26.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,210.9,177,Diesel,31.7,EEA 2024 (OBFCM)\nFord Ranger,,215.0,177,Diesel,36.64,EEA 2024 (OBFCM)\nVolkswagen Amarok,,220.1,177,Diesel,28.8,EEA 2024 (OBFCM)\nFord Ranger,,220.5,177,Diesel,33.15,EEA 2024 (OBFCM)\nVolkswagen Amarok,,222.2,177,Diesel,34.2,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,235.0,177,Diesel,32.0,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,235.0,177,Diesel,31.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,236.0,177,Diesel,41.7,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,237.4,177,Diesel,36.6,EEA 2024 (OBFCM)\nVolkswagen Amarok,,245.0,177,Diesel,26.9,EEA 2024 (OBFCM)\nFord Ranger,,245.3,177,Diesel,86.7,EEA 2024 (OBFCM)\nFord Ranger,,251.1,177,Diesel,36.43,EEA 2024 (OBFCM)\nVolkswagen Amarok,,255.1,177,Diesel,37.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,258.1,177,Diesel,35.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,272.0,177,Diesel,39.4,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,278.4,177,Diesel,42.0,EEA 2024 (OBFCM)\nFord Ranger,,284.4,177,Diesel,46.05,EEA 2024 (OBFCM)\nVolkswagen Amarok,,287.1,177,Diesel,39.7,EEA 2024 (OBFCM)\nVolkswagen Amarok,,293.0,177,Diesel,44.1,EEA 2024 (OBFCM)\nVolkswagen Amarok,,296.2,177,Diesel,34.8,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,317.2,177,Diesel,39.3,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,324.0,177,Diesel,49.2,EEA 2024 (OBFCM)\nFord Ranger,,325.6,177,Diesel,53.07,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,326.2,177,Diesel,48.5,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,331.0,177,Diesel,39.6,EEA 2024 (OBFCM)\nVolkswagen. Vw Amarok,,358.9,177,Diesel,47.3,EEA 2024 (OBFCM)\nVolkswagen Amarok,,384.4,177,Diesel,61.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,394.6,177,Diesel,63.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,408.9,177,Diesel,45.4,EEA 2024 (OBFCM)\nVolkswagen Amarok,,413.7,177,Diesel,43.6,EEA 2024 (OBFCM)\nFord Ranger,,425.0,177,Diesel,45.12,EEA 2024 (OBFCM)\nVolkswagen Amarok,,446.5,177,Diesel,68.3,EEA 2024 (OBFCM)\nFord Ranger,,452.0,177,Diesel,44.23,EEA 2024 (OBFCM)\nVolkswagen Amarok,,530.0,177,Diesel,73.3,EEA 2024 (OBFCM)\nVolkswagen Amarok,,531.0,177,Diesel,101.8,EEA 2024 (OBFCM)\nFord Ranger,,542.4,177,Diesel,72.19,EEA 2024 (OBFCM)\nFord Ranger,,557.7,177,Diesel,111.12,EEA 2024 (OBFCM)\nVolkswagen Amarok,,567.2,177,Diesel,141.9,EEA 2024 (OBFCM)\nVolkswagen Amarok,,570.4,177,Diesel,79.2,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,575.5,177,Diesel,57.3,EEA 2024 (OBFCM)\nVolkswagen Amarok,,576.4,177,Diesel,67.1,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,581.0,177,Diesel,77.1,EEA 2024 (OBFCM)\nFord Ranger,,603.3,177,Diesel,101.66,EEA 2024 (OBFCM)\nFord Ranger,,608.5,177,Diesel,85.71,EEA 2024 (OBFCM)\nVolkswagen Amarok,,623.5,177,Diesel,108.4,EEA 2024 (OBFCM)\nFord Ranger,,629.9,177,Diesel,67.65,EEA 2024 (OBFCM)\nFord Ranger,,633.4,177,Diesel,70.45,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,634.4,177,Diesel,110.9,EEA 2024 (OBFCM)\nFord Ranger Wildtrak,,647.7,177,Diesel,95.86,EEA 2024 (OBFCM)\nFord Ranger,,651.2,177,Diesel,90.93,EEA 2024 (OBFCM)\nVolkswagen Amarok,,673.7,177,Diesel,86.2,EEA 2024 (OBFCM)\nFord Ranger,,680.0,177,Diesel,72.14,EEA 2024 (OBFCM)\nFord Ranger,,725.5,177,Diesel,86.32,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,730.5,177,Diesel,93.8,EEA 2024 (OBFCM)\nFord Ranger,,730.6,177,Diesel,97.34,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,738.9,177,Diesel,94.8,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,769.5,177,Diesel,97.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,783.2,177,Diesel,107.2,EEA 2024 (OBFCM)\nFord Ranger,,790.5,177,Diesel,99.69,EEA 2024 (OBFCM)\nFord Ranger,,800.8,177,Diesel,104.96,EEA 2024 (OBFCM)\nFord Ranger,,807.6,177,Diesel,107.46,EEA 2024 (OBFCM)\nFord Ranger,,811.0,177,Diesel,100.69,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,812.6,177,Diesel,110.0,EEA 2024 (OBFCM)\nVolkswagen Amarok,,819.3,177,Diesel,109.7,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,819.6,177,Diesel,101.1,EEA 2024 (OBFCM)\nFord Ranger,,823.9,177,Diesel,122.37,EEA 2024 (OBFCM)\nEdward Davies Ranger,,857.3,177,Diesel,107.97,EEA 2024 (OBFCM)\nVolkswagen Amarok,,869.8,177,Diesel,109.72,EEA 2024 (OBFCM)\nFord Ranger,,898.4,177,Diesel,110.43,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,915.0,177,Diesel,136.8,EEA 2024 (OBFCM)\nVolkswagen Amarok,,915.0,177,Diesel,130.3,EEA 2024 (OBFCM)\nVolkswagen Amarok 3.0 / Tdi 4M Aut.,,917.0,177,Diesel,105.8,EEA 2024 (OBFCM)\nFord Ranger,,936.7,177,Diesel,97.41,EEA 2024 (OBFCM)\nVolkswagen Vw Amarok,,947.1,177,Diesel,131.0,EEA 2024 (OBFCM)\nLand Rover Defender,,57.0,183,Diesel,13.0,EEA 2024 (OBFCM)\nMazda Mazda Cx-60,,48.6,187,Diesel,14.9,EEA 2024 (OBFCM)\nJeep Gladiator,,41.4,194,Diesel,11.3,EEA 2024 (OBFCM)\nJeep Gladiator,,46.2,194,Diesel,13.0,EEA 2024 (OBFCM)\nJeep Gladiator,,54.3,194,Diesel,17.5,EEA 2024 (OBFCM)\nJeep Gladiator,,94.7,194,Diesel,25.9,EEA 2024 (OBFCM)\nJeep Gladiator,,97.0,194,Diesel,25.0,EEA 2024 (OBFCM)\nJeep Gladiator,,109.0,194,Diesel,27.8,EEA 2024 (OBFCM)\nJeep Gladiator,,109.9,194,Diesel,24.5,EEA 2024 (OBFCM)\nJeep Gladiator,,110.8,194,Diesel,24.2,EEA 2024 (OBFCM)\nJeep Gladiator,,179.7,194,Diesel,35.3,EEA 2024 (OBFCM)\nJeep Gladiator,,318.2,194,Diesel,57.5,EEA 2024 (OBFCM)\nJeep Gladiator,,366.4,194,Diesel,64.0,EEA 2024 (OBFCM)\nJeep Gladiator,,647.4,194,Diesel,90.2,EEA 2024 (OBFCM)\nJeep Gladiator,,804.5,194,Diesel,93.1,EEA 2024 (OBFCM)\nJeep Gladiator,,857.5,194,Diesel,105.7,EEA 2024 (OBFCM)\nLand Rover Discovery,,131.0,221,Diesel,22.0,EEA 2024 (OBFCM)\nLand Rover Defender,,160.0,221,Diesel,23.0,EEA 2024 (OBFCM)\nLand Rover Discovery,,751.0,221,Diesel,71.0,EEA 2024 (OBFCM)\n"
for path, text in ((TRIPS_PATH, TRIPS_CSV), (VEHICLES_PATH, VEHICLES_CSV), (SUPPLEMENTARY_PATH, SUPPLEMENTARY_CSV)):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
print("Wrote", TRIPS_PATH, VEHICLES_PATH, SUPPLEMENTARY_PATH)

## 2. Preprocessing and diagnostics code (the web application's `model/preprocess.py` and `model/diagnostics.py`)
The two files are written next to the notebook and imported, so the saved model can be loaded by the web application.

In [ ]:
os.makedirs("model", exist_ok=True)
open(os.path.join("model", "__init__.py"), "a").close()

In [ ]:
%%writefile model/preprocess.py
"""
Shared preprocessing for the fuel-consumption Linear Regression model
(Chapter III, How the Project Will Work). Imported by BOTH train_model.py and
app.py, so the exact same features and encoding are used when the model is
trained and when it serves a live prediction.

Predictors (Scope): distance traveled (km), engine power (kW) and fuel type;
the predicted value is the fuel consumed per trip in liters. The model is

    y^ = b0 + b1(Distance) + b2(Engine Power) + b3(Fuel Type) + b4(Distance x Engine Power)

with fuel type encoded as 1 = Gasoline, 0 = Diesel.
"""
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

FUEL_TYPES = ("Diesel", "Gasoline")
FUEL_TYPE_CODES = {"Diesel": 0, "Gasoline": 1}  # Encoding: 1 represents gasoline and 0 represents diesel

# The three inputs a prediction needs (from the trip and the selected vehicle's record).
FEATURE_COLUMNS = ["distance_km", "engine_power_kw", "fuel_type"]
# The four terms of the regression equation, after encoding and the interaction term.
MODEL_TERMS = ["distance_km", "engine_power_kw", "fuel_type_code", "distance_x_power"]
TERM_LABELS = {
    "distance_km": "Distance (km)",
    "engine_power_kw": "Engine Power (kW)",
    "fuel_type_code": "Fuel Type (1 = Gasoline)",
    "distance_x_power": "Distance × Engine Power",
    "actual_fuel_consumption": "Fuel Consumed (L)",
}
TARGET_COLUMN = "actual_fuel_consumption"

# Kept for code that still asks for them: every model term is numeric now.
NUMERIC_FEATURES = MODEL_TERMS
CATEGORICAL_FEATURES = []

# The local Monthly Report of Official Travel export (one row per trip) and the
# vehicle records it is merged with on the plate number.
TRIP_COLUMNS = ["trip_date", "plate_number", "distance_km", TARGET_COLUMN]
VEHICLE_COLUMNS = ["plate_number", "brand_model", "fuel_type", "engine_power_kw"]


def encode_features(frame, centers=None):
    """Encoding and interaction term: fuel type as 1 (Gasoline) / 0 (Diesel),
    plus Distance x Engine Power for the higher fuel use per kilometre of
    larger engines. Returns the four model terms in a fixed order.

    With `centers` (mean distance, mean engine power of the training set),
    distance and power are mean-centered before the interaction is formed.
    Centering removes the collinearity between a predictor and an interaction
    built from it (high VIF) without changing any prediction."""
    frame = pd.DataFrame(frame)
    distance = pd.to_numeric(frame["distance_km"], errors="coerce").astype(float)
    power = pd.to_numeric(frame["engine_power_kw"], errors="coerce").astype(float)
    fuel = frame["fuel_type"].map(lambda v: FUEL_TYPE_CODES.get(str(v).strip().title(), float("nan"))).astype(float)
    if centers is not None:
        distance, power = distance - centers[0], power - centers[1]
    return pd.DataFrame({
        "distance_km": distance,
        "engine_power_kw": power,
        "fuel_type_code": fuel,
        "distance_x_power": distance * power,
    }, index=frame.index)


class TermEncoder(BaseEstimator, TransformerMixin):
    """Pipeline step: learns the mean distance and mean engine power of the
    training data (fit) and builds the four centered terms (transform)."""

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.distance_mean_ = float(pd.to_numeric(X["distance_km"], errors="coerce").mean())
        self.power_mean_ = float(pd.to_numeric(X["engine_power_kw"], errors="coerce").mean())
        return self

    def transform(self, X):
        return encode_features(X, centers=(self.distance_mean_, self.power_mean_))


def build_feature_encoder():
    """The encoding step bundled into the saved joblib pipeline in front of the
    regression, so a prediction is always encoded exactly like the training data."""
    return TermEncoder()


def raw_equation(intercept, coefficients, centers):
    """The fitted centered equation written back in the original units:
    y = b0 + b1(D - d) + b2(P - p) + b3 F + b4(D - d)(P - p)
      = [b0 - b1 d - b2 p + b4 d p] + (b1 - b4 p) D + (b2 - b4 d) P + b3 F + b4 D P."""
    b1, b2, b3, b4 = (coefficients[t] for t in MODEL_TERMS)
    d, p = centers
    return (intercept - b1 * d - b2 * p + b4 * d * p,
            {"distance_km": b1 - b4 * p, "engine_power_kw": b2 - b4 * d, "fuel_type_code": b3, "distance_x_power": b4})


def merge_trips_and_vehicles(trips, vehicles):
    """The trip records and the vehicle records are stored separately, so they are
    merged using the plate number (each trip gets its vehicle's engine power and
    fuel type)."""
    vehicles = vehicles[["plate_number", "fuel_type", "engine_power_kw"]].drop_duplicates("plate_number")
    trips = trips.drop(columns=[c for c in ("fuel_type", "engine_power_kw") if c in trips.columns])
    return trips.merge(vehicles, on="plate_number", how="left")


def supplementary_info(supplementary_path):
    """Describe the supplementary dataset: online trip records with the same
    variables (distance, engine power, fuel type, fuel consumed)."""
    columns = list(pd.read_csv(supplementary_path, nrows=0).columns)
    missing = [c for c in FEATURE_COLUMNS + [TARGET_COLUMN] if c not in columns]
    return {"kind": "trips", "dropped_features": [], "missing_columns": missing, "columns": columns}


def load_and_merge_datasets(historical_path, supplementary_path, extra_df=None):
    """Combine the local trips (the historical CSV plus the trip records in the
    system, already merged with their vehicles) and the supplementary dataset on
    their shared variables. Each record is labeled with its source
    (`data_source`), so the local trips can be identified after merging."""
    frames = []
    historical = pd.read_csv(historical_path)
    if not historical.empty:
        frames.append(historical.assign(data_source="historical"))
    if extra_df is not None and not extra_df.empty:
        frames.append(extra_df.assign(data_source="system"))
    supplementary = pd.read_csv(supplementary_path)
    frames.append(supplementary.assign(data_source="supplementary"))
    df = pd.concat(frames, ignore_index=True)
    for col in FEATURE_COLUMNS + [TARGET_COLUMN, "trip_date", "plate_number"]:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def clean_data(df):
    """Remove records with missing values or zero distance, then extreme
    outliers with the IQR rule on the fuel used per 100 km, within each group
    of vehicles with the same engine power (large vehicles are compared only
    with vehicles of similar size)."""
    df = df.copy()
    for col in ("distance_km", "engine_power_kw", TARGET_COLUMN):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["fuel_type"] = df["fuel_type"].astype("string").str.strip().str.title()
    df = df.dropna(subset=["distance_km", "engine_power_kw", "fuel_type", TARGET_COLUMN])
    df = df[df["fuel_type"].isin(FUEL_TYPES)]
    df = df[(df["distance_km"] > 0) & (df["engine_power_kw"] > 0) & (df[TARGET_COLUMN] > 0)]

    # IQR = Q3 - Q1; lower bound = Q1 - 1.5(IQR); upper bound = Q3 + 1.5(IQR).
    def within_iqr(per_100km):
        q1, q3 = per_100km.quantile(0.25), per_100km.quantile(0.75)
        iqr = q3 - q1
        return per_100km.between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

    per_100km = df[TARGET_COLUMN] / df["distance_km"] * 100
    keep = per_100km.groupby(df["engine_power_kw"], group_keys=False).apply(within_iqr)
    df = df[keep.reindex(df.index).fillna(False).astype(bool)]
    return df.reset_index(drop=True)


def build_prediction_frame(values):
    """A single-row frame from the trip distance and the selected vehicle's
    engine power and fuel type, in the schema the pipeline expects."""
    return pd.DataFrame([{
        "distance_km": float(values["distance_km"]),
        "engine_power_kw": float(values["engine_power_kw"]),
        "fuel_type": str(values["fuel_type"]),
    }])


In [ ]:
%%writefile model/diagnostics.py
"""
Regression diagnostics (Chapter II, Theoretical Background; Chapter III, How
the Project Will Work), computed with statsmodels:

* Variance Inflation Factor (VIF) of each predictor - a value above 5
  indicates high collinearity (reported; the equation's four terms are kept).
* LINE conditions on the residuals of an OLS fit of the same equation:
    Linearity        - Ramsey RESET test
    Independence     - Durbin-Watson statistic on the residuals of the LOCAL
                       trips arranged by the date of travel
    Normality        - Jarque-Bera test (with the residual histogram)
    Equal variance   - Breusch-Pagan test
* Coefficient standard errors and p-values use heteroskedasticity-robust
  (HC3) standard errors, which stay valid when the equal-variance condition
  is not met (larger errors on longer trips).
"""
import math
import warnings

import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson, jarque_bera

VIF_THRESHOLD = 5.0
ALPHA = 0.05
VIF_CAP = 1000.0  # perfect collinearity gives an infinite VIF; stored as ">1000"


def compute_vif(frame):
    """VIF_j = 1 / (1 - R_j^2) for every column of `frame`, with an intercept.
    A column with the same value in every record has no VIF (None)."""
    usable = [c for c in frame.columns if frame[c].astype(float).std() > 0]
    out = {c: None for c in frame.columns if c not in usable}
    if len(usable) < 2:
        out.update({c: 1.0 for c in usable})
        return out
    X = sm.add_constant(frame[usable].astype(float), has_constant="add")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # divide-by-zero for perfectly collinear columns
        for i, col in enumerate(X.columns):
            if col == "const":
                continue
            value = variance_inflation_factor(X.values, i)
            out[col] = VIF_CAP if (not math.isfinite(value) or value > VIF_CAP) else round(float(value), 2)
    return out


def _p(value):
    return None if value is None or not math.isfinite(float(value)) else round(float(value), 4)


def regression_diagnostics(terms_train, y_train, vif, local_residuals_by_date=None):
    """Fit OLS on the training split with the model's terms and test the LINE
    conditions. `local_residuals_by_date` are the residuals of the local trips
    in date order (for the Durbin-Watson test); without local trips the test
    falls back to the training residuals."""
    usable = [c for c in terms_train.columns if terms_train[c].astype(float).std() > 0]
    design = sm.add_constant(terms_train[usable].astype(float), has_constant="add")
    y = np.asarray(y_train, dtype=float)
    result = sm.OLS(y, design).fit()
    resid = np.asarray(result.resid)

    reset = linear_reset(result, power=2, use_f=True)
    if local_residuals_by_date is not None and len(local_residuals_by_date) >= 3:
        dw = float(durbin_watson(np.asarray(local_residuals_by_date, dtype=float)))
        dw_note = f"Residuals of the {len(local_residuals_by_date)} local trips arranged by date of travel."
    else:
        dw = float(durbin_watson(resid))
        dw_note = "No dated local trips yet: computed on the training residuals."
    jb_stat, jb_p, skew, kurtosis = jarque_bera(resid)
    bp_lm, bp_p, _, _ = het_breuschpagan(resid, design)
    vif_values = [v for v in vif.values() if v is not None]
    max_vif = max(vif_values) if vif_values else 0.0

    tests = [
        {"key": "linearity", "assumption": "Linearity", "test": "Ramsey RESET test",
         "statistic": round(float(reset.fvalue), 3), "p_value": _p(reset.pvalue),
         "passed": bool(reset.pvalue > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": "Fuel consumed is a straight-line function of the predictors."},
        {"key": "independence", "assumption": "Independence of errors", "test": "Durbin-Watson statistic",
         "statistic": round(dw, 3), "p_value": None,
         "passed": bool(1.5 <= dw <= 2.5), "rule": "1.5 to 2.5 (close to 2 = independent)",
         "meaning": dw_note},
        {"key": "normality", "assumption": "Normality of errors", "test": "Jarque-Bera test",
         "statistic": round(float(jb_stat), 3), "p_value": _p(jb_p),
         "passed": bool(jb_p > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": f"Residuals follow a normal distribution (skew {float(skew):.2f}, kurtosis {float(kurtosis):.2f})."},
        {"key": "equal_variance", "assumption": "Equal variance of errors", "test": "Breusch-Pagan test",
         "statistic": round(float(bp_lm), 3), "p_value": _p(bp_p),
         "passed": bool(bp_p > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": "The spread of errors is the same for short and long trips."},
        {"key": "multicollinearity", "assumption": "No multicollinearity", "test": "Variance Inflation Factor",
         "statistic": round(float(max_vif), 2), "p_value": None,
         "passed": bool(max_vif <= VIF_THRESHOLD), "rule": f"every VIF ≤ {VIF_THRESHOLD:g}",
         "meaning": "Predictors do not carry the same information (an interaction term usually shares "
                    "information with the predictors it is built from)."},
    ]
    # Same coefficients; HC3 robust standard errors for the significance tests.
    robust = sm.OLS(y, design).fit(cov_type="HC3")
    coefficients = []
    for name in ["const"] + usable:
        coefficients.append({
            "feature": "Intercept" if name == "const" else name,
            "coef": float(f"{float(robust.params[name]):.4g}"),
            "std_err": float(f"{float(robust.bse[name]):.4g}"),
            "p_value": _p(robust.pvalues[name]),
            "significant": bool(robust.pvalues[name] < ALPHA),
        })
    return {
        "tests": tests,
        "coefficients": coefficients,
        "constant_terms": [c for c in terms_train.columns if c not in usable],
        "ols_r2": round(float(result.rsquared), 4),
        "ols_adj_r2": round(float(result.rsquared_adj), 4),
        "alpha": ALPHA,
        "std_err_type": "HC3 (heteroskedasticity-robust)",
    }


In [ ]:
from model.preprocess import (FEATURE_COLUMNS, MODEL_TERMS, TARGET_COLUMN, TERM_LABELS, build_feature_encoder,
                              clean_data, merge_trips_and_vehicles, raw_equation)
from model.diagnostics import VIF_THRESHOLD, compute_vif, regression_diagnostics

## 3. Merge and clean the data
The trips are merged with the vehicle records using the plate number; each record is labeled with its source.

In [ ]:
trips = pd.read_csv(TRIPS_PATH)
vehicles = pd.read_csv(VEHICLES_PATH)
local = merge_trips_and_vehicles(trips, vehicles).assign(data_source="system")
supplementary = pd.read_csv(SUPPLEMENTARY_PATH).assign(data_source="supplementary")
raw = pd.concat([f for f in (local, supplementary) if not f.empty], ignore_index=True)
df = clean_data(raw)
print(f"Records before cleaning: {len(raw)}  after cleaning: {len(df)}")
print(df["data_source"].value_counts().to_string())
df.to_csv("cleaned_merged_dataset.csv", index=False)  # the cleaned and merged dataset (Pandas output)
print("Saved cleaned_merged_dataset.csv")
df.head()

## 4. Train-test split (80/20), encoding, interaction term and multicollinearity (VIF)
Fuel type is encoded 1 = Gasoline, 0 = Diesel. Distance and engine power are mean-centered on the training set before the interaction term is formed (this removes the collinearity of the interaction without changing any prediction). A VIF above 5 indicates high collinearity.

In [ ]:
X, y = df[FEATURE_COLUMNS], df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Training set: {len(X_train)}  Testing set: {len(X_test)}")
encoder = build_feature_encoder().fit(X_train)
centers = (encoder.distance_mean_, encoder.power_mean_)
print(f"Mean distance {centers[0]:.4f} km, mean engine power {centers[1]:.4f} kW")
terms = encoder.transform(X)
vif = compute_vif(terms)
display(pd.DataFrame({"Term": [TERM_LABELS[t] for t in vif], "VIF": list(vif.values()),
                      "High collinearity (VIF > 5)": [v is not None and v > VIF_THRESHOLD for v in vif.values()]}))
plt.figure(figsize=(8, 6))
cols = [c for c in MODEL_TERMS if terms[c].nunique() > 1]
corr = terms[cols].assign(**{TARGET_COLUMN: y.values}).corr().rename(index=TERM_LABELS, columns=TERM_LABELS)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap of the Terms"); plt.show()

## 5. Train the Linear Regression model

In [ ]:
pipeline = Pipeline(steps=[("encoder", build_feature_encoder()), ("regressor", LinearRegression())])
pipeline.fit(X_train, y_train)
reg = pipeline.named_steps["regressor"]
symbols = {"distance_km": "Distance", "engine_power_kw": "Power", "fuel_type_code": "FuelType", "distance_x_power": "Distance×Power"}
coef = dict(zip(MODEL_TERMS, map(float, reg.coef_)))
D, P = f"(D - {centers[0]:.4f})", f"(P - {centers[1]:.4f})"
centered = {"distance_km": D, "engine_power_kw": P, "fuel_type_code": "(FuelType)", "distance_x_power": D + P}
print("Centered:       ŷ = " + f"{reg.intercept_:.6g}" + "".join(f" {'-' if c < 0 else '+'} {abs(c):.6g}{centered[t]}" for t, c in coef.items()))
raw_b0, raw_coef = raw_equation(float(reg.intercept_), coef, centers)
print("Original units: ŷ = " + f"{raw_b0:.6g}" + "".join(f" {'-' if c < 0 else '+'} {abs(c):.6g}({symbols[t]})" for t, c in raw_coef.items()))

## 6. Evaluate — R², MAE, RMSE and 5-fold cross-validation

In [ ]:
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
kfold = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv = cross_validate(pipeline, X, y, cv=kfold, scoring=("r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"))
cv_scores, cv_mae, cv_rmse = cv["test_r2"], -cv["test_neg_mean_absolute_error"], -cv["test_neg_root_mean_squared_error"]
display(pd.DataFrame({"Test set": [r2, mae, rmse],
                      "CV mean": [cv_scores.mean(), cv_mae.mean(), cv_rmse.mean()],
                      "CV std": [cv_scores.std(), cv_mae.std(), cv_rmse.std()]},
                     index=["R²", "MAE (L)", "RMSE (L)"]).round(4))

## 7. Regression assumption checks (LINE conditions)
Durbin-Watson is computed on the residuals of the local trips arranged by date of travel. Coefficient standard errors and p-values are heteroskedasticity-robust (HC3).

In [ ]:
local_rows = df[df["data_source"] == "system"]
local_resid = None
if not local_rows.empty:
    local_rows = local_rows.assign(_date=pd.to_datetime(local_rows["trip_date"], errors="coerce")).sort_values(["_date"], kind="stable")
    local_resid = (local_rows[TARGET_COLUMN] - pipeline.predict(local_rows[FEATURE_COLUMNS])).to_numpy()
diagnostics = regression_diagnostics(encoder.transform(X_train), y_train, vif, local_resid)
display(pd.DataFrame(diagnostics["tests"])[["assumption", "test", "statistic", "p_value", "rule", "passed"]])
pd.DataFrame(diagnostics["coefficients"])

## 8. Evaluation plots — actual vs predicted, residual plot and residual histogram

In [ ]:
residuals = y_test.values - y_pred
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(y_test, y_pred, alpha=0.4); lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax[0].plot(lim, lim, "--", color="teal"); ax[0].set(xlabel="Actual Fuel Consumed (L)", ylabel="Predicted (L)", title="Actual vs Predicted")
ax[1].scatter(y_pred, residuals, alpha=0.4); ax[1].axhline(0, ls="--", color="teal")
ax[1].set(xlabel="Predicted (L)", ylabel="Residual (Actual − Predicted)", title="Residual Plot")
sns.histplot(residuals, kde=True, ax=ax[2]); ax[2].axvline(0, ls="--", color="teal")
ax[2].set(xlabel="Residual (L)", ylabel="Number of Test Trips", title="Histogram of Residuals")
plt.tight_layout(); plt.show()

## 9. Save the trained model (Joblib) for the web application

In [ ]:
joblib.dump(pipeline, "linear_regression_pipeline.joblib")
metrics = {"r2_score": round(r2, 4), "mae": round(mae, 4), "rmse": round(rmse, 4), "cv_folds": CV_FOLDS,
           "cv_mean_r2": round(float(cv_scores.mean()), 4), "cv_std_r2": round(float(cv_scores.std()), 4),
           "cv_mean_mae": round(float(cv_mae.mean()), 4), "cv_std_mae": round(float(cv_mae.std()), 4),
           "cv_mean_rmse": round(float(cv_rmse.mean()), 4), "cv_std_rmse": round(float(cv_rmse.std()), 4),
           "train_size": len(X_train), "test_size": len(X_test), "total_records": len(df),
           "equation_terms": MODEL_TERMS, "intercept": float(reg.intercept_),
           "coefficients": coef, "centering": dict(zip(["distance_km", "engine_power_kw"], centers)),
           "raw_intercept": raw_b0, "raw_coefficients": raw_coef, "vif": vif, "diagnostics": diagnostics}
with open("metrics_colab.json", "w") as f:
    json.dump(metrics, f, indent=2)
try:
    from google.colab import files
    files.download("linear_regression_pipeline.joblib")
except ImportError:
    print("Saved linear_regression_pipeline.joblib and metrics_colab.json")

> The web application retrains this same model automatically whenever trip records or vehicle records change, so
> copying the `.joblib` file into the `model/` folder is optional. The notebook documents and reproduces the process.